In [1]:
from sage.all import matrix
from sage_periods import compute_diagonal_annihilator, compute_period_annihilator, verify_certificate
from dataclasses import replace
import random
import time

In [2]:
# In certify=True mode, compute_diagonal_annihilator verifies the certificate before returning.
# run_verify=True below checks it again outside the computation timing.
# In an existing Sage kernel, rerun the runner-definition cell after editing it; Jupyter keeps old definitions.

In [3]:
# List of functions and annihilating ODEs separated into smooth/nonsmooth and time to compute operators
var('Dt t u v w x y z')

SMOOTH_fast = [(u*y*z - y*z - 2*y + 1)*y/(u*y*z - u*z - y*z - y + 1), (x*y*z + 2*x*z + y*z + x - 1)*z/(x*y*z + x*y + x*z + y*z + x - 1), 1/((x*y - x - y)*(z + 1) + 1), -1/((x - 1)*(y - 1)*z + x + y - 1), -(x + 1)*(y + 1)/(u*(x + y + 1/x + 1/y)*x*y - 1), -1/(x + y - 1), -1/(6*u*z + 3*z - 1), -(x + 1)*(y + 1)/(u*x^2*y^2 + u*x^2 + u*y^2 + u - 1), -(x + 1)*(y + 1)/(u*x^2*y^2 + u*x*y^2 + u*x^2 + u*y^2 + u*x + u - 1), -(x + 1)*(y + 1)/(u*x^2*y^2 + u*x^2*y + u*x*y^2 + u*x^2 + u*y^2 + u*x + u*y + u - 1), -(x^2*y^2 - x*y + 1)/(x^3*y^2 + x^2*y^3 - x^2*y - x*y^2 + x*y + x + y - 1), (2*y^2 - x)/(x + y - 1), -(x - y)/(x + y - 1), -1/(x + y + z - 1), -1/(x*y + x + y - 1), -1/(x*y + y - 1), -1/(x^2*y + x + y - 1), 1/(w^2 - 2*w*z + 1), 2*(2*x^5*y^2 - 3*x^3*y + 2*x^2*y + x - 1)*x^2*y/(x^5*y^2 - 2*x^3*y + 2*x^2*y + x + 4*y - 2), 10/(5*(x + 1/x)*(y + 1/y) + 12*x + 12*y + 8/x + 8/y + 30), 1/(x^2 - 3*x - y + 3), -1/(x*y - 1)]
SMOOTH_med  = [1/((x + 40*y - 20)*(x + y - 1) - 1), -(2*x^2*y*z - (x^2*y*z - 1)*(x + 1) + x*y + x*z)/((2*x^2*y*z - (x^2*y*z - 1)*(x + 1) + x*y)*w*x*y*z + x^2*y*z - 1)]
SMOOTH_slow = [-1/((x*y*z + y*z + y + z + 1)*u*(x + 1)*(y + 1)*(z + 1) - 1), -(x*y^2 - x^2 - 1)*(x + 1)/((x*y^2 + x^2 + 1)*(x^2 + 1)*u*(y - 1) + 1), (x^2 - y)*u^2*(x + y)*(x - y)*(x + 1)*y/(u*x^2*y + u*x^2 + u*y^2 + u*y - 1)]

SMOOTH_fast_diag = [Dt^3*t + Dt^2, 1/117*(117*t^8 + 228*t^7 - 8165*t^6 + 38285*t^5 - 30755*t^4 - 7079*t^3 + 8014*t^2 - 645*t)*Dt^4 + 4/585*(936*t^7 - 2220*t^6 - 39705*t^5 + 260130*t^4 - 144950*t^3 - 82087*t^2 + 59400*t - 3225)*Dt^3 + 2/195*(546*t^6 - 4962*t^5 - 6514*t^4 + 164522*t^3 + 7149*t^2 - 114159*t + 39735)*Dt^2 - 4/195*(234*t^4 - 304*t^3 - 3647*t^2 - 16677*t + 16005)*Dt, (t^3 + 11*t^2 - t)*Dt^2 + (3*t^2 + 22*t - 1)*Dt + t + 3, (t^3 + 11*t^2 - t)*Dt^2 + (3*t^2 + 22*t - 1)*Dt + t + 3, 1/16*(16*t^4 - t^2)*Dt^3 + 1/8*(64*t^3 + 4*t^2 - 3*t)*Dt^2 + 1/8*(112*t^2 + 14*t - 3)*Dt + 4*t + 3/4, 1/4*Dt*(4*t - 1) + 1/2, 1/6*Dt*(6*t - 1) + 1, 1/64*(64*t^5 - 16*t^4 - 4*t^3 + t^2)*Dt^3 + 1/64*(448*t^4 - 112*t^3 - 20*t^2 + 5*t)*Dt^2 + 1/16*(160*t^3 - 44*t^2 - 5*t + 1)*Dt + 2*t^2 - 3/4*t - 1/16, 2*t^5 + 1/1728*(1728*t^8 - 624*t^6 + 52*t^4 - t^2)*Dt^3 - 2/3*t^4 + 1/1728*(12096*t^7 - 576*t^6 - 4080*t^5 + 348*t^3 + 4*t^2 - 5*t)*Dt^2 - 23/36*t^3 + 1/432*(4320*t^6 - 576*t^5 - 1380*t^4 + 24*t^3 + 129*t^2 + 4*t - 1)*Dt + 1/18*t^2 + 11/144*t + 1/216, 1/64*(64*t^6 + 40*t^5 - 30*t^4 - 5*t^3 + t^2)*Dt^3 + 1/64*(576*t^5 + 200*t^4 - 252*t^3 - 33*t^2 + 5*t)*Dt^2 + 6*t^3 + 1/16*(288*t^4 + 22*t^3 - 117*t^2 - 12*t + 1)*Dt - 9/8*t^2 - 9/4*t - 3/16, 1/2*t^6 - 3/2*t^5 + 3*t^4 - 13/4*t^3 + 1/4*(4*t^7 - 12*t^6 + 24*t^5 - 29*t^4 + 27*t^3 - 16*t^2 + 7*t - 1)*Dt + 9/4*t^2 - t + 1/2, 1/4*Dt^2*(4*t - 1) + 1/2*Dt, Dt, 1/27*(27*t^2 - t)*Dt^2 + 1/27*Dt*(54*t - 1) + 2/9, (t^2 - 6*t + 1)*Dt + t - 3, Dt*(t - 1) + 1, 1/4*(4*t^2 + 4*t - 1)*Dt + t + 1/2, 1/2*Dt*(2*t - 1) + 1, 1/3072*(3072*t^7 + 4864*t^6 - 4608*t^5 + 400*t^4 + 540*t^3 - 186*t^2 + 23*t - 1)*Dt^3 + 1/1024*(2048*t^6 + 5248*t^5 - 4800*t^4 + 1160*t^3 - 40*t^2 - 14*t + 1)*Dt^2 + 1/256*(48*t^5 + 200*t^4 - 200*t^3 + 60*t^2 - 10*t + 1)*Dt, 2*t^9 - 618551/8800*t^8 + 326827/1100*t^7 - 2302457/8800*t^6 - 46819/4400*t^5 + 7595217/8800*t^4 + 1/8800*(8800*t^11 - 292477*t^10 + 2876460*t^9 - 14810808*t^8 + 32408260*t^7 + 3251910*t^6 - 5516668*t^5 - 1855040*t^4 - 298612*t^3 - 21825*t^2)*Dt^2 + 80679/275*t^3 + 1/8800*(35200*t^10 - 1203505*t^9 + 8720290*t^8 - 27755281*t^7 + 25533830*t^6 - 16211085*t^5 - 14776410*t^4 - 1277995*t^3 + 266306*t^2 + 43650*t)*Dt + 295441/8800*t^2 - 133153/4400*t - 873/176, Dt*(t - 1) + 2/3, Dt*(t - 1) + 1]
SMOOTH_med_diag  = [(t^6 - 733099/103428*t^5 + 1657966189/104875992*t^4 - 2988398774599/239274575748*t^3 + 409592617116814/90984157428177*t^2 - 10592321931827/13997562681258*t + 34704126160865/727873259425416)*Dt^3 + (8*t^5 - 3588323/68952*t^4 + 30821446169/314627976*t^3 - 28287899421803/478549151496*t^2 + 10459451211737035/727873259425416*t - 8686231943365/6998781340629)*Dt^2 + (55/4*t^4 - 8406349/103428*t^3 + 17498125819/157313988*t^2 - 811291738807/18405736596*t + 19045561766633/3568006173654)*Dt + 15/4*t^3 - 227459/11492*t^2 + 176581433/17479332*t - 108705605287/79758191916, (t^41 - 141949/1680*t^40 - 3748301/1680*t^39 + 10534587313/928200*t^38 - 159060426793/1485120*t^37 + 1907934483667/571200*t^36 + 5471993338151/742560*t^35 - 802306984755719/29702400*t^34 - 287481210603661/2970240*t^33 + 445568885076661/2284800*t^32 + 4309384337277/27200*t^31 + 158876417561333/1237600*t^30 - 30014407259073/29120*t^29 - 847353981670237/309400*t^28 + 9295729227784319/3712800*t^27 + 104804245351225177/14851200*t^26 - 13942727559891779/7425600*t^25 - 125591505892892483/14851200*t^24 + 4682130304055731/1856400*t^23 + 2039131598778497/309400*t^22 - 15031234999992347/2475200*t^21 - 420837488687163/70720*t^20 + 2770473665561213/618800*t^19 + 147676402231729837/29702400*t^18 + 7569441839116399/14851200*t^17 - 56756524812252611/29702400*t^16 - 10006902614114797/7425600*t^15 + 290852308387171/1856400*t^14 + 2016309079703689/7425600*t^13 - 12738023237077/742560*t^12 - 22227148469731/928200*t^11 - 12030933779/6825*t^10 - 40500644/85*t^9 + 7680224/175*t^8 + 228096/455*t^7 + 1824768/7735*t^6)*Dt^8 + (424/9*t^40 - 61463917/15120*t^39 - 828057001/7560*t^38 + 2360726769629/4176900*t^37 - 18604934306293/3341520*t^36 + 1666327996605853/9547200*t^35 + 30410875582613/78624*t^34 - 353094663647783513/267321600*t^33 - 2585380386472843/514080*t^32 + 156442148718757019/15724800*t^31 + 329980332270769/62400*t^30 + 326508257682262819/22276800*t^29 - 1207078859190249/20800*t^28 - 3386913936079309487/22276800*t^27 + 7525623549048277573/66830400*t^26 + 44893787087446951309/133660800*t^25 - 718315311865611727/13366080*t^24 - 3580160017024905263/10281600*t^23 + 10032455109758012839/66830400*t^22 + 873529236011326429/3182400*t^21 - 7033225041067496039/22276800*t^20 - 153673717218649237/530400*t^19 + 3256772548847072327/22276800*t^18 + 57268916463445299799/267321600*t^17 + 4080592142295232997/66830400*t^16 - 16928197904334293501/267321600*t^15 - 249681904194039407/4773600*t^14 + 12162469960109789/2088450*t^13 + 334705505557642613/33415200*t^12 - 17140844473016939/33415200*t^11 - 1714194466886269/2088450*t^10 - 68406027221071/1044225*t^9 - 176063188376/9945*t^8 + 82676458336/49725*t^7 + 8921088/455*t^6 + 73193472/7735*t^5)*Dt^7 + (2458/3*t^39 - 1096658113/15120*t^38 - 6048182663/3024*t^37 + 14711244600281/1392300*t^36 - 7192100792055479/66830400*t^35 + 227329017463734803/66830400*t^34 + 335880911750506019/44553600*t^33 - 3247341473296226641/133660800*t^32 - 524561928311047681/5346432*t^31 + 1730151347921971489/8910720*t^30 + 50729091362740251/1237600*t^29 + 1015177865224518721/2227680*t^28 - 9458998380012923237/7425600*t^27 - 7766042790496416171/2475200*t^26 + 11789480064960371519/6683040*t^25 + 6665545511740823359/1113840*t^24 - 2912049719863721677/8353800*t^23 - 43812704937772824457/8353800*t^22 + 81533418157479507839/22276800*t^21 + 463369569703030087/100800*t^20 - 16396415760747306233/2784600*t^19 - 61121071096319487571/11138400*t^18 + 52451685619655257801/44553600*t^17 + 441810158024203350793/133660800*t^16 + 23463589101429224513/14851200*t^15 - 95797855259562161239/133660800*t^14 - 49516302155931106763/66830400*t^13 + 112314296118292309/1485120*t^12 + 5087265376758563/39312*t^11 - 6914042822920319/1670760*t^10 - 851514699404203/92820*t^9 - 126281293921921/149175*t^8 - 79125340951112/348075*t^7 + 2547513909376/116025*t^6 + 299388672/1105*t^5 + 61433856/455*t^4)*Dt^6 + (59960/9*t^38 - 306858173/504*t^37 - 10868662793/630*t^36 + 13470982526557/142800*t^35 - 2737391710166791/2784600*t^34 + 175009441383247997/5569200*t^33 + 61647910449163585/891072*t^32 - 9577959652133262619/44553600*t^31 - 2913604873703844007/3182400*t^30 + 39383072496589115/21216*t^29 - 241592631352281227/928200*t^28 + 134535695449194710627/22276800*t^27 - 308343909101201989339/22276800*t^26 - 42775751265748021609/1392300*t^25 + 7157932359547861247/636480*t^24 + 45509621435153125049/891072*t^23 + 32011391969722987873/22276800*t^22 - 15704225519314925321/428400*t^21 + 198998550090602570951/4455360*t^20 + 12020765839217448679/285600*t^19 - 219005105620803668791/4455360*t^18 - 634216244572764649/12376*t^17 - 2801789228639395933/655200*t^16 + 19930202778364732811/873600*t^15 + 757469574117739409/47600*t^14 - 5599987996251300307/1591200*t^13 - 7312426314182837401/1485120*t^12 + 9263376947375376307/22276800*t^11 + 8129622237315472801/11138400*t^10 + 540300523539061/795600*t^9 - 25664514083194309/696150*t^8 - 1633061920357034/348075*t^7 - 47603468097728/38675*t^6 + 960061540608/7735*t^5 + 12700841472/7735*t^4 + 6602010624/7735*t^3)*Dt^5 + (238700/9*t^37 - 1272429415/504*t^36 - 37260822271/504*t^35 + 94426973334623/222768*t^34 - 9956005653933821/2227680*t^33 + 80675331400559239/556920*t^32 + 8217513324111085/26208*t^31 - 270975355408073327/278460*t^30 - 1282177205432552873/297024*t^29 + 3168295744733682707/342720*t^28 - 1498986544454710201/318240*t^27 + 427711852437411120139/11138400*t^26 - 872023629660162469459/11138400*t^25 - 18704141028823515557/123760*t^24 + 58347092740773649117/2784600*t^23 + 1239210987036181493423/5569200*t^22 + 22670632700861257297/1237600*t^21 - 1386671107482520687591/11138400*t^20 + 156915391684589116373/556920*t^19 + 86110017745874708171/371280*t^18 - 679340175226119653261/3712800*t^17 - 1365220702490945912767/5569200*t^16 - 987943057091997301801/11138400*t^15 + 26761164086772821723/371280*t^14 + 531155320115557905683/7425600*t^13 - 190794946947943221077/22276800*t^12 - 26237860889238604859/1591200*t^11 + 9603312370843355821/11138400*t^10 + 4091074386870355979/2227680*t^9 + 4442430482186479/39780*t^8 - 15016473778699597/696150*t^7 - 111495599256118/9945*t^6 - 21022081000768/7735*t^5 + 455951958720/1547*t^4 + 2003139072/455*t^3 + 18765914112/7735*t^2)*Dt^4 + (446080/9*t^36 - 209226727/42*t^35 - 9579604939/63*t^34 + 129273006815249/139230*t^33 - 194714645795563/19890*t^32 + 179494735255178891/556920*t^31 + 375643323138467281/556920*t^30 - 1644385513511534251/742560*t^29 - 318416377415909693/31824*t^28 + 192175274391363319/8160*t^27 - 577714082698682585/27846*t^26 + 655069750784693548627/5569200*t^25 - 15956157101236109917/71400*t^24 - 2028500161644892124233/5569200*t^23 - 53434157380011389159/1113840*t^22 + 1522177053824766533/3120*t^21 + 60582896488149391309/1392300*t^20 - 548224510105728627187/2784600*t^19 + 4995006917408506361897/5569200*t^18 + 114324507564939268369/154700*t^17 - 25465202139654022675/111384*t^16 - 451253090569197948163/795600*t^15 - 1830347589299062813/5200*t^14 + 1078480945381439117923/11138400*t^13 + 821060917905352994711/5569200*t^12 - 138634519607373281543/11138400*t^11 - 6115086704580693557/222768*t^10 - 87056193592541807/1113840*t^9 + 362480061884527513/198900*t^8 + 38830600124621314/116025*t^7 + 83193012289311517/696150*t^6 - 747234359730644/69615*t^5 - 14447589896704/7735*t^4 + 1806022673664/7735*t^3 + 35706604032/7735*t^2 + 21470220288/7735*t)*Dt^3 + (114160/3*t^35 - 173512519/42*t^34 - 5598242585/42*t^33 + 13670432212459/15470*t^32 - 1709395253641441/185640*t^31 + 57158088460138537/185640*t^30 + 74864708038652911/123760*t^29 - 285668621348923493/123760*t^28 - 79345546221811736/7735*t^27 + 800322365955727681/28560*t^26 - 643923461414729941/18564*t^25 + 49446841804570074763/309400*t^24 - 2241134476991773241/7735*t^23 - 720023532634845609397/1856400*t^22 - 53562356743185299281/309400*t^21 + 37907683250239675653/77350*t^20 + 34344774601493911/1105*t^19 - 247482198850752138911/1856400*t^18 + 1477344385467568461/1105*t^17 + 1100756970593485143283/928200*t^16 + 42044680671996258453/309400*t^15 - 15484704162386643137/28560*t^14 - 919352799987929114827/1856400*t^13 + 19013009639751768847/371280*t^12 + 314089403621048499/2275*t^11 - 156355744705210257/18200*t^10 - 716198800098936751/35700*t^9 - 459877215964245077/232050*t^8 + 938883082815637/9100*t^7 + 671560731654769/2550*t^6 + 5923887953482621/46410*t^5 - 37355765035724/7735*t^4 - 43374328448/1547*t^3 + 18412199168/7735*t^2 + 10076825088/7735*t + 200724480/221)*Dt^2 + (26560/3*t^34 - 7600981/7*t^33 - 113990312/3*t^32 + 4340390867273/15470*t^31 - 3159222645977/1105*t^30 + 213635043545143/2210*t^29 + 31136414784673697/185640*t^28 - 19113736255212629/21840*t^27 - 169958616266932583/46410*t^26 + 1166773518417515051/92820*t^25 - 305094433150523563/15470*t^24 + 36142749357198712841/464100*t^23 - 125524721399596819939/928200*t^22 - 1780583972894539563/12376*t^21 - 560522845475648857/5100*t^20 + 162948923388937505501/928200*t^19 + 17630897987555294593/928200*t^18 - 7938822298927146413/185640*t^17 + 18054639963638812577/23205*t^16 + 349066459605582277667/464100*t^15 + 100578149926818786093/309400*t^14 - 136643806419993711623/928200*t^13 - 68658308677141422829/309400*t^12 + 33254260816590561463/1856400*t^11 + 16791419549574735501/309400*t^10 + 341401395248663461/309400*t^9 - 1088794425237916171/232050*t^8 - 177818068858127962/116025*t^7 - 89201422541511167/116025*t^6 + 140937039734744/4641*t^5 + 81648296585728/7735*t^4 - 14175761904128/7735*t^3 - 254546924544/7735*t^2 - 156922748928/7735*t)*Dt + 800/3*t^33 - 879065/21*t^32 - 35469373/21*t^31 + 671145489643/46410*t^30 - 2661690587383/18564*t^29 + 37261951763434/7735*t^28 + 8370767712274/1365*t^27 - 172379312351041/2730*t^26 - 808730987488888/3315*t^25 + 74005545379229237/61880*t^24 - 21026459606253233/9282*t^23 + 3568961360843352179/464100*t^22 - 63212762712241841/5100*t^21 - 9259578568536669101/928200*t^20 - 975711500193796763/116025*t^19 + 1659640073478340507/154700*t^18 + 3036899590371796619/464100*t^17 - 5883170786596936751/928200*t^16 + 11934949909869449494/116025*t^15 + 49546617260821900651/464100*t^14 + 400990171522227487/5950*t^13 - 1403782116416091271/928200*t^12 - 2219218149697496497/116025*t^11 + 1658272556343261689/464100*t^10 + 2428006624958003633/464100*t^9 + 1183419646236592/1547*t^8 - 15198683367133663/77350*t^7 - 4100551349903237/16575*t^6 - 561291716216923/3315*t^5 - 8864344586856/7735*t^4 - 832170355456/7735*t^3 - 151779938816/1547*t^2 - 20153650176/7735*t - 401448960/221]
SMOOTH_slow_diag = [(t^45 - 432151383597070952447/18331740462409920*t^44 - 1817766926880033129221189/549952213872297600*t^43 - 1850948209146487245107802449/74243548872760176000*t^42 + 272462207549568813099799323359/26515553168842920000*t^41 - 402044265154945197604368296271073/6681919398548415840000*t^40 - 2287567065832873250602859282696206241/300686372934678712800000*t^39 + 20659669000752209691137665763102357567/100228790978226237600000*t^38 - 1818823450253198360286196514197665138679/676544339103027103800000*t^37 + 36125934751944414251784365405482735255939/1623706413847265049120000*t^36 - 961564786910284327669674027630024198563/11275738985050451730000*t^35 - 540166791479931848666712293264889687895997/1873507400592998133600000*t^34 + 12303863616870165461607497499902907988489349/2810261100889497200400000*t^33 - 34545092301125593146459447329836551000032861/2087622532089340777440000*t^32 + 981692109808335376543706012147634267945214867/36533394311563463605200000*t^31 - 4764638038353746723294083605318377528976658217/73066788623126927210400000*t^30 + 1605383732507918619165726362233666313244975217/5219056330223351943600000*t^29 - 47470490804312662619268535861732376742140258831/73066788623126927210400000*t^28 + 1208967662764189420839293611833058491882529027/2810261100889497200400000*t^27 + 25647682452975023844017807516823902439158697767/24355596207708975736800000*t^26 - 21562217834678407746625083559847151259992360533/12177798103854487868400000*t^25 - 1786856394682841223793467216796534199240184547/1739685443407783981200000*t^24 + 1890538694031534022808283131743931038110287933/1124104440355798880160000*t^23 - 22621002753838086450642596242069891577108417/17488460656564606800000*t^22 + 11011117164780093324341089760788376810636093699/36533394311563463605200000*t^21 + 26863001062551350750293831049026566654538933/54043482709413407700000*t^20 + 159942703375253409276454256135402823958062229/18266697155781731802600000*t^19 + 600581520140136334591547167567856392217909713/2283337144472716475325000*t^18 + 164215535877517475215255108718973801301060949/761112381490905491775000*t^17 + 3636459054973375168482854986690492387671427/253704127163635163925000*t^16 + 1886844193923388459123013098772374975892714/285417143059089559415625*t^15 + 3036555508357617287098457923515438809335988/285417143059089559415625*t^14 + 17425117693629302015436871815738690886232/10571005298484798496875*t^13 - 3369142122837209964989567169637532206412/31713015895454395490625*t^12 - 2940165266238558413847458933304456352988/95139047686363186471875*t^11 - 18174412948988395057943246411846623424/6342603179090879098125*t^10 + 9535570476995139503648611659009608512/31713015895454395490625*t^9 - 1423506597911537428413931610714656/245837332522902290625*t^8 + 9687108469256601597741402610432/2114201059696959699375*t^7 + 51789887212898324644261888/655566220061072775*t^6)*Dt^10 + (5481/88*t^44 - 2405916166329850891167743/1613193160692072960*t^43 - 3600756638306464288896502561/16131931606920729600*t^42 - 53012414731505107130834795131/15555791192387846400*t^41 + 544552129209544117361320427599007/875013254571816360000*t^40 - 706165053008324945525061910888136177/588008907072260593920000*t^39 - 5983722047329937520885689936389737791/12250185564005429040000*t^38 + 17177418754393964188562126428380537367003/1443294590086457821440000*t^37 - 174844078054409214823178681406326625996498643/1190718036821327702688000000*t^36 + 2065139981752720719375279480987974476623078671/1786077055231991554032000000*t^35 - 9592622258574087178188580449391447779808406459/2679115582847987331048000000*t^34 - 47698296406441969653237229256181513090622786993/2143292466278389864838400000*t^33 + 1095517465651409548231084325333144876890021769603/4592769570596549710368000000*t^32 - 37306488520119855288668610112429719074125951061537/48224080491263771958864000000*t^31 + 56524969603238833339505550396977678784672769189283/48224080491263771958864000000*t^30 - 51778133002706238663349990332410518559897840481527/13778308711789649131104000000*t^29 + 315138422995113244078293973940468308197687697236679/19289632196505508783545600000*t^28 - 1446331314054587073978233172152165867725867845212587/48224080491263771958864000000*t^27 + 689845477392753292715989412659213994286065749219933/48224080491263771958864000000*t^26 + 238936198462766295057564305597895092925885641351629/3857926439301101756709120000*t^25 - 12457521072891781573175126711593719288578483604419/183710782823861988414720000*t^24 - 270135876930131399281031553116363227127998408984829/4592769570596549710368000000*t^23 + 246397646000149367336824894406682485048083681605389/3214938699417584797257600000*t^22 - 3349618381888759867132481851495954062766226645627273/48224080491263771958864000000*t^21 + 3634102057826163489129224118545629426671636696533/927386163293534076132000000*t^20 + 357581713166409450692772297935628793991376926205473/24112040245631885979432000000*t^19 - 1480010848853306089048949723683326691182945530689/219200365869380781631200000*t^18 + 49427844548170189835880601629521327651750853409997/6028010061407971494858000000*t^17 + 3355413414718448421782474628834495877331343579097/376750628837998218428625000*t^16 + 263253168136095018118072814591758956031969367/1427085715295447797078125*t^15 + 3830720250898001116670799119184837454610072263/20930590490999901023812500*t^14 + 44182970058072364919727789509336243197840229053/94187657209499554607156250*t^13 + 252714977302491028320146919831423173006034113/3622602200365367484890625*t^12 - 24119230483997956791818236701616728932547767/6279177147299970307143750*t^11 - 504735052443656139624136147776701190167207/348843174849998350396875*t^10 - 1974023846971095328699247228286781025478122/15697942868249925767859375*t^9 + 63682654002324915524246249108601315453856/5232647622749975255953125*t^8 - 157361605711124547784636076761191992672/747521088964282179421875*t^7 + 50817638793997357883236158762394112/348843174849998350396875*t^6 + 88480668217454383185419423744/29500479902748274875*t^5)*Dt^9 + (4499631/2992*t^43 - 2008310219527546089402949439/54848567463530480640*t^42 - 74829223623667343390841389267/12821743043422709760*t^41 - 169232019702497950739278246074409/1291492431554060736000*t^40 + 4012173424394941114667359588489508257/277670872784123058240000*t^39 + 372641010264189454094084683746890503459/11106834911364922329600000*t^38 - 54300632961880511177313631800255702559919/4498268139102793543488000*t^37 + 1192779239746780154848186989781297409943805523/4498268139102793543488000000*t^36 - 125135478779671847347405114721177035118127256649/40484413251925141891392000000*t^35 + 463817140391162673946722733516057255153194493141/20242206625962570945696000000*t^34 - 2252480341479168905561073242745455805384210094031/45544964908415784627816000000*t^33 - 20110302468806101392984975257237984107155061640567/33123610842484207002048000000*t^32 + 27439029961923182995680320522401758798995686374869/5605534142574250415731200000*t^31 - 1679603906658402590935101960529408263327605250064561/126124518207920634353952000000*t^30 + 10884584949736531003961806818159660847327085344679473/546539578900989415533792000000*t^29 - 39579083985975951881756531853032001994992001445401511/468462496200848070457536000000*t^28 + 1067834352775415495721254065540055346698510791068794129/3279237473405936493202752000000*t^27 - 23899038338133045464256884329012789346202602843749801/46846249620084807045753600000*t^26 + 14309392556551981137720655613098262400669944833731763/149056248791178931509216000000*t^25 + 4392742318682928330110561853338635473038706337108440637/3279237473405936493202752000000*t^24 - 523724460073468272368847678932934171360735563081276009/655847494681187298640550400000*t^23 - 781835469223572941149911794305013684108944529831803093/655847494681187298640550400000*t^22 + 784602971713591726917518396009759278861615781568648977/546539578900989415533792000000*t^21 - 23059029772884481348672410534829153698619904211158519/18017788315417233479136000000*t^20 - 2343388162387327593715278701643504514667554603892843/29278906012553004403596000000*t^19 + 129332966526884620562433572029331379199405938599826029/819809368351484123300688000000*t^18 - 19755766239549610276622462795878144290904992563805607/81980936835148412330068800000*t^17 + 984109382565393759435907202318516134797931834831567/15765564775990079294244000000*t^16 + 6960203179281479801082941812136361098763585115588909/51238085521967757706293000000*t^15 - 77805844182748959808519562609299986347011631114173/12809521380491939426573250000*t^14 + 95133340844325205468771212874794819520159917359/640476069024596971328662500*t^13 + 6107215380444307187408722674743753352920088776366/800595086280746214160828125*t^12 + 65712819720346351461376103602876516986802212824/61584237406211247243140625*t^11 - 16923928230739514865130091359522803015758255609/355820038346998317404812500*t^10 - 13574995584450533021532272680436324464497146027/533730057520497476107218750*t^9 - 30985289767103566999436221263622246347972067/15697942868249925767859375*t^8 + 72534860623991586700317479450116453508572/402511355596151942765625*t^7 - 14130103774736530094345833959663338623184/5232647622749975255953125*t^6 + 482114184550997259039657496240006784/348843174849998350396875*t^5 + 399796742511533426029983668224/9833493300916091625*t^4)*Dt^8 + (3390997/187*t^42 - 24653421295483136745472370843/54848567463530480640*t^41 - 94001856736177825836687230123167/1234092767929435814400*t^40 - 9148906827964177819416756734928877/4113642559764786048000*t^39 + 16205869862123632917279256703727632593/98001484512043432320000*t^38 + 117981259226814468357872238223263329199379/99961514202284300966400000*t^37 - 3172644608231489275850184626259065873711411/21420324471918064492800000*t^36 + 39638929840892787213681657811683789307681586569/13494804417308380630464000000*t^35 - 72719791365237582450968079761718281211316688447/2249134069551396771744000000*t^34 + 1122720279639881804290694418963032162972622067307/5060551656490642736424000000*t^33 - 1384805197530188920440401973000138920145330402919/6747402208654190315232000000*t^32 - 944421704968581196959449051653447820593938963856351/121453239755775425674176000000*t^31 + 8811751416977783610954113865342336075272752386477697/182179859633663138511264000000*t^30 - 2424522477288799336587043699014502634977623627374273/22772482454207892313908000000*t^29 + 3468363305552730057643639926218294631940789542847949/19289632196505508783545600000*t^28 - 344303616811246749377747705547309519617505976658031037/364359719267326277022528000000*t^27 + 730321124384374115892180297044970091609608002588699909/234231248100424035228768000000*t^26 - 468842060580251579135354500852533271717491550098936599/117115624050212017614384000000*t^25 - 1980118339081478378461302827213722346629640517627074243/1639618736702968246601376000000*t^24 + 2090328214754835834867935378439466288890249389488409237/156154165400282690152512000000*t^23 - 295007394894814434529428972856892113206843220855798121/149056248791178931509216000000*t^22 - 36479578855816223827608403544843215736902842644602369691/3279237473405936493202752000000*t^21 + 557130184771477665893123146218426662651976074630008091/39038541350070672538128000000*t^20 - 26345020399580565997687253405151835756216527630952727/2623389978724749194562201600*t^19 - 301411909299117622987230320498813262429707010417051373/204952342087871030825172000000*t^18 + 930807927343364773573061600289976232972680595527841981/819809368351484123300688000000*t^17 - 22118241373692827111815989270773106265526933698209163/7590827484735964104636000000*t^16 - 29581585831315023301832771461880038266625435686211/121995441718970851681650000*t^15 + 1192542659483391574562460618496179637126212058456549/1205602012281594298971600000*t^14 - 6679561204833613653834287894732492584459686002272201/51238085521967757706293000000*t^13 - 18171118597678373839644032144875435627170871094909/533730057520497476107218750*t^12 + 5152928134771783397109322505349606469611461047/89171746470532122705000*t^11 + 220668886597216309536527883789330830389554918613/29651669862249859783734375*t^10 - 237565644218625296916126158120997348017576129093/1067460115040994952214437500*t^9 - 489500945722163484762521194036050127845939901/2280897681711527675671875*t^8 - 5536246042569943592848631764902905823201596/402511355596151942765625*t^7 + 6447014581034790320626207781791918548188086/5232647622749975255953125*t^6 - 79237199905582148290403252585374465378772/5232647622749975255953125*t^5 + 39996376756692611422833084139068256/16611579754761826209375*t^4 + 1035089752958908348742657679872/4214354271821182125*t^3)*Dt^7 + (351427487/2992*t^41 - 7753447052013151343405337329/2611836545882403840*t^40 - 94201312401414882083457955319807/176298966847062259200*t^39 - 1408227246198090163258803603861950413/74045566075766148864000*t^38 + 1667452270139132355876059158027154156183/1666025236704738349440000*t^37 + 143301978820468526071449141674406640060381/11106834911364922329600000*t^36 - 97196474095422927918471849566322556194197923/99961514202284300966400000*t^35 + 235047679759419235189361833906852095278817240667/13494804417308380630464000000*t^34 - 366740455970256660386317428123502570813752349129/2024220662596257094569600000*t^33 + 135782275602395125798097967632579034277539586966193/121453239755775425674176000000*t^32 + 79129608561954040944101415369395298422437621071751/91089929816831569255632000000*t^31 - 9290983943489571889271440617768807460437184977179039/182179859633663138511264000000*t^30 + 267475087913893339567490741659585736075307146300205793/1093079157801978831067584000000*t^29 - 61167170483365247982577222322677846888518851096370859/149056248791178931509216000000*t^28 + 3299611314628439357449130615273991026577687498431034007/3279237473405936493202752000000*t^27 - 1661730849682326929114049400326382186127079688886635219/298112497582357863018432000000*t^26 + 25093629099320967528990957710912552594101850779803343847/1639618736702968246601376000000*t^25 - 47369106768409979268104877430119909194361269623110841181/3279237473405936493202752000000*t^24 - 6108207439575068432908050649746091960010061555393804017/327923747340593649320275200000*t^23 + 708036144681481173459067527745915450910768706808737551/11153868957163049296608000000*t^22 + 1862492456550967349428064206752551300962008710762963081/93692499240169614091507200000*t^21 - 171399821464009864592469942230340801640213516579685192669/3279237473405936493202752000000*t^20 + 11705497755365539331030611767624951714756662945976559381/149056248791178931509216000000*t^19 - 10492434565124489843424348618148616157737158039348893839/327923747340593649320275200000*t^18 - 97163833911638921104054732689390913492944842113892281/14905624879117893150921600000*t^17 + 7461148105870776662625382809442580237277919346604447767/819809368351484123300688000000*t^16 - 2988555668199709990722094347367275145743572136258905861/204952342087871030825172000000*t^15 - 35134251019466817501876938431985758816525546435803431/7590827484735964104636000000*t^14 + 97239748712868648980683148678551193748411064272498193/25619042760983878853146500000*t^13 - 4311876618582769171559863255389132986748105775651067/5123808552196775770629300000*t^12 - 9004499087392305901199448950990190005771207968046459/25619042760983878853146500000*t^11 + 260531762994442080575577734267216957352672753433/1232277189080513653350000*t^10 + 3524712095626775501120315497939336903260886799191/142328015338799326961925000*t^9 - 5261242598538363430154436359385372377509398259/88955009586749579351203125*t^8 - 70458916864246564101730462539764635848905953819/76247151074356782301031250*t^7 - 221434681160467437073349035670182595023995416/5232647622749975255953125*t^6 + 21127472737568398267351413299793242788897966/5232647622749975255953125*t^5 - 194464050597817601398782046914689803757348/5232647622749975255953125*t^4 - 33552087191808439790823296685222592/1423849693265299389375*t^3 + 6625943437523770751075594173952/9833493300916091625*t^2)*Dt^6 + (1227685025/2992*t^40 - 145282579312937669607684936349/13712141865882620160*t^39 - 2499333003006231964139944110253271/1234092767929435814400*t^38 - 3754030967445255492078718508463428899/44427339645459689318400*t^37 + 1068031274408543179420404067792167717203/333205047340947669888000*t^36 + 3239119401344502977930704394385671089776059/49980757101142150483200000*t^35 - 251159340789772797187333999934142263629009/73023833426993401680000*t^34 + 93837624531095434598733497913339601174245835441/1686850552163547578808000000*t^33 - 22201671649424826443308307218766146431467614243429/40484413251925141891392000000*t^32 + 29185567552104130943848300728261956624032142057681/10121103312981285472848000000*t^31 + 3512702650666640309828873655483335865689173084405949/364359719267326277022528000000*t^30 - 12610165661697947116701183688870567860860518959319643/72871943853465255404505600000*t^29 + 3556555263749974362509424321005362601943329955846701/5693120613551973078477000000*t^28 - 2361432340121014680892082050345212651503789508378086171/3279237473405936493202752000000*t^27 + 1491083502287653950962815107827690719597714788904887877/409904684175742061650344000000*t^26 - 932329469076562966065738579280669651273594637247337297/54653957890098941553379200000*t^25 + 1995746928612463065982118640417102715245974200636826893/52051388466760896717504000000*t^24 - 3094181061244614475648024448277185856065801798624107139/182179859633663138511264000000*t^23 - 276829245015335852196096403731222746719702094134867374059/3279237473405936493202752000000*t^22 + 139784013209439406855002535251649931678431315216165597021/1093079157801978831067584000000*t^21 + 205110939834962869016187609877352205509956571604502634467/1639618736702968246601376000000*t^20 - 430577010405120943855090153296943503631091400095735383829/3279237473405936493202752000000*t^19 + 370055992051514950425643425229225230332266916803530735571/1639618736702968246601376000000*t^18 - 33655448703354521790560488444395711694490691920555314183/1639618736702968246601376000000*t^17 - 158635230210938061601515875707047290909662461568279261/31531129551980158588488000000*t^16 + 5597416386351874155513478946888990342217635937414933587/117115624050212017614384000000*t^15 - 2769904218332506209029566198828802302891312403763184777/102476171043935515412586000000*t^14 - 101766750376585509632967308593812114316882297426666073/5693120613551973078477000000*t^13 + 77143289290856754243736280958036184732473304596911263/9316015549448683219326000000*t^12 - 100768546245741519008465985755686985371501816959989207/51238085521967757706293000000*t^11 - 507135329631938672137386593162057499465094744068631/376750628837998218428625000*t^10 + 105273358082262537384968064429242418483576617647891/304988604297427129204125000*t^9 + 40971150116927698850079340494917016359574705375611/1067460115040994952214437500*t^8 + 293460675451259939066530825975108471300344214421/118606679448999439134937500*t^7 - 76138384214947835538961043712619981353119761252/38123575537178391150515625*t^6 - 26426948220703619263301838751523229674600204/581405291416663917328125*t^5 + 3441539076222874250437187567266588045569778/581405291416663917328125*t^4 - 722748283899706049139110249640231142868/21357745398979490840625*t^3 - 113233148838446889846632290418413696/1107438650317455080625*t^2 + 7447315747162752550883124640256/9833493300916091625*t)*Dt^5 + (2211255661/2992*t^39 - 134025473863692812808029560033/6856070932941310080*t^38 - 306383882883254272592672036142071/77130797995589738400*t^37 - 6010178379289103319789309203825901617/31733814032471206656000*t^36 + 104363821543308982472263686830885157451/20194245293390767872000*t^35 + 7823986886174002199163224191849699945282891/49980757101142150483200000*t^34 - 15159019606747395084609351642316968830801827/2405490983477429702400000*t^33 + 413579855815053608417699752713897922691013383039/4498268139102793543488000000*t^32 - 2915594284621539534035646655612815499420218212791/3373701104327095157616000000*t^31 + 209429565087186299860367580594012308738570472779431/60726619877887712837088000000*t^30 + 1438272203828839988025793509790718720403021962917139/52051388466760896717504000000*t^29 - 2079290283574198240427412000150085197506977802101481/7144308220927966216128000000*t^28 + 814684541155849586594741950985004178771617225773297839/1093079157801978831067584000000*t^27 - 427206344233772523282226011021056234375017397128359563/819809368351484123300688000000*t^26 + 233230375072972775237106376318419439335997393656981311/30363309938943856418544000000*t^25 - 82383696148740284799554163170238009810146522926740306979/3279237473405936493202752000000*t^24 + 10920915492402976244584958058487735701467625194535986299/234231248100424035228768000000*t^23 + 9875854129857973632864401857511653636073305055777837/449826813910279354348800000*t^22 - 28054793576164165377248373615024181938869002922860515211/192896321965055087835456000000*t^21 + 196765868131926763409255928982028509083445137200845385801/3279237473405936493202752000000*t^20 + 44895712471474550947261523390926483399924021208661079199/192896321965055087835456000000*t^19 - 2516763939402022518641102484559871948752820458374223511/13116949893623745972811008000*t^18 + 683383242423973813722110674861592266040798504835488579/2335639226072604339888000000*t^17 + 9551582243393067315747446664312648764783363701043483443/163961873670296824660137600000*t^16 + 37688545235208220790255624359549701933263527252186751/4384007317387615632624000000*t^15 + 2111803930656907771295863766983668183608318821012013261/19519270675035336269064000000*t^14 - 33344787290523734721375591057492523000825657354298969/15765564775990079294244000000*t^13 - 181991401427712872892281098300100476763376805555652111/7319726503138251100899000000*t^12 + 1104774391195229056263082185733584041524369079618784161/102476171043935515412586000000*t^11 - 210225626383716510288541316121649181038933186684784/266865028760248738053609375*t^10 - 4843426538011788510624148701163142256377534295654161/2329003887362170804831500000*t^9 + 64474241552175370672760427309289693144805147417927/355820038346998317404812500*t^8 + 2696318300309955757006949082093955122870480175207/101662868099142376401375000*t^7 + 1171326992488300605592492522420776038026650334951/177910019173499158702406250*t^6 - 353018111746769002321116945467370681176432094233/177910019173499158702406250*t^5 + 35119853336532971506346335348593771454264634/5232647622749975255953125*t^4 + 15848992065529999505145730552112891449580452/5232647622749975255953125*t^3 - 7699848645945370167603150071617993016/1788939358205119745625*t^2 - 71031887602892998267458937997820416/596313119401706581875*t + 354167497791187597549086932992/1404784757273727375)*Dt^4 + (926863917/1496*t^38 - 77012890293938363313984002411/4570713955294206720*t^37 - 2252644439222491644300903887204183/617046383964717907200*t^36 - 869149272929686463363213300333692739/4442733964545968931840*t^35 + 16287402472475020507009088807607527563/4327338277155164544000*t^34 + 2851408856572554054618135102357416862348859/16660252367047383494400000*t^33 - 2456209139616985007579909141869289215334859899/449826813910279354348800000*t^32 + 74569943821186080777538273091534172676015389707/1038061878254490817728000000*t^31 - 341805691520597618769211904055444523415557058519/525771600674352492096000000*t^30 + 74593087143765351335811847425857447016941535131/53386039453088099197440000*t^29 + 11287941065851809468289068993154812735511690462517329/364359719267326277022528000000*t^28 - 7989930650181403705023691325651257180122715909104103/36435971926732627702252800000*t^27 + 7419612977260098562816673553212318693757114764393533/21020753034653439058992000000*t^26 - 8206727584365837632719248773812800512830219040266661/29811249758235786301843200000*t^25 + 140523483753399183713210523152658685347448691099642041/18017788315417233479136000000*t^24 - 7195995643651808524752444018205394269360447254605463673/468462496200848070457536000000*t^23 + 81155731281154768456648878277386159262575996874002039637/3279237473405936493202752000000*t^22 + 27729025103399830408303404305710475946693909117810881473/468462496200848070457536000000*t^21 - 46310223284807046187548933805513695342980835033303150959/655847494681187298640550400000*t^20 - 4581480201246584045091859440611856454812723303704507617/74528124395589465754608000000*t^19 + 254441545812614994581710107313640418277444273472706495623/1639618736702968246601376000000*t^18 - 29149141907523265753459211616857189209119260257354284031/182179859633663138511264000000*t^17 + 17999844568834338298945449385685633995934108913097753631/149056248791178931509216000000*t^16 + 110387404946095242755543572725305893492470167322038891/1829931625784562775224750000*t^15 - 947885219770784594487971980559858487955769562612247681/81980936835148412330068800000*t^14 + 33818473289166480357793422856554923208400101293872576283/409904684175742061650344000000*t^13 + 406309336465920973048622188683834027931776659072234771/14639453006276502201798000000*t^12 - 596489494263784845697808883906913684430733490301074807/51238085521967757706293000000*t^11 + 67540838741522577123504731328936318887792065396037329/9316015549448683219326000000*t^10 + 484834902790232397970181633764772487486658547514191/281527942428394273111500000*t^9 - 9155484167297474740876435303581961610855838060867901/8539680920327959617715500000*t^8 - 5582415895424514003552838008194892686151931411651/194083657280180900402625000*t^7 + 2907848417548967362384411705313881643980746500249/426984046016397980885775000*t^6 + 55086049583634731155414090980215708227871184664/9883889954083286594578125*t^5 - 256417285660259379625937666804845153393925569/350907335647927334718750*t^4 + 769042759495597967715233935384132188844828/38760352761110927821875*t^3 + 77349493991486281501366577789090084024696/1744215874249991751984375*t^2 + 44239175497997135920371176355750150136/12920117587036975940625*t - 5106555839933159369414845525504/151284512321786025)*Dt^3 + (26940747/136*t^37 - 3637191457516959712461947993/652959136470600960*t^36 - 3919843662961295558761692413501/3024737176297636800*t^35 - 572690398075704272989283413632966503/7404556607576614886400*t^34 + 3636099365848933987619611279235983207/3606115230962637120000*t^33 + 132778680443581151702273350118111584833113/1851139151894153721600000*t^32 - 28174531473533872437541715553695371555589929/14994227130342645144960000*t^31 + 20144805122329355005907193373396010291344955329/899653627820558708697600000*t^30 - 332977375269836194984921043292522473462434610711/1686850552163547578808000000*t^29 - 542823228038808564860135895444011785932025770059/4498268139102793543488000000*t^28 + 96481952969956718719451240695303154711302387587173/7590827484735964104636000000*t^27 - 35803838100974114089434549561494251808556236338859/583909806518151084972000000*t^26 + 29901487236755819633868806250748726943212131608393/632568957061330342053000000*t^25 - 10358054333829067592066299614175746115388780874802247/42041506069306878117984000000*t^24 + 562956265097812663893561789161830037210389917310091/195192706750353362690640000*t^23 - 373857804916318978724538265508949171916103091166571801/121453239755775425674176000000*t^22 + 87601297710068551401439978523262217406722808719033/21802280951850543144000000*t^21 + 32970328753663424502841448297756702990178356135007411451/1093079157801978831067584000000*t^20 + 2004945902633146666785077053220778449957363565336023/156154165400282690152512000*t^19 - 16075417228035185893187013069028438098645456689859282293/546539578900989415533792000000*t^18 + 1580207802738025836583179312191495744826974564549866453/34158723681311838470862000000*t^17 - 308914040962937056920103645648438491332893099111696451/6429877398835169594515200000*t^16 - 35000616364346433550128876597581508184917830876231843/24842708131863155251536000000*t^15 + 100180420292501293572755003524834857185122602501767263/12421354065931577625768000000*t^14 - 1031823704460459903727956494411402762171135795147784751/45544964908415784627816000000*t^13 + 2553343749869452008651110165794978641165171414172879/271100981597713003737000000*t^12 + 737310192682530214226834818296616109491980957077480263/68317447362623676941724000000*t^11 - 65257537796655571393553805646229727250845869778299387/34158723681311838470862000000*t^10 + 50015734036476024343271534445443995867606674706423/36108587400963888447000000*t^9 + 8440794589717635749334646939117320971482594519334671/8539680920327959617715500000*t^8 - 291349926619986407701315900726659575779881714606343/4269840460163979808857750000*t^7 - 13600531572888320387743917084848071919938219079369/711640076693996634809625000*t^6 - 66204625858715353931503203774536002581103174021/39535559816333146378312500*t^5 + 26154939158903210785429527443603063128419798671/19767779908166573189156250*t^4 - 17124553586015410577099408024155949974006336/348843174849998350396875*t^3 + 675460872987018555291606629166052327683628/581405291416663917328125*t^2 - 146886846163677648837666297714988188568936/1744215874249991751984375*t + 2988375870490647423039944244786688/5017088418834740625)*Dt^2 + (696855/44*t^36 - 96611871144697628697014179/207759725240645760*t^35 - 2431058197030012400059081104013/20568212798823930240*t^34 - 509383306059293507752337087738027/64725145171124256000*t^33 + 3012283550853377911576163653407433001/46278478797353843040000*t^32 + 458675086653361860795089487000962505061/57848098496692303800000*t^31 - 155697564990741440191876933143722901165127/871757391298990996800000*t^30 + 87115386187266791439137041357705689790602809/44100668030419544544000000*t^29 - 2260499909874743327949454769111942332142789759/132302004091258633632000000*t^28 - 97391266824481890885325128427252673224955172901/1190718036821327702688000000*t^27 + 17622924087446536831054405464910924931761543674113/12145323975577542567417600000*t^26 - 138411690162359203821635315741104722183574214093879/30363309938943856418544000000*t^25 + 13642317185685311730966812582213433272369478606439/22772482454207892313908000000*t^24 - 8966916217511363043546182917318142753998551532141481/136634894725247353883448000000*t^23 + 1708872665665006340507869255563655594194866155998763/7590827484735964104636000000*t^22 - 106328928936434861551946705020740447391847259980198527/546539578900989415533792000000*t^21 - 12809530146934643580208425645683642334719108751480889/42041506069306878117984000000*t^20 + 1217701751082594325765777263602419614256195004896668809/546539578900989415533792000000*t^19 + 187326721742613544294565935566077513158110567531258499/36435971926732627702252800000*t^18 + 504787443692041342417496270267710005989246231983991/68317447362623676941724000000*t^17 + 730417389548645020755873772869046655724561646545914903/91089929816831569255632000000*t^16 + 45850343287346646558256537279757723120266655839019227/91089929816831569255632000000*t^15 + 4977183746863349513673486621638847742369327627147919/91089929816831569255632000000*t^14 + 41884925033045879456753331789596505163269299450626609/19519270675035336269064000000*t^13 - 31871157558530756220498563959311456171657387095084609/13663489472524735388344800000*t^12 - 2912794846721388038647178735657934486955875125306273/2277248245420789231390800000*t^11 + 25732603421714123931402034293621267691258357527148703/68317447362623676941724000000*t^10 - 434227825067451411063585238719542300345372573779321/2439908834379417033633000000*t^9 - 19101345013138351196975670363724816422450350640337/271100981597713003737000000*t^8 + 180758263279033154809102208686139199343704601629863/4269840460163979808857750000*t^7 + 31932618704991280901269916729080586212775649075163/4269840460163979808857750000*t^6 - 55145762192090425414291909951535730168986189639/118606679448999439134937500*t^5 - 3065379758759733829452949738486620612345919019/5647937116619020911187500*t^4 + 2750038653088877745520371459669624159777896/64600587935184879703125*t^3 + 3620088638151234859993240411042062045884164/1744215874249991751984375*t^2 - 60673526900813285532144230002956802566532/348843174849998350396875*t + 35723778835849218446772993025241344/29500479902748274875)*Dt + 70713/748*t^35 - 2254196237418230787362949/761785659215701120*t^34 - 44877843236434423352688646289/51420531997059825600*t^33 - 624228034928311035150108536446967/9255695759470768608000*t^32 + 452382154652340148488599526721933/1577675413546153740000*t^31 + 56182486496568195771663642902489269/708344063224803720000*t^30 - 12370558004781850653050925782319935955331/7497113565171322572480000*t^29 + 6765518072578618718770098698083807358213/365772332013562656000000*t^28 - 532496070232775680557985816233766121795814193/3373701104327095157616000000*t^27 - 36373525653684501951235282817867182210532115869/20242206625962570945696000000*t^26 + 638313900534824083587311842578555264241059372019/30363309938943856418544000000*t^25 - 18703802273648173353259560089723309261912222189/379541374236798205231800000*t^24 - 382897166336620168542725556991365488165473776071/22772482454207892313908000000*t^23 - 6126952193930979505983510130756823445716866343567/3903854135007067253812800000*t^22 + 15420340551866101415894757017266434773539933781507/136634894725247353883448000000*t^21 - 19508097547866645498922757810210985757300179534827/4592769570596549710368000000*t^20 - 5465371369159499779021693837126865267143190150271889/273269789450494707766896000000*t^19 - 3004682975407785886041573883361859174574491618233209/78077082700141345076256000000*t^18 + 453119536100481684167457242660290732385852751694253/39038541350070672538128000000*t^17 - 1065690210030536936758351763371701452564255802992023/21020753034653439058992000000*t^16 + 8520268315698116867592500361113711100650130282429977/136634894725247353883448000000*t^15 + 68414653809455447088857079571745855758608149685593/1236514884391378768176000000*t^14 + 5094793603747978744817966602218406272239945786022853/136634894725247353883448000000*t^13 + 47427242955878153971564728801316576861116726984287/533730057520497476107218750*t^12 + 24281808722036316705789162136064432733992806949963/506055165649064273642400000*t^11 + 141131999165920394807262759750359839836350054323721/13663489472524735388344800000*t^10 + 44854555352678561498229275297712887428044753598531/4269840460163979808857750000*t^9 + 6990119454830147310418608507962200941623389249339/1897706871183991026159000000*t^8 - 359506976893800577603093446404261385472289423/790711196326662927566250*t^7 - 1475166729310666355135871198141001089844758911711/4269840460163979808857750000*t^6 - 1880934580012126452885572578104612800634172997/29651669862249859783734375*t^5 + 1909105507838813095786800870829069638072323/775207055222218556437500*t^4 - 74037174186219504147829476946386394994384/116281058283332783465625*t^3 - 1187674579119362531861140808432908296712/14907827985042664546875*t^2 + 1054622611914749456598647334489444040396/158565079477271977453125*t - 34337609424502126239158050862925056/737511997568706871875, (t^86 - 20383213392210324594634566815160902041/829153166768956848285044931246292080*t^85 + 62239501461776176803093533007381560319470433721/8692866772548178050709018232304055466810880*t^84 - 403450187679373019972368074903580705826385288306092843/1210307840741882830000216608483693642644078822400*t^83 + 7780473637506379690696318359396442975684030041910814953/430384810635463952389260875717130224819724288000*t^82 - 8250751503230065109207806858062121696393664660445757907510261/16892173432631324667326100111021644193949358579712000*t^81 + 398229319513284076201842888072190756370683567246091631883014771461/34354926945653751856748548492460019480672117715894272000*t^80 - 16047051100788209864145068910496684562949203901503685877448451081257/222027852334943130350795140523079381218173526727720960000*t^79 + 17297993869521007958874947484226734313472597437056479706358671827982556801/18032214055234741274570178132722415025015181146718585487360000*t^78 - 64034302263842771574523808388137831719319814157031442676929339691136882839451/5193277647907605487076211302224055527204372170254952620359680000*t^77 + 4411848459904375887105292426682814157064974602349747352566729415339653115959399/65028867939016973055562993697414260514559095001453319767982080000*t^76 - 46179880697650063502603815322248429162793934696004840086927620657797084466061023/82582672781451002592033985861133447401924740277919246576517120000*t^75 + 20297475671552476313717924087927836670932692423933754596981417644611323441427313753/6102732768284039143141193911475873364558764278011241418792632320000*t^74 - 28583144076256431888827109427543674371387456588498634388020092467702349915840040065681/1824358113436440407379032203418846378746331768285478170017891614720000*t^73 + 151130998642507960011330309400379384616661969123197529422087914403221472002217760902066041/2315718565321988357099784876872989003422010457877033623809377089617920000*t^72 - 626657488565363500485444376576985452525189532268247776662596229881047009050315070179572069/4809569327976437357053399359659284853261098643283069834065629339975680000*t^71 + 4261134624457092333571002619864440217498074648411286993168443689841742337546361909758246859089/3693749243885903890217010708218330767304523758041397632562403333101322240000*t^70 - 97094612183299020934551989670581412908538429150562120566101051169193908745962198828900527802401/28811244102310050343692683524102979984975285312722901533986745998190313472000*t^69 + 2639652752305579867397269513374297601686126673250833025554222974961826156939670318163482763357603/307319937091307203666055290923765119839736376669044283029191957314030010368000*t^68 - 185391559688618246203260129861472148972648425601544924884795419456513164907673137420236056070116891/4255199128956561281529996335867517043934811369263690072711888639732723220480000*t^67 + 264042922129326178567844011766950284089335354426913688402930529981512306867477567913233405307541819/4255199128956561281529996335867517043934811369263690072711888639732723220480000*t^66 + 25039931837865706012535177869983338454653435885317895230735272585399616340391697591795771639098975923/160923894331448135737861679610989735479716502692154097295285970375346623610880000*t^65 - 10071780853410765794477788537835419200626839153319596273041816292400981913185737420058754394288021969/168586936918659951725378902449608294312083955201304292404585302297982177116160000*t^64 - 29852254668634949728551762616682420516327590928573458592991262152321600588233105272674838891357013231/629391231162997153108081235811870965431780099418202691643785128579133461233664000*t^63 - 504745997341962698545875373064483033003647802399056321203029930213421663485226528399091395924732094863241/4078455177936221552140366408060923855997935044229953441851727633192784828794142720000*t^62 - 5255845693121735703772208415610200326385160762893452516487602104089036016931475931138992039595646252332549/8156910355872443104280732816121847711995870088459906883703455266385569657588285440000*t^61 - 748454879826474532810758726994559686357003183577477736984158526977425666654830693822507439542061104872693/436490186267421704576895401531603890942922814098188997121255131311602389703720960000*t^60 - 63240967250396163770492139976205020951918783761989008497996620532037444359001398909066459839978292561835999/30708368398578609333762758837164603151043275627143178856295361002863321063861780480000*t^59 + 2016425206814559971979619802880579955811847682945869026010517775891113106670383042297150963000489602479636683/348028175183890905782644600154532169045157123774289360371347424699117638723766845440000*t^58 + 38409342530771903255938133875499265721221696750244274441497490509915588478007265023839629382395027028103988069/16705352408826763477566940807417544114167541941165889297824676385557646658740808581120000*t^57 - 70584328861400032833291192322812656384152650687392384722173464002960582531370602386352186039348181170383449087/100232114452960580865401644844505264685005251646995335786948058313345879952444851486720000*t^56 - 3310868293550827013685167971436161774456966959900430427498288951956771341033729683928028566936609486797913259/1218627531342985785597588387167237260607966585373803474613350253049797932552520990720000*t^55 - 547712184050141584887354283466757018690158161626068667884927070944590889698461880062216362335690990528780567103/320742766249473858769285263502416846992016805270385074518233786602706815847823524757504000*t^54 - 4191771999326304190702641044399757088144948598793681682575302227930259412055575962601029448806173114554029394167/3207427662494738587692852635024168469920168052703850745182337866027068158478235247575040000*t^53 - 11488432134214564564644179362479228799437888105540670789112571991124752302089923467780436273474437324096939137857/25659421299957908701542821080193347759361344421630805961458702928216545267825881980600320000*t^52 - 34748174605359673919034330783100658236618597513332493423751093380047021536664991165803921445109750853150120078551/205275370399663269612342568641546782074890755373046447691669623425732362142607055844802560000*t^51 - 822298846073609734583100805253057253592832359847803302945161367981927154118385140159512058188507165612140503/23297624605568411033065777850589806159901345519583072034010852732463098642901720104960000*t^50 + 223255065705298231598671696716433304025420157000009204572152676650703244487453236312466431543720687787173127623/60822331970270598403657057375273120614782446036458206723457666200216996190402090620682240000*t^49 + 6183916105119018841993172495733525303981738547016522622017600220505109160244614320322430340608443051241118287661/3284405926394612313797481098264748513198252085968743163066713974811717794281712893516840960000*t^48 + 84365520400159445664958277770039634394666663878416350914648856029273263966420858310195567537663348089378318849/42654622420709250828538715561879850820756520596996664455411869802749581743918349266452480000*t^47 + 836389733929568346622747430991901941811157180705881388632553124981706776789474918713399844165197963381177758471/3284405926394612313797481098264748513198252085968743163066713974811717794281712893516840960000*t^46 + 884476492586046124272777562744760758283743417495393452304777478383171688296846606752383222749817431706774446901/13137623705578449255189924393058994052793008343874972652266855899246871177126851574067363840000*t^45 - 32838826009233393129002650544675533003458309599869795467390158369501048396921491974747438024195167107685050403/10510098964462759404151939514447195242234406675099978121813484719397496941701481259253891072000*t^44 - 2512819009005539063771168106820808090267442007896269667165828626523505669655147202569226330305494222539603257/3091205577783164530632923386602116247716001963264699447592201388058087335794553311545262080000*t^43 + 506135911396928133464303163010480778290808769627792668963283363121552658241835082406393136808898732709325861/4204039585785103761660775805778878096893762670039991248725393887758998776680592503701556428800*t^42 + 2163112237277734843701861972753313291931219260580477330903147017356700606085840139321255353202336453416864729/105100989644627594041519395144471952422344066750999781218134847193974969417014812592538910720000*t^41 - 24492589059548697841097102276402315914302019788307510724290229498881346164137325418683584472481143273104497/10009618061393104194430418585187804992604196833428550592203318780378568515906172627860848640000*t^40 - 17956522907877981795085928507013166381121654672524454258240882610682104429130636621065865436526652693616023/105100989644627594041519395144471952422344066750999781218134847193974969417014812592538910720000*t^39 - 7603625572014839098349806145011333256509586735153811402071166201903337143743924330900320479373127032251551/210201979289255188083038790288943904844688133501999562436269694387949938834029625185077821440000*t^38 - 42171947550945941064884152404462266491785049799692636514781252298213666452959229363331886024904693886193/35033663214875864680506465048157317474114688916999927072711615731324989805671604197512970240000*t^37 + 128810711608934575482522725093323182852763185458998560090062711559938195972745715348814192407196937810113/105100989644627594041519395144471952422344066750999781218134847193974969417014812592538910720000*t^36 + 1739816393892512473162489812371587495280584476442172962161657352039780238713517547121806840737118007081/26275247411156898510379848786117988105586016687749945304533711798493742354253703148134727680000*t^35 + 753993682339446180172728332291549183464371852774538130250644443473951443069440249306874799195149760487/210201979289255188083038790288943904844688133501999562436269694387949938834029625185077821440000*t^34 + 39722062961501993795115556388942942176257611075121373603435761903617933091449351304189727447731897379/105100989644627594041519395144471952422344066750999781218134847193974969417014812592538910720000*t^33 + 566161173793388433131255943319896135698478061044087498214329993104855916687764428735112613465697/55608989229961689969057881028821138847801093519047503290018437668769825088367625710338048000*t^32 - 106550319620994854654693158570473831082195370280865792688899279203991938974178157599722956691809/1194329427779859023199084035732635822981182576715906604751532354476988288829713779460669440000*t^31 - 3208297738739565697598723446801359521669142705363978194429224771600760306479172510844362810417/1946314623048659148917025836008739859673038273166662615150645318406943878092866899861831680000*t^30 + 271908924248170602355757975401201381277819525313831155356682875926092154959377713242761843/2027411065675686613455235245842437353826081534548606890781922206673899873013403020689408000*t^29 - 9966753698282200069484775585553321329210534543220091593034844079251752621293495142694862583/6568811852789224627594962196529497026396504171937486326133427949623435588563425787033681920000*t^28 - 65045664498262387594323430656599252309636351689232872635177898878624023814872199741473117/938401693255603518227851742361356718056643453133926618019061135660490798366203683861954560000*t^27 - 24613706354381731590363117075841267414739617515075437188978838178399822266872278629269/410550740799326539224685137283093564149781510746092895383339246851464724285214111689605120000*t^26 + 53097498711725606753115175954684799795546872676377037610319133104603559877478950007/5366676350317993976793269768406451818951392297334547652069794076489734957976655054766080000*t^25 + 155232551757593060468072001162230442282376660256592783618535996916265226928018909/9123349795540589760548558606290968092217366905468731008518649930032549428560313593102336000*t^24 + 2057621388010299742227304321327557591910261378415255708869862118666723571721299/8553140433319302900514273693397782586453781473876935320486234309405515089275293993533440000*t^23 + 162686108781014751731069584766982063545226173159328576356444250719319152891/570209362221286860034284912893185505763585431591795688032415620627034339285019599568896000*t^22 - 7905152512592482595695318042484846956002780930998640113075577945805525931/2138285108329825725128568423349445646613445368469233830121558577351378772318823498383360000*t^21 - 146990290254874774771467539388937017219578066882683130598313447361423/16705352408826763477566940807417544114167541941165889297824676385557646658740808581120000*t^20 + 21898962614163341357189891340709989191346860017712971461590114297/133642819270614107820535526459340352913340335529327114382597411084461173269926468648960000*t^19 + 21903542715670356972764498474142017213520941228786969015861/111369016058845089850446272049450294094450279607772595318831175903717644391605390540800*t^18 + 5344149363499045240436940156041910084575891320342910381/15354184199289304666881379418582301575521637813571589428147680501431660531930890240000*t^17 + 4463227246054368206968814598366179646459355221701/339871264828018462678363867338410321333161253685829453487643969432732069066178560000*t^16 + 2527166372967965379938100663101417257723/4515721524606962992644077744185936454787963085749222117979963454410236887040000*t^15 - 1273255328363399665827208928699993/98781408350777315464089200654067359948486692500764233830811700565223931904000*t^14 + 6107901094909158274861/61248393074638712465333085723008035682345419457319093397080667513159680*t^13 - 102035/2713222607886422458554102231366037173396569950174016062881792*t^12)*Dt^13 + (113*t^85 - 56390494953334300412021995182218170003/21819820178130443375922235032797160*t^84 + 115554989890657452250915006535138875389663710987371/147778735133319026862053309949168942935784960*t^83 - 22277143987709934203148392411287902296260707064452044729/605153920370941415000108304241846821322039411200*t^82 + 494909127021922209246497803217011766400664630002487733290163/268129737025894042338509525571772130062688231424000*t^81 - 771858073334031456059834746219529822759126677283753027292314459/16892173432631324667326100111021644193949358579712000*t^80 + 990177552735528939119309110437870646750189217141708688420933023338433/927583027532651300132210809296420525978147178329145344000*t^79 - 17659008092956709247705058075977432153031044144077774477906477150237417/2782749082597953900396632427889261577934441534987436032000*t^78 + 33826205112930038826083928312561791464959863629709481100714094178907131593/368004368474178393358575063933110510714595533606501744640000*t^77 - 3675613614453077698969097257369389668543167928547541114163865187264365975569/3249860856012268765379356259214052269840032647218368348160000*t^76 + 102749178722996475035018226717110170811422519930389231269295652505488467637340371/16805213062892026744696054551017168447582687472285689378242560000*t^75 - 27162804776311881517584729098252385405875498158626005507564576600352579157519627/509210352312332643181446555527321805428928793845312547454976000*t^74 + 309260119646413191462691378384582890504282789105558947255628177774519132638977677826271/1033802930947316230848118248604012947956254668695104296343471915008000*t^73 - 194217433445454672733428523593586687924455114649606185053849461039508117804543761986431/129225366368414528856014781075501618494531833586888037042933989376000*t^72 + 2113921570150616017582061280778840904684995583082937735519931742766171178142994765542920087/365639773471892898489439717400998263698212177559531624812006908887040000*t^71 - 63296476205731211414260422590768602206831023792915783267857282324721581088880757964842964399/4809569327976437357053399359659284853261098643283069834065629339975680000*t^70 + 45111534146005145746430264472334512641507273131612729219543788819947324512530543095897226595913/410416582653989321135223412024258974144947084226821959173600370344591360000*t^69 - 205254203155421275344232570469448771617914982608892295378399562919839710706180098075946938832313/651838101862218333567707772038528958936092427889658405746306470547292160000*t^68 + 112231640050389112752759038780346237765811349881265941646610287506481305997313799330852849460189117/131708544467703087285452267538756479931315590001018978441082267420298575872000*t^67 - 62390220794448687079550158441209615188797495150010545472993702843905997077072862807881698959605073/14774996975543615560868042832873323069218095032165590530249613332405288960000*t^66 + 3087507553401356803671586275236542010157213005130886173050928419214435509856398815227781649223274271/567359883860874837537332844782335605857974849235158676361585151964363096064000*t^65 + 13410667980070846227481547794740285344564210860482676491738430001367776048473079772259881291909554171429/885081418822964746558239237860443545138440764806847535124072837064406429859840000*t^64 - 432133764739012691482457105949977309435109300626766505154469970511994218331872830422683406092364540689/76963601636779543178977325031342916968560066504943263923832420614296211292160000*t^63 - 1355402705291985494117612019657298174489075088762377895528450251488002367595417133033081461714013863529/295027139607654915519413079286814515046146921602282511708024279021468809953280000*t^62 - 40000870471993712952562661056960676094492057109175772820445323809535339412588082344130007159397174995429637/4078455177936221552140366408060923855997935044229953441851727633192784828794142720000*t^61 - 121187555007132928121621986163436104839190766126296976347772960125551060423109695070163328636793467995388757/2039227588968110776070183204030461927998967522114976720925863816596392414397071360000*t^60 - 12037662613336882299385061604756006210427743512246963513076735468271020553986360691706196750439970706429809/71200526837948221313961661242743897104164714356442176835382042696218829526138880000*t^59 - 4807725338570491113863508910851524921273231986136671652674049270313386040762524849260301474842677546129206899/21751760948993181611415287509658260565322320235893085023209214043694852420235427840000*t^58 + 88061784619978585398999103187373138562808056110328307562832695879884466803842149006118857764769229319910357067/149154932221667531049704828637656643876495910188981154444863182013907559453042933760000*t^57 + 12860434707294186682258716345621222782221851220924380416352206231496892474074566279110940172967141997647722091/51243412297014611894377119041158110779655036629343218704983669894348609382640517120000*t^56 - 132077009948043112657831508581819928555204362600889004938255180718818663287326277766602540155782462739470704253/1542032530045547397929256074530850225923157717646082089029970127897628922345305407488000*t^55 - 29081867844442280100871326200859456449328960564926656808471148799324367999631801950293250154237650051059466740643/100232114452960580865401644844505264685005251646995335786948058313345879952444851486720000*t^54 - 12166400569356305737469630445715635421334469808310523195266017019657125588439014993582776795001046127989102727041/69726688315103012775931579022264531954786262015301103156137779696240612140831201034240000*t^53 - 69303316310241208569655113948883505385250254143484529496325406204104856190431508131651096693989752013258269235571/534571277082456431282142105837361411653361342117308457530389644337844693079705874595840000*t^52 - 49155945456660078513478289648512704389649121076527029690241506720427316944261221303642408965885809716420776786273/1115627013041648204414905264356232511276580192244817650498204475139849794253299216547840000*t^51 - 7208506549352204180698265912541799153865235634830531875324333207721496547181208906207769835487267389420268336903/458203951784962655384693233574881209988595436100550106454619695146724022639747892510720000*t^50 - 650664159696479856501645742661325643891050663012985069974231462826943876095485731561184832159587253193698143257513/205275370399663269612342568641546782074890755373046447691669623425732362142607055844802560000*t^49 + 453762208320294957226151820473041349802238639582287970750360665602920549907127604534088973788288220655807612109549/821101481598653078449370274566187128299563021492185790766678493702929448570428223379210240000*t^48 + 21511248296830717871515420719559811327054732081879831194780266352490917807493121402868899623737568077394386594037/93840169325560351822785174236135671805664345313392661801906113566049079836620368386195456000*t^47 + 19871358689004348805152680903003515840486739672175246340590870693718744922312130539409331607795598625734202166903/102637685199831634806171284320773391037445377686523223845834811712866181071303527922401280000*t^46 + 80088819736456435680649843179714738523550273064424301633057189920139843154337766869134410751824315378983720430191/3284405926394612313797481098264748513198252085968743163066713974811717794281712893516840960000*t^45 + 536775049538584131765215842606766322819942950413038968540775140086543920104648717645917384744281217302932431/86539909792361828965087440834325762813997815320960230895638336731749365503766890020864000*t^44 - 17762052270920927645882205607615135364287966498998872138711287925744711698372307481681178979814775878777948733433/52550494822313797020759697572235976211172033375499890609067423596987484708507406296269455360000*t^43 - 191653589105147881829553505623140451412472801811864101242332385916026676723205675783921011868965680569333601/1782339398396208011828778238103241629737214535866907156731360181691340547704090567639040000*t^42 - 8207714236739721067723046276276718201787608842723122790907826750727878358211850306382075492572906759556303581/7006732642975172936101293009631463494822937783399985414542323146264997961134320839502594048000*t^41 + 1158352135168456628477693399522459156549311569882978681252459568160154552674830832857525912618278009950982173/834134838449425349535868215432317082717016402785712549350276565031547376325514385655070720000*t^40 - 45855278560611359079609287505307455455315907068876961536492933144596597314804029027676425216230742380197421/442530482714221448595871137450408220725659228425262236707936198711473555440062368810690150400*t^39 + 520858854084525022435282697798122783595233439944637109707327618899758980358536526154268104384350450789233323/52550494822313797020759697572235976211172033375499890609067423596987484708507406296269455360000*t^38 - 53143494611719953066616567603598179121096134251459790988371146504009391368757132789653745970232752341832957/30028854184179312583291255755563414977812590500285651776609956341135705547718517883582545920000*t^37 - 280791311251610813949187371813070614655981594067837043843117399573460575684386955991988330265244441249397/3753606773022414072911406969445426872226573812535706472076244542641963193464814735447818240000*t^36 + 236954357684310377824119358837489325837177064842843086411222334232235296068554975649141698012598013041459/3184878474079624061864224095287028861283153537909084279337419611938635436879236745228451840000*t^35 + 800942319980857561732346009488830766228755001804356071030853790176792737053012388165198320272138950431/210201979289255188083038790288943904844688133501999562436269694387949938834029625185077821440*t^34 + 43422807527720765114442268478948105332097830141008630406571145045576380201038877779096887867784141651639/210201979289255188083038790288943904844688133501999562436269694387949938834029625185077821440000*t^33 + 3612999018342706835569811808195974209952360052228160085999929498985467699324645429803295796820704227/154560278889158226531646169330105812385800098163234972379610069402904366789727665577263104000*t^32 + 6783507091249926226406502687744314578096564861088277757521214068148494431776047674196888383669938589/10510098964462759404151939514447195242234406675099978121813484719397496941701481259253891072000*t^31 - 6945309401998845604681641373876774571998639093294183211394294302803595933111382166445381466132149/1194329427779859023199084035732635822981182576715906604751532354476988288829713779460669440000*t^30 - 145911193286362359416736251976101001709923118689861120344988109840410167686438170133444639822081/1347448585187533256942556348006050672081334189115381810488908297358653454064292469135114240000*t^29 + 117136496832602318461353964164565118159670159959598334052603697398185083311125424567893079225491/13137623705578449255189924393058994052793008343874972652266855899246871177126851574067363840000*t^28 - 220762883500877280228202198239314002926846810524942285105046271561622907622047154044493947481/2189603950929741542531654065509832342132168057312495442044475983207811862854475262344560640000*t^27 - 151983674550440611998783694188575199975553960781900186232761772868943425108721137466322343/32200058101907963860759618610438710913708353784007285912418764458938409747859930328596480000*t^26 - 6958977141231067215323072887927625113731056172087639646996065769385938771979816863240909/1642202963197306156898740549132374256599126042984371581533356987405858897140856446758420480000*t^25 + 2786282249092854800929676390793633065833860703968518094365610544829808441936376464341/4025007262738495482594952326304838864213544223000910739052345557367301218482491291074560000*t^24 + 164980760699084070558028981316005909871844053996068895673127898358513984211560374987/136850246933108846408228379094364521383260503582030965127779748950488241428404703896535040000*t^23 + 592593766867392675947248360713779760366780828489435289976282519860995366859807391/34212561733277211602057094773591130345815125895507741281944937237622060357101175974133760000*t^22 + 44622625068375014744792875514954134197674048637892302466127464972883527392077/2138285108329825725128568423349445646613445368469233830121558577351378772318823498383360000*t^21 - 682257995452036360352811170277093008190188366677043169976042798887331/2494604426142285169615812637195969087144870023086869992348673739624364500047626240000*t^20 - 18567241288914722827521880179652141169091623543778397364811489866939317/28135330372760864804323268728282179560703228532489918817388928649360247004195046031360000*t^19 + 534203155077972995888885631900077308873575559432275465558807383/42902991740165042639016220372179888575711183155482219705488735500629590134807855104000*t^18 + 14051767465186780103878726054774569837987835249453574575141013/928075133823709082087052267078752450787085663398104960990259799197647036596711587840000*t^17 + 54510078059586133209123076272970056736578012458000834851/2007854856830139841053718847045377898337444944851669386757773604033370992637116416000*t^16 + 352593191133025253325080799273370976046535905261547/339871264828018462678363867338410321333161253685829453487643969432732069066178560000*t^15 + 23586949106987333724754429678753148668429/526834177870812349141809070155025919725262360004075913764329069681194303488000*t^14 - 103133772982200036014263515360048233/98781408350777315464089200654067359948486692500764233830811700565223931904000*t^13 + 13180208188183984797079/1611799817753650328035081203237053570588037354139976142028438618767360*t^12 - 8468905/2713222607886422458554102231366037173396569950174016062881792*t^11)*Dt^12 + (47632/9*t^84 - 52371193054944052740839195864712752605582/466398656307538227160337773826039295*t^83 + 219522849909836493834889598272268294587977345299923/6157447297221626119252221247882039288991040*t^82 - 36595497910459272500635900512493267218677747681540423233/21612640013247907678575296580065957904358550400*t^81 + 273273510124769462759323435603665911178647183874386351995817/3485686581336622550400623832433037690814947008512*t^80 - 5235126764891972701513075121941438327113412674337624457513291/2993841235503847589301149303930216421558850191360*t^79 + 4658163182801867410741415161936001775690010431952463055948544910061391/115947878441581412516526351162052565747268397291143168000*t^78 - 12935434571456797463880276074981010601731943942726284849478244854683647/56330953089027406890620089633385861901506913663713280000*t^77 + 88235721164138513108363777433921857719481058753672747199258553921798959949/24500290835916768036100785506416324762248887427606773760000*t^76 - 3058817972917786854622309070910915254706353240234120981909267553297570950812697/72128856220938965098280712530889660100060724586874341949440000*t^75 + 42079484427058686723053305363520885126321196737269242380300834335058021252544251171/186957995324673797534743606880065998979357398129178294332948480000*t^74 - 7001683759876536152482357587256139707075184621892444356597382251864435341550836557761/3365243915844128355625384923841187981628433166325209297993072640000*t^73 + 214093787093380122285160153859132511437587748028199401635082215222642919187884116025673/19579600964911292250911330465985093711292702058619399551959695360000*t^72 - 456951189183151438511551683215285852886041542045491632711171593104298537633476171156368663/7753521982104871731360886864530097109671910015213282222576039362560000*t^71 + 31686780723421607682678083297582028842674139445465587025114469760146036954790484560011517047/153246081528660994219838705087183095814691868535980166281502895636480000*t^70 - 1263819217088129732463260043690540468938229181406653865747562406156509785389462566505454698243/2315718565321988357099784876872989003422010457877033623809377089617920000*t^69 + 333625663664963722294652819368638070631900465101948083406477271104780010184940654451106991511173/77952500276812906774060290920192045413894170218406118869011758653112320000*t^68 - 862329697662059196018470515917343719365501866600982598610881014115774981034447395510626152620572197/72028110255775125859231708810257449962438213281807253834966864995475783680000*t^67 + 20004081430551679475954193676241497228814872699808122640011791973014347388821137730669085496803032419/576224882046201006873853670482059599699505706254458030679734919963806269440000*t^66 - 3079872836871924021648593527759996415314121777534116556054504086694399370090553715788642231868439958169/18439196225478432219963317455425907190384182600142656981751517438841800622080000*t^65 + 83237097356052169196173929404954028616608524663516853557026783375323469211635339119994057748036687559653/442540709411482373279119618930221772569220382403423767562036418532203214929920000*t^64 + 117679677849364367130145657335341124049940847445778090905430652013392527747464099443544998785456366123621/196684759738436610346275386191209676697431281068188341138682852680979206635520000*t^63 - 24593440850254810841258725104332315001452611127878447893277884432843441811388460054411039543862760750089/117035559513780462354808494262537989439793820139748434396571614818433908080640000*t^62 - 28636382206403995367569373518489436045315806447995933818316233132407294566226878627295788211456218950322013/169935632414009231339181933669205160666580626842914726743821984716366034533089280000*t^61 - 55016626054535913881657280663689894812115438023527708776807864190513795669412968022791031601943792128771287/194212151330296264387636495621948755047520716391902544850082268247275468037816320000*t^60 - 302386317076955596297315200431531042374665118090667949311563170652061809626747459730147549223496147548555183/139434365057648600073174919933706798495655898948032596302623166946761874488688640000*t^59 - 58751594470235324881068720821096857174884545008064283521943105796691085793421809174324553280173107370263595347/8700704379597272644566115003863304226128928094357234009283685617477940968094171136000*t^58 - 40607737909096505271906677732194036663535452610549012370307365649182498802025647108165281875941813597394570867/4143192561712986973602911906601573441013775283027254290135088389275209984806748160000*t^57 + 12064026002911775568083491958018387516604447558440261052429659907201425612994445319265855629592334423094661713/493656986076441001110134184616357686588875352871332426058648829360450551381229568000*t^56 + 783955527925985087314415934771642107876384542685026226202460845080320533787283935337879241053104651703565586559/70190556339608249905743448770661950059527487147755837385817968006544733860255498240000*t^55 - 2915716504422648154016387676725193563684129279384435662949815149998757264746036783976123110583149298085864983107/742460107058967265669641813663001960629668530718483968792207839358117629277369270272000*t^54 - 153115236212881294766142370662464506711720136375952493709397433127409852836255585562278920334914454791144567780109/12149347206419464347321411496303668446667303229938828580236128280405561206356951695360000*t^53 - 171518421799695106243949036024428697730442539633438999294134364455135437376735893942803626502056583621163052677/23396852113202749968581149590220650019842495715918612461939322668848244620085166080000*t^52 - 405475703379776186615620409373574625905331890942711314896924719490246042356928406730123631600078230390392768654787/76367325297493775897448872262480201664765906016758351075769949191120670439957982085120000*t^51 - 141732225934067113456435251041338606947345453860946399602794720415325887190098909018169872101361270461591647100987/79195744752956508338095126790720209133831309943304956671168836198199213789586055495680000*t^50 - 5133097987294453815498757176395972097866337980446204476386672354642933266173517147381016888648153883333808993967383/8553140433319302900514273693397782586453781473876935320486234309405515089275293993533440000*t^49 - 233079217439699514175380013747695857634760236380140333648936010632719375865716250349257822157848689170110641686401/2012503631369247741297476163152419432106772111500455369526172778683650609241245645537280000*t^48 + 313393124762762329054968580851499973321270234621720435169606499566861322183499226591954653716307337904869104488741/10137055328378433067276176229212186769130407672743034453909611033369499365067015103447040000*t^47 + 6097772493194115358278189810160112733528864813800194090777045399295763915667068356909057728961998376354603988008233/547400987732435385632913516377458085533042014328123860511118995801952965713618815586140160000*t^46 + 2122902340342230615526068586515170475344361970177171899225559879490087440461189484387281903000252097557229856981881/273700493866217692816456758188729042766521007164061930255559497900976482856809407793070080000*t^45 + 922161044161441122027264045221037419995905331163980456603950638068950090806442310871319746325722200148137225813/1005789596201075582237783217965012559546241643230360791017214507674695389460025384632320000*t^44 + 81188677406399754299512846994642741469160476775542156815650938076564112439541091647968001350680370242328588486977/364933991821623590421942344251638723688694676218749240340745997201301977142412543724093440000*t^43 - 39169579875019982447882450860302062723330871295528258496587896853035679894779507724105727490890916230897485692167/2919471934572988723375538754013109789509557409749993922725967977610415817139300349792747520000*t^42 - 5126086926323187718747032471427024534147435679061430961925286049791978867646908269891753864123487969488153717217/1094801975464870771265827032754916171066084028656247721022237991603905931427237631172280320000*t^41 - 6149438517592210164010094011284348523233529395749447917137574466620460881896536397534893918248356394196561008903/17516831607437932340253232524078658737057344458499963536355807865662494902835802098756485120000*t^40 + 112997807075983556168194349592529034436224245180209779373690980722759689847487985443018697643184489613650719537/2919471934572988723375538754013109789509557409749993922725967977610415817139300349792747520000*t^39 - 2989812467271283406457418667403784458442524656093412516420781789862330008924810379034239147818159025941835371/3184878474079624061864224095287028861283153537909084279337419611938635436879236745228451840000*t^38 + 24389125594126395702276210664066182405832913236926504302196720138678203594968170066898914248311354532635013147/23355775476583909787004310032104878316076459277999951381807743820883326537114402798341980160000*t^37 - 143229297600474975315777882876191493468539148587293209011420065923134744291220756966984775213939072429879/7372403875184314958019037257608863104822114671085843239206989842450544992776010984325120000*t^36 - 100028697836201878618573041096183805956297159110397214483823581809743114660737948323043806784892971540696273/70067326429751729361012930096314634948229377833999854145423231462649979611343208395025940480000*t^35 + 20770827976358951297243280600064495694112595282282300119173729202683281352884761002975192632494868404676841/11677887738291954893502155016052439158038229638999975690903871910441663268557201399170990080000*t^34 + 954532341673390623108394225623891859925371308728212287954618375937257259226009960664191758905619175145211/11677887738291954893502155016052439158038229638999975690903871910441663268557201399170990080000*t^33 + 30336604642929188897059884339373470634381559012857175471639088537074414293774726675078338027700527614531/7006732642975172936101293009631463494822937783399985414542323146264997961134320839502594048000*t^32 + 38624600541334128784759952700124656567701150837724966856616914954091025986040956229618330322380018114473/70067326429751729361012930096314634948229377833999854145423231462649979611343208395025940480000*t^31 + 50097951663100695614007634195408037064628675137136648836153719598442597345080696622334883145904467589/3184878474079624061864224095287028861283153537909084279337419611938635436879236745228451840000*t^30 - 369105646750668652538088981762843538884896653587109348188890955887214571920108975864974199111067409/2502404515348276048607604646296951248151049208357137648050829695094642128976543156965212160000*t^29 - 1998060426105807461400554044554717573597609998254314834920544814728854496521607158645777240076463/729867983643247180843884688503277447377389352437498480681491994402603954284825087448186880000*t^28 + 577415982045890552452511836663451969630959390734578813026258536149642417291045880623537558467931/2502404515348276048607604646296951248151049208357137648050829695094642128976543156965212160000*t^27 - 2821026490001746945827560760193077139687028627668535731823926830352215490858441186883948583223/1094801975464870771265827032754916171066084028656247721022237991603905931427237631172280320000*t^26 - 16110447421648145844575166067095481502313806821293463228363706965664869212970218479762594957/128800232407631855443038474441754843654833415136029143649675057835753638991439721314385920000*t^25 - 259691088145958460566377558893756361973975289848675683061545321756439500468158501329170979/2189603950929741542531654065509832342132168057312495442044475983207811862854475262344560640000*t^24 + 964034829055305429063546263645105644986611895137752382256991100313859909573829033807/50911550198329183931632581508320134443177270677838900717179966127413780293305321390080000*t^23 + 437841820571962854860420140871181945262962438979433317887374038829075535308233486843/13033356850772271086497940866129954417453381293526758583598071328617927755086162275860480000*t^22 + 3197823091852329261875375029369620394628256083481086257040118171492910913607586631/6516678425386135543248970433064977208726690646763379291799035664308963877543081137930240000*t^21 + 3437521440702061663232219405595430879897191333077363644794412644064675556960819/5702093622212868600342849128931855057635854315917956880324156206270343392850195995688960000*t^20 - 3800219753356595703274720647509686746761446732331440439228643196839637183049/475174468517739050028570760744321254802987859659829740027013017189195282737516332974080000*t^19 - 4658062904748439202656794934450298963958651516061338935529512837818038841/237587234258869525014285380372160627401493929829914870013506508594597641368758166487040000*t^18 + 274656519406871600051747516669783502623861879494588346911199483539/742460107058967265669641813663001960629668530718483968792207839358117629277369270272000*t^17 + 132648712594428456058976694074498116572405262474328111729493797/285561579638064332949862236024231523319103281045570757227772245906968318952834334720000*t^16 + 293918911189643877932118880552972791772816034824727666178953/348028175183890905782644600154532169045157123774289360371347424699117638723766845440000*t^15 + 44523548793776039531928224648041802578809536009347717/1359485059312073850713455469353641285332645014743317813950575877730928276264714240000*t^14 + 3853260496745105417406599199213650913800209/2690217078489254548809237805046940866682190774488898283051893121776311336960000*t^13 - 2133763349709222058440066016050547/62928114891401379496154929545511934988683989489258948132385220936597504000*t^12 + 1691777560376845348811533/6281886469193714099008521612616208787932863534084009579187760770580480*t^11 - 308451805/2959879208603369954786293343308404189159894491098926614052864*t^10)*Dt^11 + (1210592/9*t^83 - 2463223403221842957035055436506944098811003/932797312615076454320675547652078590*t^82 + 2346057068237366768238758793614475700852901893964089/2638905984523554051108094820520873980996160*t^81 - 27943742616068300541170190563554248334457712131963703181/657776000403197190217509026349833501436999360*t^80 + 113155022380922054149369859062582199553862144504639378243375439/62244403238153974114296854150589958764552625152000*t^79 - 1964013969470948335190811109111728003781159184179501999990111748381/54899563656051805168809825360820343630335415384064000*t^78 + 31235930811812423420566501973530634661295417477469713976663309530281631/38649292813860470838842117054017521915756132430381056000*t^77 - 80729117775861486987618755233734739928421077731463330625778181556816799007/17889101245272560788264065607859538715292838439204945920000*t^76 + 679549363034065093932599332033873445478708841812135105283568399200675482210133/9016107027617370637285089066361207512507590573359292743680000*t^75 - 183911737591666386034858772033846686563109891684764263010292559065052356165043599/216386568662816895294842137592668980300182173760623025848320000*t^74 + 827861401392541358264344504418554333736730252872729761961190149859867252722783467369/186957995324673797534743606880065998979357398129178294332948480000*t^73 - 97210006378125117169409349385534995423429183365585260225334824172676210569533543125079/2243495943896085570416923282560791987752288777550139531995381760000*t^72 + 15227223371117777258257289829832165123323522266135219234278328759484953871928213997274001/71791870204674738253341545041945343608073240881604465023852216320000*t^71 - 4780999052190629895411866317500866246375759013855080160416416430245447349642555148717734669/3876760991052435865680443432265048554835955007606641111288019681280000*t^70 + 375385290006557453585245493926122885557462828477114198305749929932889213386749001183257064639/96488273555082848212491036536374541809250435744876400992057378734080000*t^69 - 84348787862791389742345125113012872112514908828137656486460505759313949177997971966653444070313/6947155695965965071299354630618967010266031373631100871428131268853760000*t^68 + 533846021268949783507973031348504774132346824962353390595020168275315552262172918305120341850104361/6002342521314593821602642400854787496869851106817271152913905416289648640000*t^67 - 5806850461104503252581202697161670215282106319146479173207304179047160589854197139565151748886324081/24009370085258375286410569603419149987479404427269084611655621665158594560000*t^66 + 145717264269511582402645569332679529088721589638127047410874337303230811115666426391207864761825168459/192074960682067002291284556827353199899835235418152676893244973321268756480000*t^65 - 64867580800224181590640161302579505932602800700124276042125133209821302636810638051115381216495675423821/18439196225478432219963317455425907190384182600142656981751517438841800622080000*t^64 + 3714174945476555006039058776564200098381182323434229708284678239276715338638775348491766237331044307151/1134719767721749675074665689564671211715949698470317352723170303928726192128000*t^63 + 22108440173347317944948607576658048613144670625727884158632697138355684865702917560105673869506692464136417/1770162837645929493116478475720887090276881529613695070248145674128812859719680000*t^62 - 28750257210814100780163843905103975127470188822938890898370736765319742376560245144779863655168664465428247/7080651350583717972465913902883548361107526118454780280992582696515251438878720000*t^61 - 21874696729265437719265715866695974619077959786134194969032105152456565376601496405707939297063062145371/7485821435796186570599618240130618063811313459447369135448746077986257633280000*t^60 - 146799210480931275209180968679894281139095591204440057415449832307446856007595119646154267181146533165304899/45316168643735795023781848978454709511088167158110593798352529257697609208823808000*t^59 - 9466279926503790814883338222542536575055225065539576419270861746511695621500550848891047945111313915452071603/236432184228186756645818342496285440927416524303185706774013196127117961089515520000*t^58 - 36329194340018846894168576795042214409144705616757538785237959635390301028248958470473028795941812116098183819/255903069988155077781356323643038359592027296892859823802461341690527675532181504000*t^57 - 1570542272081946893226022353633653931255519831502737458856245587024004463341958581275146036083819690914144424807/6692849522767132803512396156817926327791483149505564622525912013444569975457054720000*t^56 + 186256658081580010623667098871266841489073478219505062555782778732051251535486191557028314379513940383620868161403/348028175183890905782644600154532169045157123774289360371347424699117638723766845440000*t^55 + 15351996617811903299831402759844680952096436882753323373253260254667876567981788573650295347914033699905243417611/58004695863981817630440766692422028174192853962381560061891237449852939787294474240000*t^54 - 41530672901860264771686552544157575324877786443051616411230748859327459783833569605329847121454304487234493362817/464037566911854541043526133539376225393542831699052480495129899598823518298355793920000*t^53 - 19384984419913062409122318610098643651018958457837810827185862790939265253450179064609695125698456365984484192127979/66821409635307053910267763229670176456670167764663557191298705542230586634963234324480000*t^52 - 881898562326263436815351884828405804978519085080708221661591887824891190190953472881611192130804871809748236218417/5345712770824564312821421058373614116533613421173084575303896443378446930797058745958400*t^51 - 5017828168509282848118469921428824395346727976022466395756039470661009478818569551331870551912000004667057666104681/42765702166596514502571368466988912932268907369384676602431171547027575446376469967667200*t^50 - 1616203894622088844241569927916980078694244996224446796558199334085850738292595380421376399407594590104388821508673/41120867467881263944780161987489339357950872470562189040799203410603437929208144199680000*t^49 - 5273082794364188320097071196131355303577572547039608044363317610216208854412757728924789644604544025236865723312321/427657021665965145025713684669889129322689073693846766024311715470275754463764699676672000*t^48 - 15423882261970579885061473592360885683225188756382000470324095401333849037854690910464671041683896869458738158680453/6842512346655442320411418954718226069163025179101548256388987447524412071420235194826752000*t^47 + 120300075235928307875155126457755369075965970525068847640674648213363812044423488686568667834386658530432225534262309/136850246933108846408228379094364521383260503582030965127779748950488241428404703896535040000*t^46 + 101470104504947602647622093166513281622113524741761903620978849952541527098816341868701773550832408512532577594881/352026358670376453783224126287754395841184575130626276856025077686143386311008884621312000*t^45 + 2859112861490107007469482259886688213517992647656292688608079572079795778047828632194291854346730946405392316669/17037598049501552666840347236996423341312895338420861542877742718477169090653889495040000*t^44 + 38834792208230518998963050513895166471847451033226477930688797505422820827363442493211791990276097162281627919537627/2189603950929741542531654065509832342132168057312495442044475983207811862854475262344560640000*t^43 + 870379473832060849524341002666957277627740427970487041545520448521525801965704268146551237913947813072112014632737/218960395092974154253165406550983234213216805731249544204447598320781186285447526234456064000*t^42 - 20905411409738022792210009470802161426693883034012861437182668525215938576676383495862282223569098722291759024571/74858254732640736496808686000336148448963010506410100582717127631036303003571803840839680000*t^41 - 3111583716336419310891565856558508067941331717760285098164081771110510175237895497552654708549420537601564065839/31848784740796240618642240952870288612831535379090842793374196119386354368792367452284518400*t^40 - 8526654430052069451011932897666845325451190720121879479034295829038174145640378735571656566150054050823781955873/796219618519906015466056023821757215320788384477271069834354902984658859219809186307112960000*t^39 + 5429045852010077190968113187621040093901314516029003667500624689250169307639968569216010251566915203150885582043/8758415803718966170126616262039329368528672229249981768177903932831247451417901049378242560000*t^38 + 92675395675464505819577065986417323839482133862255901721049673366225771667697979151044194116107182850713066453/5838943869145977446751077508026219579019114819499987845451935955220831634278600699585495040000*t^37 + 98728746194443915334821139953996954273928589238862175763291342447156925528723083108625523854462981609399005939/3336539353797701398143472861729268330868065611142850197401106260126189505302057542620282880000*t^36 + 10669293965928663053044482599316911135930380190854760245815857814346347945555415933814847425896979615587081/28834290711831987391363345718647997921082048491357964668898449161584353749523954072027136000*t^35 - 1056349327647178972487128084776882046191504043793217707888953263925027843008268121189219789943932977520939/3046405496945727363522301308535418910792581644956515397627097020115216504841009060653301760000*t^34 + 30257322625158658521856688247190570293400876875676777157376213481043345096657668947067392922889218012430569/1347448585187533256942556348006050672081334189115381810488908297358653454064292469135114240000*t^33 + 644833009400066037370524328853673857744542263250632548448511513294151273450655947336150763883946604759943/778525849219463659566810334403495943869215309266665046060258127362777551237146759944732672000*t^32 + 1379241456905601122552771132807253867982270054031552448149024001901011911829654632487970887717123712328977/35033663214875864680506465048157317474114688916999927072711615731324989805671604197512970240000*t^31 + 442968211763214650480205060574973045202887023750316173528341014793118114718530377635805432420193191788633/70067326429751729361012930096314634948229377833999854145423231462649979611343208395025940480000*t^30 + 552565888119720207726595142940587301759590827844205684184242979660462301873151158729966643419499139513/2919471934572988723375538754013109789509557409749993922725967977610415817139300349792747520000*t^29 - 1002414701339592808686812062651016411002335803416291552645091652160139391929310568279604779126084619/530813079013270676977370682547838143547192256318180713222903268656439239479872790871408640000*t^28 - 11414035211941487091736194786515421075228264762237153420806708770883413486311650830799751313663/333653935379770139814347286172926833086806561114285019740110626012618950530205754262028288*t^27 + 1591844242474472273742432318781486432688635971243587864166691860433030472133009111871153190000057/530813079013270676977370682547838143547192256318180713222903268656439239479872790871408640000*t^26 - 20224476178242417621156280304329786235742940747778203796655980450846159661494842121576691502829/625601128837069012151901161574237812037762302089284412012707423773660532244135789241303040000*t^25 - 134659359201182945236428784868732958569290900763261460432943573525906432624197677753255465527/81096442627027464538209409833697494153043261381944275631276888266955994920536120827576320000*t^24 - 310870109897344920470635130054563169124704459966871893614336301454697065807158765220239863/182466995910811795210971172125819361844347338109374620170372998600650988571206271862046720000*t^23 + 634894741217512079784022394169283766816620952316950378646405418863189293926638717475931/2432893278810823936146282295010924824591297841458328268938306648008679847616083624827289600*t^22 + 111369616132286361481961976288741982785527121007223020726954820220685389369362918093/233932046039502301552527143751050463903009407832531564320991023846988446886161887002624000*t^21 + 322478131067562253576953020047461599454880938554557386540900183858024233730895221619/45616748977702948802742793031454840461086834527343655042593249650162747142801567965511680000*t^20 + 564003914458305791543938683176950157967848941953695353078026341066822617109959/63356595802365206670476101432576167307065047954643965336935068958559371031668844396544000*t^19 - 3876115386439633028559698074193032064191892924301617393488473210438960470131/32398259217118571592857097323476449191112808613170209547296342081081496550285204520960000*t^18 - 985971415815833684693933633234740854882976450574519484710353072224920429/3299822698039854514087296949613342047242971247637706527965368174924967241232752312320000*t^17 + 982096542860997613999764114132301629102989289227316482314942675503059/178190425694152143760714035279120470551120447372436152510129881445948231026568624865280000*t^16 + 1173347398853817535658111916196727892798653554501136563613485609/159098594369778699786351817213500420134928970868246564741187394148168063416579129344000*t^15 + 6960141660821120837391363529639799820481231441735304819171/511806139976310155562712647286076719184054593785719647604922683381055351064363008000*t^14 + 15856985046344777461280134147479927962573578080066497/29554023028523344580727292812035680115927065537898213346751649515889745136189440000*t^13 + 752238250879712442400219744174035298271641333/31610050672248740948508544209301555183515741600244554825859744180871658209280000*t^12 - 141558771269363366286793278245485241/246645214358994545478375032844113258298343801500035540151839452097937408000*t^11 + 47288911661930562684152501/10208065512439785410888847620501339280390903242886515566180111252193280*t^10 - 59279580055/32558671294637069502649226776392446080758839402088192754581504*t^9)*Dt^10 + (165367480/81*t^82 - 62142672819773572373436844353434327242673761/1679035162707137617777215985773741462*t^81 + 3562810687408098418153180757334793422677657996127997/266855661356314454606436554884133323920960*t^80 - 174292831242237149379724708027095488689232830192232286185677/272319264166923636750048736908831069594917735040*t^79 + 6639954660231190046685694238900379971962052205950400351608720243/261426493600246691280046787432477826811121025638400*t^78 - 106421567641966084293676466186985875391729096295438928050813181773/247667204463391602265307482830768467505272550604800*t^77 + 5987309718472257504280678224584247858801427338258936166195214333441438639/626118543584539627589242296275083855035249345372173107200*t^76 - 6035626283704787649873084810502596180193161840092919931607258102983981830901/112701337845217132966063613329515093906344882166991159296000*t^75 + 326266672291217893520527528069653409701768250151899100468792650009710992620367/352804188037201459719851311292395076576383978957537542144000*t^74 - 11697392487276781699977486565691353520301691629276599740164837686245680035265811393/1168487470779211234592147543000412493620983738307364339580928000*t^73 + 638570915151321441005533950577979513803689519132436544342627560940260050208014236521/12463866354978253168982907125337733265290493208611886288863232000*t^72 - 3420712171046204699285377669562255887358119225105022948467445013072730215202931543587/6471622915084862222356509468925361503131602242933094803832832000*t^71 + 309650910469627908404421725409762804263790390998484783784329368171629387679341191071731301/129225366368414528856014781075501618494531833586888037042933989376000*t^70 - 433548556910617642718712544033114443709337559252990586902985090315473112432385495325692483/28716748081869895301336618016778137443229296352641786009540886528000*t^69 + 21969669909675157752227956278716918906677942542914014630620231208366238202258127018176720606437/521036677197447380347451597296422525769952353022332565357109845164032000*t^68 - 15785655292296273155654668377832730593612361135929311730746847691395838441109440971584852792443/99245081370942358161419351865985243003800448194730012448973303840768000*t^67 + 433031438744099378318510971953908210899262323866207407124702070666719908506360904468228500384762109/400156168087639588106842826723652499791323407121151410194260361085976576000*t^66 - 6264008101715851251341090697939391208097819170321865123637145612728218625677907252903720029641659/2198660264217799934652982564415673075776502236929403352715716269703168000*t^65 + 46839994782515201546682424022579044539922253868639595643973632275638048612712380413163218134448489099/4801874017051675057282113920683829997495880885453816922331124333031718912000*t^64 - 384925533820143018106658082848700410536324093951968063801460204054844818082338350556191563257993890403/8907824263516150830900153360109133908398155845479544435628752385913913344000*t^63 + 442454896467651673984364564974615687093383221681294501518693379106076169508157925608808299173464033879329/14751356980382745775970653964340725752307346080114125585401213951073440497664000*t^62 + 26693234465143118923784156700892618934758115105128291223381311825767318833818678546068990355720953707693021/177016283764592949311647847572088709027688152961369507024814567412881285971968000*t^61 - 30358519605823680616147241089370265094404328118907971132377574751818038226355758496994150701171064454801/691807655162063309473953483427801500840989361842186642011976814510527741952000*t^60 - 14243149465005092407149892672702804575320970323629801740512109057218277584242611012459949725226723461286187/629391231162997153108081235811870965431780099418202691643785128579133461233664000*t^59 + 23204473995120637515506707766778254107527735112179345526566306136164525343107973386149593091995191201442427/9063233728747159004756369795690941902217633431622118759670505851539521841764761600*t^58 - 213040959952830547753936478184752696855556282328138271181087742294541174497696076380101387600051727379485726001/543794023724829540285382187741456514133058005897327125580230351092371310505885696000*t^57 - 1487323239188388134848371322997073428459478832282043732856524023744282689536140600687898090860415750875848289669/870070437959727264456611500386330422612892809435723400928368561747794096809417113600*t^56 - 2228755641220745142897641227509893125078635359310340484568342897903198373122377344810512333375128142742562143261/669284952276713280351239615681792632779148314950556462252591201344456997545705472000*t^55 + 78568426305430434350759852033807151719924084386886461031345669867283111793479957303716670840818394421061674152653/11600939172796363526088153338484405634838570792476312012378247489970587957458894848000*t^54 + 968359834357037941849359266469609633807795366172111221481561462424981229502332256241166706061702385302076388277/264408870035244752731353922244658817888058593560713664099789116580526221252624384000*t^53 - 1223704358514547941756656488858314501567576558422489729267801611892033476394047631862314127246918520617546175171841/1113690160588450898504462720494502940944502796077725953188311759037176443916053905408000*t^52 - 3975734801172390246680823500944423348238907972721759460634814908580314060719084366096266499889275850754664527859977/1028021686697031598619504049687233483948771811764054726019980085265085948230203604992000*t^51 - 2150403699151360230935699347272547451780776618463375737148733423421076210017634594112357273190830038936908843106899/989946809411956354226189084884002614172891374291311958389610452477490172369825693696000*t^50 - 1744614371688341620601582660423435315300188355694677833293242479909641162836049019101665487122554693994644645670273/1137385695920120066557749161356088109900768813015549909639126902846478070382352924672000*t^49 - 212270949134275220578606103287046118568482175256585721573126848710073487423305637791527858994944899747687570734849/413594798516407296930090604129486585418461386551109058050591601035082934684491972608000*t^48 - 257646809449796814505445582088478318496622483702885727124091826106128343042730420796571946550452191097255564861946199/1710628086663860580102854738679556517290756294775387064097246861881103017855058798706688000*t^47 - 14749633751016496532952798011723025495753193353173177666701283508409544697912800435391226421862457581091037647094419/570209362221286860034284912893185505763585431591795688032415620627034339285019599568896000*t^46 + 511625320035819108966753744243578113979908980812186817073398425182361077027939219735712843970919947276661995958333/36060671128618931859875725716565091273586430456398146278729841620682013551621792858112000*t^45 + 9244144011359849072854680534439225070979674557660733884042039108157235661316233704796377041805035107635824694672043/2105388414355520713972744293759454175127084670492784078888919214622896021975456983023616000*t^44 + 237982428969649987314081015337928541829555621222738525377075437421096705519893554439241612306650848923969394073813597/109480197546487077126582703275491617106608402865624772102223799160390593142723763117228032000*t^43 + 1117690685848638717785200139593188810902269753354764103462232251023777468481949514925125587200525255886151094277071/5762115660341425111925405435552190374032021203453935373801252587388978586459145427222528000*t^42 + 8097963239414954679410534934965740067100241698907319841564787565538952099670725036856839961261069552895530693003963/218960395092974154253165406550983234213216805731249544204447598320781186285447526234456064000*t^41 - 791804306735482562150545085522645242866055355098188513714030052058311758194143933663215842727868483294549920920981/218960395092974154253165406550983234213216805731249544204447598320781186285447526234456064000*t^40 - 631140972569376704107143743838979453067589589626493466803088980938822455567695116995863033303676381559175999369/564694764907734762741883704838125684624672613104447567258407732613233233489226373267456000*t^39 - 4056619563515843021626999186541394492211168599956256913586826338132114924720828233581458513493372442213574418083/27804494614980844984528940514410569423900546759523751645009218834384912544183812855169024000*t^38 + 3056613845679522032888279950258010019192623940509491748992674978532379938453617174690436233802625942727888861793/437920790185948308506330813101966468426433611462499088408895196641562372570895052468912128000*t^37 + 1418834018466636127717859496786608824722638822962565448280125367779852658715363599192992119242324877444332060177/3503366321487586468050646504815731747411468891699992707271161573132498980567160419751297024000*t^36 + 1416793932680752370236210839014526359294860379461665495059262171269621384174642812403174902782987726654964563547/3503366321487586468050646504815731747411468891699992707271161573132498980567160419751297024000*t^35 + 8777753336360094416823639095366998641041349009914637050364675922164379588742664081560137570658205491671331/756993587184007447720537274160702624764794488267068432858937245707108682058591274795008000*t^34 + 76004932275369917993824639892773337891810209694183813418896831741109305951913726764191350916261780026413989/233557754765839097870043100321048783160764592779999513818077438208833265371144027983419801600*t^33 + 51230989113817769762349436149037909200828440864707458664025484767105186400116741835780756478954748099474051/291947193457298872337553875401310978950955740974999392272596797761041581713930034979274752000*t^32 + 162546658394013699232425932320916037000810029843070148152098701508772030856669050542418337813905703165771/38080068711821592044028766356692736384907270561956442470338712751440206310512613258166272000*t^31 + 82160849410614143726020936875774849144007033629742112121425969672855361762627600864229719073586565814979/700673264297517293610129300963146349482293778339998541454232314626499796113432083950259404800*t^30 + 429194886277400723048983392450382278845449850382253199601929472493699313042970193015418210031550223237/11524231320682850223850810871104380748064042406907870747602505174777957172918290854445056000*t^29 + 2115634737402818224599143670513670763844721173670222567651419949093653660595432429995081063608110604231/1751683160743793234025323252407865873705734445849996353635580786566249490283580209875648512000*t^28 - 15619590810638133288405217112837568582202376865303460567491780882802783465664806095484422071949449471/1167788773829195489350215501605243915803822963899997569090387191044166326855720139917099008000*t^27 - 7816776528083764715842895458022854271049969328739245218614910569414560499294029248094305777291187/34346728642035161451476926517801291641288910702941104973246682089534303731050592350502912000*t^26 + 415790247360740382220104953143518401888024146544085700836325780069332389063853387742029860658869/19681833266784193640733969128178268243884656694943779254332368388384825733523373144670208000*t^25 - 183737208386766154211365872767029135227907821113115376858359067904947858324661365259352949736297/875841580371896617012661626203932936852867222924998176817790393283124745141790104937824256000*t^24 - 451544632907001014199377487170703409037305084511741089418590427566931183323030434343073573617/38080068711821592044028766356692736384907270561956442470338712751440206310512613258166272000*t^23 - 852114145918207067709688021108079006560331793899455724941575189539827780423378011338999761/62560112883706901215190116157423781203776230208928441201270742377366053224413578924130304000*t^22 + 7109815666625287313837976713175015015890162106417289235937341811084824622823177865969437/3649339918216235904219423442516387236886946762187492403407459972013019771424125437240934400*t^21 + 15049799229028321965733002975406270644830034534888883593141276422749628050775012349493/4054822131351373226910470491684874707652163069097213781563844413347799746026806041378816000*t^20 + 25521403980128860304281563680917189657473227205556265810586672054954145359234340039/456167489777029488027427930314548404610868345273436550425932496501627471428015679655116800*t^19 + 165961882309230886405602954789864528036698345005663759001087214992185365603450073/2280837448885147440137139651572742023054341726367182752129662482508137357140078398275584000*t^18 - 2261152099068368925732519838970674349442623041567674680292929129935973663843341/2280837448885147440137139651572742023054341726367182752129662482508137357140078398275584000*t^17 - 206226156032587594579999078852934473660047566593229257777554910654857566327/81458480317326694290612130413312215109083633084542241147487945803862048469288514224128000*t^16 + 692527436282895174502481832878121047008694079876562627884024798291449/15839148950591301667619025358144041826766261988660991334233767239639842757917211099136000*t^15 + 195905105517307409478070844936367252851569160911635578894529653633/2969840428235869062678567254652007842518674122873935875168831357432470517109477081088000*t^14 + 544795452048037048621477767472721723530938351696576256969199/4419405399160519438509772700375011670414693635229071242810760948560223983793864704000*t^13 + 1799923352650852927314598636468774576705706044148047997/362529349149886360190254791827637676088705337264884750386820234061580873670590464000*t^12 + 1076348934418380292524340663703736196443373039/4816769626247427192153682927131665551773827291465836925845294351370919346176000*t^11 - 52853855980990523448627484147569089051/9578803234014769984396528548273198540459315636437743886624164903294441881600*t^10 + 14825973957418332269534120597/326658096398073133148443123856042856972508903772368498117763560070184960*t^9 - 92159542525/5074078643320062779633645731385835752845533413312445624090624*t^8)*Dt^9 + (172897720/9*t^81 - 269718962057298610366618376815771149752109557/839517581353568808888607992886870731*t^80 + 2316702871671813096953511878778448537421417106220746509/18472341891664878357756663743646117866973120*t^79 - 409462560756890659671800006828504605926333402250382655260431/68079816041730909187512184227207767398729433760*t^78 + 19408891555647782273206730792935531417858385500211294144921583637/87142164533415563760015595810825942270373675212800*t^77 - 84596957221965883004220282522856605791356701100440195828400719847/26955595903135092554243776772906879032897912627200*t^76 + 4815682062844761212222891923668768684156339101866480816171328163056580613/69568727064948847509915810697231539448361038374685900800*t^75 - 9110726086061784561813182353601731504814647474862039431107402196363321283711/22540267569043426593212722665903018781268976433398231859200*t^74 + 29662959221563437007605461532812152624102794022911746276755110024135494287/4307167234014260221893434737044832756491052593709056000*t^73 - 2472787644947594209135749240325680037280362475897386292685295228589517087669382327/34367278552329742193886692441188602753558345244334245281792000*t^72 + 13548292033359799977313154097585503476807403006190635189869590200202899417861069597957/37391599064934759506948721376013199795871479625835658866589696000*t^71 - 3729825069381832973950572184066547006585868328395649450136995605441857905307475073261/954679125062164072517839694706719994788207990446867885955481600*t^70 + 2109834152212159690777956548067897844237866091255794915158124330752490594076086508783375481/129225366368414528856014781075501618494531833586888037042933989376000*t^69 - 384852146990583075802220810478721421818110030363453397226112329591571035077338684533962031/3446009769824387436160394162013376493187515562317014321144906383360*t^68 + 829270666645328940937946948563856376558033383922846949272226375462848340303242858345931332433/3046998112265774154078664311674985530818434812996096873433390907392000*t^67 - 2645487423907988882140846142681938216933860205541463192024423158364665082816575287753525071916493/2084146708789789521389806389185690103079809412089330261428439380656128000*t^66 + 3189979424867496924945402527404227752144518803227950205699807570418838766681420191540362899797869757/400156168087639588106842826723652499791323407121151410194260361085976576000*t^65 - 2703220936122027743526694720165101859557017737826594481550034667842427278226685679028941002696817999/133385389362546529368947608907884166597107802373717136731420120361992192000*t^64 + 732214891172851311178499303856423930210989279231732125784298648711525122670010735300332149821167288661/9603748034103350114564227841367659994991761770907633844662248666063437824000*t^63 - 1868601838884602488122679984822641232339065106089905990939552660213559679006469561410322530305527927269/5853713087453470546020100779500287996947359555600843486270322996457714483200*t^62 + 281802962099346523292359597098184367932781891231877395454650144289551498018753784307104756993389945327/2169317202997462614113331465344224375339315600016783174323707933981388308480*t^61 + 6530450684602371371955908820859148495524762929859777464576853928002385763075781361645016977762217789637/6020347711614221314547762050542077646079929019534384485420350556503801856000*t^60 - 2432568120822495471912771750017629445024205132268371531889052603372157582942805547945757863266459966314197/9077758141773997400597325516517369693727597587762538821785362431429809537024000*t^59 - 6126609055041329304977192695057891538971366314560848962772129553515972963986913269616456376888178651348479/435732390805151875228671624792833745298924684212601863445697396708630857777152000*t^58 + 4797240823875747865619941751217705127884312660392790387906460228006451587382992320161516087807149099015483559/11329042160933948755945462244613677377772041789527648449588132314424402302205952000*t^57 - 110845075520338137995356535386067834424275743356607933743104822706579624395336316424225979793658299767246562583/60421558191647726698375798637939612681450889544147458397803372343596812278431744000*t^56 - 19931488310614495925495292224595645990762402557088907525200925300609100611325405838125354480899225876153511/1659173222653942151900479596465160988964326486338145310694829446506090955014144*t^55 - 28016997991316351052698482401617370848246354240436366089448070592507471452218594125684653992401469088801592222857/966744931066363627174012778207033802903214232706359334364853957497548996454907904000*t^54 + 589596146175280527994550822989864007156252964972508203785781791602944548639820074589744259265992425752030958808897/11600939172796363526088153338484405634838570792476312012378247489970587957458894848000*t^53 + 88665399617212612720640063283392954890749502317880257949851920435397354954768030268421057403322579509823692279539/2900234793199090881522038334621101408709642698119078003094561872492646989364723712000*t^52 - 4061623144230631283435691046913889572964054987691294385756667842124058826514149833257756448615088116022494972571/568209265606352499236970775762501500481889181672309159789954979100600226487782604800*t^51 - 17073534372585214666116788473326039034283153424207981411717636223679375860774303349949811447340774589897963136696637/556845080294225449252231360247251470472251398038862976594155879518588221958026952704000*t^50 - 153016226432326303258658331185542180280801063679366418889029939039445104958422347072113735586448516312881184471826119/8909521284707607188035701763956023527556022368621807625506494072297411551328431243264000*t^49 - 33524606365093224105939567192548396234028414621864711468929002547653937937641743116004823301587539687774160687903983/2741391164525417596318677465832622623863391498037479269386613560706895861947209613312000*t^48 - 456010721919891280450702244805793209608601342467312360641007796617869812929894826865428176075994734026127568161/110611156804799704375933043307414338043946055462145240817293935424287176781935411200*t^47 - 29520006476372119329690133762090731113639206620867628788450409873648203205070497746316370547852977619002710653963797/25918607373694857274285677858781159352890246890536167637837073664865197240228163616768000*t^46 - 52599167974273959066801371657880075763837561429932576108437447691572166867786383098641037440780062006062840915897729/285104681110643430017142456446592752881792715795897844016207810313517169642509799784448000*t^45 + 38582661749992484990606282473760390443036653842138579388068863478803669768044825871930301555228329145636830939205179/285104681110643430017142456446592752881792715795897844016207810313517169642509799784448000*t^44 + 371468538143714215517016251945580793822067349766556558592341641396017681461614368739862341350334385280241593022759253/9123349795540589760548558606290968092217366905468731008518649930032549428560313593102336000*t^43 + 32119329365354988726228410878407352357640585093641554680204103776274600107040033170488179764214567709985476008609049/1824669959108117952109711721258193618443473381093746201703729986006509885712062718620467200*t^42 + 336755184280306975295547098272585506053151129484395544720544371810364215315856107120027372709799107710497444970711/270321475423424881794031366112324980510144204606480918770922960889853316401787069425254400*t^41 + 4803696170725504990932274925117294888158971371928624052997973579797890376735333394879065427991282273686875033063/30159834034844924828259697872036258156090469108987540524028594810024956788629135844966400*t^40 - 67497067416190469070536619510081779758859813428451161205954853590758644420000138961316890485714946720451069552407/2115559372878977335779375908705152021383737253442024581685484041746678128361811847675904000*t^39 - 488132706050685635234561192414248243314427538473444871209687804602179852247351161103900995322987172242558611189/65606110889280645469113230427260894146282188983145930847774561294616085778411243815567360*t^38 - 11856878852293290777165717177762194503475643108451932771209069488065452630983642772272135731922148689676593980059/11228738209896110474521302900050422267344451575961515087407569144655445450535770576125952000*t^37 + 17985614532775663646207507391604006313096123880110215285519166079350357246575701638714020839555241444450766392019/291947193457298872337553875401310978950955740974999392272596797761041581713930034979274752000*t^36 + 125746604269252481370120441524991358605416750761589176424854331819213520304646730322555431840569943667298204999/32438577050810985815283763933478997661217304552777710252510755306782397968214448331030528000*t^35 + 33233603771096637594834342485650308247573985454295682555239698657561813671995616613542875631442548789124775783/10913913774104630741590799080422840334615167886915865131685861598543610531361870466514944000*t^34 + 36586993428864477807871603663038509304072210800928625254507977485153956224346158341614097127469048571012081553/291947193457298872337553875401310978950955740974999392272596797761041581713930034979274752000*t^33 + 1319595310649592858286961477731575743033810618824288367763529981896284280534619629897194710621226018680055417/291947193457298872337553875401310978950955740974999392272596797761041581713930034979274752000*t^32 + 221862760963660214684339948222260973184824215249773410199819220531520778802755004679783093416194597311119/230788295223161163903204644586016584150953154920948136183870986372364886730379474291916800*t^31 + 244859697325936306081340399736315396531474266887656166390182348556524816886684995414390368083093756588051/18246699591081179521097117212581936184434733810937462017037299860065098857120627186204672000*t^30 - 2415129048190955071303529861612331368281611737205784825299836549065389026603479303253015951809534844887/5587506094876533441867059816293033089970444803349270665504244933225676205051292535488512000*t^29 + 1337569780776630404007809750304675625438599773914299779795719235217292384967203462941320584073020093661/12292513408728373572107531595844672797934978567368395464109338853096487651112843578074726400*t^28 + 303005409784789868344517030686815795967718496915395620578304107110995449599041856826117768022241663653/72986798364324718084388468850327744737738935243749848068149199440260395428482508744818688000*t^27 - 103796622745420625203534764491875066754184183121926496865147838483718870720774152519218787653770207/1880497220336868742915000807737912907896655336392910739275985814885936114099388309045248000*t^26 - 242952305587404215681754427138752960016929674625986411868082695230542796893379879140149103328239093/291947193457298872337553875401310978950955740974999392272596797761041581713930034979274752000*t^25 + 47704928628901942344066497958185187226184172005309686086769109887611160420400325234213019191324439/583894386914597744675107750802621957901911481949998784545193595522083163427860069958549504000*t^24 - 24363056957958682746603703571798208465533062596407877110715857722327004114561871335849536292501/36493399182162359042194234425163872368869467621874924034074599720130197714241254372409344000*t^23 - 190250911839368552155553612011238295944328458430712592743104907747963300750256091503401063/4175028150175166564239190518702518039540602928411048554529678060849765923233229438976000*t^22 - 996415667280098903474290735357104356246350902330191870890526868734926001706558668645255927/16219288525405492907641881966739498830608652276388855126255377653391198984107224165515264000*t^21 + 95926199891137663590941599523006350574318997364972948448835659379098325550812082092163463/12164466394054119680731411475054624122956489207291641344691533240043399238080418124136448000*t^20 + 516101814184565918943328874081081413170129954471473703537370826934320550215335868009/32011753668563472844030030197512168744622340019188529854451403263272103258106363484569600*t^19 + 7948820443919180218095500396365130996026062240800268805991419466625065019502420117/32352304239505637448753753923017617348288535125775641874179609680966487335320260968448000*t^18 + 7251907253738583391431370279021976163112764143565641828348127492835860835304021/21722261417953785144163234776883257362422302155877930972663452214363212925143603793100800*t^17 - 438340254189736062749031277539158095035038103846494887549145749699362716853131/95034893703547810005714152148864250960597571931965948005402603437839056547503266594816000*t^16 - 2273723472085218276160993240982476790552961139787787261710136558767664991/187815995461556936770186071440443183716595991960407011868384591774385487248030171136000*t^15 + 2112484447411608311360325607924676659534474611310212327702328381896077/11879361712943476250714269018608031370074696491495743500675325429729882068437908324352000*t^14 + 67350049187891644153394363302494115799476827235811661510075742247/197989361882391270845237816976800522834578274858262391677922090495498034473965138739200*t^13 + 29796678110730116331554305182118884566566450822062835855506569/46403756691185454104352613353937622539354283169905248049512989959882351829835579392000*t^12 + 9615071381575726619259462970640621605765657864910769133/362529349149886360190254791827637676088705337264884750386820234061580873670590464000*t^11 + 2553232184734250114236003963421352839694932399/2107336711483249396567236280620103678901049440016303655057316278724777213952000*t^10 - 1083708444820395553310843909527310318077/35122278524720823276120604677001727981684157333605060917621937978746286899200*t^9 + 21318600116915056048982149/82116162995996262732137537419819722718076647504366138290036088504320*t^8 - 170634661025/1607835619488250345809838359328022028679448859362379889115136*t^7)*Dt^8 + (1015452800/9*t^80 - 340029659233675005612186652817963301253612/195191253511641201787632641917431*t^79 + 1713708272227523964731808458109347082739340958160432081/2309042736458109794719582967955764733371640*t^78 - 1209040954612634308129771301923429908508316942250518675342963/34039908020865454593756092113603883699364716880*t^77 + 4498113934199918534452272825888724981102651013780542502437252241/3630923522225648490000649825451080927932236467200*t^76 - 5240114076550484678462119844530732659644175601846070843570157150271/374315206745807762514612445641956888388650559436800*t^75 + 60028508226447800854671629771976188518003329125682833164546357035501767/193246464069302354194210585270087609578780662151905280*t^74 - 27974103660252698619358379425850388089912608742902864360321395673333148260409/14087667230652141620757951666189386738293110270873894912000*t^73 + 2348794248689594669254862414092372281181920451698442548056021037437781031927297/75134225230144755310709075553010062604229921444660772864000*t^72 - 46537897733461457234052195311147849204607191253200935907741970414888463618619909033/146060933847401404324018442875051561702622967288420542447616000*t^71 + 5649122887621808743743645339199907632917663167389124914850848736866707594586446557/3570626343099193994170045967915698987382685220190570938368000*t^70 - 246118781166236769786631019261155114455105914772764352535603527916574136478872248103281/14021849649350534815105770516004949923451804859688372074971136000*t^69 + 1089229488801586535322936883371435694273586491533140627261614066363259413562428416212041087/16153170796051816107001847634437702311816479198361004630366748672000*t^68 - 32254014939529422588081222204105397544595312632861989392297315248450843773367909001909028621/64612683184207264428007390537750809247265916793444018521466994688000*t^67 + 3815900462482196869424314254685364182427086527476735232433000906410011828638727970342936475391/3618310258315606807968413870114045317846891340432865037202151702528000*t^66 - 146418095989446817854588848448402462235398032409898638319242819881909474577409268170864472373227/23683485327156699106702345331655569353179652410106025698050447507456000*t^65 + 1781610545406634981558374839809254432661917008146821002612026099116524656283180954486976709190964097/50019521010954948513355353340456562473915425890143926274282545135747072000*t^64 - 17520702583259717922141489266642185774340898215084054682523269573155749118933649092202948199840981217/200078084043819794053421413361826249895661703560575705097130180542988288000*t^63 + 3476668667572137241929189496465690506337155695170264607738758880532441621446055481941620315089627239169/9603748034103350114564227841367659994991761770907633844662248666063437824000*t^62 - 8352866498497464046033146172375243982133109400693803126095299772734364266951327280953897196647846240153/5909998790217446224347217133149329227687238012866236212099845332962115584000*t^61 + 3308164375774313399120778516173381017984482613868883507530184311762091859992793639453995385376850743/42665547284646286778572163115891311931103203758023640570483403764269056000*t^60 + 6830551829549233805624433820217699246046289172543210296857238977194339555482495237710468209965766133991211/1475135698038274577597065396434072575230734608011412558540121395107344049766400*t^59 - 8672143902012899249244499696122796941578303753306550606228289423001492732938473222784734828964573480982443/9834237986921830517313769309560483834871564053409417056934142634048960331776000*t^58 + 56513554211494656198067397160628569310612369021474327828000793959474014869465416195025366486977767603061333/59005427921530983103882615857362903009229384320456502341604855804293761990656000*t^57 + 1336641339097142598665335183646911597585697990436311749203387167857784713601803729528930314984674413814611913/314695615581498576554040617905935482715890049709101345821892564289566730616832000*t^56 - 376394970246604170735731187665950540142202916503556487978910564907715548065799084074055176441964493498327567/269739099069855922760606243919373270899334328322086867847336483676771483385856000*t^55 - 2172354498023584440081568477557123351042600146473942220390176358359029508783326327594511322288088705002863804789/45316168643735795023781848978454709511088167158110593798352529257697609208823808000*t^54 - 22473485132429000404397398964755298887882369044876238490197303619677063848821716207679932800546607536648338165699/145011739659954544076101916731055070435482134905953900154728093624632349468236185600*t^53 + 45491060215170087301343046598295699189028107270110902674159543316234849500080322516602959728925328244078999263/203668173679711438309131905521144761847587268126339747408325974191899367230668800*t^52 + 2388483128575195821231039188811168772747788290775071274985465676528031349330943371976326471176131278677848973517/15519651067286105051622947610012582789081700056824497675422404668856973856132300800*t^51 - 217794767204310434096948278261552464981720626412091701387280810577735277444413501941058784113754906703214316608137/10708559236427412485619833850908682124466373039208903396041459221511311960731287552000*t^50 - 79562082784421444046263927655946192914840359588065873760496578352070759935844776247930536832292232377239926007044917/556845080294225449252231360247251470472251398038862976594155879518588221958026952704000*t^49 - 358621247418000417366327409068920920685597779767704487846855651741483462392956880427060106251781289145496889433226943/4454760642353803594017850881978011763778011184310903812753247036148705775664215621632000*t^48 - 81031266483249760023752823475352976478474401916163310473985888437873589877091936744528755367507134078293958953215581/1370695582262708798159338732916311311931695749018739634693306780353447930973604806656000*t^47 - 289956244131178825110640715973868924170379228405924049123505799413771691807777685682078418089866427108378640760325911/14255234055532171500857122822329637644089635789794892200810390515675858482125489989222400*t^46 - 30612377429301490748392893770841556138803146424018276725354180727722867470871008687895186739631135234882222999884913/5702093622212868600342849128931855057635854315917956880324156206270343392850195995688960*t^45 - 405371407662993691908658835545916589802879897892932554833357360139167658136234250758174998955141923247686493908139/479167531278392319356541943607718912406374312262013183220517328258012049819344201318400*t^44 + 16767988724151995748174335048685984262962087765183326139288722939603169069153018044521769183443991816091981021728917/21931129316203340770549419726660980990907131984299834155092908485655166895577676906496000*t^43 + 1045125569504901668995893204599207751096454771401370519622833102096107219526347195467417254824038008533457742637567521/4561674897770294880274279303145484046108683452734365504259324965016274714280156796551168000*t^42 + 5815752701537791621706678449995942284792319084102852792825883094885233581902165952205389821362135764411896450516749/64704608479011274897507507846035234696577070251551283748359219361932974670640521936896000*t^41 + 87430177477014295337449671937102934349986200948732743612030085723240506588309782828260456196414059696433165451989557/18246699591081179521097117212581936184434733810937462017037299860065098857120627186204672000*t^40 + 25772334717699284470173977032558156054204497951985988937781609063291710009418673710319713303622038466967035171761/434445228359075702883264695537665147248446043117558619453269044287264258502872075862016000*t^39 - 14083383794223763502747000025820032992082873908202241078955175634959609087324911622358468933365039669678419207023993/72986798364324718084388468850327744737738935243749848068149199440260395428482508744818688000*t^38 - 2169804287465545474718272565876688683010571001773427166236025610018615020316539385643147888168406611196059717176927/72986798364324718084388468850327744737738935243749848068149199440260395428482508744818688000*t^37 - 7016327523842641088219122146498172271576890565018390205782057041364591464251510725918713401074186589075592429433/1658790871916470865554283382961994198584975800994314728821572714551372623374602471473152000*t^36 + 3436942969842160673473176499228167918743242932560008329375240560851405171098470067413034755841774712111013123/8484863794969160437617817815662374417314454225034857947936433322513414953322774790144000*t^35 + 20189098244421245446361290872590695084301948326925437070382042855832311092547906564490196655449180234057409673/895543538212573228029306366261690119481459328144169914946615944052274790533527714660352000*t^34 + 27966385167930083574397483622319264678695945081951807118124180658933188958990090580922940273909555247641116603/2123252316053082707909482730191352574188769025272722852891613074625756957919491163485634560*t^33 + 12770397244137662419920861849655847323561187667818595163870857807981367927575075749350202996771732336118497/18223919691466845963642564007572470596189496939762758568826267026282246049558678837657600*t^32 + 16183370971642977734193495763007095645473700610894939535928834773915150909690404654399280138955101894404657659/583894386914597744675107750802621957901911481949998784545193595522083163427860069958549504000*t^31 + 334891266314903466432297057121387756893571575445888185357793258579674502204767888067269518947251500191441/88737748771215462716581725045991179012448553487841760569178357982079508119735572942028800*t^30 + 4572642325094468473215115747420474467519750141150256441268704657060120648196573096657972077612457188732411/97315731152432957445851291800436992983651913658333130757532265920347193904643344993091584000*t^29 - 154856671190452185798471759148709542553568689673073557048294411733963254353184760236987898196219378994713/41706741922471267476793410771615854135850820139285627467513828251577368816275719282753536000*t^28 + 78704589299153145149919602465385837597382215029800959007858221662699136450664869963831773767429865159201/583894386914597744675107750802621957901911481949998784545193595522083163427860069958549504000*t^27 + 36193782794748668400114831246221388668301710113816681707366094903576206526428788450183589568437414337/4561674897770294880274279303145484046108683452734365504259324965016274714280156796551168000*t^26 - 597294147194750993101201567672378133884764236888468982064397774017148308616360686611083425920552439/4231118745757954671558751817410304042767474506884049163370968083493356256723623695351808000*t^25 - 41711808741973509780548802801634755977501160153850793421013446518280121840078778358399348513562249/22457476419792220949042605800100844534688903151923030174815138289310890901071541152251904000*t^24 + 1056330248351152028143496001192192605037153974116708905162747030302156413250807322574622073172033/6211642413985082390160720753219382530871398744148923239842485058745565568381490105942016000*t^23 - 113153715100176418753585302893665181021256411770419574340330632293241737940835894645093635547961/145973596728649436168776937700655489475477870487499696136298398880520790856965017489637376000*t^22 - 4354956388428631265886918963365634908029088783711448596136763695597167470770939931088573123849/48657865576216478722925645900218496491825956829166565378766132960173596952321672496545792000*t^21 - 3665443528072899969589172417620552738827481657660765743551549856686956436514201377503126921/24328932788108239361462822950109248245912978414583282689383066480086798476160836248272896000*t^20 + 39780585132465027737356171355941835627029153002172163223555117980235273482913835023346253/2432893278810823936146282295010924824591297841458328268938306648008679847616083624827289600*t^19 + 460247317311341564409310265064999425323147685163517549115425675783000009099348394748401/12164466394054119680731411475054624122956489207291641344691533240043399238080418124136448000*t^18 + 14046467400088010305656670323957970760541080522589523778097203176643599396385356529/24328932788108239361462822950109248245912978414583282689383066480086798476160836248272896*t^17 + 634763240228658516431286507860116261133759105527169914810185460142124838942525601/760279149628382480045713217190914007684780575455727584043220827502712452380026132758528000*t^16 - 8886607915527719933238890366926136787874468862313058050865372226776636372891673/760279149628382480045713217190914007684780575455727584043220827502712452380026132758528000*t^15 - 1509585636685515637506347628680345886753531839916746450870857112078119642831/47517446851773905002857076074432125480298785965982974002701301718919528273751633297408000*t^14 + 539027817411003038452829888686216875939964370163800719512348101920117/1827594109683611730879118310555081749242260998691652846257742373804597241298139742208000*t^13 + 11868287123765916894511796391315062518974600578910496029390156548383/11879361712943476250714269018608031370074696491495743500675325429729882068437908324352000*t^12 + 700534360548337081714216414228536748377754869748163519542977101/371230053529483632834820906831500980314834265359241984396103919679058814638684635136000*t^11 + 3333804549031965856477877571678903165542655429518494807/41431925617129869736029119066015734410137752830272542901350883892752099848067481600*t^10 + 45710777604138371735462777983435386801976565129/12260868139538905580027556541789694131787924014640312174878931076216885608448000*t^9 - 4638036256483631841972351396935686548469/46829704699627764368160806236002303975578876444806747890162583971661715865600*t^8 + 1119595519500857191229472202931/1306632385592292532593772495424171427890035615089473992471054240280739840*t^7 - 45496169893449775/126588113993548926226300193706613830361990367595318893429812887552*t^6)*Dt^7 + (10990098560/27*t^79 - 4889058566647434730625626931694859970564809420/839517581353568808888607992886870731*t^78 + 74587371039688781904656710379813086238711994719606273/27379953396736479779283592505404324901640*t^77 - 631603176341686835758352967781599364466514035051709662551741/4862844002980779227679441730514840528480673840*t^76 + 47042192025479127689127021347353543597223260001193304102807866457/10892770566676945470001949476353242783796709401600*t^75 - 20442370150699514758853078277626380762682095632098941441653692449/545648989425375747105849046125301586572376908800*t^74 + 169391485809818890827275761302205271091246301230475579636797311276111567/196152425935006148994123526401968626264175860079001600*t^73 - 12753422378463787827456235378313173374154930756519830602088300318830590965079/2012523890093163088679707380884198105470444324410556416000*t^72 + 8219575189715825249667328098580168333890290053071995275864931661837375203995961/96601146724471828256625954282441509062581327571706707968000*t^71 - 1637358126808741177770851487505846430255882979792311037813012817875717225702909083/1896895244771446809402836920455215087047051523226240811008000*t^70 + 40906772683401340254262862628168618626940832779018937140836090309025893259934283533/9676914871877525752315921681162836386095103422835315441664000*t^69 - 94343115419099277111961883772309989682808048989469787932584625346153361195960930911393/2003121378478647830729395788000707131921686408526910296424448000*t^68 + 20182460272140842786345559877435547566589528151022453268362710931950757037643485607102929/121452412000389594789487576198779716630199091717000034814787584000*t^67 - 485781170249752416406709282512304899918043433963271878340142839821514858949308094008299451/369215332481184368160042231644290338555805238819680105836954255360*t^66 + 3816043092096662373687172646932652263588417225080744111599671137470305547966654403859593235213/1550704396420974346272177372906019421934382003042656444515207872512000*t^65 - 668552853869078812097419374521138890903653550296595236511734638500281325730959416421744802896439/37216905514103384310532256949744466126425168073023754668364988940288000*t^64 + 224956589322438056020869022995909370525760925108320367582497634183691151736230596692069436212466679/2381881952902616595874064444783645832091210756673520298775359292178432000*t^63 - 721062805626848145150283581150587615741150141734982793730986634089199616650028895482746098641137963/3175842603870155461165419259711527776121614342231360398367145722904576000*t^62 + 198985138212861959414167922087811600553073627731723338265601535659780771713225483478680071647671353833/195994857838843879889065874313625714183505342263421098870658136042110976000*t^61 - 7970321222750494623791702624392313981963185442344072750452512937175511957415537645515147444264625870943/2195142407795051454757537792312607998855259833350316307351371123671642931200*t^60 - 2656641696932246200935900537234709742814589958098509131181852442694635136202176164947683796434547486409/1908819485039175178050032862880528694656747681174188093349018368410124288000*t^59 + 1089956135887102420215611435016070688703350072799902993281071148109434582328702668277605528024221197538259/95788032340147699843965285482731985404593156364377438866241649032944418816000*t^58 - 5650582414112914733817770954465713950722372017364421770908705161232826134995818685086742857692308878228197/4214673422966498793134472561240207357802098880032607310114632557449554427904000*t^57 + 780926596156669740606379145581133967181029754593757538767953896333921868052057030341569688395894798398933/122164447042507211395202103224353836458031851595148037974337175578247954432000*t^56 + 235642632959920329441572710267223875410176163850885651966380696583543133866193983930352147401659793910983457/12260868139538905580027556541789694131787924014640312174878931076216885608448000*t^55 + 70013004247473741265888990287693665366260819157585100438637839939539993411224666668939240016873627080474857/3276183389512420114501290411976193573676125445207532807862386441418277126144000*t^54 - 320855085973106935886485610046926127300948409001183293860849048200643400020526520165732264031886439336311227767/3236869188838271073127274927032479250792011939865042414168037804121257800630272000*t^53 - 566411664193003817102448041255340955173289119342793566790665281010326024434991554465972561001536264647625601731/1138239714756315102638162611703728967311476726106388541245903403647035710111744000*t^52 + 3535549272591780931775055925608172335779983117464171514901901568552255885623470050293269142317928981150778309873/6473738377676542146254549854064958501584023879730084828336075608242515601260544000*t^51 + 10696925830205431011305486060798549697513500917146991943018353855029396847691362837493299317126916009944718509401/23507475527449571481434961172207508885184540612920591717078515683830978637201408000*t^50 + 175669261725415929472409056365211797421981280754263726039533383971441646924359002177823184679234613033279535621427/19887324296222337473293977151687552516866121358530820592648424268521007927072391168000*t^49 - 844988301318282697197576783312773163469851720893218365999125018767139068849600726129481072576288497622844304413089/2272837062425409996947883103050006001927556726689236639159819916402400905951130419200*t^48 - 5400623333632468919017467244985857282344833665136569162106838808150736974341235853517735538504776859078956765074161/25455775099164591965816290754160067221588635338919450358589983063706890146652660695040*t^47 - 38572276244392943675081231108462254164108103571761273942596168466534112352805651101785624132277328786467485599603801/231416137265132654234693552310546065650805775808358639623545300579153546787751460864000*t^46 - 4247695896707598629001899442116025965350122809064935243300131910582861362140670529880369167434093006981465485911879/71204965312348508995290323787860327892555623325648812191860092485893399011615834112000*t^45 - 313730901419849999045410715514753104163538867025635508999801512471311824759585479177113220719669333242142494956038179/20364620079331673572653032603328053777270908271135560286871986450965512117322128556032000*t^44 - 14753138430821326603763586571455672971233387893042091946684229965902907464385900435264194817574031010996466790680463/5818462879809049592186580743808015364934545220324445796249138985990146319234893873152000*t^43 + 47787748865745697728853669800262036084409149977076025401135127330363058490881284167769961620870567644850551477171/19220972231554198747194934028624873786947530222874525990440761161836254947920838656000*t^42 + 70453930577149013792694154655938226740016914674479206020704396653859145101382417850387272948301260681187308635628127/93095406076944793474985291900928245838952723525191132739986223775842341107758301970432000*t^41 + 13180983545855291675847573622944712404193169906086318380695567514544116583443470186078094044696577284132373294005357/46547703038472396737492645950464122919476361762595566369993111887921170553879150985216000*t^40 + 29705010382041176358492916099841671164893101862064095947869053746257646497265541565582212997068182755621528573941609/2606671370154454217299588173225990883490676258705351716719614265723585551017232455172096000*t^39 - 6372983848935187769413382454961482097035330108153939247302022197805433560158195311350664783310970264722455471893/3218112802659820021357516263241964053692192911981915699653844772498253766687941302681600*t^38 - 1565814822500279827417187390990449328403623301321278922281504988436637917911659925757214079033146470829849171436067/2085337096123563373839670538580792706792541006964281373375691412578868440813785964137676800*t^37 - 255210065650046586120958582918592791785774345085796727916962099474092644508878674982350453331944213756606608231377/3475561826872605623066117564301321177987568344940468955626152354298114068022976606896128000*t^36 - 96674740299162279930654802269983639339823471403298137102016172705874759814476228903835920538678832635940435415253/10426685480617816869198352692903963533962705034821406866878457062894342204068929820688384000*t^35 + 9107433177874960210885761338738194279344420721918478038023214281717107345361948847768022593618744240462245021691/5213342740308908434599176346451981766981352517410703433439228531447171102034464910344192000*t^34 + 1871453674132318139736243593256191276583645175829772606380517359876814773527380176364409842544683081469162286651/20853370961235633738396705385807927067925410069642813733756914125788684408137859641376768000*t^33 + 2736379942378388966553439260482593799142434020402823317474984674624807390154069512825704866960675938359822711749/83413483844942534953586821543231708271701640278571254935027656503154737632551438565507072000*t^32 + 90917320708843407318793209681658702878829584064549291133646287655360045765392936000434002199105113001557368783/41706741922471267476793410771615854135850820139285627467513828251577368816275719282753536000*t^31 + 3520043085803517425251635783815106189956403881168488279311719007161901995256288241688117180881607211133459/40551037357774688844718921508620179033398950062504256166761135879025151984711443152896000*t^30 + 11536456827405971104616509139848623213093585427801850063296962511426560614306109398125335299998481789376733/1226668880072684337552747375635760415760318239390753749044524360340510847537521155375104000*t^29 + 1010327052946550983224301288374930497143832164494992463353966603522873507061182773230691994535977733039551/5958105988924466782399058681659407733692974305612232495359118321653909830896531326107648000*t^28 - 14670442887802062220981473724945378385677468342164365764529383871157247125224491878219328048357983122011/1668269676898850699071736430864634165434032805571425098700553130063094752651028771310141440*t^27 + 2280392737073899550711412076206518498361439732737074234765434463540090350451402704079480259008880368071/83413483844942534953586821543231708271701640278571254935027656503154737632551438565507072000*t^26 + 142412415885517629163255373244165766794016007402376063086088422816064297566940440941372938543559663493/13902247307490422492264470257205284711950273379761875822504609417192456272091906427584512000*t^25 - 181141176963390604846481331176132860086080387436201069600017423275563951618309250782484489261582913/758304398590386681396243832211197347924560366168829590318433240937770342114103986959155200*t^24 - 3141573642022075684624328215214583769175402706837042439553120749180741582495798350821868608262761/906668302662418858191161103730779437735887394332296249293778875034290626440776506146816000*t^23 + 7743775439871293248763132139302902643790429028584831241140188515182548879198021883236257299466219/41706741922471267476793410771615854135850820139285627467513828251577368816275719282753536000*t^22 + 103383527300255717547916240011592953761373856397892936207525645785217566104716099094952664511/204444813345447389592124562605960069293386373231792291507420726723418474589586859229184000*t^21 - 1705508232625284911212695362483852887072331542857570120150405324547690127211058611776142697/21128035421717967313471839296664566431535369878057562040280561424304644790413231652864000*t^20 - 12237495656979424229537711060708226655419313878448118983911162538143090192865585262772953/68148271115149129864041520868653356431128791077264097169140242241139491529862286409728000*t^19 + 1184605330850490843746662350878996373852318028948720511148960605811099004232483405780419/78990041519831945978775399188666390408808371475919748991503462597684410636885831974912000*t^18 + 25700230732332157183960909191627719390288393604693511525757365859907461427630936939689/579260304478767603844352927383553529664594724156744825937692059049685678003829434482688000*t^17 + 6871284986142022635435412171273394376756351582590706818315838828914730293919746777/10343934008549421497220587988992027315439191502799014748887358197315815678639811330048000*t^16 + 28939566128933544442864795282092495776027359346890568073888543235614613124939/27427097749941647909297013607175830003058462318027690622049813401973753693363136102400*t^15 - 540226665609275673772837735297487196559259698688189536277999501418125353242053/36203769029922975240272057961472095604037170259796551621105753690605354875239339655168000*t^14 - 18311917302900441636112463449120431830774364832496740390358778613338370909/430997250356225895717524499541334471476632979283292281203639924888158986609992138752000*t^13 - 237170752993329952499956184518055121683127601759540910695635353141747/2262735564370185952517003622592005975252323141237284476319109605662834679702458728448000*t^12 + 1401472009247835184908080291446768714606025256685117708864654399/872968967735411247113041521061730700328828372390927652900891051567451651119775744000*t^11 + 2180568863711432914942566588608134933891189707176799268951373/736567566526753239751628783395835278402448939204845207135126824760037330632310784000*t^10 + 51644192942683342681550550354681954872649694680614791/392347780465244978560881809337270212217213568468489989596125794438940339470336000*t^9 + 197988715410771861736854976282934101206295109/32111797508316181281024552847544437011825515276438912838968629009139462307840*t^8 - 21847178327978151291448314761644487011/125436709016860083129002159560720457077443419048589503277221207066951024640*t^7 + 1583804150047224018121366061/1014466137882214699218767465391437444014002806746486018999265714503680*t^6 - 12229277870507825/18084016284792703746614313386659118623141481085045556204258983936*t^5)*Dt^6 + (2603547520/3*t^78 - 475372160035565627053319490767111483210072/40966065551825931239379690278967*t^77 + 765928800592498580781558928766004119802825017688980119/128280152025450544151087942664209151853980*t^76 - 8509585771722565666941575214707587271854096361590870429507/30017555573955427331354578583424941533831320*t^75 + 16549155321025232503786505014045249139223264729504071433087382247/1815461761112824245000324912725540463966118233600*t^74 - 157616220846010377649280262614755975576414740407776361161030826599/2723192641669236367500487369088310695949177350400*t^73 + 3590579007538111678776365859916693711518464550517569768500035333207619/2435879799192886817574083007606146339228327674183680*t^72 - 650473286283369078093496966272852630088124652402797683469146833021911011/51123403192937130739209149542351219465285889458176000*t^71 + 30055992526070211561420264888341122617587701620695008303545476931981102721819/223613765565907009853300820098244233941160480490061824000*t^70 - 16463545150687050567850026161393482422411246349790775160146171968108170291256593/11709229905996585243227388397871698068191676069297782784000*t^69 + 68160209123708705767103348876687483614709238123116480354657744669028844767498160849/10116774638781049650148463575761147130917608123873284325376000*t^68 - 6971409238969201977222738214846971556045554150391045122637897398929303083684893033113/95386732308507039558542656571462244377223162310805252210688000*t^67 + 211438654357809007675178331541569813208765443303801270016354919314643286712815032502913/896501875682751476690079233790526268832083427592463349448704000*t^66 - 1500843743895723966133238923112194614111706122714499037191125204827630131480260660408499273/769198609335800767000087982592271538657927580874333553826988032000*t^65 + 1830262978014343382094724558797948947377684231911344847605449596864618121873680089416102763819/516901465473658115424059124302006473978127334347552148171735957504000*t^64 - 16741365425548355990304455039402752913872159033633870613455483423980253683418486706872130799743/563892507789445216826246317420370698885229819288238707096439226368000*t^63 + 169431744764084892493175578228506284452128175716020512549367429941978315477455621700511938456953629/1190940976451308297937032222391822916045605378336760149387679646089216000*t^62 - 3008395704424036212947678335678703790496028928523924883733352199403915239429170188080229200295841/8724842318324602915289613350855845538795643797338902193316334403584000*t^61 + 786965328291642502246667700494142180210526669129240032217017471063734613772256885223439510802320531/497088407562285202617196058041804347566861375305778149309640200106803200*t^60 - 162182720306408012715794982305530518930162070685104244631937947964539150480726187452691314880176260837/32092725260161570975987394624453333316597365984653747183499577831456768000*t^59 - 45986998964714696658532280295674070899146075344584512353166782903019460518024080821578984553390953567/9619379525832828460813049046067519714527869558940912828007761278140416000*t^58 + 2671330733860756703239082940936061371810020350888088515555480732471431773282952317650430710564606608470599/175611392623604116380603023385008639908420786668025304588109689893731434496000*t^57 - 745067842244173196965278265177000906604889166959632876101106143025720974054034770693258265277046767281/1756113926236041163806030233850086399084207866680253045881096898937314344960*t^56 + 73311434018291586854609377170135669512825932036299064866282847119173585561637013813429294584685444615804819/4214673422966498793134472561240207357802098880032607310114632557449554427904000*t^55 + 1188511840278152220891966133409005438405464292723012737948415878212020155062012736128581099116691937829749/27546885117428096687153415432942531750340515555768675229507402336271597568000*t^54 + 5441345195263769322478093581236131442983986684368659330476422849544886735850046120622461753672176366675838473/67434774767463980690151560979843317724833582080521716961834120919192870846464000*t^53 - 42784244223252895416521193742689010379856337340076377938402422600030224495519810044983471832216787111872855193/539478198139711845521212487838746541798668656644173735694672967353542966771712000*t^52 - 15751585910740728891671754720774097520846153873857930493465921491261761375563208224162788731964270368422353359883/17263302340470779056678799610839889337557397012613559542229534955313374936694784000*t^51 + 362376912170912521114321867439922894920542004269319346050097019014257723888032073382589812739417409974004625479/548041344141929493862819035264758391668488794051224112451731268422646823387136000*t^50 + 413324568916607762894168110996689493306476969788006401520167040546023335659272364185016833080347201571640656550679/552425674895064929813721587546876458801836704403633905351345118570027997974233088000*t^49 + 5935396228932210547635993570966323889617531666933007821897957672819408470598888950860993373951703329529423902171/31870712013176822873868553127704411084721348330978879154885295302116999883128832000*t^48 - 6341381946693017406367043502969669649510143403907633956601547056153117137003562893795984425022151662573831109987303/13258216197481558315529318101125035011244080905687213728432282845680671951381594112000*t^47 - 9794247060975723583551234478421093648223986924732951959415335276298201933632955456489795956841401942101281595962293/35355243193284155508078181603000093363317549081832569942486087588481791870350917632000*t^46 - 106842585613554210746813716178027281922351831400680947434871550516933079688647864548912785748624228347859656499204183/424262918319409866096938179236001120359810588981990839309833051061781502444211011584000*t^45 - 32842755178369094148782340164044760312034180515500851361386502471887033774068609724466111909758667889492112117089609/339410334655527892877550543388800896287848471185592671447866440849425201955368809267200*t^44 - 28610651989099724686730914992281581759873112874786887643329123343166203515991295861304967753820271077496109733063673/1131367782185092976258501811296002987626161570618642238159554802831417339851229364224000*t^43 - 16504772383659498537448179076486484001354552321069797762399246953637620839055629215349114677528536049300867913557539/3394103346555278928775505433888008962878484711855926714478664408494252019553688092672000*t^42 + 29527146830865368630132480790215320127588353595118723739200543913033283892541826196337637130436432850929509445098139/6788206693110557857551010867776017925756969423711853428957328816988504039107376185344000*t^41 + 185713542468471221201503743216723145031882518181489486198867404092890484327648722323758224251090254731825865326743/135764133862211157151020217355520358515139388474237068579146576339770080782147523706880*t^40 + 40565321552932239240923268505186117319041215776317077808207376802716815332107965766718054778249919060410865776589/78081457289553505191097177486999487284048533989496516796058419174562231937971257344000*t^39 + 37814817817995883003759279650574645892198649392838347221024532657415725841553015501830642219400689015333013445653/2129633472348410308251297527145417388472774721164503036535632570035609110308196450304000*t^38 - 126659650764027350743020355307018460266124589260314505485657982994413156357819451547380718736916453494155985610903/18888922972133726212315856327724571619497654048589505193620393229881054717516177211392000*t^37 - 4125304654749012064703353402839772645417205049144832683460243047778863571893554517678614188714163291209698245749/2406898772072441567220303022369335995836266166856280440184316034832488966774914547712000*t^36 - 135901848479253314142185094172642270333920290893341978473939843965715085364964899130046802176634797129764537015/1158520608957535207688705854767107059329189448313489651875384118099371356007658868965376*t^35 - 1116049832745235405834489787064634264707876356760426256224741704052551846840992600684910850132594188501889944321/102222406672723694796062281302980034646693186615896145753710363361709237294793429614592000*t^34 + 197248499619017360994889157317593216079633589362663931932435252844369289222963600934201776874537419985966161233/44558484959905200295719455952581040743430363396672678918284004542283513692602264190976000*t^33 + 1181227339007634371930980676421027661779649216108810563709424831559690356516086909817243283539469471914114569/5081230741041821086353973047224153768987673018918814262611333851313032263191486267392000*t^32 + 107292404245150051054342029781078475764381962064114912116242695251605967706434838874467913022378507616647605449/2317041217915070415377411709534214118658378896626979303750768236198742712015317737930752000*t^31 + 608694191297306044825757136556073363295488778760432929944176346552080205130782618517570082254199365570637/159898869473344020199950200786774069653458241854088560711545470845515001289242574848000*t^30 + 326217119732705540676804901139018335178486793626048786404066895064826040798360220418961542071929099115438839/2317041217915070415377411709534214118658378896626979303750768236198742712015317737930752000*t^29 + 85765964804289926455859198364990921575346560085878324299887669610697071854083521071238176794809714730267051/6951123653745211246132235128602642355975136689880937911252304708596228136045953213792256000*t^28 + 14601749025696502590466453902232246948123867919082752787709316596693929339166659168940499470843991676589/45432180743432753242694347245768904287419194051509398112760161494092994353241524273152000*t^27 - 58601058331382319180493001196142466618959387783403094312302319298305510064041259964263636589569959314917/6951123653745211246132235128602642355975136689880937911252304708596228136045953213792256000*t^26 - 10185028319665788872887275652284807394885375014730394797542496327540445138754862048219766833088058017/165502944136790743955529407823872437047027064044784235982197731157053050858236981280768000*t^25 + 40588792999661823207902521016709982107300790920790145150408973566546256831953798423476413226449103/3550114225610424538371928053423208557699252650603134786134987083041995983680262111232000*t^24 - 5241605388036958843420658239545164713076106553211784879407558593826471369572564598174306031879763/20687868017098842994441175977984054630878383005598029497774716394631631357279622660096000*t^23 - 1198821282512130004000236205085188415027432086421067829577203642975095345426450306604329556837517/210640110719551855943401064503110374423488990602452663977342566927158428365028885266432000*t^22 + 26746962973578759968558677908367023489074098298537794866513848024569976220406541748948462749387/248254416205186115933294111735808655570540596067176353973296596735579576287355471921152000*t^21 + 2142407101212647467502491857847091980305585980206722055291582194889577425142288299222189621/1253810182854475332996434907756609371568386848824122999865134326947371597410886221824000*t^20 - 34179692318567649696660816860724969808803046428979869111116589592569897481836376462593310003/1158520608957535207688705854767107059329189448313489651875384118099371356007658868965376000*t^19 - 20987767562145135858708144843428922987996740132668135217349321666131574061848580198745999/289630152239383801922176463691776764832297362078372412968846029524842839001914717241344000*t^18 + 560175467322192320520633518882778926455535700196376152020583696410333721916744382435851/144815076119691900961088231845888382416148681039186206484423014762421419500957358620672000*t^17 + 85304390537780690858749287967230137175848673921930898215865344829460924650766797433/3861735363191784025629019515890356864430631494378298839584613726997904520025529563217920*t^16 + 4558576194231193065606646696032014805925162776142510674650198467454320266229576379/14481507611969190096108823184588838241614868103918620648442301476242141950095735862067200*t^15 + 70050528851960000903529690162647573559727699688908608151467386389878891576301/120679230099743250800906859871573652013457234199321838737019178968684516250797798850560*t^14 - 108278609374229662638942062349992201273086788392015145582700414340526500801/13165006919971990996462566531444398401468061912653291498583910432947401772814305329152*t^13 - 3769959682267786797812023771961483781411622426262889370772135983385366083/150849037624679063501133574839467065016821542749152298421273973710855645313497248563200*t^12 - 940320754291275815521883669086408712994792382422661754757946555247/1203582747005418059849470012017024454921448479381534295914420003012146106224712089600*t^11 + 74801623470321601060800491686937583037126312323167580185349464771/56568389109254648812925090564800149381308078530932111907977740141570866992561468211200*t^10 + 22541754656460047840449379171896897740831876335167486253383/10101498055224044430879480458000026675233585451952162840710310739566226248671690752*t^9 + 242064920197640198073997481955421531752377599140536203/2301773645396103874223839948111985245007652935015141272297271327375116658225971200*t^8 + 150696819971677664066728359107160479654531243/30582664293634458362880526521470892392214776453751345560922503818228059340800*t^7 - 34234987120273041985652809472799611953/222998593807751258896003839219058590359899411641936894715059923674579599360*t^6 + 44887259254768667073843180371/31110294895054584109375535605337414949762752740225571249310815244779520*t^5 - 453359264701925/695539087876642451792858207179196870120826195578675238625345536*t^4)*Dt^5 + (3066492800/3*t^77 - 134527232236019907729961941030492797912724160/10364414584611960603563061640578651*t^76 + 187601862923047572285991559095617911638966600321284989/25656030405090108830217588532841830370796*t^75 - 519602463746078375734832398146328150129955131448995096801/1500877778697771366567728929171247076691566*t^74 + 360960758928848810200145967316191646909574279731042791598478293/33008395656596804454551362049555281163020331520*t^73 - 1124304528844937053208014966219476605788389683554049049305628581/23176107588674352063833935056070729327227041280*t^72 + 3290050435290110178502993341656607722349652213819105986892500224066771/2179471399277846099934705848910762514046398445322240*t^71 - 12433845389413669872412281502446011170549473657273378873108634992932238749/828199131725581517975188222586089755337631409222451200*t^70 + 1171045798871520045547280663784343320955319608847910590516118094923147034069/9938389580706978215702258671033077064051576910669414400*t^69 - 8641926101849609558779718698206168252139893642295776505801483412200564216603469/6440076448298121883775063618829433937505421838113780531200*t^68 + 136282874611784115975983472659412480999619967250771632177237076513840155653577329793/22256904205318309230326619866674523688018737872521225515827200*t^67 - 2813715568448512818107340687115012382975705981785858472275531672760890075358804232307/44513808410636618460653239733349047376037475745042451031654400*t^66 + 928581096575715984306253605172679886322831004664918151349754531441836474025762665298443/5127990728905338446667253217281810257719517205828890358846586880*t^65 - 117393149384701917283749158772202087204958278439788855626383538616907850919385250810561809/76919860933580076700008798259227153865792758087433355382698803200*t^64 + 1997924557205293361673393053418721712754446205682073864985019330808748756219715357023877147/604563117513050427396560379300592367225879923213511284411387084800*t^63 - 32030886882996964630841898337908387882028762320135016649740352413483561313174739390570167510567/1240563517136779477017741898324815537547505602434125155612166298009600*t^62 + 27197557551369592638078621419961737577061763692299382995570139851265564271918556293769141883640729/238188195290261659587406444478364583209121075667352029877535929217843200*t^61 - 141455828649928957848698450460992112388870067212940288085553102513534887833009798321353839274433387/476376390580523319174812888956729166418242151334704059755071858435686400*t^60 + 14016179242598413655232311340488658428267956080030627164344307807540197534020630876976019891710484771/11433033373932559660195509334961499994037811632032897434121724602456473600*t^59 - 1220038587570544539327451225027055368113680240926003622912436459411244310027492479574068641021528994003/365857067965841909126256298718767999809209972225052717891895187278607155200*t^58 - 847048373030661130213358838977906767762551553833252109398263165223269163203376908799375592742393410057/146342827186336763650502519487507199923683988890021087156758074911442862080*t^57 + 357861085858562930345706285036461254412981935525933534833069535077721776623375240403202145794061849540497/35122278524720823276120604677001727981684157333605060917621937978746286899200*t^56 + 23057076981633890070943196645043798731135286991620706066482474250026937293166266475265727808599442901467/35122278524720823276120604677001727981684157333605060917621937978746286899200*t^55 + 1488632949841623515020146971311005869497194318414694689165006458831372856389400763110573231819147553469597/70244557049441646552241209354003455963368314667210121835243875957492573798400*t^54 + 166833270350625873404349112578812115939866490113208627025459244784837564213901847553891502697835441190819/3705207404805713224733602251639742732133713301127566866034841808746861035520*t^53 + 80403465510188709353041048436017004277065649236725408119324084658459078628267507700670072776316392399492209/770683140199588350744589268341066488283812366634533908135247096219347095388160*t^52 + 15081097308238301729229066045327939339546280753472661248681601300433885879688922569743328749204678249280599/781852461072046152929293460635864553331403850208947443035757923700786908364800*t^51 - 1005941104381298252166258903932969313273173122434041883461111109976304209070571801645631714503307553880268058003/1150886822698051937111919974055992622503826467507570636148635663687558329112985600*t^50 + 1984359218838952553909042122146452357476369119146615182448692244190974691963661380070090944218158666596420274559/6905320936188311622671519844335955735022958805045423816891813982125349974677913600*t^49 + 22938919865863347147790120704207125136276376510711957604876356354492921742541614342311127419337190031410674656663/36828378326337661987581439169791763920122446960242260356756341238001866531615539200*t^48 + 63160271492259730137969796694808051681936741408515157105096943873406287183028021634696850788636732513283822258999/165727702468519478944116476264062937640551011321090171605403535571008399392269926400*t^47 - 553064835679300158790634486811410647359435123014524666938174046674319347440691677974299809157922271828056592766013/2651643239496311663105863620225007002248816181137442745686456569136134390276318822400*t^46 - 2520827088616286782867932644637585091812518376241055175664073674620082204535347107206843059784004799799475756469009/21213145915970493304846908961800056017990529449099541965491652553089075122210550579200*t^45 - 2055848023949273292066185667976667309007486505561062217518704467681577178559266232566771760864989054361485813796007/12121797666268853317055376549600032010280302542342595408852372887479471498406028902400*t^44 - 8520647261966724347334134755712007062169322520210595281147696813114117129673595528528796698230035515927608497735657/113136778218509297625850181129600298762616157061864223815955480283141733985122936422400*t^43 - 620007616140024613185920152351643169774263992303160986556386139968059685703323029090631262046831266961620066592733/29513942143958947206743525512069643155465084450921101865031864421689147996119026892800*t^42 - 3730807442563998035214890720413249095074196279178004978834488106001678295375276373666485521541929772632705633851203/678820669311055785755101086777601792575696942371185342895732881698850403910737618534400*t^41 + 9644133754585048627433845957035353694048349830918493744694858256239693723045724571810195789237365834640170115244977/2715282677244223143020404347110407170302787769484741371582931526795401615642950474137600*t^40 + 2126044670600823390433652014482332888971787085040489920127879382140434720305893830168457713912174074529779836419043/1810188451496148762013602898073604780201858512989827581055287684530267743761966982758400*t^39 + 134548289841194208744370862145286990362906240811039369503346638165283209478571039988709563304249308776135068793629/271528267724422314302040434711040717030278776948474137158293152679540161564295047413760*t^38 + 29892132524088087431918786196876112592554286182137891265840574597929378200832580312198316113694724261861662466637/1670943185996445011089479598221789027878638627375225459435650170335631763472584907161600*t^37 - 1976360435850930368853221886545310323433949882818611474697928649595288930576671942066703711011698132878360795827/238706169428063573012782799745969861125519803910746494205092881476518823353226415308800*t^36 - 119135265845401942776728098044133274563319076389257902470088983237460504992262731718228942345854944476311727284021/57926030447876760384435292738355352966459472415674482593769205904968567800382943448268800*t^35 - 21431731860077931912654649386093057446534896882584680238980147554974702671449791400423937943629518364149868372123/173778091343630281153305878215066058899378417247023447781307617714905703401148830344806400*t^34 - 1357632852995160548706094804061764270823699001821016579910444050168237283723735733750855837619571135399773066637/173778091343630281153305878215066058899378417247023447781307617714905703401148830344806400*t^33 + 2075593789293254250268160834707503119961574110268823241501621434934316026953350647682763444967785443210259659931/347556182687260562306611756430132117798756834494046895562615235429811406802297660689612800*t^32 + 119388197434107024020256157107280701369627804437618967739691904156484966475675572956203145860261374917143054309/347556182687260562306611756430132117798756834494046895562615235429811406802297660689612800*t^31 + 1089996616777589488850649976456026959081374937791944745211132970642349582163653472293668206513484418576045109/29579249590405154238860575015330393004149517829280586856392785994026502706578524314009600*t^30 + 1289524862570924976057982969072986576838233044023772175249162781769972114425745317131791740815997652148411541/347556182687260562306611756430132117798756834494046895562615235429811406802297660689612800*t^29 + 164686064923880829592424953945946852712974173937996961808425973925008903137766754711130338290859966427907117/1390224730749042249226447025720528471195027337976187582250460941719245627209190642758451200*t^28 + 4551121121251942659293265278919970530005975500221598535061136576540460890389674148744899493271253837775989/695112365374521124613223512860264235597513668988093791125230470859622813604595321379225600*t^27 + 19529911416383681930105287254365924612917594704498432180197546792556328508776473686871649596659095234699/81777925338178955836849825042384027717354549292716916602968290689367389835834743691673600*t^26 - 704163014939601442743971322565737176786839943397567485746537769736988164662586675116331829284407387/268642460048124106130714401105416129699522190913272962753712259269419444871341187006464*t^25 - 82519391398857622529512535783176377143319379095247596058978758260627285044146882603245176843307644819/1390224730749042249226447025720528471195027337976187582250460941719245627209190642758451200*t^24 + 461386569957823031272116293128623150182543494087377891026908564959137642369645494558871946969998153/57926030447876760384435292738355352966459472415674482593769205904968567800382943448268800*t^23 - 89498232857033225486261813640075042939406486133857530622265994731167462631075573187121071835427369/695112365374521124613223512860264235597513668988093791125230470859622813604595321379225600*t^22 - 3558472420864817516127135181218687846831422574520875785790247956334997593600114924455406295109551/695112365374521124613223512860264235597513668988093791125230470859622813604595321379225600*t^21 + 5446062499305498466282587174084444168152606392785801984292639289449444006161777720099673406573/173778091343630281153305878215066058899378417247023447781307617714905703401148830344806400*t^20 + 5019201124146036252481073663367388034099662399020962469505096717040807438600617913111286461/3861735363191784025629019515890356864430631494378298839584613726997904520025529563217920*t^19 - 11262696315054182274887714010000981711618803253111283259592502131726581824223170178761067/1980377109329120013143086931225824033041349484296563507479289090768156164115656186265600*t^18 + 656100652770471868083037583370256231071638238526387759121144201953440122993242978079/47636538197267072684568497317726441584259434552363883711981254856059677467420183756800*t^17 - 28730320087642878560894615473962896604626009536288956984061763595684470812958416909/55167648045596914651843135941290812349009021348261411994065910385684350286078993760256*t^16 + 2463896101481201909543568977347426878880852500700183214518527989186933445031917833/891169699198104005914389119051620814868607267933453578365680090845670273852045283819520*t^15 + 2422227289812260010989399836171914322428074324585954601732097168651107438956623/53635213377663667022625271054032734228203215199698594994230746208304229444799021711360*t^14 + 183486549877837992231346957374559679239130139542620705467413438667722397994727/1810188451496148762013602898073604780201858512989827581055287684530267743761966982758400*t^13 - 344734944362397563421958246716428271368224674357568773861405087217149876547/241358460199486501601813719743147304026914468398643677474038357937369032501595597701120*t^12 - 129605888441324255351947462149556405479751003580148394893723382417058407/27427097749941647909297013607175830003058462318027690622049813401973753693363136102400*t^11 - 131570947367560614739806483389079075502324240556711575542557671852321/226273556437018595251700362259200597525232314123728447631910960566283467970245872844800*t^10 + 90134585154064479946682072607228762252920208453454933819071071/175678227047374685754425747095652637830149312207863701577570621557673499976898969600*t^9 + 53538801329048532410218423387984880558383551499664428228399/80352825439282171609268594552272939462085338822346749869286562701094981523524812800*t^8 + 876582146970247229906082390673704782000396339147879/25575262726623376380264888312355391611196143722390458581080792526390185091399680*t^7 + 165732408332014874101948329588200158246499273/107039325027720604270081842825148123372751717588129709463228763363798207692800*t^6 - 126569671500836103507522629562196547/2186260723605404498980429796265280297646072663156244065833920820339015680*t^5 + 67079472343859667484039745/115223314426128089293983465204953388702825010148983597219669686091776*t^4 - 2521929734157875/9042008142396351873307156693329559311570740542522778102129491968*t^3)*Dt^4 + (5349299200/9*t^76 - 687372156218414295139826422251512196022039520/93279731261507645432067554765207859*t^75 + 258697673032158356252936034905869987306499957237644879/57726068411452744867989574198894118334291*t^74 - 124748075828337203649904283233350374079846144108379677073/587300000359997491265633059240922769140178*t^73 + 11052029534276800378082711020364483112634322700170763293835685/1650419782829840222727568102477764058151016576*t^72 - 3429756638857444706388890310886983481674278806071037239947459/170768769335445424383391766456206774411528680*t^71 + 2442852962825650790240891877001706789396042173361226379288608804567881/2823406130882664265824505304270760529560107076894720*t^70 - 51975345656592446802495583457691230368004224977435025659450566508416519369/5590344139147675246332520502456105848529012012251545600*t^69 + 3359956752697410059447201261116895573529121304085149660263553495370726519013/61923812002866556574760227104129172476013671520324812800*t^68 - 5903596512327685081686258883602671408035298582487832269132426814338796635050759/8280098290669013850567938938494986491078399506146289254400*t^67 + 5462279209338036481116030274764621079141690287195780066754532473690105155876272873/1854742017109859102527218322222876974001561489376768792985600*t^66 - 989534793234811094401777462843122281603057172864420226219929636771126060498854997937/33385356307977463845489929800011785532028106808781838273740800*t^65 + 143250916037447413691353805994585597679255486546469676938642516988687960385413523629481/2136662803710557686111355507200754274049798835762037649519411200*t^64 - 5112582073372683624110257015835985780389708560820890149409378068968630384030483271760609/8546651214842230744445422028803017096199195343048150598077644800*t^63 + 25162477113293933677096175784475076992995882767239630885317778896620970888137276818746537971/12922536636841452885601478107550161849453183358688803704293398937600*t^62 - 3273097215161780247786187340962743823703567188513738465111096515676507480525473341841677104147/310140879284194869254435474581203884386876400608531288903041574502400*t^61 + 301472270524093431562202189358852758642954410373015320678142377367478556659987306946538408910047/6616338758062823877427956791065682866920029879648667496598220256051200*t^60 - 15321082522937238650898554138238343069673021032814664082056514992297614015598218164238647768487/111043447687767673467322351738165306857399102875222391551298801500160*t^59 + 90208542811328101117519665957853524731671351865408734899745139055963763639344522640899538523840897/243256029232607652344585304999180850936974715575168030513228183030988800*t^58 - 68978060439095082135844810875356457616822353976973708759099034588119920225903193952893896132949246407/91464266991460477281564074679691999952302493056263179472973796819651788800*t^57 - 96427522571553035808101783410429229906813191249326241301995789652988128121079707939147722324900725099/38511270312193885171184873549343999979916839181584496620199493397748121600*t^56 + 2115731667259751471647091768896217181868997504694470179859680574700472668388916376232903515492644616293/675428433167708139925396243788494768878541487184712709954268038052813209600*t^55 + 1429814807191278910269286861677885977364033620329245585009451519658000471251990436357778859791898279579/5403427465341665119403169950307958151028331897477701679634144304422505676800*t^54 + 12146861905390052012681073616440127383215718926054264845144643451009663003270558971944238636946803156587/1170742617490694109204020155900057599389471911120168697254064599291542896640*t^53 + 119054380783135127125526227891069618536388936356603900609762500410065851522690457897809329411033386121822903/6743477476746398069015156097984331772483358208052171696183412091919287084646400*t^52 + 587699262123906754036045176158122145916207346827571636728027152978341445657153619036243786736053851777826787/13486954953492796138030312195968663544966716416104343392366824183838574169292800*t^51 + 1050320037147275408972917846060765879649385620370854341878031041134514798318519981255373267880539799312275219/23976808806209415356498332792833179635496384739741054919763242993490798523187200*t^50 - 8248308191279370478290291252531039974909478659415331812092615858380773561111130099138550345337537232401310513/22132438898039460329075384116461396586612047452068666079781455070914583252172800*t^49 - 1827218729048303935663495020985410610712073174482296547441185239881631181451669453811089418653554630015162399/66397316694118380987226152349384189759836142356205998239344365212743749756518400*t^48 + 474966990744365619907515875551084089775865930986085911618726568526855952867033338456627219507245272548687318553/2124714134211788191591236875180294072314756555398591943659019686807799992208588800*t^47 + 22594769805546924436241740705258431074115634272702613537238274321719633582618220657800354846217924277451076402577/82863851234259739472058238132031468820275505660545085802701767785504199696134963200*t^46 + 16633386054006650801545739738194016050960838653023167928959739261695457833742952181270278676105754876680671373393/331455404937038957888232952528125875281102022642180343210807071142016798784539852800*t^45 + 205348525429748473547587176107165280759674998312066587955338925733125382620058471564568827343157141685591890719997/5303286478992623326211727240450014004497632362274885491372913138272268780552637644800*t^44 - 550712864002690616058386323355208442946835152901956216384701724986400158205836481908992175836250796822868083294149/21213145915970493304846908961800056017990529449099541965491652553089075122210550579200*t^43 - 37600600479640641606174544214637541084158276576415521054136099418600694543249233527454696456151853971972084665351/1885612970308488293764169685493338312710269284364403730265924671385695566418715607040*t^42 - 10203291491600404548237934660856621042785990924987036037924043270897087811788615834602759716404476958228349949431/1475697107197947360337176275603482157773254222546055093251593221084457399805951344640*t^41 - 1071644006687654979248572514820186966897251809948615919158311485110157868726466981726682888925327989015185283290177/339410334655527892877550543388800896287848471185592671447866440849425201955368809267200*t^40 + 1215435387708081795731686368463569404244040048307136803471610843861311524921072646858845937223464440854746644308973/1357641338622111571510202173555203585151393884742370685791465763397700807821475237068800*t^39 + 65372895238452211888908071356662825285926574368213443722334182538426023651634643896519702860625790031238071964769/201132050166238751334844766452622753355762056998869731228365298281140860417996331417600*t^38 + 2108910541854600414634519714947717382654120683207901545972005541610488595949404384403747835213154012936507471466201/10861130708976892572081617388441628681211151077938965486331726107181606462571801896550400*t^37 + 13827219379055362710144536681118395342500034663792590755741129133388051536147039793891752787159609186174368165943/1551590101282413224583088198348804097315878725419852212333103729597372351795971699507200*t^36 - 80701968847483604607183889985191081861870638351775396146686338461415711252948750813561956330118838737097808081793/21722261417953785144163234776883257362422302155877930972663452214363212925143603793100800*t^35 - 429373779051961733871578939076148247792738316359893780969510017048566851334836755574287346312328109894587977993/386173536319178402562901951589035686443063149437829883958461372699790452002552956321792*t^34 - 3231712824424987055894078583078140622147873776509213466014719940177828526086493062215333057516406141027554440071/43444522835907570288326469553766514724844604311755861945326904428726425850287207586201600*t^33 - 14858274798275903457939156622273163866290754721277298222317938244374377037809180039305080159255366059026618847/3103180202564826449166176396697608194631757450839704424666207459194744703591943399014400*t^32 + 327982647003856176338097988682902677164639654561812967968897245705181350356970295200329156698765278286544046001/86889045671815140576652939107533029449689208623511723890653808857452851700574415172403200*t^31 + 389232210528852723761307550862942759197160429699990999347454471120154943937289483630780676411892672350382449/1579800830396638919575507983773327808176167429518394979830069251953688212737716639498240*t^30 + 11706106633839116093457875604849919293334520691977747626122040025733086774188579758343124912208299572984478623/695112365374521124613223512860264235597513668988093791125230470859622813604595321379225600*t^29 + 343329184861056350737737300101552022736013854498970702658771305837140155827230152213886585629639086727596641/173778091343630281153305878215066058899378417247023447781307617714905703401148830344806400*t^28 + 118742405030839695928997886812178492040816490913513773214884404262420380896141429101667496786411921016389/2138807278075449614194533885723889955684657443040288588077632218029608657244908681166848*t^27 + 9540665370037190866162275486408930589393628762019215368385813945854391104353799860258931836063175391197/20444481334544738959212456260596006929338637323179229150742072672341847458958685922918400*t^26 + 4038124314315275347076653778086097829399241119676690393133311633919446596766853636561501122055342502211/86889045671815140576652939107533029449689208623511723890653808857452851700574415172403200*t^25 + 3712515532611844960817986859837880850657178042534665286055661422612989595425723548209567240971228101/5792603044787676038443529273835535296645947241567448259376920590496856780038294344826880*t^24 - 16686236175206158244925166119659985328211469676234568773014072250540049424690257925856919791308096691/695112365374521124613223512860264235597513668988093791125230470859622813604595321379225600*t^23 + 3232805373760566931963469364489442877219873275674008938843520152565855247323025086694309393331013/1655029441367907439555294078238724370470270640447842359821977311570530508582369812807680*t^22 - 5993917146502700821721308953476575873480038523428176485175008408140435541144483236642368943741161/347556182687260562306611756430132117798756834494046895562615235429811406802297660689612800*t^21 - 287583195289798545487666134334646610139166715830045946303163263487406970622310579037788796064561/173778091343630281153305878215066058899378417247023447781307617714905703401148830344806400*t^20 + 606315003483116182895238636096278307583235967925342659464089375409676993926959254791459579/1373739852518816451804789551107241572327102112624691286808755871264076706728449251737600*t^19 + 8921865544596673643102117552382437876035644215734456993319365398776046261872312635080827471/19308676815958920128145097579451784322153157471891494197923068634989522600127647816089600*t^18 - 50550313955856977663562595217986207688477276959897950986751725978853046759890549939001/905094225748074381006801449036802390100929256494913790527643842265133871880983491379200*t^17 + 1969352906226342928168599918597067187028738591186039037126163810904359266558804768189/197027314448560409470868342647467186960746504815219328550235394234586965307424977715200*t^16 - 81355001453536357065197340593714992864037641805729845606239307003189949072490376393/965433840797946006407254878972589216107657873594574709896153431749476130006382390804480*t^15 - 291705704303075195342830522400404347305684231419808153143897022888609987886869/646495875534338843576286749312001707214949468924938421805459887332238479914988208128*t^14 + 3041671434184710798206654478018227231297793420094907408840555479060639287729673/1206792300997432508009068598715736520134572341993218387370191789686845162507977988505600*t^13 + 2097910530764850355564971916655846882154277238772319412723274025777653537533/905094225748074381006801449036802390100929256494913790527643842265133871880983491379200*t^12 - 725055012476474335428773108949294158883842961353455496244620724980021347/15084903762467906350113357483946706501682154274915229842127397371085564531349724856320*t^11 - 575801126734089362207398219821099516947942450413522521738119885794249/5028301254155968783371119161315568833894051424971743280709132457028521510449908285440*t^10 - 80118111011190805024233476953600190179654735272511868146149690131/1028516165622811796598638010269093625114692336926038398326868002574015763501117603840*t^9 + 62261116638378719568954603296779514131203958892186028677176051/744320909332298010696382770589475649754053664880685682999707107125932460428440371200*t^8 + 165547249849001235429167307269874504527055392079148440469/3090493286126237369587253636625882287003282262397951918049483180811345443212492800*t^7 + 28203428135321338205990952867856234593082768925801/8852975559215784131630153646584558634644818980827466431912582028365833300869120*t^6 + 26683863357791310276842633247634185502337/231686850709351957294549443344476457516778609498116254249412907713848934400*t^5 - 128577694445290586855972596576421417/18583216150645938241333653268254882529991617636828074559588326972881633280*t^4 + 120418534478800253610716485/1555514744752729205468776780266870747488137637011278562465540762238976*t^3 - 75544163125/1858965489801881552900320043858873213727537118117347471654912*t^2)*Dt^3 + (140313600*t^75 - 165743798563056801607731073684018716903092000/93279731261507645432067554765207859*t^74 + 428465665964520817309624635142820846180254347333941/377294564780736894562023360777085740747*t^73 - 733428616952996061384944437973920366880884325743530901049/13507900008279942299109560362541223690224094*t^72 + 552729188035283422275823443676934266011597249287853963292301/318502063353127060526372791706235169116862848*t^71 - 724843075416773788041692739406738845159062987436703396914573079/204239448125192727562536552681623302196188301280*t^70 + 799002608778565507884458169669904756621340248583575151407076985930397/3450829715523256324896617594108707313906797538426880*t^69 - 14057526487663424478207850774545290974842539330569166682135737279596443749/5590344139147675246332520502456105848529012012251545600*t^68 + 71031980551193402166752467066978465950666130495270634268576259213671631279/5963033748424186929421355202619846238430946146401648640*t^67 - 10614782315857259842818993266563997570620680339095226623391936681785577901581947/57960688034683096953975572569464905437548796543024024780800*t^66 + 21404931781316066102899329944862941734858583613978708239853784927930668118283843/32539333633506300044337163547769771473711605076785417420800*t^65 - 39582369944896553490852444358657703035275531252672436557481037398504890361684245107/5564226051329577307581654966668630922004684468130306378956800*t^64 + 61078481246444061027390414224145075531082895127910515355353698993724393151239622726993/6409988411131673058334066521602262822149396507286112948558233600*t^63 - 624938178885285161662433542947991858490610469500385465211776910559478107624249066822001/5127990728905338446667253217281810257719517205828890358846586880*t^62 + 223413413645516711798353078740992659346568833282368238154651398182923843946134996706096943/391592019298225845018226609319701874225854041172387991039193907200*t^61 - 9717223279526220375335400637934983897275298836742329436061628924804863143421875060672109183/5441068057617453846569043413705331305032919308921601559702483763200*t^60 + 174711638225230295280666665000673557013456037177627850232986714469617071059015429458441796373279/19849016274188471632283870373197048600760089638946002489794660768153600*t^59 - 8186076597436765463183770379937013331425808302978522723959998925461995080937249674187284074339/265538679253357480030553449808656168572041332962488327622671047065600*t^58 + 4879075735922529297796336374111519393198451779431017682794720868996361444985170217391227203295579/346455556785835141218045737423075757395085200970693861640052260680499200*t^57 + 24656174115162754659281809685242385671963196715518139815392498992187056816860062704558034359207119/1451813761769213925104191661582412697655595127877193324967838044756377600*t^56 - 976706245221589213971883007082355241359112149098611608090705532985147849140321706093347123244359/3574568324043399209831522215132076207222373934783123770316513798520832*t^55 + 1267921908517456043890168304407133800229171782584363536536687012419037705729282381023814353775215092803/2926856543726735273010050389750143998473679777800421743135161498228857241600*t^54 + 12200000132061185812043832354357135098963068300562192776481650890964137941711125179491104929716867633/2926856543726735273010050389750143998473679777800421743135161498228857241600*t^53 + 15246844684017886846699253414898805141014699209885915495419011922436231275727675423804819720236808354309/9365940939925552873632161247200460795115775288961349578032516794332343173120*t^52 + 54581928372943241087399545357107667373942947069787367474920786891676674428773414673964111262789291161549/43227419722733320955225359602463665208226655179821613437073154435380045414400*t^51 + 167215328367709387663027989343176358313764762060757660842790297496553884927488549396929222010062842003653/236613244798119230491759863087169535876609059931655147234505687435764459110400*t^50 + 9620111719237449590553806468489414488151715462423658836754165790097014166487802511333539721621088869631639/1027577520266117800992785691121421984378416488846045210846996128292462793850880*t^49 - 962210620723872914837499173187876372487013606409114528834225004464422908943968525635362852292862313941208711/19181447044967532285198666234266543708397107791792843935810594394792638818549760*t^48 - 128036639318935881475748253128122805600919504991603441387450528711431338718641683842552159168443477124047/5828101314100489877612623430440733990154687588658496577482557849657461964800*t^47 + 231789834169293254430258775927448287601468151499613156245377586917581640505455789173910273724757795532925223629/9207094581584415496895359792447940980030611740060565089189085309500466632903884800*t^46 + 127715813738981719981213217483418275485203444285596225634169358336526660938248103012667143141340848978257951039/2209702699580259719254886350187505835207346817614535621405380474280111991896932352*t^45 + 7304113642058169168239696049119769724300207296458707392396710838013508140798632108623774000199164889249225291407/220970269958025971925488635018750583520734681761453562140538047428011199189693235200*t^44 + 43264695774886862815062688493735168045927893255358407014063363436312880922758896131875438905414880054798568475243/1767762159664207775403909080150004668165877454091628497124304379424089593517545881600*t^43 + 10560645495256922188455730586608191355260584221407426505479595972623991520955851273358181057740094757157201471929/1414209727731366220323127264120003734532701963273302797699443503539271674814036705280*t^42 + 4538697580271677854480205752611477781617991314401425563748447787515746515821236370395757869209754599411571531293/9428064851542441468820848427466691563551346421822018651329623356928477832093578035200*t^41 - 2346205158075178953230663894941491904593280857757749819970531991071985602819436795304388114213815884339285135621/5142580828114058982993190051345468125573461684630191991634340012870078817505588019200*t^40 - 7551188106901564100455126182473472813035532677636962150905546198879780292967060613869428922715169510899721355303/10285161656228117965986380102690936251146923369260383983268680025740157635011176038400*t^39 - 3529349391107945857180849234313493073193260435009932568488375101243265919488744269265686675665522007846145167323/32324793776716942178814337465600085360747473446246921090272994366611923995749410406400*t^38 - 5212428416466003044435549288171310945481177648848773035812754952676114615403535728019524748865175784710692032591/164562586499649887455782081643054980018350773908166143732298880411842522160178816614400*t^37 + 235598190794498471995179401366458894015467936461150727811085017717390031679982226402609322512100433533001220701/17746945602903419235427479392878478237273122676370858637796938083630075919234970419200*t^36 + 3188092213562626936559932385281667857564880533390294936245085065465926459107304966969886429525875478004185221/3226717382346076224623178071432450588595113213885610661417625106114559258042721894400*t^35 - 1314732334507401465416429688361703505675276838159909091954189419552353913148232491670555186712770019068901036793/3620376902992297524027205796147209560403717025979655162110575369060535487523933965516800*t^34 - 5917435105934939755695625885963151421591680018780363361390664436670035940220857364834706864355671267191351731571/28963015223938380192217646369177676483229736207837241296884602952484283900191471724134400*t^33 - 10527940643925295578297658739578831389004515703696244524939168845695197420597617011080988607823636194857299/576998470474507534309858282914528577640244963898263632498298728035785399238813286400*t^32 - 1626119986521241143855521368618540189273329943938319666105788911735644337821994187158010600097418464567582727/877667127998132733097504435429626560097870794176886099905594028863160118187620355276800*t^31 + 377410442023739774188719060389911198243818395176319833671484401417196348015685273389618514997634771640493283/419753843825193915829241251727212702655503423301989004302675405108467882611470604697600*t^30 + 144998252948273267463180921690708122575168747409874895414449693356590640656652079870103465053062499250193191/2145408535106546680905010842161309369128128607987943799769229848332169177791960868454400*t^29 + 3128339507491352625785959998793955230867415180115860914498647276847663220655032974114416272941406514496039/812996918566691373816635687555864603038027683027010282017813416210085162110637802782720*t^28 + 97629954196776765312269824143970342469942998139461592346147809026613192053177623522486573044948375258029/198037710932912001314308693122582403304134948429656350747928909076815616411565618626560*t^27 + 95702674554829243620984383740195032861557306359577815698070275240657291315902605608214963091593048395671/6620117765471629758221176312954897481881082561791369439287909246282122034329479251230720*t^26 - 186949754120921279268382349688318538140504371962319891804063496250196743836591070581052062149799825553/752286109712685199797860944653965622941032109294473799919080596168422958446531733094400*t^25 - 222239281079697521945420322519114228165177258311568876205218827942081486514183145246801815433287589/216951424898414832900506714375862745192732106425747125819360321741455310113793795686400*t^24 + 248661916324700971452935864317619682976230451197747082369086207072936360077210959161020573718584549/508123074104182108635397304722415376898767301891881426261133385131303226319148626739200*t^23 - 3431203091297689617907630485082076010072480630002661294798488639820785773838284608968384981277047/671606150120310265326786002763540324248805477283182406884280648173548612178352967516160*t^22 - 798501612126300433854046013312526998587837416949365105855887379468366178796801198646525618010763/7723470726383568051258039031780713728861262988756597679169227453995809040051059126435840*t^21 + 17859233585388148224566893017586941243994389519392634907924421159558224028903444228257797357203/10532005535977592797170053225155518721174449530122633198867128346357921418251444263321600*t^20 - 412512249849023134014497884857605063229761570070280782816903912097854041245422016150102472033/8275147206839537197776470391193621852351353202239211799109886557852652542911849064038400*t^19 - 244574012251018297821110297290214036819263143167914421296257461810022186971582512270612541/390074279110281214710001971302056248932387019634171599958041790605848941416720157900800*t^18 + 424111006290073053292124718395088589365189921909309820700474084929027991928524217407394047/9654338407979460064072548789725892161076578735945747098961534317494761300063823908044800*t^17 + 769758725835174193329442626685118997323797925533662310101318985358569332090902072519/2979734076536870390145848391890707457122400844427699721901708122683568302488834539520*t^16 + 16840724052101050018707121227734584742803465993066560941942316291475609725737103797641/9654338407979460064072548789725892161076578735945747098961534317494761300063823908044800*t^15 - 82593797931194485453729858529444598232222156210930925090050964045250526662557839/80452820066495500533937906581049101342304822799547892491346119312456344167198532567040*t^14 - 47976583061138151678093663544104451299246133613759339229026580166824159196049039/603396150498716254004534299357868260067286170996609193685095894843422581253988994252800*t^13 - 12638246235587237906844350958206578029568564431743719704734124599073086825459/301698075249358127002267149678934130033643085498304596842547947421711290626994497126400*t^12 + 34741124570664708836364702678502661002644010620487314055910774118751686349/150849037624679063501133574839467065016821542749152298421273973710855645313497248563200*t^11 - 45671592648626096792902960257384399376880407725801741016027515687503077/18856129703084882937641696854933383127102692843644037302659246713856955664187156070400*t^10 - 20805301644185332862466648112783719112296846408337545577945133593747/7542451881233953175056678741973353250841077137457614921063698685542782265674862428160*t^9 + 5317687084242693842876784621487653655459632677490376545677948987/1346866407363205924117264061066670223364478060260288378761374765275496833156225433600*t^8 + 9207283018098839910302931240092739703811275736377202754268811/2357016212885610367205212106866672890887836605455504662832405839232119458023394508800*t^7 + 12805834596001644645020069394961930258558277509356043739/24552252217558441325054292779861175946748297973494840237837560825334577687743692800*t^6 + 3438763288477436586072984789868482480862811446123/95907235224837661425993331171332718541985538958964219679052971973963194092748800*t^5 - 9299272682881177862630797183470766798987/8919943752310050355840153568762343614395976465677475788602396946983183974400*t^4 - 65561895789630263399568035894971/489032003964366795824569822848812698157674148337580909462850709812674560*t^3 + 42686246082235577913965/23568405223526200082860254246467738598305115712292099431296072155136*t^2 - 94382375/83499631929635341619636124901462390214712069135294567284736*t)*Dt^2 + (9139200*t^74 - 11837585968364305020700938640419018836569280/93279731261507645432067554765207859*t^73 + 524433968856930593086568015252097969856421943142306/6414007601272527207554397133210457592699*t^72 - 1587775489028555533458222271638681639094376487891780617/397291176714115949973810598898271285006591*t^71 + 117189640746383341026472411081049193234280353749563093516831/889932235839619727941335741532127678414763840*t^70 - 12828640611582628078999874680274023061417169700852272788470819/62842907115443916173088170055884092983442554240*t^69 + 33593269671444284059527281522060147710837824993609247917146552893967/1725414857761628162448308797054353656953398769213440*t^68 - 30966268876519480413601399472108037676503744327798714146586099111648607/147114319451254611745592644801476469698131895059251200*t^67 + 3468202730692363066842908709820353052231349755078491870414040867224103543/3726896092765116830888347001637403899019341341501030400*t^66 - 65276072895925995923943981242336611717524449089974633679475981493223644601051/4140049145334506925283969469247493245539199753073144627200*t^65 + 1459734003663900280156764805355861734136162706356197531457309731207792541846471/28102151774391804583745732154892075363660022566314678681600*t^64 - 1788938613078480715633541237757911553340424733376528796446714359877232814803794789/2782113025664788653790827483334315461002342234065153189478400*t^63 + 3439321472319890090140717908592076893981412141031511643727596086017233178958772923/10305447606321017778672132671386274633680701780202753936588800*t^62 - 75274967863638416374973238791725029395335260811983049592351124519202730666869905921887/6409988411131673058334066521602262822149396507286112948558233600*t^61 + 556191181708779027034165531741892504856566541131473253246258086170326406168606326291951/10183244000663083440190290076871679944407551898099924116858470400*t^60 - 1208007906576602246549570156955811341790440908735203232575886216687667159594388067661228587/10338029309473162308481182486040129479562546686951042963434719150080*t^59 + 7446433092788306556858373767114555063520597747650098602971609104959356861515208623678543593033/9924508137094235816141935186598524300380044819473001244897330384076800*t^58 - 24712818535308902116462813641436555255813422943060599548385513392459205166678100153981543003159/9924508137094235816141935186598524300380044819473001244897330384076800*t^57 - 148524794219363169457211707407816134966909226244101470218091542845173924126407530861958572474751/38110111246441865533985031116538333313459372106776324780405748674854912*t^56 + 69169448741781242780603046850321053378573694061829255518928091568049313433403052108597475172451/8380453270245599897522821575929265159639224212595123646048542864179200*t^55 - 406865331164015749823652445681514236708048670589079115397390975933337534628580338976992091024985391/121952355988613969708752099572922666603069990741684239297298395759535718400*t^54 + 114019841988453029958838647363408016978679130966649320013002410745165525905412997929547226682895073/4099238856760133435588305867997400558086386243417957623438601538135654400*t^53 + 5257552928182774106388366209833133701815902390902559691763003218636665348008551731439290647583813879/212862294089217110764367301072737745343540347476394308591648108962098708480*t^52 + 1175456730950836466648582842701084977503457881570358470286238264397176550616368539960894441265934317823/11707426174906941092040201559000575993894719111201686972540645992915428966400*t^51 - 49850132652541956687946139882873966101599757028700760312928207800321337196733388052925457499571864649603/1123912912791066344835859349664055295413893034675361949363902015319881180774400*t^50 - 1925999663534291124932009799349531942057489016395249193953614133216660309270744304314090595747089053530093/2247825825582132689671718699328110590827786069350723898727804030639762361548800*t^49 + 959603393513472512427971463513452383729091860429027164392424052030298088645146989101474159513884895979391/3996134801034902559416388798805529939249397456623509153293873832248466420531200*t^48 - 48505628683841197863492145262368018539563041380443026124964397968346577089218989627702713556646951712766767/143860852837256492138989996756999077812978308438446329518579457960944791139123200*t^47 - 798664476942663366259238649564558454954544628432911944673898400232516920401287192140932368116732346795357/6393815681655844095066222078088847902799035930597614645270198131597546272849920*t^46 + 359101524000421855544969167655177334416813579004278323291400628252578758147130284299834419567588888982407899/1534515763597402582815893298741323496671768623343427514864847551583411105483980800*t^45 + 153143023753294544362178577200732660443770309549039373776827950903036245177558390223544875330501840803007193/111827059695357273241644046062120740648145081863083786508369457200410525905715200*t^44 + 253830866092262555916325134565555369597377943176098529831050206721873621798114568144885862712026581659539783/228275072270687987526331234523502668926378803472575993946836825855383470237286400*t^43 + 25401828117135619287373668801174069039257841253036014138191042930949603028995635631348512782169876639064296783/25253745138060111077198701145000066688083963629880407101775776848915565621679226880*t^42 + 706726054852994281648424940038075905361860600188722687852274780342063673068784892619188420267252897296685854639/1178508106442805183602606053433336445443918302727752331416202919616059729011697254400*t^41 + 2606788308435202061505718077620272225678493356061095681028528275930560107976150061359201895686476668905882162683/14142097277313662203231272641200037345327019632733027976994435035392716748140367052800*t^40 + 26349727842086246814886267294100474806269608931866460047428081758072744816315072780906706583839948726861722409/2175707273432871108189426560184621130050310712728158150306836159291187192021594931200*t^39 - 1173505350837807115039737633012812191660016801175233350997049019056865451614634921595608865362973725965976929691/28284194554627324406462545282400074690654039265466055953988870070785433496280734105600*t^38 - 602638003932451411358902403501727884237673025164873817923188615963559425429875224508281632890642489891049794031/25141506270779843916855595806577844169470257124858716403545662285142607552249541427200*t^37 - 199950707175654120110835041860848167862285971098219988524688877009537783387400100784323344775182603389221826209/23818269098633536342284248658863220792129717276181941855990627428029838733710091878400*t^36 - 6923572278845878303519072382250020225061737443142102341643587551994517228986305716999400433257952532813222941/3724667595671087987682310489863384321403001055534624652377135153354460378111043174400*t^35 - 1506170584362170187656264626760551217398253034830696256184819895227742880765332343734071267923752325073821857/14309790130404338039633224490700433045073980339840534237591206992334132361754679705600*t^34 + 2769583837538229868376230552031313435412113705684274680331492664429348965160791113403493858500587307961056097/113136778218509297625850181129600298762616157061864223815955480283141733985122936422400*t^33 - 131409301320751385621378721587412032806001693273767563738329310001353676958361217697415021641560880685526189/22986520018998714438267973308871171812087092228442254997527462660701812619199580733440*t^32 - 412264405080732948191929357379613758498902346695556196134298170176375984560656920819953457016311875672169379/413757360341976859888823519559681092617567660111960589955494327892632627145592453201920*t^31 - 198849117621556561943284213301678186748083861812629165301943988004884427684591729973545819471264041813364679/965433840797946006407254878972589216107657873594574709896153431749476130006382390804480*t^30 + 671181426838284343106539977985641509422470531899313525003224398717520383926035223246750894611423468556631/13433680530583664282104659725963671838232716237401317855697867788721838543688066662400*t^29 + 32211992098671346469605351843980284923777648486138978449253319390720245927554687426535690563916654311569/7375354016791031370567264163274172773931687346024252940383143099690421161240507187200*t^28 + 278193111299878234786294454492410096057938637223154760743664264593513147387304443176091469089298991421311/1259261531475581747487723755181638107966510269905967012908026215325403647834411814092800*t^27 + 321723463896786893311116938187421744313726531779022822560846341500644132645333578681556159894971372052139/9654338407979460064072548789725892161076578735945747098961534317494761300063823908044800*t^26 + 27475335884178183751995648911883356556437767152471661121398385162271299669282937933618049849923273787/25185230629511634949754475103632762159330205398119340258160524306508072956688236281856*t^25 - 407224097736015390452206854259116220072631153850961103302524266966709548542132393704825669327757583/20821721943880934717625913996533196609079609063865737812282245113216595183459001958400*t^24 + 7188206866835322007552000953567974338474421855203190145452817684394140297612697105050417251244519797/28963015223938380192217646369177676483229736207837241296884602952484283900191471724134400*t^23 + 19738722822805420088160312395717982141919305370985956435233324307929122754174251910767951950517393/295540971672840614206302513971200780441119757222828992825353091351880447961137466572800*t^22 - 7045995167101004058047350539989226616183565393193446479052430387264938302689713594962949319049831/14481507611969190096108823184588838241614868103918620648442301476242141950095735862067200*t^21 - 121765821672647330719198152167760215380297772188306208559066016304230115224247513584477730006071/2896301522393838019221764636917767648322973620783724129688460295248428390019147172413440*t^20 + 516156619748585413602658781761099752302348045985511138312928510310603816635562375344787712657/9654338407979460064072548789725892161076578735945747098961534317494761300063823908044800*t^19 + 2141352350883152217654633248385124045001271101658214061714509321410932070633224041176577823/148528283199684000985731519841936802478101211322242263060946681807611712308674213969920*t^18 + 27400652971976134593986734781955607698856878795482868234238000200461411921165934694906871/804528200664955005339379065810491013423048227995478924913461193124563441671985325670400*t^17 + 3505635037294632704776728487377891183475831849386325025299348556416639314923847545437/371320707999210002464328799604842006195253028305605657652366704519029280771685534924800*t^16 + 1719254739477670446451187530054478732560239633957938055483122941977593972076784883/128930801388614584189003055418347918817796190383890853351516216846885166934613032960*t^15 + 20002974737667909859337206352696830145790886875134709741510092473150648828361550011/268176066888318335113126355270163671141016075998492974971153731041521147223995108556800*t^14 + 1406162759130630665658182086404543565433362956443718098479823408578383722095849/6094910611098143979843780801594628889568547181783931249344402978216389709636252467200*t^13 - 2218745209372054173923800637924990163955231423951085671808840187324801998010451/603396150498716254004534299357868260067286170996609193685095894843422581253988994252800*t^12 - 256405576735804816968301018866109055575815181776906966092228798496514438471/24135846019948650160181371974314730402691446839864367747403835793736903250159559770112*t^11 + 10724105079918113475037801912603178998190385797501296910905315674735959/5387465629452823696469056244266680893457912241041153515045499061101987332624901734400*t^10 + 168396107080586884812342043190590172790375480372602273329585932617/97953920535505885390346477168485107153780222564384609364463619292763406047725486080*t^9 + 1515787070699661597130952387912711897948410511728817770378426418903/37712259406169765875283393709866766254205385687288074605318493427713911328374312140800*t^8 + 34630375450846262623091253861172676974984742667996336282542987/1047562761282493496535649825274076840394594046869113183481069261880941981343730892800*t^7 + 977735169971039291786432443728340705336781597160171259339/157134414192374024480347473791111526059189107030366977522160389282141297201559633920*t^6 + 1025732528682425832579018985949876861027520051479173/1444250130444614195591428987050657408632252821970284719872797695607916334573158400*t^5 + 4889482522009817569878579709257733267575759749/95907235224837661425993331171332718541985538958964219679052971973963194092748800*t^4 - 192870318870917425468077536616581137/104940514733059415951060630220733454287011487831499715160028199376272752640*t^3 - 141654063887825831299264178147/1858321615064593824133365326825488252999161763682807455958832697288163328*t^2 + 9718776415140147175/11271845976469052213541860726571527155711142297183177988880730161152*t - 2550875/2854169236867535313543925723904532610975612544988250663550976)*Dt + 179200/3*t^73 - 94798432023572571911604707435042545613440/93279731261507645432067554765207859*t^72 + 12077846021415178229690582666347477657593877135402/19242022803817581622663191399631372778097*t^71 - 30978243579256321140964515310044703787781602181944972/964850000591424449936397168752944549301721*t^70 + 5640209955306352359934805180645144517131101737425763026833/5042949336424511791667569202015390177683661760*t^69 - 139447135323517273965977136970817359788042361088411994889709/81695779250077091025014621072649320878475320512*t^68 + 52596976793361924073485137990968056811861976738652398504947463981/272433924909730762491838231113845314255799805665280*t^67 - 6220010455331470004804869300056190332724016037591530252459137954028507/2795172069573837623166260251228052924264506006125772800*t^66 + 693412053691821682248729082235990126099873128439802072144693595412479363/67084129669772102955990246029473270182348144147018547200*t^65 - 2518553773072132026601268823587973447590588225519781448688110274540740379101/14490172008670774238493893142366226359387199135756006195200*t^64 + 2258605032724373655587736513513716950503281946285705567934918910168739931019/3473299657509099442934865771952953134834384811566982758400*t^63 - 1392246231274481069268574191683718951003034371854990290600963731657289641840839/168612910646350827502474392929352452181960135397888072089600*t^62 + 7252457921828772924511778623658115879140340209508082833068426100013642219818310921/3204994205565836529167033260801131411074698253643056474279116800*t^61 - 248313376328990350050125673792397652984112973379388739072033117775738749579117413431/1281997682226334611666813304320452564429879301457222589711646720*t^60 + 3255203047475614777994982470582166976726964434803556836982594620081790329157916719019879/4307512212280484295200492702516720616484394452896267901431132979200*t^59 - 1137000801610943640956526884835735317755476168981766350062626301835829476373834158833497/671300604511244305745531330262346070101464070581236556067189555200*t^58 + 37410426312157686363538101483081332722266834488136545608701546013619758120185026738505796387/3308169379031411938713978395532841433460014939824333748299110128025600*t^57 - 917628597895222406515315065405600555202455882063739494835905060258168399775598571271706093/29019029640626420515034898206428433626842236314248541651746580070400*t^56 - 89393851563928548561209059334226884620778148827016100588216657119369733727159955012240710696237/952752781161046638349625777913458332836484302669408119510143716871372800*t^55 + 1831845176109223302477715299466390457900809574346946470769654005767315094292824660919016526839107/15244044498576746213594012446615333325383748842710529912162299469941964800*t^54 - 17773404361446668300185474332816263867483835871533904503049615508163786490679807061679178219393/225837696275211055016207591801708641857537019892007850550552584739880960*t^53 + 94733223612777282083216254677585287342373401070448438897024975680522645335159395695344761670263807/162603141318151959611669466097230222137426654322245652396397861012714291200*t^52 + 67342669570408527764051440264638555665883421608057012173584547457676800567193848663999082764830453/69687060564922268405001199755955809487468566138105279598456226148306124800*t^51 + 64878689233521000901535802970063161388671027447867549609299953054136639305890981835195891021571149871/23414852349813882184080403118001151987789438222403373945081291985830857932800*t^50 + 93152875593217685743428091589486801926195678192187662006392385645272706464266847520426030598321612933/70244557049441646552241209354003455963368314667210121835243875957492573798400*t^49 - 8974211123979535726491999770724358917216145167489411884473438626934237589402808030073183633417482827219/449565165116426537934343739865622118165557213870144779745560806127952472309760*t^48 + 61786286853091614222972521576073872953217212853425696228347627510248259187467521178159954548937749473/176300064751539818797781858770832203202179299556919521468847374952138224435200*t^47 + 521014127099436588598163319870775465446035047488756573179061323824084827610744013025607248877666397553263/28772170567451298427797999351399815562595661687689265903715891592188958227824640*t^46 + 211784540514292351671637134287625698037380966026651186726301118287112203679217316537239354127649317348381/10656359469426406825110370130148079837998393217662691075450330219329243788083200*t^45 + 1776447088294897051122604185005286480081576813067030958901627865041540763298854951294132790778190154056271/657649612970315392635382842317710070002186552861468934942077522107176188064563200*t^44 - 6005623466124944980029479228750507553557177535302410836457895145997013825494755142152225367628382751269407/1255512897488783931394821789879264679095083419099167966707602542204609086305075200*t^43 - 88687989703869300349849541367111648096396835777957444251642687877968871251895597821741699270025267510890001/8498856536847152766364947500721176289259026221594367774636078747231199968834355200*t^42 - 8983449627831817792439561790162922367007390207725272544271636512314783904741656282030556262364261013305564093/883881079832103887701954540075002334082938727045814248562152189712044796758772940800*t^41 - 5819441016668522704455898884499002844190764664761453423867267794543215121087377739495351939383084382214286339/1178508106442805183602606053433336445443918302727752331416202919616059729011697254400*t^40 - 26952696811759874674405834270183665485584961794844217469872146735897139215033542052177824273237260695597263/57722846029851682462168459760000152429906202582583787661201775654664149992409661440*t^39 + 9598877879588826961587792683190971075817466701837829532517421064019443129209253834785698163369327330538199281/28284194554627324406462545282400074690654039265466055953988870070785433496280734105600*t^38 + 53242533055468201791970618998136390620453803617964405660803406479043159338175942870559420332997817777818097/197791570312079191653584232744056466368210064793468922755166923571926108365599539200*t^37 + 691388044835016678232538541587602060840572229580530045435817238086249090423601938279017903960331102838906497/12570753135389921958427797903288922084735128562429358201772831142571303776124770713600*t^36 + 7885764361814364491869973306750354003363658593984017977139027843534009968844145281843759117913930629490914259/452547112874037190503400724518401195050464628247456895263821921132566935940491745689600*t^35 - 647668545850077408573955935889980873572794751371616747221703235212413506050174884958594170771891501820798677/50283012541559687833711191613155688338940514249717432807091324570285215104499082854400*t^34 - 541581646830189868419670776681918770912427201014066577708008952571213986724626827882748741794551633129228769/329125172999299774911564163286109960036701547816332287464597760823685044320357633228800*t^33 + 16043927452349593424407710197013093148974851703733801455801776870454886285624914662614893125811400959355421/38514647904173377915183040384544782557486351340209097469261440096388675399190786867200*t^32 + 28432340857781137109193344625294855857099309529243167444784769808261879870629652143751677088087918517696097/804528200664955005339379065810491013423048227995478924913461193124563441671985325670400*t^31 - 18467117662782801536413468102084227551002602434130692831781988239639201301281254990899208304366240900175409/14481507611969190096108823184588838241614868103918620648442301476242141950095735862067200*t^30 - 138042483266866867097403638957954682752166063619064562838071781599800275333962424276333030090582309960613/70987782411613676941709917571513912949092490705483434551187752334520303676939881676800*t^29 + 167734218446704006133606424978651788478909940450126067808535040313169712755647797058308127752143660530859/1810188451496148762013602898073604780201858512989827581055287684530267743761966982758400*t^28 + 16592894132007134743292872400671361616564796589819710712027421525164384962924059390291218436084575767637/1609056401329910010678758131620982026846096455990957849826922386249126883343970651340800*t^27 - 109116675927419751571217791013418159981619868834013558599510728507026671618563418168421023504226883559/243386682554104035228719717388047701539745682418800347032643722289783898320936737177600*t^26 + 573244340511523189679015282153860953345334080949487977728442374530010386174508120824357379095983470721/9654338407979460064072548789725892161076578735945747098961534317494761300063823908044800*t^25 - 5162876991280379912298682099209216446025740614251887249861507370073291878808013459680143721978907/304873844462509265181238382833449226139260381135128855756680031078781935791489176043520*t^24 - 2227863094259626926190791602343509096949712835672400152005227200702256337447274897382612921406441/11139621239976300073929863988145260185857590849168169729571001135570878423150566047744*t^23 + 383965061668663437305284060028545816944150245959965618459297334219053118228737185975738697009593887/28963015223938380192217646369177676483229736207837241296884602952484283900191471724134400*t^22 + 8046017763041012208913078610503482055241951015529879364321459688384617042793512010762301022653857/7240753805984595048054411592294419120807434051959310324221150738121070975047867931033600*t^21 + 5352220211795681963033996469035050293503844311448798763867002933058338450365121597368071950647/1810188451496148762013602898073604780201858512989827581055287684530267743761966982758400*t^20 - 171056130760929500371348732143857827022102603881689539058042096452665911208416416420620315/327635918822832355115584234945448828995811495563769697928558856928555247739722530816*t^19 + 531953591267430534701276454313653205959474732887589143689528577300676081615214532899824781/643622560531964004271503252648392810738438582396383139930768954499650753337588260536320*t^18 + 1022840029653998706165033925615797328071564667353560944960522129319641227609727968290223401/4827169203989730032036274394862946080538289367972873549480767158747380650031911954022400*t^17 + 6598685197889803373669495475726919370083025199813203100456322675007813352941975033187443/4827169203989730032036274394862946080538289367972873549480767158747380650031911954022400*t^16 + 52716964848153430925166120719195763462970431573946103034089412543887680793270349970201/2413584601994865016018137197431473040269144683986436774740383579373690325015955977011200*t^15 + 35349489820978367368035210631442565670689870767512229919717997452388524681687557819/482716920398973003203627439486294608053828936797287354948076715874738065003191195402240*t^14 - 206242694855846117464745106641301729229586413527438542816944675939613385584179/521517848313497194472371909557362368251759871215738283219616158032344495465850470400*t^13 - 58999948841370350488743245640712838315417324261812082607512908793105714657267/20113205016623875133484476645262275335576205699886973122836529828114086041799633141760*t^12 - 102988944470331360179269412876532264478336248904237096269389651431519783127/24135846019948650160181371974314730402691446839864367747403835793736903250159559770112*t^11 + 8488757020380068541124703507446006096046513142378244373971067868681223/1984855758219461361857020721571935066010809773015161821332552285669153227809174323200*t^10 + 124595941886774746698387991443041522356009358295960412669931121418831/9428064851542441468820848427466691563551346421822018651329623356928477832093578035200*t^9 + 440872975112884976185671198223626597454499505834422781843398594227/37712259406169765875283393709866766254205385687288074605318493427713911328374312140800*t^8 - 1035466221279801381550991612790710800234949634623968332546569/9428064851542441468820848427466691563551346421822018651329623356928477832093578035200*t^7 + 1742716068893585043277000871201470441898064407005119689/2238381968552336531059080823235206923920072749720327315130489875813978592614809600*t^6 + 7590441562175856364383667966860092488830275903516629/24552252217558441325054292779861175946748297973494840237837560825334577687743692800*t^5 + 348034661816366476221645443238610387497348661/5641602072049274201529019480666630502469737585821424687003115998468423181926400*t^4 - 33712099562632482165518687478566083/162180795496546370106184610341133520261745026648681377974589035399694254080*t^3 - 60693463686200287452532141729/619440538354864608044455108941829417666387254560935818652944232429387776*t^2 + 227050905042510745025/259252457458788200911462796711145124581356272835213093744256793706496*t - 2550875/2854169236867535313543925723904532610975612544988250663550976, (t^66 - 1464760907/7662281088*t^65 + 693666239859483463815462629/43056336252079038115458000*t^64 + 13113177316378091715924702192733/3438995689126056932357861376000*t^63 + 3056683310291928806751991243493623/49128509844657956176540876800000*t^62 + 10550252977269807618568521609566001167/371411534425614148694649028608000000*t^61 - 5974101601465442635090268191039798509894397/7799642222937897122587629600768000000000*t^60 + 181639692673986124441629696192663719637619213/5490948124948279574301691238940672000000000*t^59 + 68750591009707387518010737798596955085872143582917/29980576762217606475687234164616069120000000000*t^58 - 1352631773064569289123614164240593981034675919909/4996762793702934412614539027436011520000000000*t^57 + 1850796819491468432438696977301286815671897422532397/959378456390963407221991493267714211840000000000*t^56 + 310357974209169696351702791705097843536727832545953/328929756476901739618968511977502015488000000000*t^55 - 2822888105132007938583837655375653502085216698743433/130824334962404100984817021809233756160000000000*t^54 + 179670820075128689926062807212230808934958533862535279/46050165906766243546655591676850282168320000000000*t^53 + 5424830155967125085652720742488742214894572277694062899/100473089251126349556339472749491524730880000000000*t^52 - 902852686709836726374943738584525995782827226502497940213/123782845957387662653410230427373558468444160000000000*t^51 - 14306690170769247509465280572300351299505643183928397329829/165043794609850216871213640569831411291258880000000000*t^50 - 452554166838918475806168252961729719493952750424569985763/70733054832792950087662988815642033410539520000000000*t^49 + 158582260121537304147818521877546050183556065494084432652057/1697593315987030802103911731575408801852948480000000000*t^48 + 15417222047007073131066180846212843994383233541769332252539/848796657993515401051955865787704400926474240000000000*t^47 - 5074572517687371960054674757064343961913284172051394412288407/81484479167377478500987763115619622488941527040000000000*t^46 - 59306399350553702893051800664727526144932037719192731152091/5092779947961092406311735194726226405558845440000000000*t^45 + 4308847833773493640629693918257762116964640903113611972460023/162968958334754957001975526231239244977883054080000000000*t^44 + 931210815709599542156606967357671591985237017911215415581067/695334222228287816541762245253287445238967697408000000000*t^43 - 65847709646462087289787911168946902717294778708261664155489633/8344010666739453798501146943039449342867612368896000000000*t^42 + 61911236819400228622116120475357320179824699659956791026923809/41720053333697268992505734715197246714338061844480000000000*t^41 + 290543511064036501821406516674137673127697257347914179887412703/166880213334789075970022938860788986857352247377920000000000*t^40 - 1522499052768940420693383260074795905223216817950759708868153/2317780740760959388472540817510958150796558991360000000000*t^39 - 182786483680043971942764694988086841895106012128533280102179/713163304849525965683858713080294815629710458880000000000*t^38 + 89391204247944205544600379935232836560379414328129951047334121/876121120007642648842620429019142181001099298734080000000000*t^37 + 16509146962977091344728814657599752878730946109451858300377451/700896896006114119074096343215313744800879438987264000000000*t^36 - 934696909155191915683286294896942978866019917507888647997463/125160160001091806977517204145591740143014185533440000000000*t^35 - 25331006537022480865016684943244772235157525873378701424849/20025625600174689116402752663294678422882269685350400000000*t^34 + 6617630377408739881707551012485551185753841086968037808427/25032032000218361395503440829118348028602837106688000000000*t^33 + 80385622362920045341912532044351745090446303668485003737/2395409760786446066555353189389315600823238000640000000000*t^32 - 1468323130990692598627011384693623745014162290947559271/379273212124520627204597588319974970130346016768000000000*t^31 - 37625036815898739525573028206601899024724288925955183901/250320320002183613955034408291183480286028371066880000000000*t^30 + 40098417467626217582320368869621945906927111818119401/2503203200021836139550344082911834802860283710668800000000*t^29 - 398546271224952589442914937141088997565165316488306573/50064064000436722791006881658236696057205674213376000000000*t^28 - 229388089855827392424616770561030061942671238339457/782251000006823793609482525909948375893838659584000000000*t^27 + 51849170191621410242571401442064669267195978893889/3129004000027295174437930103639793503575354638336000000000*t^26 - 29371073247405921631833302351232206672136017657/15964306122588240685907806651223436242731401216000000000*t^25 + 1627443302399230737451242635018240319571068774313/3129004000027295174437930103639793503575354638336000000000*t^24 + 95640005986215967078147796826847967176601211931/1368939250011941638816594420342409657814217654272000000000*t^23 + 194110201993476099340116478492264256996529859/99559218182686664641206866933993429659215829401600000000*t^22 - 1219358061694438625601955269197437518014779/3911255000034118968047412629549741879469193297920000000*t^21 - 530401049394023518347144844142186805967813/15645020000136475872189650518198967517876773191680000000*t^20 + 1736147652577154298480868215480336263/12536073718058073615536578940864557305991004160000000*t^19 + 18340684815928877577191998719918077657/279375357145294212003386616396410134247799521280000000*t^18 + 3390096579941874313509716158499683/97781375000852974201185315738743546986729832448000000*t^17 - 26307099824177269198919353515720803/391125500003411896804741262954974187946919329792000000*t^16 + 28831193202185283357410887297/349219196431617765004233270495512667809749401600000*t^15 + 4363132765982618324817029563/107452060440497773847456390921696205479922892800000*t^14 + 628784455876197318869/54822479816580496860947138225355206877511680000*t^13 - 661127752954932607/88601987582352318159106486020776091923251200*t^12 + 83411221/41202561189709969382024965597459120128*t^11)*Dt^15 + (100*t^65 - 146025281335/7662281088*t^64 + 756323060758516917412913777977/459267586688843073231552000*t^63 + 1362220080750739385870046475133371/3438995689126056932357861376000*t^62 + 27721070125391713318668167347279549/4245673690279082632540569600000*t^61 + 74578469203874242584752281742062648517/24760768961707609912976601907200000*t^60 - 157860506036240889493524900740927432053398583/1949910555734474280646907400192000000000*t^59 + 77982460894553184799195085783989837399879333/22596494341350944750212721147904000000000*t^58 + 2447979941872126212277125011216109056217432473896607/9993525587405868825229078054872023040000000000*t^57 - 1728132321992893303595268064320412844524877299413421/59961153524435212951374468329232138240000000000*t^56 + 105599021593598515994602194417548883065682344116165073/479689228195481703610995746633857105920000000000*t^55 + 50028501629745477117395660116508233174287185308497589/479689228195481703610995746633857105920000000000*t^54 - 18489893448727418736024507630117616091163589213767942487/7675027651127707257775931946141713694720000000000*t^53 + 498960873634631245536491701059936162208430480690125609/1137041133500401075226063992020994621440000000000*t^52 + 117883906491889690727374074820442220266868171534019787701/19389543539691049914381301758673803018240000000000*t^51 - 286189813602991819734847866965170423691257925020571535649/352657680790278241177806924294511562588160000000000*t^50 - 4886726635571510871623122724840813353002514164372486155804753/495131383829550650613640921709494233873776640000000000*t^49 - 2809954803253721906904621255154285582001667308013843203267/3536652741639647504383149440782101670526976000000000*t^48 + 9102733990883181968317327707779512003136681507240726406325679/848796657993515401051955865787704400926474240000000000*t^47 + 586460102901753397895379548316795278480560692417986909902073/271614930557924928336625877052065408296471756800000000*t^46 - 21591956979640692603724526139419482308833317554931035490293343/3017943672865832537073620856134060092183019520000000000*t^45 - 36941637314724188928393402506676614367155598654023637716933639/27161493055792492833662587705206540829647175680000000000*t^44 + 15628393279240079720048842273443495681532742856670463651045627/5173617724912855777840492896229817300885176320000000000*t^43 + 462088082750314151115817882784079143483915145931689927232650373/3476671111141439082708811226266437226194838487040000000000*t^42 - 37416103352958283333886420989409265388993708383329806316401284367/41720053333697268992505734715197246714338061844480000000000*t^41 + 45454306143517898802955458234652146389930170652777856305492911/243976920080100988260267454474837700083848314880000000000*t^40 + 32831113475539628435167469534463690889594689746481606250762860059/166880213334789075970022938860788986857352247377920000000000*t^39 - 49084904920634832553464292081458691378661649120521895164077351/632122020207534378674329313866624950217243361280000000000*t^38 - 67189286305384980172985968574539449278567992730781742721026867/2384003047639843942428899126582699812247889248256000000000*t^37 + 3247059989650403953716121889829718110852746490658215867135742459/292040373335880882947540143006380727000366432911360000000000*t^36 + 3947480688473885401423263894793145161763376710358731537744141/1602416314600169453758793651612514277093917327360000000000*t^35 - 3414786393257559943202511286801441309959546249596293593226759/4635561481521918776945081635021916301593117982720000000000*t^34 - 295881834269459169693755781235465267585727495598560817019981/2418553816446218492319173026967956331265974599680000000000*t^33 + 324629518831482733429432685611111183178608723437425608893971/13906684444565756330835244905065748904779353948160000000000*t^32 + 94991599298611763847354498303280505056598764996974480448603/33376042666957815194004587772157797371470449475584000000000*t^31 - 2162687523014902600725170868252966294101294116776842950501/6953342222282878165417622452532874452389676974080000000000*t^30 - 139822005422631955948351883704371027155714487098166776731/27813368889131512661670489810131497809558707896320000000000*t^29 + 6637080121358784674119507885745507033731262800877038559/4172005333369726899250573471519724671433806184448000000000*t^28 - 227035000653410993319026401658806375527624643540535567/340571863948549134632699875226099973178269892608000000000*t^27 - 6105753494308395224023570301298229144935201804916913/260750333335607931203160841969982791964612886528000000000*t^26 + 921424195096827158349330727869330274604880314630449/1043001333342431724812643367879931167858451546112000000000*t^25 - 5118151260638178567952547143111308220840855734653/23704575757782539200287349269998435633146626048000000000*t^24 + 44485744187728967038425662950722900638997673189471/1043001333342431724812643367879931167858451546112000000000*t^23 + 2867895099476475576670364855674116757607276245007/456313083337313879605531473447469885938072551424000000000*t^22 + 73316044358684381406217972650421978916529946621/365050466669851103684425178757975908750458041139200000000*t^21 - 35761779843535084359208168592727359217801601/1303751666678039656015804209849913959823064432640000000*t^20 - 16045294898128565425784781780197164800036479/5215006666712158624063216839399655839292257730560000000*t^19 + 1983842875302344096168848188559581780697/162968958334754957001975526231239244977883054080000000*t^18 + 1311282189960727957947776778164675010061/217291944446339942669300701641652326637177405440000000*t^17 + 110187959606791607066987237464615933/32593791666950991400395105246247848995576610816000000*t^16 - 820377117766146981596931858340458493/130375166667803965601580420984991395982306443264000000*t^15 + 883305001410670806902514087413/116406398810539255001411090165170889269916467200000*t^14 + 5536318599770304451973846287/1432694139206636984632751878955949406398971904000*t^13 + 20632793959425150564583/18274159938860165620315712741785068959170560000*t^12 - 64130109506587016479/88601987582352318159106486020776091923251200*t^11 + 4087149829/20601280594854984691012482798729560064*t^10)*Dt^14 + (36191/9*t^64 - 13160771103493/17240132448*t^63 + 840455489144797051580222918876333/12400224840598762977251904000*t^62 + 257249446719146910647251973887859149/15475480601067256195610376192000*t^61 + 53295786263416335092724168955303045973/193443507513340702445129702400000*t^60 + 431524330049846355446659081697635776890197/3342703809830527338251841257472000000*t^59 - 242604831871477497752324140456200146266995116477/70196780006441074103288666406912000000000*t^58 + 3549395787159889506591421844218206962692445355769/24709266562267258084357610575233024000000000*t^57 + 4081356983431728421796825930530264604622950519981657/385464558371369226115978724973635174400000000*t^56 - 16088720581988411524355655377125013267953189177017473/12848818612378974203865957499121172480000000000*t^55 + 17479454078262921765629187881714546455301136031493378081/1726881221503734132999584687881885581312000000000*t^54 + 484261000688151841798515628229305392243460290870062447933/103612873290224047979975081272913134878720000000000*t^53 - 45255591959101314005336972071089191468343566684264223318269/414451493160896191919900325091652539514880000000000*t^52 + 33205222665554715899749647642608743178256628033434254529319/1657805972643584767679601300366610158059520000000000*t^51 + 5546172622491353853759958413856024746378298547250588801394409/19893671671723017212155215604399321896714240000000000*t^50 - 5860023801699234588127070539472580426952758674869384361400457/159149373373784137697241724835194575173713920000000000*t^49 - 97346363347452790447060269004992827669377995809004974469383437/212199164498378850262988966446926100231618560000000000*t^48 - 3172533298364341377460111582764975856059055288536422991559939/79574686686892068848620862417597287586856960000000000*t^47 + 1539581669188323858275820440329843288516104945212119565433100123/3055667968776655443787041116835735843335307264000000000*t^46 + 1599959154151527609854885893043866954817760596980551932204424073/15278339843883277218935205584178679216676536320000000000*t^45 - 1545452598826058200652262379790540859730842237325846032557396323/4583501953164983165680561675253603765002960896000000000*t^44 - 190603745068305653675101945218140190781113882826487648387282293463/2933441250025589226035559472162306409601894973440000000000*t^43 + 1665143463901410828388394945550747642134322882804921596699140316503/11733765000102356904142237888649225638407579893760000000000*t^42 + 1222591060221713777752382757011077544753522765357126834412609517/235263458648668810108115045386450639366568017920000000000*t^41 - 15772495307070258218604857800292071704385170824872197259713680888803/375480480003275420932551612436775220429042556600320000000000*t^40 + 3599781639849717225443132333242794324381642369080336307717745165829/375480480003275420932551612436775220429042556600320000000000*t^39 + 599605382414752043580639389455492218884194307847879433337112090557/65300953044047899292617671728134820944181314191360000000000*t^38 - 866890216788790698592799257617540958936926979578860501393304503/229231062273061917541240300632951905023835504640000000000*t^37 - 45744179502757896576360911992164934873298159567105189379752326943/35760045714597659136433486898740497183718338723840000000000*t^36 + 24354379348978580460773442159594014861619022549067164921516243741/48975714783035924469463253796101115708135985643520000000000*t^35 + 95377860167450491686908297192424230244789909973533534701484778721/901153152007861010238123869848260529029702135840768000000000*t^34 - 33185049151213956408193141520138160187777085804812655865767162999/1126441440009826262797654837310325661287127669800960000000000*t^33 - 21557510567658082770218216900933064474031016433651150912251443337/4505765760039305051190619349241302645148510679203840000000000*t^32 + 917012116954235705955687962380574600780628033822843933759627667/1126441440009826262797654837310325661287127669800960000000000*t^31 + 417550294502063502797981290847514241346626133471762977376584813/4505765760039305051190619349241302645148510679203840000000000*t^30 - 162884907135572713998249012639317880979697982356718815312699/17067294545603428224206891474398873655865570754560000000000*t^29 + 453909625154374335887057471696512189348663836548803187378587/2252882880019652525595309674620651322574255339601920000000000*t^28 + 1013023896225598405259604259940090019163152733488981392207/16092020571568946611395069104433223732673252425728000000000*t^27 - 9612768353956557060847258745543534962701805108709380701481/450576576003930505119061934924130264514851067920384000000000*t^26 - 4735013876870967913407373044162252677090220948454668973/7040259000061414142485342733189535383044547936256000000000*t^25 + 66297325431379566096645899053289957162715230733570989/28161036000245656569941370932758141532178191745024000000000*t^24 - 3142730179128497991306379955004562388537302343559083/306098217393974527934145336225631973175849910272000000000*t^23 + 36649132289043522705144231905890296730737928861016669/28161036000245656569941370932758141532178191745024000000000*t^22 + 56080207593257393001574996915677471115140837174407/251437821430764790803047954756769120823019569152000000000*t^21 + 1078893120245950070908683757870326377028187532201/128004709092025711681551686057991552418991780659200000000*t^20 - 29295051482603069431352014129520050910133817/31234511978976992646341360839350201344474480640000000*t^19 - 1191724955734201264862950257753302959703341589/10831167692402175603823604204906977512376227594240000000*t^18 + 605541328808735235288473378648605920257239/1466720625012794613017779736081153204800947486720000000*t^17 + 3854075250816958026310287103145043717524347/17600647500153535356213356832973838457611369840640000000*t^16 + 115671061837967210076407127466069906583/880032375007676767810667841648691922880568492032000000*t^15 - 821543490654210721365348394200158149783/3520129500030707071242671366594767691522273968128000000*t^14 + 865680319665362290555778897669477/3142972767884559885038099434459614010287744614400000*t^13 + 141539246759087560478191163291207/967068543964479964627107518295265849319306035200000*t^12 + 399211424208442985814007/8970951242713172213609531709603579307229184000*t^11 - 22382372615382646238003/797417888241170863431958374186984827309260800*t^10 + 65644630927/8427796606986130100868742963116638208*t^9)*Dt^13 + (756683/9*t^63 - 273792663528079/17240132448*t^62 + 1200675337542276382800018565355189/826681656039917531816793600*t^61 + 9650651671123705117987027405005263/26453812993277361018137395200*t^60 + 1557907630638056174416732905649868756329/257924676684454269926839603200000*t^59 + 16796736742719130735055266443273260802811/5803305225400221073353891072000000*t^58 - 22285127024092450650164830524997130072886451781/288875637886588782318060355584000000000*t^57 + 6884006975714316196421886519438367060481424286379/2246296960206114371305237325021184000000000*t^56 + 105127113028977882896690465478937336355729023606010851/438027907240192302404521278379130880000000000*t^55 - 202021514757372471972641243105789669660748508931334037/7008446515843076838472340454066094080000000000*t^54 + 697520969110875007370231671718534625019802699180499105287/2878135369172890221665974479803142635520000000000*t^53 + 2280814078763683056752098751112946810974396001976630648897/20722574658044809595995016254582626975744000000000*t^52 - 1079698368272381194593513552208480252018537773514092359218291/414451493160896191919900325091652539514880000000000*t^51 + 796658518410387062757963845540410691285367267217141270450399/1657805972643584767679601300366610158059520000000000*t^50 + 26887258009942021382656349109569300160238299494085752816389661/3978734334344603442431043120879864379342848000000000*t^49 - 352324659044029877823342227326744647491486085390168240402071/397873433434460344243104312087986437934284800000000*t^48 - 13843648616069113182771258248239537296283130104530606465013267/1224225949029108751517244037193804424413184000000000*t^47 - 133644972215821991859276175863166059479442141403499492673495247/127319498699027310157793379868155660138971136000000000*t^46 + 64140763184231538429878734565630002853987480620963598378992783039/5092779947961092406311735194726226405558845440000000000*t^45 + 164564243679431664509421281195632611054161773689477533189144943251/61113359375533108875740822336714716866706145280000000000*t^44 - 6220116359935026571146574253153837639420550481426367606065989088031/733360312506397306508889868040576602400473743360000000000*t^43 - 4844064496759907105013176007311686255256545064252956915252947545227/2933441250025589226035559472162306409601894973440000000000*t^42 + 364216510393360184798484729648823584299452823063802026510803343801/102032739131324842644715112075210657725283303424000000000*t^41 + 14778244717454245653799897541870562365496460352088809206680694343/144861296297559961779533801094434884424784936960000000000*t^40 - 2610531922563227314361925200647412448898728761767147991293585071869/2470266315811022506135207976557731713348964188160000000000*t^39 + 4942830723974497165787708578891784938085552257539275607319539109229/18774024000163771046627580621838761021452127830016000000000*t^38 + 28897898313948571136898845751656240497802878403590765993646762753909/125160160001091806977517204145591740143014185533440000000000*t^37 - 618936245509749691230003562594815448677254379288635006169564580907/6258008000054590348875860207279587007150709276672000000000*t^36 - 3917282662161255460670771301275284295493743871205868304296144112169/125160160001091806977517204145591740143014185533440000000000*t^35 + 64527282113685366868569244958280724449712304466109292937040449379/5415583846201087801911802102453488756188113797120000000000*t^34 + 687559870299457820581717039338574571703711759602703486751520198393/281610360002456565699413709327581415321781917450240000000000*t^33 - 50080059146998482583962258194642773160494647363903153209963820113/80460102857844733056975345522166118663366262128640000000000*t^32 - 221965053361018554366826641579990106909306214282419034383234634057/2252882880019652525595309674620651322574255339601920000000000*t^31 + 116553722124840858659946561627316417924678787342112430531370647/8046010285784473305697534552216611866336626212864000000000*t^30 + 1633359244667557168795571310129192172065243958679053112890919771/1126441440009826262797654837310325661287127669800960000000000*t^29 - 537699093762873656923835930814477370991400621170123158166099/3754804800032754209325516124367752204290425566003200000000*t^28 + 13943039609388182376545616632077904151587213225416362357566161/1126441440009826262797654837310325661287127669800960000000000*t^27 + 4491288564793066373319935542317521046139422422581941176039/3520129500030707071242671366594767691522273968128000000000*t^26 - 9714427927506835323183518773479659097053780734481439541/29643195789732270073622495718692780560187570257920000000*t^25 - 26696654883045955603857632135480853225630197149802188441/3520129500030707071242671366594767691522273968128000000000*t^24 - 9492805533852512707217359703226608006808143855040032551/14080518000122828284970685466379070766089095872512000000000*t^23 - 12912282011293514379302582748764455722322544790968419/50287564286152958160609590951353824164603913830400000000*t^22 + 124923013699581775794769417718937772762174431593824393/7040259000061414142485342733189535383044547936256000000000*t^21 + 3550237321030932969972153996486022789484808015631091/880032375007676767810667841648691922880568492032000000000*t^20 + 132262652167348754113871869732459497271421544071017/704025900006141414248534273318953538304454793625600000000*t^19 - 5377961758434240610196114097849237128465848043/338473990387567987619487631403343047261757112320000000*t^18 - 35164981740735872161128499259836161771208955729/17600647500153535356213356832973838457611369840640000000*t^17 + 4999386279861689887808530651508425094229751/733360312506397306508889868040576602400473743360000000*t^16 + 35551834258690088882292041794333505519744923/8800323750076767678106678416486919228805684920320000000*t^15 + 578743383794519173267840502368393184701/220008093751919191952666960412172980720142123008000000*t^14 - 556602538012029175401035447238698201243/125718910715382395401523977378384560411509784576000000*t^13 + 319664484562840703331244434573131/62859455357691197700761988689192280205754892288000*t^12 + 17896695083087215717212611407909/6279665869899220549526672196722505515060428800000*t^11 + 2547309506069635670624269/2803422263347866316752978659251118533509120000*t^10 - 24297452207405988431/43150318627768986116447963971157187624960*t^9 + 2673913511597/16855593213972260201737485926233276416*t^8)*Dt^12 + (2953190/3*t^62 - 530742080610545/2873355408*t^61 + 112726974786350182105582748383669/6458450437811855717318700*t^60 + 582667274417905671988835438678227967/128962338342227134963419801600*t^59 + 57242474573376261602586988997103697579321/773774030053362809780518809600000*t^58 + 8155921821162555667487672283306648948492103/222846920655368489216789417164800000*t^57 - 1137115734762990048687839562194989136091494714309/1169946333440684568388144440115200000000*t^56 + 295653109976975202464410022817182989136522722747/8319618371133756930760138240819200000000*t^55 + 901111082387884999922642130060466332020665032989206297/292018604826794868269680852252753920000000000*t^54 - 98951904193417613128271164912662659251332855872273017/259572093179373216239716313113559040000000000*t^53 + 1558187808931530919789182611360061746030850743867010953501/479689228195481703610995746633857105920000000000*t^52 + 198826277501367603609862206899748944571879483639607744351/134912595429979229140592553740772311040000000000*t^51 - 139757397302955953121312180631644201085587750703558865553403/3947157077722820875427622143730024185856000000000*t^50 + 1549643461999438167496774191759024570490488527485758049437341/236829424663369252525657328623801451151360000000000*t^49 + 138835496662426326729447979089300351434984995836903787575041/1478533754866073371397637726079473942528000000000*t^48 - 4824180740735395972577350629449083622224486738592989984707647/392961415737738611598127715642455741169664000000000*t^47 - 102176083266357351017876923242890508718162100910258712715272404493/636597493495136550788966899340778300694855680000000000*t^46 - 40012209764712451907874336879020901261462392783559797245386336167/2546389973980546203155867597363113202779422720000000000*t^45 + 1855106227415332394091304784540016478481479183593485502862396545803/10185559895922184812623470389452452811117690880000000000*t^44 + 886443916148301729963029075205526668306108740778368353816904047/22223039772921130500269389940623533406074961920000000*t^43 - 10102762307206055570815665753046437926720360369329587545162957840377/81484479167377478500987763115619622488941527040000000000*t^42 - 2065745572292269310771511432888672234515168173667698085138883536257/85027282609437368870595926729342214771069419520000000000*t^41 + 136635346096738433085781171488200188328143411424709660754781406012173/2607503333356079312031608419699827919646128865280000000000*t^40 + 405109590161162606056900280156142831168475656601906268953639058443/386296790126826564745423469585159691799426498560000000000*t^39 - 11142635534149047165265633744870513552984391856243386168125784124853/715200914291953182728669737974809943674366774476800000000*t^38 + 526424961018427644991367767330969279885738745605026258427790087277139/125160160001091806977517204145591740143014185533440000000000*t^37 + 30006031608240389039414157984998409939477450862866801160615377683989/8783169122883635577369628361094157203018539335680000000000*t^36 - 4994600835564055344467815441987747891728868539374285308217674026043/3293688421081363341513610635410308951131952250880000000000*t^35 - 67837810360438619082853493764645563428711513188984226307757654691321/150192192001310168373020644974710088171617022640128000000000*t^34 + 1611676808559749893949635538448201509012415254483092914043074027911/9627704615468600536732092626583980011001091194880000000000*t^33 + 49777600894179446273760161511444218040619019628690935946796010687599/1501921920013101683730206449747100881716170226401280000000000*t^32 - 2849089214228109766528370419644715817910127905700672505451049651871/375480480003275420932551612436775220429042556600320000000000*t^31 - 92105221010161696953710797701553532897182417260009948450112657339/79048522105952720196326655249847414827166854021120000000000*t^30 + 5903824944664169458839393164937003454841619642612434644953013621/41720053333697268992505734715197246714338061844480000000000*t^29 + 3188907297625548014863809104370585058386813391578017904740924359/300384384002620336746041289949420176343234045280256000000000*t^28 - 214324020941541408328417895101355130003068189943230433740435639/187740240001637710466275806218387610214521278300160000000000*t^27 + 19434337661866138698522234227297315971833976631015246034962239/83440106667394537985011469430394493428676123688960000000000*t^26 + 105633240279655580400212442979092093442445620040001404305159/7509609600065508418651032248735504408580851132006400000000*t^25 - 120842781922579918685099204105711542949519812332472425084257/50064064000436722791006881658236696057205674213376000000000*t^24 + 1357521416967859815783814726186270149586851179652177991/469350600004094276165689515545969025536303195750400000000*t^23 - 51439579336121150565275886013814245856206761540485790049/3129004000027295174437930103639793503575354638336000000000*t^22 - 1721118395646605244975064597308330405415173045517811183/469350600004094276165689515545969025536303195750400000000*t^21 + 267363540473682777861305327040066961621434740438007097/3129004000027295174437930103639793503575354638336000000000*t^20 + 146651688998320037442827376568310273842959514921261/3644026397547315808739825431257523490188689408000000000*t^19 + 126758665787573405632593817954464246638171243767903/52150066667121586240632168393996558392922577305600000000*t^18 - 1642812838653482577065903096750414326130673230189/11733765000102356904142237888649225638407579893760000000*t^17 - 930160274464792538024140657459262674518227443091/46935060000409427616568951554596902553630319575040000000*t^16 + 81693182697798753874056726736880707534442813/1466720625012794613017779736081153204800947486720000000*t^15 + 34088834928602417068601611976895308025668069/838126071435882636010159849189230402743398563840000000*t^14 + 1221522629675134495508013743534253372671/41906303571794131800507992459461520137169928192000000*t^13 - 407756230237921203700243972020426919909/8822379699325080379054314201991898976246300672000000*t^12 + 10732289920004244121192893784936757/209531517858970659002539962297307600685849640960000*t^11 + 128746405985075962855194677604319/4186443913266147033017781464481670343373619200000*t^10 + 17251523897392101477324793/1661287267169105965483246612889551723560960000*t^9 - 7266910042575536919287/1150675163407172963105279039230858336665600*t^8 + 754621316387/416187486764747165474999652499587072*t^7)*Dt^11 + (19265930/3*t^61 - 3430418651350265/2873355408*t^60 + 134355915603647629292710494043799/1148168966722107683078880*t^59 + 27117768485442062381297982480233353/859748922281514233089465344*t^58 + 129281431228934716111028390788175909807617/257924676684454269926839603200000*t^57 + 2146279231664156133569060863668238218621971/8253589653902536637658867302400000*t^56 - 59904931772106218643214719580396413964699720817/8666269136597663469541810667520000000*t^55 + 5298024696153398583529816744208571068006145064427/24958855113401270792280414722457600000000*t^54 + 440316069533916206752220530390598325187520345169931501/19467906988452991217978723483516928000000000*t^53 - 90908976342084484811244089679989461901762121343217441/31148651181524785948765957573627084800000000*t^52 + 48479790120059000974324460313920856239366500027109153231/2012682076344678476689492643218980864000000000*t^51 + 1298791049744321580540546833992295032091155584266283737023/115125414766915608866638979192125705420800000000*t^50 - 544032142322699987855626058479328689297563716304653539668311/1973578538861410437713811071865012092928000000000*t^49 + 132932514789097063110352378113075460807911281579258105630071/2631438051815213916951748095820016123904000000000*t^48 + 14308932743530131381934675543774024094225924981083036031850551/18946353973069540202052586289904116092108800000000*t^47 - 1058623246496147359421694529765065929055014540427269595208866901/10609958224918942513149448322346305011580928000000000*t^46 - 18783475495205451789197289132185600156995200481681116624082921949/14146610966558590017532597763128406682107904000000000*t^45 - 7615238577107579570512730872220163029244945205365217420996928231/56586443866234360070130391052513626728431616000000000*t^44 + 1050349214291348832229745677454911029573081373033970930448282579937/679037326394812320841564692630163520741179392000000000*t^43 + 10273586775430755204605835754643910119180387892033786224747663/29762758115047658156544584380020316490956800000000*t^42 - 26196586863227633729055414794540344519997497829735263597644701045413/24445343750213243550296328934685886746682458112000000000*t^41 - 3293675993490818380305840054578879127041895065435765411493360339629/15645020000136475872189650518198967517876773191680000000*t^40 + 715868554240923207825555999486563616370334591757365085529791223969477/1564502000013647587218965051819896751787677319168000000000*t^39 + 32087572771261268584137266116270262053031157288489009570816468581503/6258008000054590348875860207279587007150709276672000000000*t^38 - 491278180608793272845222151706500909918250672681651431340387757715913/3576004571459765913643348689874049718371833872384000000000*t^37 + 5280649331311987470257400634743900134105545855513113994183411325849/132444613757769107912716618143483322902660513792000000000*t^36 + 3044788172309457632329422578371042029346055184040369567442731785816587/100128128000873445582013763316473392114411348426752000000000*t^35 - 58136440389417343971569608096815601586948944137509165036414415741361/4172005333369726899250573471519724671433806184448000000000*t^34 - 39452277207103152053184136960445604246261865364564615171869670790089/10012812800087344558201376331647339211441134842675200000000*t^33 + 106294259384071642812191463363814821483570699382457416593359969654671/75096096000655084186510322487355044085808511320064000000000*t^32 + 16345734215131505963497898159835551321616365525055936744805500780157/60076876800524067349208257989884035268646809056051200000000*t^31 - 4104217450384908890873764765604412498889499270160013021627053979681/75096096000655084186510322487355044085808511320064000000000*t^30 - 44100678489946493777267744805035724460639500691641525376804345241/5461534254593097031746205271807639569876982641459200000000*t^29 + 3753383691677947419048227468182526036593926015323053564334231573/5006406400043672279100688165823669605720567421337600000000*t^28 + 1725161782230671975802230345350169062588936131707437404768361367/100128128000873445582013763316473392114411348426752000000000*t^27 - 40055785551779869412126820780531259971544072812139447556353669/7509609600065508418651032248735504408580851132006400000000*t^26 + 27508623541612621285945660200691743055637227857058268855323243/13653835636482742579365513179519098924692456603648000000000*t^25 + 1269869078247634383474853818145716934942832380285205134639/15809704421190544039265331049969482965433370804224000000*t^24 - 175821941466693299671453296125648686132252536824506784301109/30038438400262033674604128994942017634323404528025600000000*t^23 + 131091615349854542336665171329967248950806187778980375183/156450200001364758721896505181989675178767731916800000000*t^22 - 100290375878649687068356895842148821644920824218521004943/625800800005459034887586020727958700715070927667200000000*t^21 - 570023189456993123364635124785799935145536171005146901/18774024000163771046627580621838761021452127830016000000*t^20 - 694026404578433770756381296036000971315861613317367539/1877402400016377104662758062183876102145212783001600000000*t^19 + 26454061757525829559757569418067695009702329958140651/117337650001023569041422378886492256384075798937600000000*t^18 + 76392484060060975438899643317751120468750871561303/4081309565252993705788604483008426309011332136960000000*t^17 - 42894908733462528763235370707258399344641220379/71113727273347617600862047809995306899439878144000000*t^16 - 337930335975233736360399630993418528824729698263/3129004000027295174437930103639793503575354638336000000*t^15 + 52936823154767114903351565467039507483894179/293344125002558922603555947216230640960189497344000000*t^14 + 37360216159986449947361535325596748637048699/167625214287176527202031969837846080548679712768000000*t^13 + 1492932917368709122472104670264381701753/8381260714358826360101598491892304027433985638400000*t^12 - 66964397221099246508755730792575001701/252067991409288010830123262914054256464180019200000*t^11 + 41088802024641074687286479205791/147039661655418006317571903366531649604105011200*t^10 + 7352596905451024791967727678693/39870894412058543171597918709349241365463040000*t^9 + 66601507498560329729387053/996772360301463579289947967733731034136576000*t^8 - 45884424294912633693829/1150675163407172963105279039230858336665600*t^7 + 10985841684247/936421845220681122318749218124070912*t^6)*Dt^10 + (63382480/3*t^60 - 198604806449105/51309918*t^59 + 3861599036965547322729107007271/9763341553759419073800*t^58 + 43621404857992522040049377218024147/383816483161390282629225600*t^57 + 99741850335014074183349448105697333187/59815555817359524565593600000*t^56 + 416065040207498781553780172929162106904037/442156588601921605588867891200000*t^55 - 9122946424500696248880032269505214514198259677/357126475409244373744854835200000000*t^54 + 18066293362625180016972493619318222335312409227/40517621937339725312143530393600000000*t^53 + 7682096652921460801436497133113691856203447239554337/86910299055593710794547872694272000000000*t^52 - 236248290040173628177829075635132129184272412449499233/19467906988452991217978723483516928000000000*t^51 + 80930453521176992515043907871167186864028666640098901413/934459535445743578462978727208812544000000000*t^50 + 1086230844588961300066646919992854581181800213146650344053/23494982605492981401354893712678715392000000000*t^49 - 907314233329402233907580892371827822890868715928765145748173/789431415544564175085524428746004837171200000000*t^48 + 118135804135676029611987237982403385978159152367545076336749/584764011514491981544832910182225805312000000000*t^47 + 882300057934980841215351539991683302843091561210837368620527621/265248955622973562828736208058657625289523200000000*t^46 - 221708884687517766884967235193301373069794905232333573092186189/482270828405406477870429469197559318708224000000000*t^45 - 129356498214579795782825609472250323647240741271339774202472566289/21219916449837885026298896644692610023161856000000000*t^44 - 104533616648475056529600989801141510901401553532198588243365093287/169759331598703080210391173157540880185294848000000000*t^43 + 1675824871876888384181694439309399622060419476897672608263195907749/226345775464937440280521564210054506913726464000000000*t^42 + 972873256837982772196616681217744330828035226331815829244309542789/582031994052696275007055450825854446349582336000000000*t^41 - 129193750713023075858133864038379559760343644955385955462289735776879/24445343750213243550296328934685886746682458112000000000*t^40 - 104499994024447241259508552478741497011808312829956592199196894427/100624003088091560793604647016973035232034816000000000*t^39 + 514357424661262048096501429238508423337518005792280621657064795328147/223500285716235369602709293117128107398239617024000000000*t^38 + 21129524692446335477186710750875184694856414272121464940130884779581/3129004000027295174437930103639793503575354638336000000000*t^37 - 800323464050938931703668437130217245303780815636387512576046591475191/1137819636373561881613792764959924910391038050304000000000*t^36 + 9264021425216270484617845164298683462693739565691614857535926978747/42571482993568641829087484403262496647283736576000000000*t^35 + 305232513779633679437531189092282761110797059320571054818608278325403/1925540923093720107346418525316796002200218238976000000000*t^34 - 29250899622981087998429602673503286224677814443973058196405718493549/391125500003411896804741262954974187946919329792000000000*t^33 - 10586160484400496188002737594215769041142455621852944111056433178537/521500666671215862406321683939965583929225773056000000000*t^32 + 132565521902567481965406658394923688731355258158402337233907699788433/18774024000163771046627580621838761021452127830016000000000*t^31 + 99529377002272606561381008329987957749363176858813548045665915079023/75096096000655084186510322487355044085808511320064000000000*t^30 - 4268487592410804572864533733184633241923717282950496818585056799639/18774024000163771046627580621838761021452127830016000000000*t^29 - 19722280101147907548898703414491489479430882571707349390288298001/620628892567397389971159689978140860213293481984000000000*t^28 + 78627950412958925529001078288180512995279458490346705622642447/42571482993568641829087484403262496647283736576000000000*t^27 - 155947464195328219288277560979749964633219868780504396751843/669753364554337428642232530544972522504423735296000000*t^26 - 9244300213748945084824791646522153585557747565901102444696953/494053263162204501227041595311546342669792837632000000000*t^25 + 6143733516503195142818326749934435597636955421612220436889547/766286693884235552923574719258724939651107258368000000000*t^24 + 63053847766467765955377032681105734911269343170243767185769/375480480003275420932551612436775220429042556600320000000*t^23 + 9462595836719988380722603404529239916971607349003595298657/395242610529763600981633276249237074135834270105600000000*t^22 + 35224792561797243796144532947058574585887144635267769/4524297281705169425156058565124050757049384960000000*t^21 - 104833390859056178923108056081386112335470544273116323373/156450200001364758721896505181989675178767731916800000000*t^20 - 3307499103430451273209127371228281878913365241806990713/23467530000204713808284475777298451276815159787520000000*t^19 - 2622284324438908904507861672182868705656523425869305251/469350600004094276165689515545969025536303195750400000000*t^18 + 267338151939647223384900672563327937361736330429707/380966396107219380004618113267832001246999347200000000*t^17 + 1976710602644887169353524475585747962135793970514557/23467530000204713808284475777298451276815159787520000000*t^16 - 36667125177861789833634702034027564203575360861/39112550000341189680474126295497418794691932979200000*t^15 - 245653177426484349115795902904196182116584365679/782251000006823793609482525909948375893838659584000000*t^14 - 1969178581130733619718733020817880012481701/10476575892948532950126998114865380034292482048000000*t^13 + 26257243796390176240216700089595619573531781/41906303571794131800507992459461520137169928192000000*t^12 + 1156561432350789346918622146784316476321/2095315178589706590025399622973076006858496409600000*t^11 - 511440329457874644291028410565660672037/644712362642986643084738345530177232879537356800000*t^10 + 1895281688110394354515873345450439/2494422831654412607173094789253661912926781440000*t^9 + 1155889827126129391059313908317/1993544720602927158579895935467462068273152000*t^8 + 8154679257089460333430259/35599012867909413546069570276204679790592000*t^7 - 12911995467465695315263/95889596950597746925439919935904861388800*t^6 + 153373800604507/3745687380882724489274996872496283648*t^5)*Dt^9 + (20969200*t^59 - 63100752857855/17103306*t^58 + 81095644219009379647726424067383/205030172628947800549800*t^57 + 211856576957208170799954397953023/1523081282386469375512800*t^56 + 652364285050910110427527599764462850143/511755310881853710172300800000*t^55 + 457412653519620987918152464812370842504749/442156588601921605588867891200000*t^54 - 156971379983970645218452221983392712919893616977/4642644180320176858683112857600000000*t^53 - 196823669548449403603149575827258360532761750709/148564613770245659477859611443200000000*t^52 + 1273582293490739245546475238367635621261069641713493/9012919902061570008323483094220800000000*t^51 - 11881172841668897460883839196913495372980403805104011/564287159085593948347209376333824000000000*t^50 + 1863623900745999811139966913854201076927507066419000527/34609612423916428831962175081807872000000000*t^49 + 238216902331678753787499137323030372848620822136721345703/3045645893304645737212671407199092736000000000*t^48 - 2562247976396313240000607946041338934233121908186983013502261/1315719025907606958475874047910008061952000000000*t^47 + 1400325440128742618282987443148496410780609292611521065068017/5262876103630427833903496191640032247808000000000*t^46 + 573413212233874540681805248619853958148846614419729779462541163/88416318540991187609578736019552541763174400000000*t^45 - 11797218997029089569996952286051992691161601750156653301891377/10983393607576545044668165965161806430208000000000*t^44 - 274546032116957944811633246000608941885376892992456069685498523419/21219916449837885026298896644692610023161856000000000*t^43 - 95275933610468032349935818785244565838190078586040276635520254171/84879665799351540105195586578770440092647424000000000*t^42 + 43906919988790121844923208372017918518426994075262434523205545707/2552771903739895943013401100113396694515712000000000*t^41 + 1977458497530100539523960809522431089866907887732901263194055440831/509277994796109240631173519472622640555884544000000000*t^40 - 71026061359245798422792289585881713203340264695918251359345304142571/5432298611158498566732517541041308165929435136000000000*t^39 - 7688488874106626062775581087140236529023227490813873540374745566913/2963071969722817400035918658749804454143328256000000000*t^38 + 14046073455235523614791505675581294462553419143496406708177880511281/2370457575778253920028734926999843563314662604800000000*t^37 - 265716515195739701897607052493885917610288901075480262108786121623/29800038095498049280361239082283747653098615603200000000*t^36 - 867671501819275921329724261706727589408954864626033237620601762043619/463556148152191877694508163502191630159311798272000000000*t^35 + 1275720869277531725579093713339030151678355535566987387947459470606279/2086002666684863449625286735759862335716903092224000000000*t^34 + 6940941687557394818098027042783756033700024193942811905345976208029/15893353650932292949525994177217998748319261655040000000*t^33 - 1258124962541399420043283559063000761098937991715285357330593075407/5926143939445634800071837317499608908286656512000000000*t^32 - 14628583163186846514998409695590409139663351959024558617348056264191/260750333335607931203160841969982791964612886528000000000*t^31 + 39621594108765488335876775523485908956853094954947333300549707495709/2086002666684863449625286735759862335716903092224000000000*t^30 + 5843232129936570596356791857336652643933638507363176117784826029511/1668802133347890759700229388607889868573522473779200000000*t^29 - 1036157422089298055598359876280857309009724844954890847699910151227/2086002666684863449625286735759862335716903092224000000000*t^28 - 75050028790500642053989514489613514356765344880266810715788273843/1192001523819921971214449563291349906123944624128000000000*t^27 + 45694759431525342872390397335901967146272117739732268010491847/160461743591143342278868210443066333516684853248000000000*t^26 - 1341470889023957619377090955672271405160071993140435148423115779/927112296304383755389016327004383260318623596544000000000*t^25 - 52000903059638721471942158609165105667039679632078018213348911/1043001333342431724812643367879931167858451546112000000000*t^24 + 2412187497437942882252157800016609915977375501456334635608469/219579228072090889434240709027353930075463483392000000000*t^23 - 233798990378410189154581623441688463310567651543074281201/608164042765266311844106920046607094961196236800000000*t^22 + 149501665875086630990931636692605494728778426823995327143591/834401066673945379850114694303944934286761236889600000000*t^21 + 413413283321670640229492460720876736370530662233212657211/13037516666780396560158042098499139598230644326400000000*t^20 - 5047168017123008555078767099571104960354203068608817053/7450009523874512320090309770570936913274653900800000000*t^19 - 19242611035001559133494431979405413682859433342267849/57944518519023984711813520437773953769913974784000000*t^18 - 1116120337910354078296888054609809804069618825098128343/52150066667121586240632168393996558392922577305600000000*t^17 + 1266080944035790624889454510504301771767168038614129/1086459722231699713346503508208261633185887027200000000*t^16 + 535714938844847909782900820858104392026620052475913/2607503333356079312031608419699827919646128865280000000*t^15 + 4754593970354349706450368929366930281781979487/9312511904843140400112887213213671141593317376000000*t^14 - 2575263222791878758530093626190898254913048233/5321435374196080228635935550407812080910467072000000*t^13 - 197461809897333425321934552235444853308469/89543383700414811539546992434746837899935744000000*t^12 + 31355807797320140190847584896115591632153/38481454152244381818648294269477979923939328000000*t^11 + 48667781220527758583745352257986113439/77604265873692836667607393443447259513277644800000*t^10 - 1000577195536708443574138197582174142237/931251190484314040011288721321367114159331737600000*t^9 + 652165781144005244994424718035009/831474277218137535724364929751220637642260480000*t^8 + 18888262243047340819412616922513/23258021740367483516765452580453724129853440000*t^7 + 1391754686477569940698727/3955445874212157060674396697356075532288000*t^6 - 3989659635606756111647/19177919390119549385087983987180972277760*t^5 + 82960383350495/1248562460294241496424998957498761216*t^4)*Dt^8 + (-47657120*t^58 + 75024509793140/8551653*t^57 - 760382078576411768013789651859/759371009736843705740*t^56 - 12056618497042078263414040181252783/51175531088185371017230080*t^55 - 51414987236942424070808602369574131387/8435527102448138079763200000*t^54 - 3193810892700023708560563093016232909459/1169726424872808480393830400000*t^53 + 11475405844882128723731824462249828423790715719/257924676684454269926839603200000000*t^52 - 594036255640402977789444676641929505402568511743/74282306885122829738929805721600000000*t^51 - 27659570015473338118622421246332985818441873302685801/405581395592770650374556739239936000000000*t^50 + 155714608585113150535879732253354567048721536451147859/12978604658968660811985815655677952000000000*t^49 - 8769004007468147548939971391198724428955275904944792033/17304806211958214415981087540903936000000000*t^48 - 404271085409109391998579466984787216096980004687078260087/10964325215896724653965617065916733849600000000*t^47 + 139812627232475907760186831331511376486741675150435639741051/109643252158967246539656170659167338496000000000*t^46 - 542340636091868552769968983430526671037757332015844648082723/877146017271737972317249365273338707968000000000*t^45 - 630572702828215552553087929655819565232783949128190607443419/775581741587641996575252070346952120729600000000*t^44 - 50390797271828008740993268170212688450220927005611110395504481/58944212360660791739719157346368361175449600000000*t^43 - 64039036608295289951788406359702973804383924077449026664262331/35366527416396475043831494407821016705269760000000*t^42 + 11813057020037337823918057556956897199708497615926081926999185367/11317288773246872014026078210502725345686323200000000*t^41 + 1789086352768336298855912302876293827524694122776308069670655367533/226345775464937440280521564210054506913726464000000000*t^40 + 2280966169137920528253696614454214187482682485055926273975042404451/1358074652789624641683129385260327041482358784000000000*t^39 - 47730247108091691200084212536397971667573813253019727357399449157847/5432298611158498566732517541041308165929435136000000000*t^38 - 90406175962201039240444546591625752884210880360390090011685003764203/43458388889267988533860140328330465327435481088000000000*t^37 + 163185443734784527644619965976825858893002597171217864556795987793743/34766711111414390827088112262664372261948384870400000000*t^36 + 9218837242317791123884661796974278229008295117787024008557060387609/57944518519023984711813520437773953769913974784000000000*t^35 - 180887543984414496758800858705234149956959253312016635671376325290843/109789614036045444717120354513676965037731741696000000000*t^34 + 500828539111471017527663190379575607030662300917431515570103693391/903031457439334826677613305523749928881776230400000000*t^33 + 23245042735666855368866111074726762292894183058694831667417216093983/54181887446360089600656798331424995732906573824000000000*t^32 - 5741099292720472439533049200979914552665505307848332210346659524659/26075033333560793120316084196998279196461288652800000000*t^31 - 30187157870575073720729250667657388530370451761633146234851172138817/521500666671215862406321683939965583929225773056000000000*t^30 + 6774331273273658620788076567131912123235527636482392748983076046581/347667111114143908270881122626643722619483848704000000000*t^29 + 5427662466678815077702984559633107827039039207328235642761683541/1535519077427209017022662300890586923604639744000000000*t^28 - 398394701617427965491535959636212854337838903678630502738427867953/1043001333342431724812643367879931167858451546112000000000*t^27 - 49752453597704674113955003807591138663939318946468334920079439717/1390668444456575633083524490506574890477935394816000000000*t^26 - 143726820796169286140174588826663121144101869697518862431937979/23177807407609593884725408175109581507965589913600000000*t^25 - 1015253699161292424823575813533813578365579864389597158329946819/379273212124520627204597588319974970130346016768000000000*t^24 - 6330772263483687264014294653614328560457101603547054438517/238673073991403140689392075029732532690721177600000000*t^23 - 11359585017604266583682116947463281129499756792247931045194627/2086002666684863449625286735759862335716903092224000000000*t^22 - 78126829181837643941178333833941163207251437447516650802823/34766711111414390827088112262664372261948384870400000000*t^21 + 54546932236510375981187848398218604045291334541773917888007/139066844445657563308352449050657489047793539481600000000*t^20 + 388265644762295704549208240165852414859308544055565929/6512246087302895384694326722527042756358963200000000*t^19 + 38174393370697998103402308073354779233109571997450483/19102588522755159795103358386079325418652958720000000*t^18 - 132789758021071917364414618335543744692313191198804597/434583888892679885338601403283304653274354810880000000*t^17 - 289794123649662960595950967935380232547457592147939959/8691677777853597706772028065666093065487096217600000000*t^16 + 391555965426370010469329888563518269541790056135571/543229861115849856673251754104130816592943513600000000*t^15 + 281060327356695330938579167822789052002461926766971/1303751666678039656015804209849913959823064432640000000*t^14 + 1975459456081336008459925443565220822262099967/1552085317473856733352147868868945190265552896000000*t^13 - 768083470496107786077228534540334118321111639/1693183982698752800020524947857031116653330432000000*t^12 - 529643445482766069709297003161667979654081/194010664684232091669018483608618148783194112000000*t^11 + 439712934872434685161701595533230133663809/776042658736928366676073934434472595132776448000000*t^10 - 42444064149514912232043204216037716937/116406398810539255001411090165170889269916467200000*t^9 - 88456168489617601229004399825505453181/155208531747385673335214786886894519026555289600000*t^8 - 8769910689436414451208825996697/83147427721813753572436492975122063764226048000*t^7 + 3351648928937182334457546587027/11629010870183741758382726290226862064926720000*t^6 + 544596319243899906648703/5933168811318235591011595046034113298432000*t^5 - 6085121441666743759/83382258217911084282991234726873792512*t^4 + 680218507255/26011717922796697842187478281224192*t^3)*Dt^7 + (-104322400*t^57 + 155778103811350/8551653*t^56 - 427617115998458110947950585831/189842752434210926435*t^55 - 33517511998890488256074560733115133/51175531088185371017230080*t^54 - 1893146268129017167958448057960900414787/153526593264556113051690240000*t^53 - 34263358972381888582243906630904991636769/4912850984465795617654087680000*t^52 + 7047971866192358574815001707581120753276381589/51584935336890853985367920640000000*t^51 - 171552610306263138835206869500357320211248204241/14856461377024565947785961144320000000*t^50 - 21598659924847984359587383491441481175959730896611859/57940199370395807196365248462848000000000*t^49 + 138532762990888356381878737170242488809490852383459089/1854086379852665830283687950811136000000000*t^48 - 1219371779859365977693163753623122023418348154019187073/1059477931344380474447821686177792000000000*t^47 - 1766307117817766138859808323528580076436769393074602617573/7831660868497660467118297904226238464000000000*t^46 + 4779918172546591278747820033822250977709363808162726011/686987795482250918168271745984757760000000*t^45 - 89372977608846649010381052957601745116088176232396684294831/46165579856407261700907861330175721472000000000*t^44 - 36896478072128925492908988067304084574281953780843743559491337/2105150441452171133561398476656012899123200000000*t^43 + 313979468577577420805417708522895878281747040567089242600611/647738597369898810326584146663388584345600000000*t^42 + 7549243643976568920482299629427191964815844063396204718109721867/252618052974260536027367817198721547894784000000000*t^41 + 45700572845917299402137113560145263754364674351351611310174890709/8083777695176337152875770150359089532633088000000000*t^40 - 950218825986005936420108116278694812617595679110283867803053148821/32335110780705348611503080601436358130532352000000000*t^39 - 79310820410503343087425365214590309345080355920735859085969711/11972271810196364805246435273595689526886400000000*t^38 + 13341644558949903707905583820342477927917210913935214586415572199027/776042658736928366676073934434472595132776448000000000*t^37 + 524404007131170207227668007203599618663069549246660941764134500189/269927881299801171017764846759816554828791808000000000*t^36 - 166364224207203430218210854640083439327442627271697260328526174147643/24833365079581707733634365901903123044248846336000000000*t^35 + 119825471777481166089153810402068671420836140586976637941151491559/131393466029532845151504581491550915578036224000000000*t^34 + 7607536291106564878629838163550184641641681872838897065495754317597/3973338412733073237381498544304499687079815413760000000*t^33 - 36966445955100563089981797315609936725467481019819200413382335099361/49666730159163415467268731803806246088497692672000000000*t^32 - 75044198184146747799344005296777328098903245231103775607823158558687/198666920636653661869074927215224984353990770688000000000*t^31 + 46039767662772069790155980294632012231428048609906961143614274499/269927881299801171017764846759816554828791808000000000*t^30 + 70778761671829205133311268853693206937792698820195133983361694997/1655557671972113848908957726793541536283256422400000000*t^29 - 1966097299432110635099001951745408914621339496901989886429261708561/149000190477490246401806195411418738265493078016000000000*t^28 - 1465385347171423166927356084024027692623379017642768115326279090671/596000761909960985607224781645674953061972312064000000000*t^27 + 358678931982914085582806186287742274986536531117140099211694593/1041959373968463261551092275604326841017434112000000000*t^26 + 1071169964521307086018166909609237840044278274632504255172015427/22074102292961517985452769690580553817110085632000000000*t^25 - 455264477522170427707995775826579159966727957968242513900219871/149000190477490246401806195411418738265493078016000000000*t^24 + 1073584775495520658509028143847183186359258798588823141238213/2167275497854403584026271933256999829316262952960000000*t^23 + 10582739519243927011679825593439670325683879669049812945075971/74500095238745123200903097705709369132746539008000000000*t^22 - 2226518842048994621148328140242577309130522686906623683441699/298000380954980492803612390822837476530986156032000000000*t^21 - 79359003219277079205931549578048316818390930412425934133/38701348175971492571897713093874996952076124160000000*t^20 + 2397700498785590952729892475326779653392833348918068949863/19866692063665366186907492721522498435399077068800000000*t^19 + 858847601749773906792141360098782347899895378030121049/37250047619372561600451548852854684566373269504000000*t^18 + 439018092286543021024855702483416609689396879031274571/286538827841327396926550375791189881279794380800000000*t^17 - 34891909390159910812648886707459230705509042785187/1591882376896263316258613198839943784887746560000000*t^16 - 98631564660458726210899658142864926301619399114217/5847731180435252998501028077371222066934579200000000*t^15 - 179805692353182482336871865225287084029517596857799/232812797621078510002822180330341778539832934400000000*t^14 - 660844884455571731680275122851833869890387314391/62083412698954269334085914754757807610622115840000000*t^13 + 317527685997822871960244088822421380798249931/4656255952421570200056443606606835570796658688000000*t^12 - 2295774233285996284185481683932003182780869131/18625023809686280800225774426427342283186634752000000*t^11 + 596391288455152538051510942751501830150603/582031994052696275007055450825854446349582336000000*t^10 + 1013519072044791069750883539071806151675573/2328127976210785100028221803303417785398329344000000*t^9 - 105381669782883760331677937011090488863/116406398810539255001411090165170889269916467200000*t^8 - 1716480824611160479129447325363865643/4703288840829868888945902632936197546259251200000*t^7 - 2726895663024468086464558337773/7558857065619432142948772088647460342202368000*t^6 - 448355559330754445600058236027/11629010870183741758382726290226862064926720000*t^5 - 1013218913338010782850023/5933168811318235591011595046034113298432000*t^4 + 74923317269777351117/1065439966117752743615999110398942904320*t^3 - 3856517802935/156070307536780187053124869687345152*t^2)*Dt^6 + (-26073600*t^56 + 612123832400/150029*t^55 - 4150429933782219384037675929509/6834339087631593351660*t^54 - 1803465580029668566203031275894617/8529255181364228502871680*t^53 - 41963625685884037445390543089360270321/12793882772046342754307520000*t^52 - 8668114461506040083013218629137061343159/3684638238349346713240565760000*t^51 + 862194743193912372548348403968306168895828223/19344350751334070244512970240000000*t^50 - 881285309175967732862075396023875112209497687/309509612021345123912207523840000000*t^49 - 23744275624906435420326357680972951199278838336173/175576361728472143019288631705600000000*t^48 + 69519216328555543586863709090813276030108016201167/1872814525103702858872412071526400000000*t^47 - 163121679737644729158457569698452642133392881833149013/370817275970533166056737590162227200000000*t^46 - 876400183999255120183041055261203561282604826248573987/9668717121602049959405306054600294400000000*t^45 + 10219456798724405414124631744810240748447665983737586501827/3132664347399064186847319161690495385600000000*t^44 - 144866507872869332088665691343407980738150308528350131819/171317581498386322718212766654948966400000000*t^43 - 649093748481519040245262497749450352847623161933625090727189/65785951295380347923793702395500403097600000000*t^42 + 191090608251379268013065945681734768706282824046844013259797/526287610363042783390349619164003224780800000000*t^41 + 980160810366740286445601463540411112498071291156378836186900623/50523610594852107205473563439744309578956800000000*t^40 + 2595548227531334637095393030000979672239083670013632955945670667/808377769517633715287577015035908953263308800000000*t^39 - 298440293009738336554784121445553164915253211164548177070358973/13091137967896902271863595385196906125721600000000*t^38 - 4710944936759932413102575727957891420125609247535255774784726203/1021108761495958377205360440045358677806284800000000*t^37 + 1681803287516125333615021605856097545214646463185021844481425987/104729103743175218174908763081575249005772800000000*t^36 + 1001244952136689630933311228811750844136237293248445071751157081889/620834126989542693340859147547578076106221158400000000*t^35 - 330955029497565593463381182344786279588284861097658497532614093991/45151572871966741333880665276187496444088811520000000*t^34 + 630324024107557145597263936810904199390902624017303116137302882603/620834126989542693340859147547578076106221158400000000*t^33 + 5998900638665587074295673019497895083232636133459966204954387531061/2483336507958170773363436590190312304424884633600000000*t^32 - 1247128714965774322436637258997949742788508034374164634278442151193/1241668253979085386681718295095156152212442316800000000*t^31 - 8268379949201054623772059045981034849799640883537669078812594673/14480096256315864567716831429681121308599910400000000*t^30 + 2817573787470818926702260835716176866487249845227941836514224843/9854509952214963386362843611866318668352716800000000*t^29 + 25935521131298719442893209229278472212212526007748570994484508509/354762358279738681909062370027187472060697804800000000*t^28 - 84003560258530251148293827287855877013691818454102138640210009/3537516393102807369463584886310986188639436800000000*t^27 - 7019257024174635257487471198825543393858223662598859321655029971/1655557671972113848908957726793541536283256422400000000*t^26 + 101804840786572327699271094522946359878525180525769217466690323/248333650795817077336343659019031230442488463360000000*t^25 + 6952598205144092372617243974820141064545821701100460886355077/382051770455103195902067167721586508373059174400000000*t^24 + 12953200270099369550905679345108232391889564296017871472433013/1241668253979085386681718295095156152212442316800000000*t^23 + 20725990178686717552990184928932590710227110650890078703523499/4966673015916341546726873180380624608849769267200000000*t^22 + 7039816631242685187755746026210402035136899403461797751549/206944708996514231113619715849192692035407052800000000*t^21 + 60666118807329169139360546387735261487121718408780737688969/2483336507958170773363436590190312304424884633600000000*t^20 + 463776921882063668464514550822607731075593053879604485307/124166825397908538668171829509515615221244231680000000*t^19 - 343476372230512188013965844671687958862736944072868229531/496667301591634154672687318038062460884976926720000000*t^18 - 648928285195457045062965691331056076322467059250183851/7760426587369283666760739344344725951327764480000000*t^17 - 2177477906892134085957875746561165513926433356019061/295635298566448901590885308355989560050581504000000*t^16 - 17243562270553727271290510190607915157457695542279/1552085317473856733352147868868945190265552896000000*t^15 + 46853098333809195861222460872531161944997911083373/2069447089965142311136197158491926920354070528000000*t^14 - 51145013814178360057247746710411261250063664581/129340443122821394446012322405745432522129408000000*t^13 - 3223135953264649776767458169519215741842534401/23878235653443949743879197982599156773316198400000*t^12 + 41369655232222161261144931629395911137479771/38802132936846418333803696721723629756638822400000*t^11 + 550323570276755080807465839852047780793943/1567762946943289629648634210978732515419750400000*t^10 + 40026165955962920720111714276554765051/230965077005038204367879147153116843789516800000*t^9 - 116198099070598691263028098327597332541/554316184812091690482909953167480425094840320000*t^8 - 4411449420725989497008057450802359/19797006600431846088675355470267158039101440000*t^7 - 155267619759788966579540715814697/2399637163688708616809133996396019156254720000*t^6 - 97712600590049327470857419/1363430206641311714096098861588647247872000*t^5 + 30771440581155821879348837/512742983694168507865199571879491272704000*t^4 - 8056537178271748411/422590371176512506482307339461119180800*t^3 + 3184888795459921529/399539987294157278855999666399603589120*t^2 - 224793240595/52023435845593395684374956562448384*t)*Dt^5 + (23923200*t^55 - 10858151390200/2850551*t^54 + 3782495937123586594232866078381/6834339087631593351660*t^53 + 1612731576577111550892488375805253/8529255181364228502871680*t^52 + 3479641089244682136590003077216365323/1163080252004212977664320000*t^51 + 7685560280318572283160550266530722821367/3684638238349346713240565760000*t^50 - 77079570083895243676473034029753395162891303/1934435075133407024451297024000000*t^49 + 2562083955345225488322567024473710460073983/1018123723754424749711208960000000*t^48 + 77902650827042469890098526063919701923263511269163/643779993004397857737391649587200000000*t^47 - 84427984439411286549973986663075917826193130807227/2687081709931399754034330363494400000000*t^46 + 5125873133123304523465201292298695812847967904326317/13733973184093820965064355191193600000000*t^45 + 107645970563850339363216679304296615247448590181073921/1338745447606437686686888530636963840000000*t^44 - 409510097427044391042366217059606040126250826645267438341/149174492733288770802253293413833113600000000*t^43 + 23909588703240785445583939814989557581518616779980537703/33225227926959771678683688078535557120000000*t^42 + 88666519044610945104460206697559564417068405771389193751243/10964325215896724653965617065916733849600000000*t^41 - 62563554897726201634513694229849311196814914158881509554293/175429203454347594463449873054667741593600000000*t^40 - 1037270201721607985136696102674735866703205939475150042941451/66566021864100272997988884637344281395200000000*t^39 - 30361490087826757112306044084027786812015984311276991616917373/11715619848081648047646043696172593525555200000000*t^38 + 719603120808506662976315325961840323966242730182877159307470161/39919889852722652606793926668439948309299200000000*t^37 + 24773507431964262346814969043708825536971997911967929608861314161/6467022156141069722300616120287271626106470400000000*t^36 - 108024612742392934533734833698137958860349095179748101000494668309/8622696208188092963067488160383028834808627200000000*t^35 - 28231845187520062477034213795032707673882511217169385875190969843/18813155363319475555783610531744790185037004800000000*t^34 + 4681492511465647675005012483304226595373502746067837168583333357617/827778835986056924454478863396770768141628211200000000*t^33 - 128864550827440909577052021884722572320512693924282051533178862299/206944708996514231113619715849192692035407052800000000*t^32 - 1529896767093084091046490921232883061726459774201266773501467482117/827778835986056924454478863396770768141628211200000000*t^31 + 26780507185772983452744359642864434106690336709016376321391287053/37626310726638951111567221063489580370074009600000000*t^30 + 724017014423505907835608278944395262324145567574943384692587437297/1655557671972113848908957726793541536283256422400000000*t^29 - 3369779081604033415656215243737334092708793041114651619269504823/15918823768962633162586131988399437848877465600000000*t^28 - 49172780171311864745953736132492959518997518962382173241955306229/827778835986056924454478863396770768141628211200000000*t^27 + 67642091121294023594149617433310444300124520014284390251473593/3311115343944227697817915453587083072566512844800000*t^26 + 28571180593108291351258890685974424868969882113948991920298729/6812994534864666044892830151413751178120396800000000*t^25 - 745872282684735844713478320892603564036624997307503356039861/1021949180229699906733924522712062676718059520000000*t^24 - 62917914258253698603517406220577258964792988107680108303773037/551852557324037949636319242264513845427752140800000000*t^23 + 950315942206182506713519914556148336608016178938127326920037/82777883598605692445447886339677076814162821120000000*t^22 - 243925626172808108053423178425730883794129893972874795357939/551852557324037949636319242264513845427752140800000000*t^21 - 15048896310968150015865292601881216816796150534745906564781/41388941799302846222723943169838538407081410560000000*t^20 + 30072283866277845449799285091363730535043528292197436728609/827778835986056924454478863396770768141628211200000000*t^19 + 64208668609518193419882361198333314855153520731909079147/8277788359860569244544788633967707681416282112000000*t^18 - 8280727630290313298415173139115081401045329048134592843/33111153439442276978179154535870830725665128448000000*t^17 - 212978507035191743878177826808515277206650809500265251/2586808862456427888920246448114908650442588160000000*t^16 - 856825900839813296536078903382472918483228561777659/108918267892902226901905113604838258966003712000000*t^15 - 4577312559540127389850879714789584601098660932131/517361772491285577784049289622981730088517632000000*t^14 + 26318677683260648653620856204113741791866284474529/689815696655047437045399052830642306784690176000000*t^13 + 147242561205270778343578614904356509089380805181/129340443122821394446012322405745432522129408000000*t^12 - 1786585181344559687021527776302842962379577123/34490784832752371852269952641532115339234508800000*t^11 - 7238970886044072928205665413446569865287/3832309425861374650252216960170235037692723200*t^10 + 3452179626894495204293557277685293555332361/51736177249128557778404928962298173008851763200000*t^9 - 200831099724517181160499197565293444911/230965077005038204367879147153116843789516800000*t^8 - 46241145955810093532737132675592737181/184772061604030563494303317722493475031613440000*t^7 - 1484025371757355017821548324033/66656587880241906022475944344333865451520000*t^6 + 474873520533415381918933782646217/2399637163688708616809133996396019156254720000*t^5 + 442789193290099415264330929/14997732273054428855057087477475119726592000*t^4 - 10240797764568579695934079/170914327898056169288399857293163757568000*t^3 + 104450125323050745343/5493674825294662584269995412994549350400*t^2 - 3184888795459921529/399539987294157278855999666399603589120*t + 224793240595/52023435845593395684374956562448384)*Dt^4 + (4300800*t^54 - 1544402850800/2850551*t^53 + 132081696912595200993700076/1230082629163353735*t^52 + 103073950145125147662087225707/2302714681793798191920*t^51 + 3931866218289362367424177450467227/6908144045381394575760000*t^50 + 1627617832975880998979805639898741/3070286242391730922560000*t^49 - 20374797636739103168017858647373760587942811/2149372305703785582723663360000000*t^48 + 4810653099456684697352092037481038307533/7735033038969988601794560000000*t^47 + 3123156404929337951736662497271495085209685726209/107296665500732976289565274931200000000*t^46 - 117673295396384163301140646629249025266366436120817/10300479888070365723798266393395200000000*t^45 + 1154365644869977734235968841927944177083320549845101/8828982761203170620398514051481600000000*t^44 + 10165334514708950444359919621248716967563410119644949/478123374145156316673888760941772800000000*t^43 - 542195737356048353161846560015140012632480022849113671853/522110724566510697807886526948415897600000000*t^42 + 456867702554290450348029244183901677561487787048404374141/1827387535982787442327602844319455641600000000*t^41 + 79179875008916706879416819181909815904710718559524936415699/21928650431793449307931234131833467699200000000*t^40 - 2821174505018449501934015132276180867323670276192412505691/87714601727173797231724936527333870796800000000*t^39 - 9423388380369990307465744621762586696046656327257260789057791/1202943109401240647749370558089150228070400000000*t^38 - 15054521946987117902579914428952189248528546378958730997105959/12248148022994450231629954773271347776716800000000*t^37 + 5397386651627536442067544357106145934436257113600245762669356637/538918513011755810191718010023939302175539200000000*t^36 + 11205249868521625923711245104778980146777914928362799780711941/6807391743306389181369069600302391185375232000000*t^35 - 33201619982908104085725940461119722443032432968296657134539539251/4311348104094046481533744080191514417404313600000000*t^34 - 2741256073271585230006136397005878587345259017642591362215930691/7959411884481316581293065994199718924438732800000000*t^33 + 145082801864525405734418649562614889294309732190855554029758369621/37626310726638951111567221063489580370074009600000000*t^32 - 316863151235926196336471404953325965305936468782461309652560313/422336140809212716558407583365699371500830720000000*t^31 - 576682887060720009694139151561772102617670144955744139022050702547/413889417993028462227239431698385384070814105600000000*t^30 + 65300655148478231679078912528827450211134909897305156236214122971/103472354498257115556809857924596346017703526400000000*t^29 + 29979999357801940425177344737076015172177662616745847823681528577/82777883598605692445447886339677076814162821120000000*t^28 - 3294346149249106993322918108083035834141407009060274521033001911/17245392416376185926134976320766057669617254400000000*t^27 - 2174475097572055219459392325689332911856012113309629271183947119/41388941799302846222723943169838538407081410560000000*t^26 + 344729573289099498310297143874164532466088147056013790826560267/17245392416376185926134976320766057669617254400000000*t^25 + 261284034556776447277266031360722937276348698910645749365666357/68981569665504743704539905283064230678469017600000000*t^24 - 1246427317753801704593803602244859535528924716595053859308149/1591882376896263316258613198839943784887746560000000*t^23 - 32147090645500077441424885826682381819880950054717352699465307/413889417993028462227239431698385384070814105600000000*t^22 + 15588683793549275891906536477441260150069081154825100447087/1175822210207467222236475658234049386564812800000000*t^21 - 3990169425974052949414247052486145151033922755118092449009/2069447089965142311136197158491926920354070528000000*t^20 - 693568042000975846811901336509610240275283020601063932303/1916154712930687325126108480085117518846361600000000*t^19 + 314247888742119059518049264207440158829711466699018946553/15918823768962633162586131988399437848877465600000000*t^18 + 20639315345571798083271259813690442015223260564025053/4619301540100764087357582943062336875790336000000*t^17 + 21218629088780335411384522848912870386629760072142607/49746324278008228633081662463748243277742080000000*t^16 - 520764739430437944891634601073865687193357164069691/646702215614106972230061612028727162610647040000000*t^15 - 34227331019820815933162875943156144593839271916851/19161547129306873251261084800851175188463616000000*t^14 - 456582836896704970567084063371459351760900042121/32335110780705348611503080601436358130532352000000*t^13 + 52441129312433877090843685962591306461314420447/8622696208188092963067488160383028834808627200000*t^12 + 6718892341710825217723518583415573616706741/119759669558167957820381780005319844927897600000*t^11 - 605170744385572756850179295466312639427619/51122704791629009662455463401480408111513600000*t^10 - 27961013338841182171100611050163224330809/202094442379408428821894253758977238315827200000*t^9 + 258382121764450730902324312856235816499/53891851301175581019171801002393930217553920000*t^8 + 163222676835337437163689072063182633/3849417950083970072797985785885280729825280000*t^7 + 9664352881743669224673069712487471/4199365036455240079415984493693033523445760000*t^6 + 4780853381736966363602301091/3749433068263607213764271869368779931648000*t^5 - 69527602626742309901667661/42728581974514042322099964323290939392000*t^4 + 25117903453809373781/50867359493469098002499957527727308800*t^3 - 3184888795459921529/11098332980393257745999990733322321920*t^2 + 224793240595/1445095440155372102343748793401344*t)*Dt^3 + (-13081052759313/47440000*t^49 + 31350194815011653/82925120000*t^48 - 6733035054639413208040382122902231781/238580564513684713367040000000*t^47 + 2052491789230437271375708440490041867437/204702124352741484068920320000000*t^46 + 725113028333292584386155430010135984356387/101896168566697983180973670400000000*t^45 - 47235092700744660610868901616580176204421643403/792344606774643517215251261030400000000*t^44 + 8466516413719688594310364330579312016103845656441/4754067640647861103291507566182400000000*t^43 - 13613568947882370327400773655863789768643774115731/3346863619016094216717221326592409600000000*t^42 - 7438121054860459706417446902847529124832329623711610707/522110724566510697807886526948415897600000000*t^41 + 54536035416773514130296154740331881712511692832459507/37293623183322192700563323353458278400000000*t^40 + 5771135714127283879462560424691460997720964841327124057/89504695639973262481351976048299868160000000*t^39 + 102237954845891439088972365832213570491968741011561860319/17542920345434759446344987305466774159360000000*t^38 - 1368731720605002839976156925608043971511325659642848518443783/8420601765808684534245593906624051596492800000000*t^37 - 3073910419661663162007563680161629763970802580060965481918803/134729628252938952547929502505984825543884800000000*t^36 + 124103220528572326842941431405398662002071342553351401980111779/538918513011755810191718010023939302175539200000000*t^35 + 55425403642897918635133028429082269298921093348660907586415289/3233511078070534861150308060143635813053235200000000*t^34 - 843068609102526528954254203470529197156917203347678295509702349/4311348104094046481533744080191514417404313600000000*t^33 + 1797687274658868459020607162168003782819790093483462979089704039/103472354498257115556809857924596346017703526400000000*t^32 + 44837742388085804720352573411937113592780232528689358164308387161/413889417993028462227239431698385384070814105600000000*t^31 - 3852218523471404100267459368199027914311658534836645459199510131/103472354498257115556809857924596346017703526400000000*t^30 - 133722479046717595894986873660658797422620112639503993481275913/3111950511225777911483003245852521684742963200000000*t^29 + 236446532370031195011191052091272982179906532015814915537859599/9406577681659737777891805265872395092518502400000000*t^28 + 262632874831888151428200917850289638950137066810215369005750521/21783653578580445380381022720967651793200742400000000*t^27 - 133549345062148155583579132079831836497192117086482401437354569/17245392416376185926134976320766057669617254400000000*t^26 - 381398778869870347799989842717345769225858076569213969416205013/206944708996514231113619715849192692035407052800000000*t^25 + 16561103297464721245236045440060610397974886213705964427630581/17245392416376185926134976320766057669617254400000000*t^24 + 1892133014867252306935134196278445345345238440982790106306703/13796313933100948740907981056612846135693803520000000*t^23 - 1145589246325387505682619672508645502117406597663651618749647/20694470899651423111361971584919269203540705280000000*t^22 - 883417184768705056163764171056935553389685314677083717951973/413889417993028462227239431698385384070814105600000000*t^21 + 22937817914803530521866571024729860236939399902535340110731/12934044312282139444601232240574543252212940800000000*t^20 - 6784581991327233364517221337271629159611178433874705832903/51736177249128557778404928962298173008851763200000000*t^19 - 24800059720898374036213268034569610958505866502767443643/638718237643562441708702826695039172948787200000000*t^18 + 50851773436344843689926468800755927448007548353853005599/41388941799302846222723943169838538407081410560000000*t^17 + 4481389050834556141374563481014865682649950348026269/12436581069502057158270415615937060819435520000000*t^16 + 3934110808225354781971982463973687550915064590545739/92386030802015281747151658861246737515806720000000*t^15 + 210848786928476764108456350544298449371193144461247/129340443122821394446012322405745432522129408000000*t^14 - 1288365333855710120907615171173684689309770738509/19161547129306873251261084800851175188463616000000*t^13 - 263378885059251154528325950272463706310675395479/32335110780705348611503080601436358130532352000000*t^12 - 35805825372288506804112652328942599967404357/74979967027722547504934679655504598563553280000*t^11 - 12475514761234933451591408733769012177069/403231210633562147543372996650908568780800000*t^10 - 1945076446510091864658125449204742451383/4647518617420819060223223945589128010137600000*t^9 + 31961581360671751439161729742137791916169/202094442379408428821894253758977238315827200000*t^8 + 461946829817938856696517757339229545757/53891851301175581019171801002393930217553920000*t^7 - 12351267406885558993176544745599003/349947086371270006617998707807752793620480000*t^6 - 25858456506742549814889958058986031/4199365036455240079415984493693033523445760000*t^5 - 18640481773224002211707081/340857551660327928524024715397161811968000*t^4 + 69527602626742309901667661/42728581974514042322099964323290939392000*t^3 - 25117903453809373781/50867359493469098002499957527727308800*t^2 + 3184888795459921529/11098332980393257745999990733322321920*t - 224793240595/1445095440155372102343748793401344)*Dt^2]

NONSMOOTH_fast = [1/((x - 1)*(y - 1) + (x - 1)*(z - 1) + (y - 1)*(z - 1)), -1/((2*x + 1)*(x + y - 1)), x*y^2/((2*x*y^2 + 2*x*y + y^2 - 1)*(x - 1)*(y - 1)^2), 12/((3*x + y - 4)*(2*x + y - 3)), -24/((3*x + y - 4)*(2*x + y - 3)*(x + y - 2)), -9*x*y/((3*x*y - x - y)*(x + y - 3)), 3/(3*x*y*z - x*y - x*z - y*z - x - y - z + 3), 1/(x^2*y^2 - 2*x^2*y - 2*x*y^2 + 5*x^2 + 14*x*y + 5*y^2 - 20*x - 20*y + 19), -1/((x*y - 1)*(x - 1)*(y - 1)), 1/((2*x + y - 3)*(x + 2*y - 3)*(x + y - 2)^2), 9/((2*x + y - 3)*(x + 2*y - 3)), 2/((x*y - 1)*(x + y - 2)), ((x - 1)*(y - 1) + (x - 1)*(z - 1) + (y - 1)*(z - 1))^(-2), x*y*z/((x*y*z - 1)*(x*z + x - 1))]
NONSMOOTH_med  = [((x^2 + 2*x*y + 1)*u*y - 1)*(x + 1)/(((x^2 + x*y + 1)*u*y - 1)*((x^2*y + x*y^2 + x + y)*u - 1)*(y - 1)), 1/(3125*u*v*w*x*y*z - u - v - w - x - y - z + 1), (x^2 - y)*(x*y - 1)*(y^2 - x)*u/((u*x^2*y + u*y^2 + u*x - 1)*(x - 1)*(y - 1)), (x^2 - y)*(x*y - 1)*(y^2 - x)*u/((u*x^2*y + u*x*y^2 + u*x^2 + u*y^2 + u*x + u*y - 1)*(x - 1)*(y - 1)), z/(((x + y + 1/x + 1/y)*z - 2*z^2 - 2)*(y*z - 1)), -(x - 1)^3*x*y/((x - 1)^4 - (x^3 + x^2*y - x^2 - x + 1)*x*y)]
NONSMOOTH_slow = [(x*y*z - (x*y - x - y)*((x*y - y - 1)*z - x + 1))/((x*y*z^2 - (x + y + 1)*z + 1)*(z - 1)), -(x + y + z)/((4*x + 3*y + 2*z + 1)*(2*x + 3*y + 4*z - 1)*(x - y + z + 1)), (2*x*y^2*z - 1)*(x + 1)/((x*y^2*z - 1)*((x*y^2 + x^2 + 1)*z - 1)*(y - 1))]

NONSMOOTH_fast_diag = [(t^3 - 3*t^2 + 2*t)*Dt^2 + (3*t^2 - 6*t + 2)*Dt + 8/9*t - 8/9, 1/16*(16*t^2 + 8*t - 3)*Dt^2 + 1/8*Dt*(28*t + 5) + 3/2, 6*t^8 + 48*t^7 - 60*t^6 - 16*t^5 + 1/9*(9*t^11 + 42*t^10 - 132*t^9 + 42*t^8 + 122*t^7 - 98*t^6 + 4*t^5 + 14*t^4 - 3*t^3)*Dt^3 + 160/3*t^4 + 1/3*(27*t^10 + 156*t^9 - 372*t^8 + 120*t^7 + 150*t^6 - 68*t^5 - 28*t^4 + 16*t^3 - t^2)*Dt^2 - 128/3*t^3 + 2/3*(27*t^9 + 186*t^8 - 330*t^7 + 84*t^6 + 60*t^5 + 2*t^4 - 14*t^3 + t)*Dt + 12*t^2 - 2/3, 1/24*(24*t^3 - 83*t^2 + 95*t - 36)*Dt^3 + 1/48*(384*t^2 - 889*t + 511)*Dt^2 + 55/24*Dt*(6*t - 7) + 15/4, 1/24*(24*t^4 - 107*t^3 + 178*t^2 - 131*t + 36)*Dt^4 + 1/48*(744*t^3 - 2496*t^2 + 2777*t - 1025)*Dt^3 + 7/96*(936*t^2 - 2103*t + 1175)*Dt^2 + 35/32*Dt*(84*t - 95) + 105/4, 1/2*t^5 - 79/18*t^4 + 1/72*(72*t^7 - 590*t^6 + 1535*t^5 - 1535*t^4 + 590*t^3 - 72*t^2)*Dt^2 + 7/3*t^3 + 1/36*(90*t^6 - 682*t^5 + 1163*t^4 - 372*t^3 - 92*t^2 + 18*t)*Dt - 7/3*t^2 + 79/18*t - 1/2, (t^4 + t^3 - t^2 - t)*Dt^2 + (3*t^3 + 3*t^2 - t - 1)*Dt + t^2 + 10/9*t - 1/9, 6*t^5 + 75*t^4 + 1/32*(32*t^7 + 956*t^6 + 6400*t^5 - 69339*t^4 - 208698*t^3 + 346544*t^2 + 250810*t - 326705)*Dt^2 - 1809/4*t^3 + 3/8*(16*t^6 + 339*t^5 + 652*t^4 - 25101*t^3 - 92469*t^2 + 75266*t + 54365)*Dt - 108915/16*t^2 - 113601/4*t + 102375/16, Dt*(t - 1) + 2, 1/8*(8*t^3 - 25*t^2 + 26*t - 9)*Dt^3 + 1/16*(176*t^2 - 369*t + 193)*Dt^2 + 7/4*Dt*(17*t - 18) + 35/2, 1/8*(8*t^2 - 17*t + 9)*Dt^2 + 1/4*Dt*(14*t - 15) + 3/2, Dt*(t - 1) + 3/2, (t^5 + 7*t^4 - 30*t^3 + 26*t^2 - 4*t)*Dt^2 + 20/9*t^3 + (4*t^4 + 45*t^3 - 94*t^2 + 22*t - 4)*Dt + 40*t^2 - 56/3*t + 112/9, (t^2 - t)*Dt + 1]
NONSMOOTH_med_diag  = [(t^31 + 2124287/40860*t^30 - 140616731/163440*t^29 + 8061643465/588384*t^28 - 1081225447/290560*t^27 - 1421625204539/282424320*t^26 + 8980409604053/225939456*t^25 + 81609418233899/1694545920*t^24 - 83782990783543/753131520*t^23 - 22494923963659/423636480*t^22 + 139076444274073/1506263040*t^21 + 139010328686351/6778183680*t^20 + 7233362289661/847272960*t^19 - 1422052212827/338909184*t^18 - 84014317460407/13556367360*t^17 - 2335275211369/2259394560*t^16 + 76214863669/105909120*t^15 + 2001926347/7845120*t^14 - 8343300353/847272960*t^13 - 6152200157/423636480*t^12 - 488080763/211818240*t^11 + 467965/21181824*t^10 + 4176941/52954560*t^9 + 312013/26477280*t^8 + 637/1103220*t^7)*Dt^7 + (145/4*t^30 + 316191883/163440*t^29 - 21577052467/653760*t^28 + 1270702283209/2353536*t^27 - 5788124747797/31380480*t^26 - 6309595119065/75313152*t^25 + 7087408204026457/4518789120*t^24 + 44114178507065/21181824*t^23 - 43251699513574013/9037578240*t^22 - 979692337217663/502087680*t^21 + 60668223211978547/18075156480*t^20 + 16172174586353887/13556367360*t^19 + 7704493008227551/13556367360*t^18 - 818604043281389/6778183680*t^17 - 14642510483838439/54225469440*t^16 - 476099167653659/6778183680*t^15 + 58502448578435/2711273472*t^14 + 10015317480269/847272960*t^13 + 468423661723/677818368*t^12 - 4343202269/9414144*t^11 - 373255115/3138048*t^10 - 845229289/105909120*t^9 + 2361487/1103220*t^8 + 6570113/13238640*t^7 + 220363/5883840*t^6 + 637/735480*t^5)*Dt^6 + (465*t^29 + 277641911/10896*t^28 - 73614470531/163440*t^27 + 5953078045259/784512*t^26 - 656211648431/207360*t^25 + 116614759271077/376565760*t^24 + 50782852538119079/2259394560*t^23 + 9166396989253829/282424320*t^22 - 101401819890539561/1355636736*t^21 - 334106818809405229/13556367360*t^20 + 1190992115526834151/27112734720*t^19 + 149094463000861847/6778183680*t^18 + 21756027402060233/1807515648*t^17 - 836534970841181/1355636736*t^16 - 4665093964662073/1129697280*t^15 - 2462153592748903/1694545920*t^14 + 12214411446433/75313152*t^13 + 49948787107217/282424320*t^12 + 3579983045629/141212160*t^11 - 367254679823/105909120*t^10 - 148559932243/84727296*t^9 - 2463956681/10590912*t^8 + 834301427/105909120*t^7 + 144911917/26477280*t^6 + 3594409/6619320*t^5 + 37583/2206440*t^4)*Dt^5 + (5259/2*t^28 + 8064338567/54480*t^27 - 356631153025/130752*t^26 + 186239936490601/3922560*t^25 - 148607122508539/6276096*t^24 + 4044819044618669/376565760*t^23 + 41029938585008357/282424320*t^22 + 519408769070023321/2259394560*t^21 - 1444972534638160177/2711273472*t^20 - 116126418123592919/903757824*t^19 + 13009017797141735/50208768*t^18 + 2321748638937315959/13556367360*t^17 + 962411866084304651/9037578240*t^16 + 80479984000835347/13556367360*t^15 - 752301472623407809/27112734720*t^14 - 7012171765924901/564848640*t^13 - 248977337419651/1355636736*t^12 + 907677280574431/847272960*t^11 + 430992052223593/1694545920*t^10 + 400983455653/70606080*t^9 - 216711929563/23535360*t^8 - 7189512167/3530304*t^7 - 1508507809/13238640*t^6 + 215904221/13238640*t^5 + 67900313/26477280*t^4 + 12103/122580*t^3)*Dt^4 + (6672*t^27 + 5259563191/13620*t^26 - 977440805363/130752*t^25 + 132406005495161/980640*t^24 - 2488526824519099/31380480*t^23 + 1730032721109893/31380480*t^22 + 60613169513486423/141212160*t^21 + 12030299196150991/15690240*t^20 - 604219977103661251/338909184*t^19 - 846774764704865371/3389091840*t^18 + 596478088306344917/847272960*t^17 + 67143705605101501/112969728*t^16 + 93181095610457477/225939456*t^15 + 181933299682321613/3389091840*t^14 - 286726531869208001/3389091840*t^13 - 78873328612425619/1694545920*t^12 - 8417652772953731/1694545920*t^11 + 1118605898338817/423636480*t^10 + 420983426604691/423636480*t^9 + 3258080300251/26477280*t^8 - 2604851695441/211818240*t^7 - 41383423433/6619320*t^6 - 2599559723/3530304*t^5 - 39043507/2647728*t^4 + 883207/275805*t^3 + 195559/1103220*t^2)*Dt^3 + (6894*t^26 + 1865161031/4540*t^25 - 368072629433/43584*t^24 + 104068280693503/653760*t^23 - 3409759696249411/31380480*t^22 + 2239498748651347/23535360*t^21 + 2183869798717633/4184064*t^20 + 317865548048833379/282424320*t^19 - 2934805627821453823/1129697280*t^18 - 6566311283421293/112969728*t^17 + 200730815757075007/251043840*t^16 + 243050786953251623/282424320*t^15 + 1466833900617878317/2259394560*t^14 + 130978466461944649/1129697280*t^13 - 4162106775796967/37656576*t^12 - 20384788715845471/282424320*t^11 - 79153562998747/6276096*t^10 + 5237933801263/2206440*t^9 + 22136344582123/14121216*t^8 + 11838228186449/35303040*t^7 + 142829155343/7060608*t^6 - 43108637837/8825760*t^5 - 9310005709/8825760*t^4 - 11141507/147096*t^3 - 394043/275805*t^2 + 7007/183870*t)*Dt^2 + (2232*t^25 + 155829171/1135*t^24 - 66398101321/21792*t^23 + 1976379946321/32688*t^22 - 246318690284137/5230080*t^21 + 231849341982269/4707072*t^20 + 6148216325683561/31380480*t^19 + 4195224873218561/7060608*t^18 - 4706802428858623/3530304*t^17 + 45434282721888061/282424320*t^16 + 322915974983241493/1129697280*t^15 + 3691875670228187/8825760*t^14 + 371425052227708301/1129697280*t^13 + 12085860992007431/188282880*t^12 - 934949135650591/17651520*t^11 - 3561363056919427/94141440*t^10 - 2303022064367843/282424320*t^9 + 10491730439165/14121216*t^8 + 64647334424189/70606080*t^7 + 264996548831/980640*t^6 + 1331464414541/35303040*t^5 + 4029305137/2206440*t^4 - 161614027/980640*t^3 - 60639917/2206440*t^2 - 171457/110322*t - 7007/183870)*Dt + 108*t^24 + 1565124/227*t^23 - 3718377671/21792*t^22 + 394644140959/108960*t^21 - 5254741538893/1743360*t^20 + 190709802182977/47070720*t^19 + 422838603672937/47070720*t^18 + 18147341164423387/282424320*t^17 - 25361360073225829/188282880*t^16 + 23503995490509871/564848640*t^15 + 6946610807460473/564848640*t^14 + 6934384921198609/188282880*t^13 + 7707272729506669/282424320*t^12 + 51917191566035/14121216*t^11 - 840195135508889/141212160*t^10 - 536549684462453/141212160*t^9 - 27354775639063/35303040*t^8 + 490787200291/3922560*t^7 + 1179785618039/8825760*t^6 + 156879518663/3530304*t^5 + 660492149/81720*t^4 + 3969327259/4412880*t^3 + 69711733/1103220*t^2 + 815243/275805*t + 7007/91935, (t^12 + 28/15625*t^11 + 316/244140625*t^10 + 356/762939453125*t^9 + 38/476837158203125*t^8 - 44156/931322574615478515625*t^7 + 80812/14551915228366851806640625*t^6 - 59156/227373675443232059478759765625*t^5 + 1/227373675443232059478759765625*t^4)*Dt^5 + (15*t^11 + 67/3125*t^10 + 109/9765625*t^9 + 327/152587890625*t^8 - 13/95367431640625*t^7 - 42703/186264514923095703125*t^6 + 111843/2910383045673370361328125*t^5 - 20203/9094947017729282379150390625*t^4 + 2/45474735088646411895751953125*t^3)*Dt^4 + (65*t^10 + 216/3125*t^9 + 208/9765625*t^8 - 88/152587890625*t^7 - 522/476837158203125*t^6 + 8072/186264514923095703125*t^5 + 96304/2910383045673370361328125*t^4 - 34056/9094947017729282379150390625*t^3 + 1/9094947017729282379150390625*t^2)*Dt^3 + (90*t^9 + 39/625*t^8 + 219/48828125*t^7 - 2781/762939453125*t^6 - 1101/11920928955078125*t^5 + 9741/186264514923095703125*t^4 - 77211/2910383045673370361328125*t^3 + 4809/9094947017729282379150390625*t^2 + 3/45474735088646411895751953125*t)*Dt^2 + (31*t^8 + 154/15625*t^7 - 26/9765625*t^6 - 382/762939453125*t^5 - 136/11920928955078125*t^4 + 17614/931322574615478515625*t^3 + 316186/14551915228366851806640625*t^2 + 47366/45474735088646411895751953125*t + 1/227373675443232059478759765625)*Dt + t^7 - 1/15625*t^6 - 3/48828125*t^5 + 19/152587890625*t^4 + 2131/11920928955078125*t^3 + 49929/931322574615478515625*t^2 + 33751/14551915228366851806640625*t + 481/45474735088646411895751953125, (t^11 - 5/3*t^10 - 29/27*t^9 + 17/81*t^8 - 91/81*t^7 + 7/27*t^6 + 1/81*t^5 + 1/81*t^4)*Dt^6 + (22*t^10 - 42*t^9 - 122/9*t^8 - 170/27*t^7 - 598/27*t^6 + 10/3*t^5 + 2/27*t^4 + 2/9*t^3)*Dt^5 + (152*t^9 - 328*t^8 - 200/9*t^7 - 1088/9*t^6 - 3640/27*t^5 + 56/9*t^4 - 8/9*t^3 + 32/27*t^2)*Dt^4 + (384*t^8 - 928*t^7 + 1184/9*t^6 - 13736/27*t^5 - 7904/27*t^4 - 256/9*t^3 - 160/27*t^2 + 56/27*t)*Dt^3 + (312*t^7 - 840*t^6 + 808/3*t^5 - 1768/3*t^4 - 1768/9*t^3 - 184/3*t^2 - 8*t + 8/9)*Dt^2 + (48*t^6 - 144*t^5 + 208/3*t^4 - 1088/9*t^3 - 208/9*t^2 - 16*t - 16/9)*Dt, (t^25 - 88733/24024*t^24 + 18302971/3123120*t^23 - 192072299/37477440*t^22 + 114735811/56216160*t^21 - 2647776799/3597834240*t^20 + 8410692799/2398556160*t^19 - 185509224049/28782673920*t^18 + 648559812811/115130695680*t^17 - 21374345101/8528199680*t^16 + 3613692679/17712414720*t^15 + 844976463949/1842091130880*t^14 - 10098750505/30701518848*t^13 + 8635736099/68225597440*t^12 - 319231073971/9824486031360*t^11 + 38613915199/6549657354240*t^10 - 3401315227/4366438236160*t^9 + 31328169/396948930560*t^8 - 118775313/17465752944640*t^7 + 2636253/4990215127040*t^6 - 2170557/69863011778560*t^5 + 243/268703891456*t^4)*Dt^6 + (81/4*t^24 - 2470725/32032*t^23 + 1598157409/12492480*t^22 - 18126589097/149909760*t^21 + 17718198749/299819520*t^20 - 156317362111/4797112320*t^19 + 754475399/7454720*t^18 - 6791337594661/38376898560*t^17 + 3550666866457/21929656320*t^16 - 666170676581/7872184320*t^15 + 599628494903/27910471680*t^14 + 6406311903997/2456121507840*t^13 - 1148064034681/245612150784*t^12 + 5149325461213/2456121507840*t^11 - 1066976816857/1871330672640*t^10 + 2784488302177/26198629416960*t^9 - 7145315119/499021512704*t^8 + 26122059027/17465752944640*t^7 - 726550767/5374077829120*t^6 + 138002643/12702365777920*t^5 - 35489907/55890409422848*t^4 + 19197/1074815565824*t^3)*Dt^5 + (993/8*t^23 - 7807097/16016*t^22 + 131519627/156156*t^21 - 63597109561/74954880*t^20 + 559706111407/1199278080*t^19 - 380461481651/1199278080*t^18 + 403531520131/456867840*t^17 - 6508637047207/4264099840*t^16 + 223980920697419/153507594240*t^15 - 29068774051483/34112798720*t^14 + 8860922640811/29239541760*t^13 - 7671725583163/153507594240*t^12 - 44582188194737/4912243015680*t^11 + 20305223015521/2456121507840*t^10 - 11507742096327/4366438236160*t^9 + 914803352893/1746575294464*t^8 - 183708660659/2495107563520*t^7 + 285462907011/34931505889280*t^6 - 77155281/95965675520*t^5 + 297638577/4366438236160*t^4 - 78793479/19960860508160*t^3 + 112995/1074815565824*t^2)*Dt^4 + (519/2*t^22 - 2104559/2002*t^21 + 589784375/312312*t^20 - 2688619987/1338480*t^19 + 351998607893/299819520*t^18 - 570098272427/599639040*t^17 + 2200168547069/799518720*t^16 - 2207576662339/456867840*t^15 + 184927705846531/38376898560*t^14 - 26043248986857/8528199680*t^13 + 5073365859451/3936092160*t^12 - 10952901823477/30701518848*t^11 + 1245443441357/22328377344*t^10 - 109582518781/614030376960*t^9 - 2450639221207/1091609559040*t^8 + 14665730877/23991418880*t^7 - 438845069617/4366438236160*t^6 + 519547653/38986055680*t^5 - 27712417059/17465752944640*t^4 + 5180549589/34931505889280*t^3 - 590764347/69863011778560*t^2 + 54675/268703891456*t)*Dt^3 + (297/2*t^21 - 191745/308*t^20 + 4329795/3718*t^19 - 208570057/160160*t^18 + 5012012419/6662656*t^17 - 77911798079/99939840*t^16 + 717435535007/266506240*t^15 - 5236642209023/1066024960*t^14 + 13058809668517/2558459904*t^13 - 839173776773/243662848*t^12 + 5501983099507/3411279872*t^11 - 27685574914279/51169198080*t^10 + 17963247127027/136451194880*t^9 - 318753319631/13645119488*t^8 + 3335303973921/1091609559040*t^7 - 637244897203/2183219118080*t^6 + 11556172503/873287647232*t^5 + 23157473373/8732876472320*t^4 - 6833568051/8732876472320*t^3 + 1614171969/17465752944640*t^2 - 12734901/2495107563520*t + 3645/38386270208)*Dt^2 + (9*t^20 - 12333/308*t^19 + 1189671/14872*t^18 - 199018109/2082080*t^17 + 1954633859/49969920*t^16 - 17620835471/199879680*t^15 + 1501640249/3331328*t^14 - 348057782867/399759360*t^13 + 3008157689959/3198074880*t^12 - 50737993563/76144640*t^11 + 8546959377877/25584599040*t^10 - 1151306680019/9303490560*t^9 + 2380188318603/68225597440*t^8 - 65543881609/8528199680*t^7 + 184735244929/136451194880*t^6 - 103855410117/545804779520*t^5 + 21625576641/1091609559040*t^4 - 256625037/218321911808*t^3 - 3710367/272902389760*t^2 + 33230007/4366438236160*t - 3645/9596567552)*Dt, (t^27 + 535/14*t^25 + 1807/14*t^23 + 93055/448*t^21 + 24309/448*t^19 - 133005/448*t^17 - 24097/64*t^15 - 65195/448*t^13 + 30895/448*t^11 + 71585/448*t^9 + 55203/448*t^7 + 2055/56*t^5)*Dt^5 + (49/3*t^26 + 29593/42*t^24 + 118159/42*t^22 + 8782909/1344*t^20 + 3130011/448*t^18 + 794159/448*t^16 - 5692927/1344*t^14 - 9431561/1344*t^12 - 7358069/1344*t^10 - 2795881/1344*t^8 - 47359/448*t^6 + 685/8*t^4)*Dt^4 + (701/9*t^25 + 68249/18*t^23 + 154693/9*t^21 + 192629663/4032*t^19 + 4506099/64*t^17 + 86028205/1344*t^15 + 139907269/4032*t^13 + 16571069/4032*t^11 - 43900897/4032*t^9 - 28153019/4032*t^7 - 1179323/1344*t^5 - 685/24*t^3)*Dt^3 + (3242/27*t^24 + 1253680/189*t^22 + 917893/27*t^20 + 327675743/3024*t^18 + 126714523/672*t^16 + 13123927/63*t^14 + 890443529/6048*t^12 + 165031823/3024*t^10 - 41566691/6048*t^8 - 15262759/1512*t^6 - 1886995/2016*t^4 - 3425/63*t^2)*Dt^2 + (415/9*t^23 + 368915/126*t^21 + 3387164/189*t^19 + 787710325/12096*t^17 + 86042725/576*t^15 + 840661243/4032*t^13 + 243726965/1344*t^11 + 393569293/4032*t^9 + 219262411/12096*t^7 - 1350499/1728*t^5 + 4200965/4032*t^3 + 37675/504*t)*Dt + 35/27*t^22 + 5185/54*t^20 + 174308/189*t^18 + 43874699/12096*t^16 + 27318623/1344*t^14 + 143042693/4032*t^12 + 392900323/12096*t^10 + 170512729/12096*t^8 - 26536043/12096*t^6 - 35648075/12096*t^4 - 932695/1344*t^2 - 37675/504, (t^12 - 31/10*t^11 - 1491/5*t^10 - 7231/10*t^9 - 2281*t^8 - 104054/5*t^7 - 261888/5*t^6 + 7680*t^5 + 344064/5*t^4)*Dt^4 + (19/2*t^11 - 1541/20*t^10 - 49077/20*t^9 - 62788/5*t^8 - 44359/5*t^7 + 12866/5*t^6 - 1039392/5*t^5 - 1990656/5*t^4 - 344064/5*t^3)*Dt^3 + (21*t^10 - 1344/5*t^9 - 20433/4*t^8 - 290019/10*t^7 - 355263/5*t^6 + 57108*t^5 + 1960032/5*t^4 + 2054016/5*t^3 + 258048*t^2)*Dt^2 + (9*t^9 - 1497/10*t^8 - 12411/5*t^7 - 64377/5*t^6 - 148581/5*t^5 - 611874/5*t^4 - 2540904/5*t^3 - 4617216/5*t^2 - 516096*t)*Dt - 54/5*t^5 - 55449/5*t^4 + 352458/5*t^3 + 2533992/5*t^2 + 4617216/5*t + 516096]
NONSMOOTH_slow_diag = [(t^38 - 46264/1305*t^37 + 311496703/648672*t^36 + 2275029628717/165075840*t^35 - 652590680249927/3191466240*t^34 + 197505034147519667/143615980800*t^33 - 22924373761136713/3470719536*t^32 + 47005111142150358506927/1940826364531200*t^31 - 364498022423244044093173/4657983274874880*t^30 + 13287487502716415420153629/46579832748748800*t^29 - 27007141373391364417316449/23289916374374400*t^28 + 121108719070085047283372899/27947899649249280*t^27 - 592539604634964035841986483/46579832748748800*t^26 + 87782179045383854671122779/3105322183249920*t^25 - 6287967273139930303790717327/139739498246246400*t^24 + 493743182892274219457435623/11644958187187200*t^23 - 1837769822778006202675433/145561977339840*t^22 - 27284441097104249185223369/1704140222515200*t^21 + 84041209004219722932054899/4657983274874880*t^20 - 886429930387247145075694909/23289916374374400*t^19 + 4491975090242615996885111539/34934874561561600*t^18 - 1246313288275767430268675711/4657983274874880*t^17 + 225673096976073032404134739/582247909359360*t^16 - 39275102350048330821118315/96372067756032*t^15 + 138400760524639267016972323/465798327487488*t^14 - 212186006458800499656489329/1552661091624960*t^13 + 113855993494054914625734953/3493487456156160*t^12 - 4790586011013544598122091/582247909359360*t^11 + 109262840278656200092397/12130164778320*t^10 - 71688802742748902013877/12130164778320*t^9 + 9894266133245534663/5615817027*t^8 + 3713730904118480378/5615817027*t^7 - 12744031862692063360/16847451081*t^6 + 1069716987056354560/5615817027*t^5)*Dt^10 + (31*t^37 - 1370828/1305*t^36 + 42828282203/3243360*t^35 + 1192846399985377/2393599680*t^34 - 1251065964148595/212764416*t^33 + 37339108613467483/997333200*t^32 - 20494124559216484493/115690651200*t^31 + 615696158321310121029677/970413182265600*t^30 - 671019384875753460279629/310532218324992*t^29 + 2445790283299626577409077/291123954679680*t^28 - 28605670026082386976031299/803100564633600*t^27 + 3081118754298728919083736701/23289916374374400*t^26 - 5928491211575176269402577007/15526610916249600*t^25 + 16783546913331062070777983/20077514115840*t^24 - 11941520164803283597238102537/9315966549749760*t^23 + 13308323774147638539177264589/11644958187187200*t^22 - 381379715785271388886766503/1455619773398400*t^21 - 732900332433233321358773821/1455619773398400*t^20 + 930420665975518089918774557/1552661091624960*t^19 - 131701235567386358150403761/86258949534720*t^18 + 419687742890342021927549407/86258949534720*t^17 - 3835611664158990465648520807/388165272906240*t^16 + 1836635582900516242820553299/129388424302080*t^15 - 6912174449945058965879309983/465798327487488*t^14 + 5020859328405758886289722803/465798327487488*t^13 - 719934695698672863555739201/145561977339840*t^12 + 7804912734464245600203829/6469421215104*t^11 - 20062924808897512163534171/58224790935936*t^10 + 17993508411938206053530911/48520659113280*t^9 - 971923824994156827115351/4043388259440*t^8 + 254332047424499517277/3743878018*t^7 + 173161019283057341018/5615817027*t^6 - 184032460259734748480/5615817027*t^5 + 45997830443423246080/5615817027*t^4)*Dt^9 + (314*t^36 - 4420907/435*t^35 + 570298635377/4865040*t^34 + 145708783640921/24933330*t^33 - 4445980294776277/82537920*t^32 + 74639513969096343/221629600*t^31 - 3043162202305772779/1994666400*t^30 + 2360677530299111927759/438307670400*t^29 - 260083194480412667150893/13385009410560*t^28 + 126243913109859357176426263/1552661091624960*t^27 - 4180611683790176094970383461/11644958187187200*t^26 + 125892503116588278734552603/95843277260800*t^25 - 86245270016162996148786043769/23289916374374400*t^24 + 24454588656015613226701930279/3105322183249920*t^23 - 21100529690602018305233522425/1863193309949952*t^22 + 70148673126538506443624062447/7763305458124800*t^21 - 17263283010184428151144175597/23289916374374400*t^20 - 122754893104234923353154882473/23289916374374400*t^19 + 10832027708308610569435364551/1552661091624960*t^18 - 2481642319070504841325531333/116449581871872*t^17 + 12281471958900511190604591437/194082636453120*t^16 - 72449944428824003476793012587/582247909359360*t^15 + 34214997995689491893888594423/194082636453120*t^14 - 212326036699658334893775973127/1164495818718720*t^13 + 2832304690133345073927390971/21564737383680*t^12 - 13949357937924434645965219819/232899163743744*t^11 + 8790200920371698076419838227/582247909359360*t^10 - 1954802559752497800112698413/388165272906240*t^9 + 514067390224196282016460477/97041318226560*t^8 - 2014151136775211068847261/599020482880*t^7 + 14524055821452296099771/16847451081*t^6 + 17623945574273515907885/33694902162*t^5 - 2833750552286094176960/5615817027*t^4 + 233465732425049382720/1871939009*t^3)*Dt^8 + (1218*t^35 - 16390381/435*t^34 + 383591739041/973008*t^33 + 5190586304618617/199466640*t^32 - 438257427496836073/2393599680*t^31 + 2378692747056994991/1994666400*t^30 - 41922469810864716367/8465169600*t^29 + 475801589343441972765959/26955921729600*t^28 - 26122612526246384784475063/388165272906240*t^27 + 13213215019949774682923189/43129474767360*t^26 - 8174099106612252143948242499/5822479093593600*t^25 + 9655518563730389893134060119/1940826364531200*t^24 - 9881958573003240955147487333/727809886699200*t^23 + 3509197932445227697274896163/129388424302080*t^22 - 19918713316389324697693833799/582247909359360*t^21 + 19281233987339197478870538427/970413182265600*t^20 + 50649687776000279098396808579/5822479093593600*t^19 - 65366504790580462685282466239/2911239546796800*t^18 + 4678136717529002213670632077/129388424302080*t^17 - 36721121230895426792240265107/291123954679680*t^16 + 4584402567839387527904123909/13385009410560*t^15 - 187280590273279982873693372479/291123954679680*t^14 + 86266098980730038003082924797/97041318226560*t^13 - 261972873571511639849739921403/291123954679680*t^12 + 61695627784284669334315695959/97041318226560*t^11 - 41548270759675492881691577927/145561977339840*t^10 + 44209266178518450994372077319/582247909359360*t^9 - 6042885181898749913555586599/194082636453120*t^8 + 306768425480162059589057743/9704131822656*t^7 - 155444981250821995021356809/8086776518880*t^6 + 65757188786877193568228/16847451081*t^5 + 68046548696002857876175/16847451081*t^4 - 19128496226542476267200/5615817027*t^3 + 1546810763283488693760/1871939009*t^2)*Dt^7 + (1596*t^34 - 20610674/435*t^33 + 362661181001/810840*t^32 + 26458139840363/687816*t^31 - 82124187214688423/398933280*t^30 + 991846681485675463/664888800*t^29 - 313694441249358072799/57845325600*t^28 + 6145345185646945875387/299510241440*t^27 - 60242269411677553335667/743611633920*t^26 + 1233284060843632918753717/3032541194580*t^25 - 933077764971196693353661793/485206591132800*t^24 + 71159377812840632555048627/11154174508800*t^23 - 360188402313635840146472557/21564737383680*t^22 + 1230933285239436707124458507/43129474767360*t^21 - 211054593096939566002058149/8625894953472*t^20 - 13152863584409789919610014769/1940826364531200*t^19 + 72122405669442964935167188001/1940826364531200*t^18 - 754944538472356033214834617/19408263645312*t^17 + 16557864649121533875128661523/194082636453120*t^16 - 13067693341680756864777498569/43129474767360*t^15 + 56750891407394091619435225663/77633054581248*t^14 - 475731086359795973772041837/371805816960*t^13 + 162984995194793828791149400279/97041318226560*t^12 - 156743844455724171099433010857/97041318226560*t^11 + 52432471462472962539559729261/48520659113280*t^10 - 7467308176546920737165761421/16173553037760*t^9 + 6667647916435409802342328409/48520659113280*t^8 - 7419942271051342424099354227/97041318226560*t^7 + 1750342046743397455074894209/24260329556640*t^6 - 1079825314442324188660873/27885436272*t^5 + 2932271458324171217282/1871939009*t^4 + 243759683027704108482350/16847451081*t^3 - 55932353517236022951040/5615817027*t^2 + 4358027005267588477440/1871939009*t)*Dt^6 + (420*t^33 - 349286/29*t^32 + 5558255241/54056*t^31 + 74334626401651/6648888*t^30 - 1230925609942819/26595552*t^29 + 870245089558481/2216296*t^28 - 2790352890741635065/2313813024*t^27 + 2790743599149217189583/539118434592*t^26 - 262101258770004676024751/12938842430208*t^25 + 1469555981866196388225733/12938842430208*t^24 - 21511038653209118689553381/38816527290624*t^23 + 21379979314966138687786931/12938842430208*t^22 - 163304090587239463310552131/38816527290624*t^21 + 65420183965033417029622315/12938842430208*t^20 + 18506523776388907319451041/38816527290624*t^19 - 166888736119817789915378149/12938842430208*t^18 + 277849551221295486473089/14792883876*t^17 - 208960594215999747765639331/9704131822656*t^16 + 873389612189329947236600369/12938842430208*t^15 - 8052677872329108449313912547/38816527290624*t^14 + 693579687343828498532447849/1617355303776*t^13 - 3258373149653765141259551311/4852065911328*t^12 + 6773229468532659795989389/8766153408*t^11 - 1495173079382239905741331595/2426032955664*t^10 + 9092240036657554649668943/29951024144*t^9 - 399448183017953923910672875/4852065911328*t^8 + 845096713493432789682826673/19408263645312*t^7 - 33264286060269893500335709/718824579456*t^6 + 16395823735117754896993675/539118434592*t^5 - 124095069652712742443195/269559217296*t^4 - 119819036545379643891160/5615817027*t^3 + 39946971474784025825110/1871939009*t^2 - 19248902621139144858240/1871939009*t + 4043530211073020236800/1871939009)*Dt^5 + (-940800/29*t^26 + 2012629645600/1763577*t^25 - 87232650415960/5290731*t^24 + 45929742180654745/410913441*t^23 + 917239489602173413445/151627059729*t^22 - 4555253224314270961945/202169412972*t^21 + 17507809856751668272651/151627059729*t^20 - 54377125263306311095021/202169412972*t^19 + 134267295820758495275111/303254119458*t^18 - 231946628424380407315393/1617355303776*t^17 - 189541465173309776099783/167312617632*t^16 + 1708622246624683393139465/303254119458*t^15 - 22996120273782462170570395/1617355303776*t^14 + 4970571588938162236356733/167312617632*t^13 - 64460255561987595029989661/1617355303776*t^12 + 250141091150264661424878605/9704131822656*t^11 + 25849819017061369923039811/3234710607552*t^10 - 123520805547458356661298773/4852065911328*t^9 + 20410202181437421914207773/1617355303776*t^8 + 11439682019748468889531499/9704131822656*t^7 - 75359851113593248511003611/9704131822656*t^6 + 12556321112602553909826385/808677651888*t^5 - 1078755972068640959375765/44926536216*t^4 + 447765112523161332995480/16847451081*t^3 - 107833360843317124097740/5615817027*t^2 + 14561121583412599883520/1871939009*t - 2439268926932562278400/1871939009)*Dt^4 + (940800/29*t^25 - 8495580800/7569*t^24 + 27930292748120/1763577*t^23 - 542806724001640/5290731*t^22 - 1692356672894650/277037*t^21 + 4119603136625689160/216919971*t^20 - 2517700784617101566/24102219*t^19 + 45211719848594712266/216919971*t^18 - 93118264008998216177/289226628*t^17 - 37472996367372752851/867679884*t^16 + 240309929756765397662/216919971*t^15 - 49789062405711417943/9973332*t^14 + 9826446398783289543737/867679884*t^13 - 6694030954980332823265/289226628*t^12 + 45885087164051863036253/1735359768*t^11 - 49154388020232432959/4702872*t^10 - 12190147993726096484741/867679884*t^9 + 5007900327836914518017/289226628*t^8 - 4484546372229594379093/1735359768*t^7 - 4644867098022994326211/1735359768*t^6 + 898692132345325523065/144613314*t^5 - 95808097993873727245/8034073*t^4 + 412146956260995820480/24102219*t^3 - 133902754824952504160/8034073*t^2 + 76660954642143406080/8034073*t - 18063083692188057600/8034073)*Dt^3, (t^27 - 11339312189612050317138656641699737530110096783495024416201429459001220752907800427/57043960554768992541879463440212830328265333684390699231890891507951490191461897200*t^26 - 6894640110811751955308521935608319168251479621985823497960378605540134506526619338120239/31923454601304816527219462245582866023776800424461882620443867723703384504357630068138800*t^25 - 573759713456616850123434419736139440887703634931359760362346220238618731050960324298640562131529/300711280715187058530440801682451658056131753566329063432752327429032193218767829410651142688000000*t^24 + 51255548352144273470766640362318732483685910999405012338434728774574221158789235801167657500181031/45758233215494297406382075322679727300874715167676405819017145823784398734789171375320748879024000000*t^23 + 7358061978728979545305342878632784729355561645243445704614855119775320792211527025714819859455171051077/320234419335315290968824315938241803542441606629466558483809593333172736105548536953044728954961561600000000*t^22 - 10656307116457444861648938442034126577832186309005330882269729379489711337745706802403377446307522985823397/9320743009173686858938600539698465933906305402557253651229762023555325657088095716555319880963111211929600000000*t^21 - 43084833647461637007150852511862407923632858758612627411280190494914118607091276102423668852726939299586827/447395664440336969229052825905526364827502659322748175259028577130655631540228594394655354286229338172620800000000*t^20 + 89823808769016574618287503556808005108452491689040372962584271700696687187317883725572821131310201459813/29826377629355797948603521727035090988500177288183211683935238475377042102681906292977023619081955878174720000000*t^19 + 428022898223544803097586177765242356791541769023504361921199388449054673110769429967982199927093377309/16570209794086554415890845403908383882500098493435117602186243597431690057045503496098346455045531043430400000000*t^18 - 92845158584777773289011358583826996181088896973833619718579336645084370687516366131831522345650769788023/21474991893136174522994535643465265511720127647491912412433371702271470313930972530943457005739008232285798400000000*t^17 + 146020827734442474703779845546225688075829357149563814625874842961758383733792541116908212055992086559/3579165315522695753832422607244210918620021274581985402072228617045245052321828755157242834289834705380966400000000*t^16 + 15839421862041468833578165964807321724043159883966976452732814264560216043351392238382303704369462179/6135711969467478435141295898132933003348607899283403546409534772077562946837420723126702001639716637795942400000000*t^15 + 12135982426876278035030741526141229765793635673635048903477859720035270696092196602534034685216827/1590740140232309223925521158775204852720009455369771289809879385353442245476368335625441259684370980169318400000000*t^14 - 5942391579198180110663050473646926172033802780944571026133068479905390864918764918458527707429087/10737495946568087261497267821732632755860063823745956206216685851135735156965486265471728502869504116142899200000000*t^13 - 74764800703839597733790070544238837089252308175552858845620995844849840846642003643846446176697/14316661262090783015329690428976843674480085098327941608288914468180980209287315020628971337159338821523865600000000*t^12 - 1243210636237301702591571707204159300986964028515405544955756869238527814127903784982271527/48531055125731467848575221793141842964339271519755734265386150739596543082329881425860919786980809564487680000000*t^11 + 6834646211844156698195403471143343749650462146764385525956405968788702032948814460861987/99421258764519326495345072423450303295000590960610705613117461584590140342273020976590078730273186260582400000000*t^10 - 1304372153484200697469985884633621667969971608617552904343990841856143410631134003097/4208307249291823343718309943849748287619072633253363200555236469189000649408381840278945131440134868172800000000*t^9 - 16863479468260144124310719015069166909133732243170750619505028838406703576523273/88729369713984227126590872310085054257028639857751633746646552061213869114032147234797035903858265292800000000*t^8 + 87384104151361511981940322846625169630665391846157008820843735551076330237/108814091109052760808319184422828893370765028194346713962347278680271145633343206567496364953017671680000000*t^7 + 15117447045223204324783376180059699598746238873534801962663169284224047/113650272936121772399800037063843510853910140558539901249562713288283196550380682414940647839818457088000000*t^6 + 215185253720995104680439256872572758206949329379572716729053/1029488435934284757717974626217728453523760070062474817673304774842707376431163780346199397253120000*t^5)*Dt^12 + (5902/57*t^26 - 1353714806246140374044983792199569837683214200951711871669583869613706326961226231927/55110266976641230082832701967663242859510576610343556885047132473783643066327595600*t^25 - 394137257420276197886821839019631691312154348921348701703983162753760707022185213266414759/19357839492280580234590524978704503865481676853131141588992558087777584220727499083871400*t^24 - 761649567196104155885102182511446328158551485788840235631374500577140034073375773329189290720154631/2448649000109380333747875099414249215599929993325822373666697523350690716209966610915302161888000000*t^23 + 373755565642362067370850102073271494757046046911155637578449723706623969637121725205533134927511540957/3912328939924762428245667440089116684224788146836332697525965967933566091824474152589924029156552000000*t^22 + 49221731979408686712417936370340248186530149283219621336464941868851503314724617693014379724004288108693199/18253361902112971585222986008479782801919171577879593833577146819990845958016266606323549550432809011200000000*t^21 - 7745598792772108456401961825074840349561057443962431300225155625028766410909433725186998037432429918148135889/132820587880725037739875057690703139558164851986440864530024108835663390613505363960913308303724334769996800000000*t^20 - 215135029812318037741863365307986672620328655246239683852172524257577620753537623316241706018811376176180382103/25501552873099207246056011076615002795167651581396645989764628896447370997793029880495355194315072275839385600000000*t^19 + 1572011605882754430130327497610280364015139111212460767464914419182198571110566990301704843677655468395713/6548934995659786144338985895381356650017373287467038004562051591280783512530310703773845709890876290662400000000*t^18 + 289237780023262696074606126630290713519408952669635207353234431684203172757554815462449126940576926558817/236125489565733400426444547005694470325626403531450425831153971263401583312898424819401436984398817368883200000000*t^17 - 52513338845859287052417869406616990122724889431737199697329139053474205475500831467202411990018756911906831/153009317238595243476336066459690016771005909488379875938587773378684225986758179282972131165890433655036313600000000*t^16 + 4890225749676495174059343993787174590162855677983544557034122213153251107340147416143611616383826234931/3709316781541702872153601611144000406569840230021330325783946021301435781497167982617506210082192331031183360000000*t^15 + 5629426695629208812164386218132817270922815190011244052937619092942845164194733976110299016925714421193/30601863447719048695267213291938003354201181897675975187717554675736845197351635856594426233178086731007262720000000*t^14 + 753975037119399437359707681205310279484280902182843907820165095365610309151696496161622861821531409/588780441514556011452952636689523874058704798416084178695864447825624727221772695653572414298760687465267200000000*t^13 - 61529642596905375445815411238312290520976316603434653616067802716555266394240145926438539238776120473/2448149075817523895621377063355040268336094551814078015017404374058947615788130868527554098654246938480581017600000000*t^12 - 13177330198075010492554372939524112114448516531484565940770209683143140375792818401872906370804001/46631410967952836107073848825810290825449420034553866952712464267789478395964397495762935212461846447249162240000000*t^11 - 2558559955063894779730028956266382596460557376911771344103661264760498141768294750211419759929307/1632099383878349263747584708903360178890729701209385343344936249372631743858753912351702732436164625653720678400000000*t^10 + 184390454229665539533276822100033579915480771579506351286513759492984074594328473416424386033/45336093996620812881877353025093338302520269478038481759581562482573103996076497565325075901004572934825574400000000*t^9 - 22610898613566453954926211114971628517994595777908688555677629363334125254748959645003067/1119409728311625009429070445064033044506673320445394611347692900804274172742629569514199404963075874933964800000000*t^8 - 2197015899607075208076160527522264103950091356713923084371090654776312102269123097/239873513209633930591943666799435652394287140095441702431648478743773037016277764895899872492087687485849600000*t^7 + 33490060072479287510429216752025181139342404900173159204213256812170378154569967/1166051800324609384821948380275034421361118042130619386820513438337785596606905801577291046836537369722880000000*t^6 + 9411011906933298744178460330587784568378492481474747350102310840197462643/1850875873531126007653886317896880033906536574810506963207164187837754915249056827900461979105614872576000000*t^5 + 497293121349219686916495122632515644216259900196192548360841483/58680840848254231189924553694410521850854323993561064607378372166034320456576335479733365643427840000*t^4)*Dt^11 + (752116/171*t^25 - 5905267492649322526928132623442934090622594304640008917436041934793968352314088302950861/4877258627432748862330694124138196993066686030015404784326671223929852411369992210600*t^24 - 11882090597184962821931741856658175575574400069817478295035446924065166935212618248005844825589/15163640935619787850429244566651861361293980201619394244710837168759107639569874282365930000*t^23 - 6801949385202304678938095276486366146478879393129833682548356493868315020867763124900145503066855857/408108166684896722291312516569041535933321665554303728944449587225115119368327768485883693648000000*t^22 + 1394561967401549103463273550435613427040954492550465541668499004081283677650771297952777148445060440554629/422531525511874342250532083529624601896277119858323931332804324536825137917043208479711795148907616000000*t^21 + 68672188884829780304641977024784921545423133766974140723419908895776395999912973898846929409752702765450919/570417559441030362038218312764993212559974111808737307299285838124713936188008331447610923451025281600000000*t^20 - 20050314359123709628917147447724761571674252746663219138185401992106158257553641880286743003012766692140453819/37948739394492867925678587911629468445189957710411675580006888238760968746715818274546659515349809934284800000000*t^19 - 609930895503841732926165756883643032522600618106321308006076605052455012388591647063150317737887754475261380371/2125129406091600603838000923051250232930637631783053832480385741370614249816085823374612932859589356319948800000000*t^18 + 20061838927146140219526698204901652872173881286865655687090583408197835697575314212422285274956733719677614941/2428719321247543547243429626344285980492157293466347237120440847280701999789812369570986208982387835794227200000000*t^17 + 796330125926259394097690521222996724451540743118962695550310878949842996841546994217526896408420996794958191/34002070497465609661408014768820003726890202108528861319686171861929827997057373173993806925753429701119180800000000*t^16 - 1532343979238597231050270172254532094061285759866558926784492063063369074927741585721713046915302459092679519/136008281989862438645632059075280014907560808434115445278744687447719311988229492695975227703013718804476723200000000*t^15 - 10125141259358871325031014588537377538915832319477540430301705793260733403060596303413343863727652925184299/408024845969587315936896177225840044722682425302346335836234062343157935964688478087925683109041156413430169600000000*t^14 + 957719703791817834007731136952350864529570848429517627903680551385322779255078368909949129208818015243481/195851926065401911649710165068403221466887564145126241201392349924715809263050469482204327892339755078446481408000000*t^13 + 1770481563902768052776190693211466718998302172873574276851094116544499724207830707550953387977632748237/34002070497465609661408014768820003726890202108528861319686171861929827997057373173993806925753429701119180800000000*t^12 - 3392570191341488954715192474827467323838653339040884561491215121074904688863768470557860128019936076133/9792596303270095582485508253420161073344378207256312060069617496235790463152523474110216394616987753922324070400000000*t^11 - 18212952449677672477801983688811709931160723062697830264512870826680135012658237788569501808887611999/3264198767756698527495169417806720357781459402418770686689872498745263487717507824703405464872329251307441356800000000*t^10 - 245675573018310421997982593482469037833072135669566677186002482725405891053715978269285430674879/6800414099493121932281602953764000745378040421705772263937234372385965599411474634798761385150685940223836160000000*t^9 + 534938626890914157007710417948982096000035127104146424040297570643215021177992694821688760287/5495284120802522773560891275768889491214578118550119001161401513039164120736545159433342533455099749675827200000000*t^8 - 246877766987543936718834407756145179331041807247474747980247225872324605900874631062945501/503734377740231254243081700278814870028002994200427575106461805361923377734183306281389732233384143720284160000000*t^7 - 168617327620943637837222455317970734106298817813324176500821323050164426625053057421/999472971706808044133098611664315218309529750397673760131868661432387654234490687066249468717032031191040000000*t^6 + 148627031265904585157653355186786531731475687376989087007617335759558175002679/454601091744487089599200148255374043415640562234159604998250853153132786201522729659762591359273828352000000*t^5 + 8223402730874032628081402551414702380349254263110983939875388106155936577/123391724902075067176925754526458668927102438320700464213810945855850327683270455193364131940374324838400000*t^4 + 508482754542711432359877963989889427643021265323930329630752239/4191488632018159370708896692457894417918165999540076043384169440431022889755452534266668974530560000*t^3)*Dt^10 + (51356260/513*t^24 - 18332354400193278052898184804573182059055227500617826016856948258393754811762242197326021/585271035291929863479683294896583639168002323601848574119200546871582289364399065272*t^23 - 2193462136168262004317189242356492726211518576765239498994971439733122697982960304398757956895853/136472768420578090653863201099866752251645821814574548202397534518831968756128868541293370000*t^22 - 33411535094864243105571205985432976563227922424960702163483029712908112497979126948649183040456975077/78148372343916393630251332960029230285104148723164543840426516702681618602445742901552196656000000*t^21 + 9561706048529648081850765829230639890894508177572690127979478690152604902983231971115260029861155565702403/158449322066952878343949531323609225711103919946871474249801621701309426718891203179891923180840356000000*t^20 + 200301383332583593280445676666392345080319482771871691967388541766991521153430494211133739348860830713865991/72819262907365578132538508012552325007656269592604762633951383590389013130384042312460968951194716800000000*t^19 + 17443329748494575499359345680932360983615178979260639722130601788211778506291457545549467286662637048962694027/843305319877619287237301953591765965448665726898037235111264183083577083260351517212147989229995776317440000000*t^18 - 948581660578661938857582554508176935717761419559594673641138599434342880230815305485212606315585318472836614611/193193582371963691258000083913750021175512511980277621134580521942783113619644165761328448441780850574540800000000*t^17 + 37764656758781396514103201476782530032904045680981143037159316888765318929582524574943221259277723475575068381/232890893818257600420602840882328792649932891154307269312918985355683753404502555986258951546256367815884800000000*t^16 + 25148858995955847997596306822254589725193815623551099880147252493877321397416796260437460733906874896686169877/51003105746198414492112022153230005590335303162793291979529257792894741995586059760990710388630144551678771200000000*t^15 - 79908232392836321241366027399217956153298748022541935266870913940124414054877106393580408344356757510606779047/408024845969587315936896177225840044722682425302346335836234062343157935964688478087925683109041156413430169600000000*t^14 - 261510577026447433244950252966098619999720586476246348617103735463536494352388336642853577447348596815663887/163209938387834926374758470890336017889072970120938534334493624937263174385875391235170273243616462565372067840000000*t^13 + 18617121861749651369698568798746598504952378724908349633806164497113158582776291739859113159882472235202779/306018634477190486952672132919380033542011818976759751877175546757368451973516358565944262331780867310072627200000000*t^12 + 731159301706388059820875597071607786183197738113356512622009899130502668843091373708400395620168621078573/816049691939174631873792354451680089445364850604692671672468124686315871929376956175851366218082312826860339200000000*t^11 - 331959194762688739983235069548383166090949021268352205175888875597734906911356667267215390950155995973/612037268954380973905344265838760067084023637953519503754351093514736903947032717131888524663561734620145254400000000*t^10 - 204688930499162598925589330749234870234446517659877966533886488855718605372592544343861403225822259/4121463090601892080170668456826667118410933588912589250871051134779373090552408869575006900091324812256870400000000*t^9 - 218778138400460390492666695511904615515934538009789196851515588773535143866568251558682325476695699/544033127959449754582528236301120059630243233736461781114978749790877247952917970783900910812054875217906892800000000*t^8 + 7310423702046010559119098144568442201548110240693399503656115525616620019079293275100696060181/6044812532882775050916980403345778440336035930405130901277541664343080532810199675376676786800609724643409920000000*t^7 - 409952557602698262047922999352054413484707042787599760858550876754673053736187054406063187/71962053962890179177583100039830695718286142028632510729494543623131911104883329468769961747626306245754880000000*t^6 - 3869211062604637949306646898441144674175154859041765070135394795231034291084977975567/2544113018890056839611523738781893282969712091921351389426574774555168574415067203441362284006990624849920000000*t^5 + 166357932384668458307417026826650268583205613112971385865504709921771545052937943/155473573376614584642926450703337922848149072284082584909401791778371412880920773543638806244871649296384000000*t^4 + 4389585164399511596946348127858535619080077373327302386150499033177464559/12339172490207506717692575452645866892710243832070046421381094585585032768327045519336413194037432483840000*t^3 + 50138164116991859390542346851309452662219193745440442997869349/66531565587589831281093598292982451078066126976826603863240784768746395075483373559788396421120000*t^2)*Dt^9 + (2061077200/1539*t^23 - 29392258325174463115090135345316973789062555632500286668391892418151956832605163239049147/62707610924135342515680353024633961339428820385912347227057201450526673860471328422*t^22 - 2816821861706880129392798706636761482704656250173804615744803839674657937214812982546720596551087/14622082330776223998628200117842866312676338051561558735971164412731996652442378772281432500*t^21 - 782966175256138013497466757603167629439083694817715285552479010939657499135567343583743985003217549027/131177625005859660736493308897191922264281963928169055732144510179501288368391068441891187244000000*t^20 + 87202300255819370021222569125367291942131765343112795170876771313598768084427460426309382223112216206649993/135813704628816752866242455420236479180946217097318406499829961458265222901906745582764505583577448000000*t^19 + 40897240995187421443454178071436063114332662417776520431956046444000647266542724087740411540022270207989487921/1140835118882060724076436625529986425119948223617474614598571676249427872376016662895221846902050563200000000*t^18 + 1848481800370829336259351247266822561137282703908395961750712292009889238702735374105639634944886938275247597089/3162394949541072327139882325969122370432496475867639631667240686563414062226318189545554959612484161190400000000*t^17 - 2002826355848836297777316502602613974280499224976881976607401546607457090559398859583660527767939778914354066943/44273529293575012579958352563567713186054950662146954843341369611887796871168454653637769434574778256665600000000*t^16 + 8446539948633115377837462907323156821769439798095104579428573703926888610046929323064962843294397893637504180269/4250258812183201207676001846102500465861275263566107664960771482741228499632171646749225865719178712639897600000000*t^15 + 101851134857652076656603170898752403426791679946131273937516652538926814999387348697130094102585510242044989379/8500517624366402415352003692205000931722550527132215329921542965482456999264343293498451731438357425279795200000000*t^14 - 13054221515241940329420872623514888925444442017127845453048833073447825806603716973975221008066481036104889793/6800414099493121932281602953764000745378040421705772263937234372385965599411474634798761385150685940223836160000000*t^13 - 2626715910969064110100428558056490571351535776047752696121030764228066654603399361360303474884465038304231/97613599514255338740884252924842115962364216579508692783788053192143046881504420595197531844268219237662720000000*t^12 + 143128082254992474048433322273713558355737820468589867927020967607779910163463523248750863562728278242865013/408024845969587315936896177225840044722682425302346335836234062343157935964688478087925683109041156413430169600000000*t^11 + 6101726168510638680528109987513268796396007921773287129370950685843768635557596491545257231656660863533139/816049691939174631873792354451680089445364850604692671672468124686315871929376956175851366218082312826860339200000000*t^10 + 38910308910723592833729405919432535131004112291314093913776489230482896438622616910725122711105071609383/1632099383878349263747584708903360178890729701209385343344936249372631743858753912351702732436164625653720678400000000*t^9 - 20974607025400969347146004616459935478079161580948552559855274136063874867050511814937021020637812241/108806625591889950916505647260224011926048646747292356222995749958175449590583594156780182162410975043581378560000000*t^8 - 417347468000516901346149267577500295944614817733583400333664810673316595137533292510915972850705039/181344375986483251527509412100373353210081077912153927038326249930292415984305990261300303604018291739302297600000000*t^7 + 6289704417536243440184160263568395060359014054855565250332551700999921562029720504877712733691/755601566610346881364622550418222305042004491300641362659692708042885066601274959422084598350076215580426240000000*t^6 - 22796404890142631652946156460210914648952968256756368357146777525083723454990658123260287/674343209826280126162090629556646412353417662918912416474513795665225405266644318984457472869322816225280000000*t^5 - 1222269281096332696876739338818417434827758594215068788378981173358960488532487495787/172748414862905094047696056337042136497943413648980649899335324198190458756578637270709784716524054773760000000*t^4 - 264265687833940559863419689935817299727525607968425071519750071946387198529986863/77736786688307292321463225351668961424074536142041292454700895889185706440460386771819403122435824648192000000*t^3 + 3704325839813678323192625656288767676329148278376916429940957105946161683/5608714768276139417132988842111757678504656287304566555173224811629560349239566145152915088198832947200000*t^2 + 1388160071754139420293513646084966863193030123827623595619120903/698581438669693228451482782076315736319694333256679340564028240071837148292575422377778162421760000*t)*Dt^8 + (50239287280/4617*t^22 - 3956239409745626874152163584683722933565581710553811920641297147166394031311238590898689313/940614163862030137735205295369509420091432305788685208405858021757900107907069926330*t^21 - 12227537538094805289701876704629115677746621068498723528168953511807760887439280249559241518996099/8773249398465734399176920070705719787605802830936935241582698647639197991465427263368859500*t^20 - 3733733008077635771594449996380825751485876891150841359763655341031050492885311845824203771294757817617/78706575003515796441895985338315153358569178356901433439286706107700773021034641065134712346400000*t^19 + 168527141165684992896843754756892365199099819841001481126802436923169830990318794554519509991925589586206113/40744111388645025859872736626070943754283865129195521949948988437479566870572023674829351675073234400000*t^18 + 190439728121444909779727525077739928111844810993177335573991762676583944576596904677672662509691435520094947109/684501071329236434445861975317991855071968934170484768759143005749656723425609997737133108141230337920000000*t^17 + 45338044845964531662506809658498005044274097562884969678353712074713831506702583322360305199559926419863355731/7106505504586679386831196238132859259398868485095819397005035250704301263429928515832707774410076766720000000*t^16 - 19288125094846579758145963298352009396964146099257183521683381728472217994424338963215115801911801597638882696131/88547058587150025159916705127135426372109901324293909686682739223775593742336909307275538869149556513331200000000*t^15 + 812174701578413160844709459926302216196148384536716558327047489424741019187702141870447586790781603867980725853/52472331014607422316987677112376548961250311895877872406923104725200351847310761070978097107644181637529600000000*t^14 + 360929265465477509385950297655271773445120634957936631917284865922405254554976358025641220567034719352118454451/2125129406091600603838000923051250232930637631783053832480385741370614249816085823374612932859589356319948800000000*t^13 - 125350669547480262573414829636143233229536857167737601336339274188130404725896769464704246302206284008384251/11993675660481696529597183339971782619714357004772085121582423937188651850813888244794993624604384374292480000000*t^12 - 14517310994334441258527213191997524733979072983645914393380668330962108903915288857227822751087689564942095151/68004140994931219322816029537640007453780404217057722639372343723859655994114746347987613851506859402238361600000000*t^11 + 60842317871447182097014600692736677788706308878641507600372171461288746122125036725951678305155257303241963/102006211492396828984224044306460011180670606325586583959058515585789483991172119521981420777260289103357542400000000*t^10 + 8574186723931572826932104218177800888676984210987321780269113984832982832680567806593447150474752460707/283350587478880080511733456406833364390751684237740510997384765516081899975478109783281724381278580842659840000000*t^9 + 159443925281936878329558891612876749667828864892955557914509161553693711397652240532900231344899683301857/816049691939174631873792354451680089445364850604692671672468124686315871929376956175851366218082312826860339200000000*t^8 - 465358902206874456789876076407828573905353383262851001571124660923636801976096526512411391121595961/2248070776691941134638546430996363882769600139406866864111482437152385322119495747040912854595268079412838400000000*t^7 - 601178763749534456431707947946669019251938017756640094766030861445931315442279857226656889500400667/90672187993241625763754706050186676605040538956076963519163124965146207992152995130650151802009145869651148800000000*t^6 + 31810159339820302199777702738626319512886254004045549150342250924716564331357434748839033039369/1007468755480462508486163400557629740056005988400855150212923610723846755468366612562779464466768287440568320000000*t^5 - 1683290247520934339822883507377174324620538458938808023897842648845036745938097875555963603/16791145924674375141436056675960495667600099806680919170215393512064112591139443542712991074446138124009472000000*t^4 - 37357647195495516865992082872257759553254505136713959653097770450713865826259635430773/2332103600649218769643896760550068842722236084261238773641026876675571193213811603154582093673074739445760000000*t^3 - 7616674430635750179122997267291771249269787427767609405475825800465249997610927/294457525334497319399481914210867278121494455083489744146594302610551918335077222620528042130438729728000000*t^2 - 4528226764661288117324046217336725186821186076217310367111502243140797/62319052980845993523699876023463974205607292081161850613035831240328448324884068279476834313320366080000*t + 12695929969538711176145916155481792734210010433394790287014127/7353488828102033983699818758698060382312571929017677269095034106019338403079741288187138551808000)*Dt^7 + (743728425440/13851*t^21 - 1684403534603018404155597646836942441070400583456109242896524607556994777426141156875852119/74259012936476063505410944371277059480902550457001463821515106980886850624242362605*t^20 - 240233235984019422511294793844222051385566224346515814886595841927889431040882228617487224892729897/39479622293095804796296140318175739044226112739216208587122143914376390961594422685159867750*t^19 - 227922296788489781912066784767375271478136954978456954759210148863986130161049107693617025157456996159273/1062538762547463251965595802067254570340683907818169351430370532453960435783967654379318616676400000*t^18 + 60622815222754733188993805271809011618962152620011899113524884753802734576886274065691052631017315204437601/3704010126240456896352066966006449432207624102654138359086271676134506079142911243166304697733930400000*t^17 + 20156064865490584788609031791181375448300905690242216842382701768925619209295267625869855803324842955984466801/15556842530209918964678681257227087615272021231147381108162341039764925532400227221298479730482507680000000*t^16 + 114860950573208992932215611960997488816482683710165312465238561576063853890497805487177971860450360080586494407/3194338332869770027414022551483961990335855026128928920876000693498398042652846656106621171325741576960000000*t^15 - 1163526009884474972254576903933730709949163183399884809773002493082242041421586177972031168845506857314434472977/2683244199610606823027778943246528071882118221948300293535840582538654355828391191129561783913622924646400000000*t^14 + 58274367825024364081090052044572088855650471607968254030501753280869031608438621291914592401262280447725829019/775877840851259804248996320938755105122540208756135024636869566035273548673269742013367262818396990259200000000*t^13 + 96287527692216868011202370796238664173012605726070780308197275054098007978180285719464587848779116841798372239/77277432948785476503200033565500008470205004792111048453832208777113245447857666304531379376712340229816320000000*t^12 - 43918597388112486530696044388658157886287636518696280577675176873665465759229929617625638683234914503591415503/1545548658975709530064000671310000169404100095842220969076644175542264908957153326090627587534246804596326400000000*t^11 - 1343765226795763510663025155274906205884388670428802632762593809529550351522941073557689275550426533419956913/1545548658975709530064000671310000169404100095842220969076644175542264908957153326090627587534246804596326400000000*t^10 - 44058862903741958910155566764685370285619224781911484657817815410997472143892929523397507815229886729084929/18546583907708514360768008055720002032849201150106651628919730106507178907485839913087531050410961655155916800000000*t^9 + 32939428691052418251925814881945161765163084880234934780169078639089470543483064420096201067383266003957/650757330095035591605895019498947439749094777196724618558587021280953645876696137301316878961788128251084800000000*t^8 + 44365104796122714643532454566085652001333131564086619166970233763626827530035383358798655020032934584541/74186335630834057443072032222880008131396804600426606515678920426028715629943359652350124201643846620623667200000000*t^7 + 194953661701215401556631377703734698555946343584835651371557074602576831546548107728428148006424037/749356925563980378212848810332121294256533379802288954703827479050795107373165249013637618198422693137612800000000*t^6 - 6415215782346975193646573139999431141302446580784314040308421645629045602451090365913527614884439/749356925563980378212848810332121294256533379802288954703827479050795107373165249013637618198422693137612800000000*t^5 + 2856815436104365451241092373224398506929749349614228612391244343192579891284944021598690111989/45794034340021023113007427298074079093454817654584325009678345941993034339471209661944521112125831247298560000000*t^4 - 252987849203997505585892801760804084843933116714177149860014525826369455413715712232943921/1908084764167542629708642804086419962227284068941013542069931080916376430811300402581021713005242968637440000000*t^3 - 34599427466115273048104568721577464577605111858872297624937615883637184049977756829/2650117728010475874595337227897805503093450095751407697319348723494967265015695003584752379173948567552000000*t^2 - 17511164026590079161846497487196206761671756918965080830767488270832456426855893/392610033779329759199309218947823037495325940111319658862125736814069224446769630160704056173918306304000000*t - 3520473661686825675510068013272936391457560181277263557511391610171019/5281275676342880807093209832496946966576889159420495814664053494943088841091870193176002907908505600000)*Dt^6 + (342747685280/2187*t^20 - 303568508548677264065556461063756281800579369769229299445908960698329790608074260398931061409/4232763737379135619808423829162792390411445376049083437826361097910550485581814668485*t^19 - 1849749405239058209701334543613051327932012966647075664046016101157104495247923852335767398445385911/118438866879287414388888420954527217132678338217648625761366431743129172884783268055479603250*t^18 - 11834279785059474412915165247978458518635031254069496870961920051726190112661896670202994649425118345066971/22313314013496728291277511843412345977154362064181556380037781181533169151463320741965690950204400000*t^17 + 22078922802755202871712592539635495748800140558208594450085609189369874474849231934104003522541397684136631/555601518936068534452810044900967414831143615398120753862940751420175911871436686474945704660089560000*t^16 + 55242111032291184100825116321946504125831673399523186441593239177069395832716242812761288494101224752166272447/15556842530209918964678681257227087615272021231147381108162341039764925532400227221298479730482507680000000*t^15 + 348580941025953452153306689540816224428805890226706931005624077144674880439658131576252152732355987450006288117/3194338332869770027414022551483961990335855026128928920876000693498398042652846656106621171325741576960000000*t^14 + 2418982030447144725941064041428043633479936079477412235855425864636237171753148936911700740835744437887336231863/8049732598831820469083336829739584215646354665844900880607521747615963067485173573388685351740868773939200000000*t^13 + 4667426364970865958220147602464420976145524790009597681452118720219993712522370357099072500848841290194709988129/21465953596884854584222231545972224575056945775586402348286724660309234846627129529036494271308983397171200000000*t^12 + 305234357428787960007705999088899645863100395314999397881150466794559964325961546086698510610906818864565708111/64397860790654563752666694637916673725170837326759207044860173980927704539881388587109482813926950191513600000000*t^11 - 11890938080965184076285127331513102680551021990947716751270958582029371333676226164918649506559571643813468299/515182886325236510021333557103333389801366698614073656358881391847421636319051108696875862511415601532108800000000*t^10 - 5447052262079277884374073238534633905579848113413128190900196842398663502731098989046016187150561065400505613/3091097317951419060128001342620000338808200191684441938153288351084529817914306652181255175068493609192652800000000*t^9 - 114582814530714857238551984745588499227877844348649444296190511832049126589823614136210643063581540858009/10407735077277505252956233476835017975785185830587346593108714986816598713516184014078300252755870738022400000000*t^8 + 3716516036026275280657894364715684492529055283927230403390676682244193823342893857326253671413627568303/883170662271834017179428955034285811088057197624126268043796671738437090832659043480358621448141031197900800000000*t^7 + 164508339558677869210623199300962851721458561471730412475287380380933513566999569378762620429859389087/206073154530094604008533422841333355920546679445629462543552556738968654527620443478750345004566240612843520000000*t^6 - 1144012690511517013948285296959801341172210759453843341003873158587482988857121108019563926447913/6360282547225142099028809346954733207424280229803378473566436936387921436037668008603405710017476562124800000000*t^5 - 14186784307390734128421046954399023642876036054429178537197003903858896702173709615443350282953/4519148125659969386152048746520468331590935952755032073323520981117733651921500953481367215012417557299200000000*t^4 + 2269441850740419294806558251208862155540651516713930234429523104447461187644186648473840213/40170205561421950099129322191293051836363875135600285096209075387713188017080008475389930800110378287104000000*t^3 - 24394887299692021255742722953875911169558757013310317684356288486309325516241226231877/446335617349132778879214691014367242626265279284447612180100837641257644634222316393221453334559758745600000*t^2 + 57017311795713534687454231782031902758293329147717986216971394681477633890971077/14877853911637759295973823033812241420875509309481587072670027921375254821140743879774048444485325291520000*t - 22155626528739511319262495939352738307555681525297182509413948239384950203/1000662759728124784501871336683632056825094788100725522778452241147111569891091194496505814130032640000)*Dt^5 + (1678803526400/6561*t^19 - 319719632291923297433559169806020945962878622834337170418687329175437998938610930236651746336/2539658242427481371885054297497675434246867225629450062695816658746330291349088801091*t^18 - 6706549150562028957849932442695380608876008137777272200875016547418543614299591999339948323246312/300606261114942676113930002422657911504259741669158948632909725236368459098434690496141125*t^17 - 65150686371127493415386282958101331511178689560244970620887399580250299823851875942279456818816860134047/99613008988824679871774606443805115969439116357953376696597237417558790854746967598061120313412500*t^16 + 13418909204184658707889005730077227319191229377387882454165912242938856051005380222635203688201183750506099/231500632890028556022004185375403089512976506415883647442891979758406629946431952697894043608370650000*t^15 + 15403942266897866416316497372336600983866454908354365943869297945850676313188134201187874125031143298622291/2820312278863292053059949466502372664117480281208734791182440362538964019651962875507338602335480000000*t^14 + 1717738119631135617884993813086088749605174316977535510991903655224969966901083923923682193650724079813669607/9982307290218031335668820473387381219799546956652902877737502167182493883290145800333191160392942428000000*t^13 + 13580684805130774802291581191087492719305256626287713193669974751156752290177798926325439999546423076509111257/5323897221449616712356704252473269983893091710214881534793334489163996737754744426844368618876235961600000000*t^12 + 40429712011955846640785123028164089558789579150808836459736223830737292196632043735549374467393144170094921339/114996179983311720986904811853422631652090780940641441151536024965942329535502479619838362167726696770560000000*t^11 + 222508967332109996911985643496141395143889458032398241962162870356245219472220695282933911841729810512054903/24467272336874834252532938692217581202572506583115200245007664886370708411809038216986885567601424844800000000*t^10 + 197445024042669123023524598343649350175013346844955496296953349656600824805033935194816709480541053867999511/4599847199332468839476192474136905266083631237625657646061440998637693181420099184793534486709067870822400000000*t^9 - 7021278931982046048932835384346071210236512143762146929315938540245555749990317698378441516651296348424261/4599847199332468839476192474136905266083631237625657646061440998637693181420099184793534486709067870822400000000*t^8 - 265132534940320793243189083542378928469917174247125017769112896608438729951787499329114335982296916846963/18399388797329875357904769896547621064334524950502630584245763994550772725680396739174137946836271483289600000000*t^7 - 387959926293016870431276636560657167778985591669776481296844294487069240565686176784890620160083223221/6133129599109958452634923298849207021444841650167543528081921331516924241893465579724712648945423827763200000000*t^6 + 104218722431781850522838510652127494653233470343581093055860408955690086345789097907898995370584909/209083963606021310885281476097132057549255965346620802093701863574440599155459053854251567577684903219200000000*t^5 - 17044862196619190954300546417967304229785340719902633001910628172479398320591929016932190248239/10395134913745692292601564913303740714313290932487361912003256494096481765921128101228326523636311572480000000*t^4 + 92218491125799571597022033836574532347891668261791743333205036429010889435310659325223214011/71732509931110625177016646770166163993506919885000509100373348906630692887642872277482019285911389798400000000*t^3 + 13521977978968294877204888720640933279579893924187109125452607345082969262326551831529493/797027888123451390855740519668512933261187998722227878893037210073674365418254136416466880954570997760000000*t^2 + 9962808233244575904532749150591732167891558797047374627882870077530366278560252097/2415236024616519366229516726268221009882387874915842057251627909314164743691679201262020851377487872000000*t + 188202082892058962681522884985047413713215760079411859195401370363239952589/32489050640523531964346471970247794052762817795478101388910787050230895126334129691444993965260800000)*Dt^4 + (4168900736000/19683*t^18 - 848037225575396444443372262272632522622744249835537815381550951523651234546766582238142215104/7618974727282444115655162892493026302740601676888350188087449976238990874047266403273*t^17 - 318206822691153947778411560995607071483072866307481310227301987139794172948921973754266804386554632/19739811146547902398148070159087869522113056369608104293561071957188195480797211342579933875*t^16 - 10886264663846523806445396664146338530556521490362225602635033275362520470332123270265307234778617908421/33204336329608226623924868814601705323146372119317792232199079139186263618248989199353706771137500*t^15 + 785059934593657353476490325639322343852679424983371267393032772460073009244509321309655110333351240699333/16535759492144896858714584669671649250926893315420260531635141411314759281887996621278145972026475000*t^14 + 2404045856037874149421668996160587757364671302521671139032364476119099731550897002271087787759562463161805173/555601518936068534452810044900967414831143615398120753862940751420175911871436686474945704660089560000000*t^13 + 10914988809207047846933252666057951717149876869930995667880982741981732689747440621961442858006810151342465259/85562633916154554305732746914748981883996116771310596094892875718707090428201249717141638517653792240000000*t^12 + 94742273051106184494853479390825093539995005679003370573061654927969629124952917947712776685013354633942519/27490002864627969254853894591084010243854174063071677460206546415648864394602811154101731250651820800000000*t^11 + 23625792429429758092356581937643762181845439645971329694659667529022270636074287964063737762538434643388731569/82140128559508372133503437038159022608636272100458172251097160689958806811073199728455972976947640550400000000*t^10 + 166650210289015444284711017094588103951407475078690764930296976826109342293797532496347729320422038009438091/20535032139877093033375859259539755652159068025114543062774290172489701702768299932113993244236910137600000000*t^9 + 54602072783647971546945598507977167082845145302527237518584261111196497396351172012995302553779792595599411/657121028476066977068027496305272180869090176803665378008777285519670454488585597827647783815581124403200000000*t^8 - 4055373197200168816477261220826606874241061793414191509103495551856921060737922743802715598165684071877/14719511037863900286323815917238096851467619960402104467396611195640618180544317391339310357469017186631680000*t^7 - 134282546107351607644398400382527818494254316126464816533827585763747749102206220775223477586960250187/22275289100883626341289067671365158673528480569615775525721263915921032355545274502632128264934953369600000000*t^6 - 22710036261818477556910414992011608777529382742598940801076393153760238906198349003922273012248158661/511094133259163204386243608237433918453736804180628627340160110959743686824455464977059387412118652313600000000*t^5 + 1718427913506061934025474271147491944898207301587706479379143648911353242071654182508797728596631593/9199694398664937678952384948273810532167262475251315292122881997275386362840198369587068973418135741644800000000*t^4 - 2316371167519653320371561683100324381429393549091000438505065743187142418891715943366230078342177/2044376533036652817544974432949735673814947216722514509360640443838974747297821859908237549648474609254400000000*t^3 + 9829525149410585695610110570032025903243852162617395670124238669169839170499532178308623/15526517301106195925761178954581420777815350624458984653760465131305344780875080579541562615998136320000000*t^2 - 691428902782804068233459359792448731058620389881954173795728483832112345534771084029/3622854036924779049344275089402331514823581812373763085877441863971247115537518801893031277066231808000000*t + 13436493489919957903127139963860633295666219383000265551937285180210280836239/48733575960785297946519707955371691079144226693217152083366180575346342689501194537167490947891200000)*Dt^3 + (4404989388800/59049*t^17 - 951961755620812838475311377505922938890138557044352819460168330004465898452691661214870656192/22856924181847332346965488677479078908221805030665050564262349928716972622141799209819*t^16 - 726712243804007402029043685248685240518962154002280455860307492110461456253146489343841062585248/146220823307762239986282001178428663126763380515615587359711644127319966524423787722814325*t^15 - 33078648703803479515994496675231408300311201153736620951566093680237331107698026530847391289769608217/1106811210986940887464162293820056844104879070643926407739969304639542120608299639978456892371250*t^14 + 10206541530212437030411783881308242256271742802004092872420597038743035325231348316918861527547794934869/551191983071496561957152822322388308364229777180675351054504713710491976062933220709271532400882500*t^13 + 1404806863332328620583574382755595923298381037072485797396859776971571325545907304389793837770698158783363/926002531560114224088016741501612358051906025663534589771567919033626519785727810791576174433482600000*t^12 + 10425636478510287927353508595378298635329008854315700792545249842787803565420742531405252754908495607867031/285208779720515181019109156382496606279987055904368653649642919062356968094004165723805461725512640800000*t^11 + 912393325393215853620911919593981222747066092655830893341767694659424769873581564293232271715172089655509/570417559441030362038218312764993212559974111808737307299285838124713936188008331447610923451025281600000*t^10 + 696356748567270750472153739779756728412096311348544421072469068556357537856194940385729075700362256075317/6845010713292364344458619753179918550719689341704847687591430057496567234256099977371331081412303379200000*t^9 + 242236267432515414589292990566844119403174964589010217433772447494667463501340461250958921465068385441/84506305102374868450106416705924920379255423971664786266560864907365027583408641695942359029781523200000*t^8 + 1018716342540214233047707422276816254234727615261462139963619812208574763007859538333402160215648503403/27380042853169457377834479012719674202878757366819390750365720229986268937024399909485324325649213516800000*t^7 + 519972370370885127993126071670445938598170710735611887060066326881859808518963497082706494438498597/3549264814299744474904469501648846655928727806809921023195556326109331158503162951229579079250823974400000*t^6 - 3496716646615742038793368064415248181760732650748184534770721873114262193198250529708724305211807/14197059257198977899617878006595386623714911227239684092782225304437324634012651804918316317003295897600000*t^5 - 298674185015937059929065752544752470351991783134678274312917503423777536326611558734234029236731/42591177771596933698853634019786159871144733681719052278346675913311973902037955414754948951009887692800000*t^4 + 357654853068884354386183538630699310269848331766315951101416421728362317703137881952640065393/9464706171465985266411918671063591082476607484826456061854816869624883089341767869945544211335530598400000*t^3 - 87801736636676583572123856729610379404492030678947491631711755392555344523803318381369149/525817009525888070356217703947977282359811526934803114547489826090271282741209326108085789518640588800000*t^2 - 10289472098669366338237515164048444935251947606970422293544049318587075034644360963/419311809829256834414846653866010591993470117172889246050629845367042490224249861330211953364147200000*t - 329514992840884254187350332180179130078422099457589437776333117081909429919/5640460180646446521587929161501353134160211422826059268908122751776197070544119738098089230080000)*Dt^2 + (1392341350400/177147*t^16 - 45525722671206225391170076524555543598038683419996534761293557503538233032855281340197720384/9795824649363142434413780861776748103523630727427878813255292826592988266632199661351*t^15 - 111708513609110903575289146326878921149188978638924702347122672178173731378802144926737837393376/241711973223035539569160042764341259454445588199282909717074350496181985479149526643835925*t^14 + 180888259987352935413861228586235919604423545120375608605514365059355417742576379588527335301951/25814838740220195625986292567231646509734788819683414757783540632992236049173169445561676790*t^13 + 151258269174228093325501207492439898661431637829507297229274928812548978025689851274173915526023857/62493422116949723577908483256506610925649634600983599892800987949035371435706714366130559229125*t^12 + 2933029527457709095777700588225385523398349202068358402257211557926125463380984802999680093784162059/16798231865036085697741800299348977016814621780744391651184905560700707841917964821615894320788800*t^11 + 1209943427149226866941582571544386307954758994978979193827568698204372366448973596539001739125064371/443473321236952662420383527902812993243906015011651939591281506802498687026634271290659610068824320*t^10 + 37329452267863158029869200910603401990114682925591668015571867942172952890651762225517902879600961/173231766108184633757962315587036325485900787113926538902844338594726049619779012222913910183134500*t^9 + 1038816495916027264315040211065100989257149504630712139984270099400968203049132461809833498427861481/88694664247390532484076705580562598648781203002330387918256301360499737405326854258131922013764864000*t^8 + 52458693057142558054135181026386776758064643445642878931007640812155730403372761967117715749770567/177389328494781064968153411161125197297562406004660775836512602720999474810653708516263844027529728000*t^7 + 4534479663243603117094323495036840130866807087224649105080906018874989971627583179804345278300979/1241725299463467454777073878127876381082936842032625430855588219046996323674575959613846908192708096000*t^6 + 1630008049657315079646015620516275102149006163618559014344276879608919988791127979986472003321/68984738859081525265392993229326465615718713446256968380866012169277573537476442200769272677372672000*t^5 + 81429432206547063855832040508597693460120609645198759583767431416542060237480854306706207931/1241725299463467454777073878127876381082936842032625430855588219046996323674575959613846908192708096000*t^4 + 14171965097386359883818183355125664955045799450814158191822049029329615653450486548652733/275938955436326101061571972917305862462874853785027873523464048677110294149905768803077090709490688000*t^3 + 32799546959042749760170609490141971017997464274943121587451922424963182186868748640823/15329941968684783392309554050961436803493047432501548529081336037617238563883653822393171706082816000*t^2 + 1180308512600142196768477334847955850711759236187504841777954821307863830703717329/2554990328114130565384925675160239467248841238750258088180222672936206427313942303732195284347136000*t + 117351810378567979683873739497049953893880894979431269752266694056094853/164444903225843921912184523658931578255399749936619803758254307631959098266592412189448665600)*Dt + 16502886400/177147*t^15 - 191972324889165527808658786900550003674285675369552052435387577766038770415950070284655104/3265274883121047478137926953925582701174543575809292937751764275530996088877399887117*t^14 - 2158214288165436317844275361321783985409567894315091442274511673941621851983333257106391936/460403758520067694417447700503507160865610644189110304222998762849870448531713384083497*t^13 + 165142917896297147740947939960779388960884875091881976011824908801377668844520390645261566904/614639017624290372047292680172182059755590209992462256137703348404577048789837367751468495*t^12 + 15609244382933533035989128480227717349986283587745927328225962364213862053701110807458403502784/357105269239712706159477047180037776717997912005620570816005645423059265346895510663603195595*t^11 + 1046804609515715776453200537098547836809985335444793355668970021806761103178747085253943481523/357105269239712706159477047180037776717997912005620570816005645423059265346895510663603195595*t^10 + 508155188579725532880345868932369659603754785891335569023721563192759756853347380137960149219/23568947769821038606525485113882493263387862192370957673856372597921911512895103703797810909270*t^9 + 15462824276603079195266372196695056868220332524092709048549128030479183460255429920971283902/3928157961636839767754247518980415543897977032061826278976062099653651918815850617299635151545*t^8 + 2442966831131677525543243043299475561373757255388152130834120076191361279121433442862813304/11784473884910519303262742556941246631693931096185478836928186298960955756447551851898905454635*t^7 + 126243345644088572295742940396524289115343817376676952916230425315135301397407340781506899/27497105731457878374279732632862908807285839224432783952832434697575563431710954321097446060815*t^6 + 2735211986399781098356235359530802160783804919796267283779363292218953572525932212925561/73325615283887675664745953687634423486095571265154090540886492526868169151229211522926522828840*t^5 + 7402129695218084125017081647403586032355556428255814810304578284332423125314159920955/43995369170332605398847572212580654091657342759092454324531895516120901490737526913755913697304*t^4 - 12396906503283824737461228223375808653809644277746972568500891204859062312436104531/48883743522591783776497302458422948990730380843436060360590995017912112767486141015284348552560*t^3 + 216023602226394902209584234386727591967330704323027506243503516413937299604709/46029890322591133499526650149174151592024840718866346855547076288052836880872072519100139880*t^2 + 412474137154635397103793803106648014408808160237429770488737005403068255553/452627254838812812745345393133545823988244267068852410746212916832519562661908713104484708820*t + 9671877008795095598599669189069940837178516365333903874391855438789/6088609831030573214223101871583882485717571523659569689887179403181592180009533401997373, (t^54 + 24427931652403/4770518770710*t^53 - 2322604945469/7269361936320*t^52 - 226745178472975643/6594765148629504*t^51 - 2735819423735606039/23980964176834560*t^50 - 24241834093308768589/61665336454717440*t^49 - 79736103819507265208233/75971694512211886080*t^48 - 122554480425349475376341/75971694512211886080*t^47 - 815815681608439636616341/607773556097695088640*t^46 - 90080678978727013362271/54024316097572896768*t^45 - 2654377357846805088376867/540243160975728967680*t^44 - 76020372139869383978283139/19448753795126242836480*t^43 + 114954105107157686528822107/12965835863417495224320*t^42 + 3094351258219054295131513409/155590030361009942691840*t^41 + 132613342974167776083230597/12965835863417495224320*t^40 - 84838706361840929926650319/9220149947319107715072*t^39 - 785065315307828717007050941/62236012144403977076736*t^38 - 19574258925361600616971723651/4978880971552318166138880*t^37 + 201711133321529885940265361/1659626990517439388712960*t^36 - 332590912236952456078208321/4978880971552318166138880*t^35 + 337034816392170818833327619/711268710221759738019840*t^34 + 11539147298407105257754708763/9957761943104636332277760*t^33 + 153479067855454444119279469/311180060722019885383680*t^32 - 257773506178648365161232469/1244720242888079541534720*t^31 - 1527887258849310252303676331/9957761943104636332277760*t^30 - 11192442391622902098563459/331925398103487877742592*t^29 + 76772386224655015838329/26343285563768879185920*t^28 + 1178539888793877821627971/226312771434196280279040*t^27 + 10190928290433726386986661/9957761943104636332277760*t^26 + 1892113428998959212640589/4978880971552318166138880*t^25 - 11820881441283535902355/142253742044351947603968*t^24 - 5147053866674334203503/184402998946382154301440*t^23 + 313476836595748681895339/9957761943104636332277760*t^22 + 2056990720049846348321/2489440485776159083069440*t^21 - 403683904362128678857/138302249209786615726080*t^20 + 400099496183060540981/4978880971552318166138880*t^19 - 276181253456955812941/1422537420443519476039680*t^18 - 46441640724900128753/1106417993678292925808640*t^17 + 118737151317741613637/4978880971552318166138880*t^16 - 2941469779735424099/1659626990517439388712960*t^15 + 273491894885565113/829813495258719694356480*t^14 - 98743043179273/8890858877771996725248*t^13 + 344251150989457/17287781151223326965760*t^12 + 1182512036473/12965835863417495224320*t^11 + 5590255801/12965835863417495224320*t^10 - 19187612569/1080486321951457935360*t^9 - 9906821/9647199303138017280*t^8 + 659749/3376519756098306048*t^7)*Dt^8 + (445/9*t^53 + 1655527523245532/7155778156065*t^52 - 33633456097758397/4121728217893440*t^51 - 19499578542850841/12880400680917*t^50 - 13555776399734585061317/2374115453506621440*t^49 - 777638399728941191849/37095553961040960*t^48 - 4343693242658211267459787/75971694512211886080*t^47 - 486945565993337652355031/5064779634147459072*t^46 - 9241942236590955870569861/86824793728242155520*t^45 - 37595927903094829945402375/243109422439078035456*t^44 - 21611482557092689270657363/63145304529630658560*t^43 - 1029243573166471238976283831/3241458965854373806080*t^42 + 11163762289354701196453187017/38897507590252485672960*t^41 + 71325298934292316188590918749/77795015180504971345920*t^40 + 3092235236780916288266375057/4714849404879089172480*t^39 - 95202611223169816968663425447/622360121444039770767360*t^38 - 169856269047463074999306650851/414906747629359847178240*t^37 - 24060200775163974620830536313/138302249209786615726080*t^36 - 2508103633513149469338708953/59272392518479978168320*t^35 - 9514857653046228609924967471/237089570073919912673280*t^34 + 171933089265653331078461407/226312771434196280279040*t^33 + 33390217752682781914827108149/711268710221759738019840*t^32 + 81524179164052366924405516709/3319253981034878777425920*t^31 - 610375157362195451476451989/138302249209786615726080*t^30 - 1625798393123553822276670313/301750361912261707038720*t^29 - 2766723060606704248758597397/1991552388620927266455552*t^28 + 48581567974441652304899639/829813495258719694356480*t^27 + 12019497841572610822394453/248944048577615908306944*t^26 - 189241722526942004636487403/9957761943104636332277760*t^25 + 111611769612484802479473703/9957761943104636332277760*t^24 + 196261833978184004345201/237089570073919912673280*t^23 - 7271598160747209306053/71126871022175973801984*t^22 + 9942998529701370263500879/9957761943104636332277760*t^21 + 537449854942375736243039/3319253981034878777425920*t^20 - 156211815001987946880191/2489440485776159083069440*t^19 + 41516202382487100848807/4978880971552318166138880*t^18 - 53310693745675581881117/9957761943104636332277760*t^17 - 10146934587222030459689/9957761943104636332277760*t^16 + 3397807588423589639807/9957761943104636332277760*t^15 - 33967973496351850369/829813495258719694356480*t^14 + 2111246196723790691/553208996839146462904320*t^13 - 519874206385619/853717587714732195840*t^12 + 56498907372255763/138302249209786615726080*t^11 + 405238095821089/103726686907339961794560*t^10 + 2045856622717/4321945287805831741440*t^9 - 1799259815599/4321945287805831741440*t^8 - 62987809721/2160972643902915870720*t^7 + 3298745/666966865402134528*t^6)*Dt^7 + (8093/9*t^52 + 3888773277319133/1022254022295*t^51 - 621866823704732167/4121728217893440*t^50 - 307268813531403692453/12365184653680320*t^49 - 49901062448036709451061/474823090701324288*t^48 - 321284294763408052246631/791371817835540480*t^47 - 5734111042348014445500715/5064779634147459072*t^46 - 8619516381108541231588733/4220649695122882560*t^45 - 1620732869572568015259800183/607773556097695088640*t^44 - 3427034390295810173103404849/810364741463593451520*t^43 - 6799984057889979658020206207/810364741463593451520*t^42 - 15475977730959090460339559879/1768068526829658439680*t^41 + 28654731050289921362387575837/19448753795126242836480*t^40 + 20021394956969743518561130127/1414454821463726751744*t^39 + 2027938838400023481340972228001/155590030361009942691840*t^38 + 925347647832383172424456991687/622360121444039770767360*t^37 - 2350134489704754062906451397813/622360121444039770767360*t^36 - 60458469546562968650043938333/27660449841957323145216*t^35 - 155521665596824639740017037691/150875180956130853519360*t^34 - 1321407935055672020408312933/1178712351219772293120*t^33 - 44027939924819723493126741493/138302249209786615726080*t^32 + 3093899008200489461041938633953/4978880971552318166138880*t^31 + 1950290220998139867811992899989/4978880971552318166138880*t^30 - 24756961907558009615371182685/1991552388620927266455552*t^29 - 127582640323395639309779509715/1991552388620927266455552*t^28 - 182710293165021999871429224733/9957761943104636332277760*t^27 + 3503494780524526800938168017/3319253981034878777425920*t^26 - 427555559477463497287846651/1422537420443519476039680*t^25 - 8307344010295538249782675393/9957761943104636332277760*t^24 - 6622443390706828250964295/663850796206975755485184*t^23 - 8094190840344838717162153/3319253981034878777425920*t^22 - 5694021925089575338312907/1422537420443519476039680*t^21 + 23243139123226291166935825/1991552388620927266455552*t^20 + 597458768628177115033589/221283598735658585161728*t^19 - 1842948245473853200811423/3319253981034878777425920*t^18 + 141312756759233412575207/905251085736785121116160*t^17 - 492299685039068860161907/9957761943104636332277760*t^16 - 934820413423597016923/181050217147357024223232*t^15 + 291988158240345355979/905251085736785121116160*t^14 - 248310659628730707149/995776194310463633227776*t^13 - 93294068214048894133/4978880971552318166138880*t^12 - 1041682838399315627/138302249209786615726080*t^11 + 308443625021004467/138302249209786615726080*t^10 + 1289587970481233/25931671726834990448640*t^9 + 2989482952529/270121580487864483840*t^8 - 1343065993739/432194528780583174144*t^7 - 23213131751/80036023848256143360*t^6 + 240808385/6002701788619210752*t^5)*Dt^6 + (7578*t^51 + 3244920484989053/113583780255*t^50 - 58180267373003635/22898490099408*t^49 - 52757397170748723727/274781881192896*t^48 - 354544165068289890859237/395685908917770240*t^47 - 8564956866418637658379193/2374115453506621440*t^46 - 8872144710297611256474881/863314710366044160*t^45 - 67603212385658711259049577/3453258841464176640*t^44 - 8645841482457448452040959683/303886778048847544320*t^43 - 16695767302838745888346086623/347299174912968622080*t^42 - 221737257177336723536235243149/2431094224390780354560*t^41 - 671179350769177466690740778107/6482917931708747612160*t^40 - 62814037644845567131807771493/2431094224390780354560*t^39 + 54222430979483272322424205357/617420755400833105920*t^38 + 1481570410905344695989077226557/14144548214637267517440*t^37 + 5512518661328326050223576915253/155590030361009942691840*t^36 - 4047271664613210228679339638347/622360121444039770767360*t^35 - 5108827249178529570376963188179/622360121444039770767360*t^34 - 109150595294662713354531953293/14818098129619994542080*t^33 - 54198517955502334113599180088509/4978880971552318166138880*t^32 - 23221714296074188662928016444171/4978880971552318166138880*t^31 + 1648567270899184137952337462555/497888097155231816613888*t^30 + 1629047878414389695186461115837/622360121444039770767360*t^29 + 523712816315469150952185569827/2489440485776159083069440*t^28 - 519484592031232014613305779981/1659626990517439388712960*t^27 - 116434323371987575810478852137/1244720242888079541534720*t^26 + 77011650241050487166930631193/4978880971552318166138880*t^25 - 349832965784453412601015157/155590030361009942691840*t^24 - 6571444545209209072593428701/995776194310463633227776*t^23 - 991790069009062583878370591/995776194310463633227776*t^22 - 1597854778709097321781876693/4978880971552318166138880*t^21 - 24971068911558029045787443/237089570073919912673280*t^20 + 10364447379586725713990143/150875180956130853519360*t^19 + 271635669377681455441753/17287781151223326965760*t^18 - 13104981431134580090171867/4978880971552318166138880*t^17 + 112017759178990593415373/110641799367829292580864*t^16 - 392868105144492986518117/2489440485776159083069440*t^15 + 73408426527437152918301/4978880971552318166138880*t^14 - 7476708052356921360853/995776194310463633227776*t^13 - 591953681654328107419/4978880971552318166138880*t^12 - 12492240564968304887/45262554286839256055808*t^11 - 5729383431025095139/248944048577615908306944*t^10 + 5009813646006137/3143232936586059448320*t^9 + 5777477111634353/25931671726834990448640*t^8 + 1100460262957039/12965835863417495224320*t^7 - 19255648516319/2160972643902915870720*t^6 - 228025642399/180081053658576322560*t^5 + 1627600783/13506079024393224192*t^4)*Dt^5 + (91850/3*t^50 + 6858610406412005/68150268153*t^49 - 1439964667411813285/68695470298224*t^48 - 993367247692741063223/1373909405964480*t^47 - 5954190032861644029637/1628337073735680*t^46 - 36166805292310909380414731/2374115453506621440*t^45 - 5247513036559792594147945/118705772675331072*t^44 - 475668784391687278817501677/5426549608015134720*t^43 - 13867406249775312378343194673/101295592682949181440*t^42 - 28227916972793747906613997651/115766391637656207360*t^41 - 1105154812882385641775451766429/2431094224390780354560*t^40 - 708393768673286224711349429755/1296583586341749522432*t^39 - 2624513226817006550256491342843/9724376897563121418240*t^38 + 7791924313775822115135992117759/38897507590252485672960*t^37 + 9093776023577609457597413016941/25931671726834990448640*t^36 + 9254157181239798332131669213961/51863343453669980897280*t^35 + 1105927989981930320042885857409/25931671726834990448640*t^34 + 380965422297705767242108020079/77795015180504971345920*t^33 - 13782774044591182795397052765677/829813495258719694356480*t^32 - 13795305987192954378361713950699/311180060722019885383680*t^31 - 38375426242458769964942095581749/1659626990517439388712960*t^30 + 2873577544157692306057962329213/414906747629359847178240*t^29 + 706406266558612616726610456151/92201499473191077150720*t^28 + 2147300118762716256208497298637/1659626990517439388712960*t^27 - 296813551822307741693639811685/497888097155231816613888*t^26 - 45774376021290823610629189909/355634355110879869009920*t^25 + 574798949348176558856580093079/4978880971552318166138880*t^24 + 24040621356062706959708298929/2489440485776159083069440*t^23 - 35920330272491533816956121297/2489440485776159083069440*t^22 - 14532740271998292725815772753/4978880971552318166138880*t^21 - 285959993319324480568351883/155590030361009942691840*t^20 - 1509669571029451918209964441/2489440485776159083069440*t^19 + 72081963507130537577241901/311180060722019885383680*t^18 + 26492922853550404377629021/622360121444039770767360*t^17 - 4592191514387084686950209/995776194310463633227776*t^16 + 681493026881765867476369/276604498419573231452160*t^15 + 15882065710824178690721/452625542868392560558080*t^14 + 576837722354931087666127/4978880971552318166138880*t^13 - 6712578829895667518483/497888097155231816613888*t^12 + 7430261069989627855519/4978880971552318166138880*t^11 - 401281424084175360269/622360121444039770767360*t^10 + 44784342461457346043/1244720242888079541534720*t^9 - 119344762424663/14032289895473479680*t^8 + 62698955473037/308710377700416552960*t^7 + 75556166028967/294678087804943073280*t^6 - 22274869083403/2160972643902915870720*t^5 - 20648977163/8441299390245765120*t^4 + 740898127/6753039512196612096*t^3)*Dt^4 + (165160/3*t^49 + 10327886729536568/68150268153*t^48 - 2392628582335026383/34347735149112*t^47 - 633208399156218768263/515216027236680*t^46 - 24185743400916587659409/3663758415905280*t^45 - 8395541187493077836890607/296764431688327680*t^44 - 263006907311245985529638837/3165487271342161920*t^43 - 799723705537386485605065541/4748230907013242880*t^42 - 41751972628882394780247593377/151943389024423772160*t^41 - 156413259640067266216608570733/303886778048847544320*t^40 - 35095470730007692400151660281/36834760975617884160*t^39 - 3812018479983853783616319281059/3241458965854373806080*t^38 - 128480580943400858170310358787/176806852682965843968*t^37 + 508812325491266881510526315011/4862188448781560709120*t^36 + 3171228068052696978459873894043/7072274107318633758720*t^35 + 44971462568628011621552351293787/155590030361009942691840*t^34 + 1134853895949775368269716586083/8890858877771996725248*t^33 + 1096462071969562151895837538399/20745337381467992358912*t^32 - 43269904816432819051241862337/11525187434148884643840*t^31 - 22808382800995796287259323287439/311180060722019885383680*t^30 - 12206143149664994758267483911443/311180060722019885383680*t^29 + 4472693973866521897966171486639/622360121444039770767360*t^28 + 3101470085483029422099253113289/311180060722019885383680*t^27 + 2428777790256387581741237286113/1244720242888079541534720*t^26 - 53481097863189301348041150853/124472024288807954153472*t^25 + 87886471746207696853000322947/414906747629359847178240*t^24 + 92112730500021214660257619519/248944048577615908306944*t^23 + 272345606746979700338574097/3951492834565331877888*t^22 + 113987168721485536288211083/35563435511087986900992*t^21 + 160792280467147890185625395/82981349525871969435648*t^20 - 420041561875063404164648267/124472024288807954153472*t^19 - 147909462813161673903382109/124472024288807954153472*t^18 + 101456565862530864304915331/248944048577615908306944*t^17 + 66734050884143103840567229/1244720242888079541534720*t^16 + 153758177104097817432329/82981349525871969435648*t^15 + 106955368267268409749833/59272392518479978168320*t^14 + 957114666845325858389711/1244720242888079541534720*t^13 + 4541900184160671429667/37718795239032713379840*t^12 + 146672090102441256205/8890858877771996725248*t^11 - 3664771612616365859/15366916578865179525120*t^10 - 546819914344870222589/1244720242888079541534720*t^9 + 26100423310141243/164645534773555494912*t^8 - 107355737821863863/12965835863417495224320*t^7 - 3183787023920267/5186334345366998089728*t^6 + 14486226549959/48021614308953686016*t^5 - 504227684977/98226029268314357760*t^4 - 58068151409/33765197560983060480*t^3 + 659749/1125506585366102016*t^2)*Dt^3 + (34200*t^48 + 527006326234032/7572252017*t^47 - 301193259455008273/3816415016568*t^46 - 3442538574785907137/4770518770710*t^45 - 132025113338993428677113/32973825743147520*t^44 - 864809994964784630036449/49460738614721280*t^43 - 160384366115059645906957103/3165487271342161920*t^42 - 45221679969194398877670911/452212467334594560*t^41 - 2735752714698056654178401321/16882598780491530240*t^40 - 4123019354839867567528965877/12661949085368647680*t^39 - 80136633002241198039090239447/135060790243932241920*t^38 - 203102302739681037742540218781/294678087804943073280*t^37 - 297138615779825974658422297567/810364741463593451520*t^36 + 202027240393222315432215833/2046375609756549120*t^35 + 647933363376180139399260454543/2881296858537221160960*t^34 + 6032651465095668527925151449179/51863343453669980897280*t^33 + 4408626304088369470841541679243/103726686907339961794560*t^32 + 747167262636112960121692399723/51863343453669980897280*t^31 - 1221126987791890962937942277893/207453373814679923589120*t^30 - 52643559661402263674351733959/1234841510801666211840*t^29 - 148852674289807553246908428671/17287781151223326965760*t^28 + 916716566451893767302154746833/69151124604893307863040*t^27 + 410710115199729594487539198347/59272392518479978168320*t^26 - 137101155665399677950852629057/414906747629359847178240*t^25 - 3396148551326178903270835775/3457556230244665393152*t^24 + 7577534325661935349282382917/34575562302446653931520*t^23 + 4215818805557009376835410401/11525187434148884643840*t^22 + 235381798257866423862230423/2881296858537221160960*t^21 + 8784937061330889503135229449/414906747629359847178240*t^20 + 129567204660033866702433301/23050374868297769287680*t^19 - 236603462612005609330429697/59272392518479978168320*t^18 - 4500990920558018710513309/3704524532404998635520*t^17 + 4392298518565849918176127/18859397619516356689920*t^16 + 199288444238721724570657/207453373814679923589120*t^15 + 19093809110259759429067/41490674762935984717824*t^14 - 153469387393578688488589/207453373814679923589120*t^13 + 1196577972136780280677/1796133106620605399040*t^12 - 4092563868601258077887/207453373814679923589120*t^11 + 109126196915414351179/3771879523903271337984*t^10 - 1551307692153383809861/414906747629359847178240*t^9 - 100580452419135647/179613310662060539904*t^8 + 189722880836058271/3241458965854373806080*t^7 - 73774254346092011/25931671726834990448640*t^6 - 3970835678842801/4321945287805831741440*t^5 + 68987211018871/540243160975728967680*t^4 + 239222655215/108048632195145793536*t^3 - 18229963/234480538617937920*t^2 - 659749/562753292683051008*t)*Dt^2 + (1040*t^47 - 12155874425912/2065159641*t^46 - 2161169416714013/130105057383*t^45 - 49750362579081961/8178032178360*t^44 + 436938883844944886539/16486912871573760*t^43 + 4914912705508150092853/14131639604206080*t^42 + 2289737863451126823022031/791371817835540480*t^41 + 9404697795532454214129223/791371817835540480*t^40 + 718660568510557596747316153/25323898170737295360*t^39 + 946229963868882757450528447/20259118536589836288*t^38 + 1358761491771433624874008147/13506079024393224192*t^37 + 48128543841362113720348891217/231532783275312414720*t^36 + 495472105561223004466201668641/1620729482927186903040*t^35 + 503766459833811828302430041/1875844308943503360*t^34 + 46500806602904336036387130721/392904117073257431040*t^33 - 17711464583157221588705290285/5186334345366998089728*t^32 - 318414366387788177077958559929/5762593717074442321920*t^31 - 931407472517563003391346440249/17287781151223326965760*t^30 - 18890627427602069209231393427/523872156097676574720*t^29 - 327807456218503729990936770043/29636196259239989084160*t^28 + 486723471560537646382383942571/41490674762935984717824*t^27 + 1218759680528619833103064762453/103726686907339961794560*t^26 + 6907652179017565478957186941/3143232936586059448320*t^25 - 131200545598926299428044562013/69151124604893307863040*t^24 - 61672910568038203235636068397/41490674762935984717824*t^23 - 9237074910662360203768147477/29636196259239989084160*t^22 - 101674721204090698221154867/207453373814679923589120*t^21 - 93085103804837253245646107/103726686907339961794560*t^20 + 418599492207816383024692049/69151124604893307863040*t^19 + 8877588776468183989013953/7409049064809997271040*t^18 - 197452961972896293346074269/69151124604893307863040*t^17 - 3391809806967085937061763/4321945287805831741440*t^16 - 984496045608773913360253/34575562302446653931520*t^15 - 142802126704585553044309/6286465873172118896640*t^14 - 95903528691180211470943/13830224920978661572608*t^13 - 243707278331807261188247/207453373814679923589120*t^12 + 2079589004420634341/41161383693388873728*t^11 - 2762733657785736668749/103726686907339961794560*t^10 + 156076441953663196355/13830224920978661572608*t^9 - 95906434201566006919/207453373814679923589120*t^8 - 18108618325585039477/69151124604893307863040*t^7 - 219605192803914637/25931671726834990448640*t^6 - 1053210963877573/648291793170874761216*t^5 - 1008633467658301/4321945287805831741440*t^4 + 4507265716189/360162107317152645120*t^3 + 962316061541/540243160975728967680*t^2 + 440817857/5627532926830510080*t + 659749/562753292683051008)*Dt - 1040*t^46 - 4805970092168/2065159641*t^45 + 240023604023965/130105057383*t^44 + 4097071361140800781/171738675745560*t^43 + 71589562233680291335/471054653473536*t^42 + 10953031362032460230867/14131639604206080*t^41 + 723496099910044432708699/263790605945180160*t^40 + 97222489530249556568357/14388578506100736*t^39 + 326870102290236446256260647/25323898170737295360*t^38 + 13751782576062948600153493/535955516841000960*t^37 + 1139053823010727038588104359/22510131707322040320*t^36 + 127282182641180912750591964433/1620729482927186903040*t^35 + 22710340080146293936802680271/270121580487864483840*t^34 + 96757487923066348428368081/1688259878049153024*t^33 + 8156003085155691167677264069/392904117073257431040*t^32 - 81527658098897277895701789169/25931671726834990448640*t^31 - 104987847410647343956314327793/7409049064809997271040*t^30 - 329500970189051364152747097931/25931671726834990448640*t^29 - 168106329966371752340212934815/20745337381467992358912*t^28 - 29588535560049216804905693/101295592682949181440*t^27 + 580192739544147989633430592919/207453373814679923589120*t^26 + 20090352245642862513530189695/10372668690733996179456*t^25 + 3042736533585027607413190069/12965835863417495224320*t^24 - 1654884830836536452038174637/5927239251847997816832*t^23 - 2051958902168289877469571413/10372668690733996179456*t^22 - 4800692629964620181279569073/103726686907339961794560*t^21 - 267064987704880393137858361/29636196259239989084160*t^20 - 24109539962040249530448307/9429698809758178344960*t^19 + 48589690109306832485729503/51863343453669980897280*t^18 + 96770567141135172264995839/207453373814679923589120*t^17 - 6285408663200721825089189/25931671726834990448640*t^16 - 3990859213614687157068881/51863343453669980897280*t^15 - 44247464340124146710719/9429698809758178344960*t^14 - 3223422333314075270899/10372668690733996179456*t^13 - 29711333467262335374599/41490674762935984717824*t^12 - 7041139644232504729/60979827693909442560*t^11 - 65680370618535006797/9878732086413329694720*t^10 - 71141984482561802105/41490674762935984717824*t^9 + 2409762924985048511/2593167172683499044864*t^8 + 642623933991045113/3292910695471109898240*t^7 + 124717312097837033/8643890575611663482880*t^6 + 26043714787983569/12965835863417495224320*t^5 + 73005887954299/1440648429268610580480*t^4 - 11680535335949/1080486321951457935360*t^3 - 920630906309/540243160975728967680*t^2 - 62031481/803933275261501440*t - 659749/562753292683051008]

# Corrections made on 7-14-2026, after adding diagonal COV + pivot optimization.
SMOOTH_fast_diag[1] = 1/6*(6*t^6 + 63*t^5 - 37*t^4 + 25*t^3 - 2*t^2)*Dt^4 + 1/3*(21*t^5 + 153*t^4 - 99*t^3 + 83*t^2 - 5*t)*Dt^3 + 1/3*(24*t^4 + 108*t^3 - 72*t^2 + 109*t - 3)*Dt^2 + 2*Dt*(t + 3)
SMOOTH_med_diag[1] = 6*t^6 + 1/8*(8*t^10 - 20*t^9 - 40*t^8 + 185*t^7 - 146*t^6 - 100*t^5 + 170*t^4 - 65*t^3 + 8*t^2)*Dt^4 - 6*t^5 + 1/4*(52*t^9 - 124*t^8 - 284*t^7 + 1276*t^6 - 1135*t^5 - 101*t^4 + 499*t^3 - 211*t^2 + 28*t)*Dt^3 - 108*t^4 + 3/4*(60*t^8 - 132*t^7 - 412*t^6 + 1720*t^5 - 1605*t^4 + 391*t^3 + 119*t^2 - 81*t + 12)*Dt^2 + 363*t^3 + 3/2*(28*t^7 - 52*t^6 - 280*t^5 + 1056*t^4 - 965*t^3 + 373*t^2 - 52*t)*Dt - 609/2*t^2 + 243/2*t - 18
NONSMOOTH_med_diag[4] = 35/9*t^8 + 118/3*t^6 + (t^11 + 4*t^9 - 6*t^7 - 4*t^5 + 5*t^3)*Dt^3 - 124/9*t^4 + 2*(4*t^10 + 23*t^8 - 15*t^6 - 7*t^4 - 5*t^2)*Dt^2 + 1/9*(125*t^9 + 950*t^7 - 348*t^5 + 82*t^3 + 55*t)*Dt + 26/3*t^2 - 55/9
SMOOTH_slow_diag[2] = 1/12350218589030685081600*(12350218589030685081600*t^34 - 8063475734001808834560*t^33 + 11416520225483390976000*t^32 + 39712866854075923169280*t^31 - 17108829215147903942656*t^30 - 8507834081108144685056*t^29 - 80868139238710129983488*t^28 + 127090901125013669543936*t^27 - 137770731179865108217856*t^26 - 56894538655154317582336*t^25 + 147403464312899901272064*t^24 + 25521974715234578214400*t^23 + 12216525945283400480640*t^22 + 2271880559881032527040*t^21 + 1800998083432556449296*t^20 - 170463021621432938184*t^19 - 380222187864621387802*t^18 + 1442514151081723734*t^17 + 3676351705421207526*t^16 - 1943368677368186184*t^15 + 532109748501217161*t^14 + 92526410826807930*t^13 - 2817652114373985*t^12 + 586382256180000*t^11 + 40144342940100*t^10 - 2159733731400*t^9 + 558438300*t^8 + 91683900*t^7 + 2083725*t^6)*Dt^10 + 1/6175109294515342540800*(345806120492859182284800*t^33 - 228265281095422716149760*t^32 + 343965147984927632916480*t^31 + 1163981713105399924654080*t^30 - 494651804003639482122240*t^29 - 217422309107971475374080*t^28 - 2526059911499019087249408*t^27 + 3986312719089709345210368*t^26 - 4484950416452293466619904*t^25 - 1724382250571121909555200*t^24 + 4697442493747916080307200*t^23 + 817198370995009338803200*t^22 + 573951537044941036195200*t^21 + 106257430240860633272960*t^20 + 86974363110150019072800*t^19 - 1528099944455324963320*t^18 - 10186969410091764113472*t^17 + 96061676401389708747*t^16 - 72730299413566301886*t^15 - 68080833650230717230*t^14 + 11065937994839651040*t^13 + 1742209007797814535*t^12 - 71838692056277685*t^11 + 13148491720329000*t^10 + 887321646225000*t^9 - 48621597475500*t^8 + 12060600300*t^7 + 2150404200*t^6 + 50009400*t^5)*Dt^9 + 1/2058369764838447513600*(2420642843450014275993600*t^32 - 1613728914147934658887680*t^31 + 2601066134430075353825280*t^30 + 8652485107877998556610560*t^29 - 3598232868858253509918720*t^28 - 1361009583308260491919360*t^27 - 19956063123128170945183744*t^26 + 31811476389045879932583936*t^25 - 36897128741853375971328000*t^24 - 13312362613146539517890560*t^23 + 38149305827727293024885760*t^22 + 6700232188543477133653760*t^21 + 6121252444455113612748800*t^20 + 1149068970995204344570080*t^19 + 983630090371645729072800*t^18 + 27030924164749548986100*t^17 - 60298868256227854906086*t^16 + 2338679050395774847479*t^15 - 1515669876232092474030*t^14 - 545431904974033825230*t^13 + 53672183670532513590*t^12 + 6626911801443315165*t^11 - 455530822305770925*t^10 + 70346733996420000*t^9 + 4601403391977000*t^8 - 259546910647500*t^7 + 59436171900*t^6 + 12135614400*t^5 + 291721500*t^4)*Dt^8 + 1/102918488241922375680*(1210321421725007137996800*t^31 - 813719786187414967418880*t^30 + 1416633097369464990597120*t^29 + 4683090521756430730526720*t^28 - 1885086436071293945118720*t^27 - 582743416469100439797760*t^26 - 11445692438140784367960064*t^25 + 18545984668846117732958208*t^24 - 22025628497986000111804416*t^23 - 7516148190424905212941312*t^22 + 22674879732840758327242752*t^21 + 4044056383456441940072960*t^20 + 4474128840634630991564288*t^19 + 858782534514495634399056*t^18 + 768191415600355926488496*t^17 + 44070937180370426686014*t^16 - 18916934248203833434047*t^15 + 3005554574383363682811*t^14 - 1102544998271374536792*t^13 - 285938416809414856872*t^12 + 17096724045887677443*t^11 + 810435855702991770*t^10 - 205692272666636175*t^9 + 25207322943360600*t^8 + 1531708850655900*t^7 - 90985735436400*t^6 + 17253243000*t^5 + 4609199700*t^4 + 116688600*t^3)*Dt^7 + 1/4900880392472494080*(288171767077382651904000*t^30 - 195000110102562188820480*t^29 + 372949410029531991900160*t^28 + 1239576669052746827038720*t^27 - 475678785965180840837120*t^26 - 107237094563001784074240*t^25 - 3200546563090432689373184*t^24 + 5306288371819330929049600*t^23 - 6407922715003358397440000*t^22 - 2085221507202826344512512*t^21 + 6626954846365458794708480*t^20 + 1208685393350678212986240*t^19 + 1539959239732415207421184*t^18 + 303962948331185384447168*t^17 + 282484684586062065022128*t^16 + 22430029633165452487472*t^15 + 295785158219392394346*t^14 + 1556684456866533786897*t^13 - 288353846749197832434*t^12 - 62619543800365389570*t^11 + 2512542037874834520*t^10 - 294477956480532825*t^9 - 44355822258521025*t^8 + 3955294425952800*t^7 + 195543108898200*t^6 - 13043184991200*t^5 + 1150216200*t^4 + 750141000*t^3 + 20837250*t^2)*Dt^6 + 1/2450440196236247040*(345806120492859182284800*t^29 - 234820813156927552880640*t^28 + 508119528451509269299200*t^27 + 1719737710164498962186240*t^26 - 615501941735003155070976*t^25 - 71097215761595059142656*t^24 - 4675324068577004584763392*t^23 + 7986417487617745842782208*t^22 - 9736053047049450027737088*t^21 - 3049386235131413537907712*t^20 + 10209410147116649403213312*t^19 + 1920957129800749567610240*t^18 + 2708024493901175177447168*t^17 + 552870418084030731404256*t^16 + 529121251688807711860272*t^15 + 51595798551160007344140*t^14 + 10425652598995571708496*t^13 + 3784195474893003808161*t^12 - 270228431715637585050*t^11 - 49082460887173214862*t^10 + 2580213151717143948*t^9 - 706703912952347835*t^8 - 49099762948079655*t^7 + 2951961471849600*t^6 + 61646022459000*t^5 - 6599590489800*t^4 - 2150404200*t^3 + 483424200*t^2 + 16669800*t)*Dt^5 + 1/2450440196236247040*(345806120492859182284800*t^28 - 234529974517767717519360*t^27 + 605748516380582011207680*t^26 + 2117025983058464871546880*t^25 - 683484890710199044669440*t^24 + 29366630565463085547520*t^23 - 6037710452070112577650688*t^22 + 10704296281747787777261568*t^21 - 13071652281040506987945984*t^20 - 3988140107197641007631360*t^19 + 14038872033479084190174720*t^18 + 2757061046542118328534400*t^17 + 4148916404302914346408960*t^16 + 883505057370800223348480*t^15 + 859796593250509263917520*t^14 + 98501790519176973703560*t^13 + 28711952156551923072738*t^12 + 7670747552368410383427*t^11 - 10887053575879834866*t^10 + 19802986535764775250*t^9 + 4581104654204132670*t^8 - 586953981181159515*t^7 - 48008828129824485*t^6 + 2415912587676000*t^5 - 49569335802000*t^4 - 131608071000*t^3 - 5651062200*t^2 + 66679200*t + 8334900)*Dt^4 + 1/204203349686353920*(8233479059353790054400*t^27 - 5533014359291744747520*t^26 + 19021175144592409886720*t^25 + 69666933996964691312640*t^24 - 19181527101844467220480*t^23 + 5764346290659115663360*t^22 - 207545773087063140335616*t^21 + 384804858007449238175744*t^20 - 466587400434192213770240*t^19 - 141352823326473175664640*t^18 + 519268650509338021537280*t^17 + 108402591030089855277440*t^16 + 167609924203064878471680*t^15 + 37811432229208342117680*t^14 + 36649156726996694897760*t^13 + 4911704568864883490490*t^12 + 1574184029346783616761*t^11 + 405509208633414295506*t^10 + 14334996767020792290*t^9 + 4978116548498452590*t^8 + 265527020455393095*t^7 + 17859612920536485*t^6 - 1315013213052000*t^5 + 114297419061000*t^4 + 3375634500*t^3 - 3000564000*t^2 - 216707400*t - 8334900)*Dt^3

# Corections 10-11-2026, to keep us up to date and eliminate time needed for display:
SMOOTH_fast_diag[0] = Dt^2
SMOOTH_fast_diag[1] = 1/3*(3*t^4 + 32*t^3 - 14*t^2 + t)*Dt^3 + 1/3*(12*t^3 + 94*t^2 - 50*t + 3)*Dt^2 + 2*Dt*(4*t - 3)
SMOOTH_med_diag[1] = (t^3 - t)*Dt^3 + 3*(3*t^2 - 1)*Dt^2 + 18*Dt*t + 6
NONSMOOTH_med_diag[0] = 12*t^6 + 1/144*(144*t^10 - 480*t^9 - 537*t^8 + 62*t^7 + t^6 + 30*t^5 + 2*t^4 - 2*t^3)*Dt^4 - 359/6*t^5 + 1/72*(1008*t^9 - 3708*t^8 - 3621*t^7 - 358*t^6 - 162*t^5 + 197*t^4 + 24*t^3 - 7*t^2)*Dt^3 - 119/3*t^4 + 1/12*(648*t^8 - 2627*t^7 - 2256*t^6 - 749*t^5 - 263*t^4 + 97*t^3 + 16*t^2 - t)*Dt^2 - 35*t^3 + 1/12*(720*t^7 - 3224*t^6 - 2441*t^5 - 1436*t^4 - 532*t^3 + 41*t^2 + 10*t + 1)*Dt - 85/6*t^2 - 4/3*t - 1/6
NONSMOOTH_med_diag[2] = 1/3*(3*t^3 + 2*t^2 - t)*Dt^3 + (6*t^2 + 3*t - 1)*Dt^2 + 2*Dt*(3*t + 1)
NONSMOOTH_med_diag[3] = 1/12*(12*t^3 + 4*t^2 - t)*Dt^3 + 1/4*(24*t^2 + 6*t - 1)*Dt^2 + Dt*(6*t + 1)
SMOOTH_slow_diag[0] = (t^4 - 34*t^3 + t^2)*Dt^3 + 3*(2*t^3 - 51*t^2 + t)*Dt^2 + (7*t^2 - 112*t + 1)*Dt + t - 5
SMOOTH_slow_diag[2] = 1/64*(64*t^4 + 48*t^3 - 4*t^2 - 3*t)*Dt^4 + 1/16*(96*t^3 + 88*t^2 - 3)*Dt^3 + 3/16*(32*t^2 + 36*t + 3)*Dt^2
NONSMOOTH_slow_diag[0] = (t^4 - 6*t^3 + 10*t^2 - 8*t)*Dt^4 + 4*(t^3 - 3*t^2 + 6*t - 5)*Dt^3
NONSMOOTH_slow_diag[1] = 1/172923846954494613066983651914688578780035890185464491654374158039280393670983680000*(172923846954494613066983651914688578780035890185464491654374158039280393670983680000*t^20 - 90499930342982829860567816335749126873864674091649777375948405588243279402097049600*t^19 - 36861665369761919090409498358062455682299755528582800383542491476635121605662081024*t^18 + 8285633727008891829072323584883117843302188500171607701908245560150010919123968000*t^17 + 636325013323763672030105576793994319629577540329411979517215440936475514121359936*t^16 + 15967232784050380721901146598535090031044858426391792280434254680135295139606304*t^15 - 1956390347970639738532299698048195959197179046523279314393447517888000340896440*t^14 - 94362860239063690105543908518764239359962708669439692393796417948766867046884*t^13 + 1970247608056798766883768248784261641663033436960982630195390941680003784582*t^12 + 89438760084769368078907501175647354215396292703103554342525825024720061009*t^11 - 343197745303361993262871187559376829745842362871708989428187901273045095*t^10 - 22637669768209013385269179674948334525502587625610718774053680803939698*t^9 - 70820301271056741028088293475394387251765747001330042364756440191397*t^8 + 4666616291437462080937253433676406504471399480701825449184172868459*t^7 - 50974045880595587057684600109333610986848787365142164843283802596*t^6 - 304844924772098654077614318857402699546001366122354733337609656*t^5 + 121800754702143131690088378233373418349895002661754571678304*t^4 + 699823241795873225697512615092982852035764343241303174400*t^3)*Dt^10 + 1/345847693908989226133967303829377157560071780370928983308748316078560787341967360000*(25938577043174191960047547787203286817005383527819673748156123705892059050647552000000*t^19 - 14857871144219299939911227404059781301739455981868113233044664982798513726821603737600*t^18 - 4611415000473723930627422702629799018763072736236206954082586682290313962633341861888*t^17 + 1100254045788152025096412525740145514882703921239182079541710863636650317180588408832*t^16 + 95455774477857831090323633524505844342230545316935662124816103138972980966869178752*t^15 + 3489007813286054034259087385513200439610669702381190286673646487994782845591698432*t^14 - 202127000420938036117398979512368008766017582839556288664676361843309877606204592*t^13 - 12846007337994911339340154906016579703065780026052284001054786284729375577183184*t^12 + 119004996009169213867480887805251088450652820617654820273229225980334345555092*t^11 + 8712141274468114988338365997980009863625216969937573601789495763830210890440*t^10 + 15533273157832256235095469169654240512714783344545348601270162797726622966*t^9 - 1778692660885470818608368101028184734573706410806280821373210054545278632*t^8 - 5016136321971237112382641078126452117920378378933930691336753248925741*t^7 + 448081754255084982912635524274470510827479116993168263031181834682081*t^6 - 3470905063622654033726022572557705270886672112649236063696398869968*t^5 - 23587130641143907636450896442059951124044001271781309457527623016*t^4 + 3894570737060453896229886317873548748290900547097045759706944*t^3 + 29392576155426675479295529833905279785502102416134733324800*t^2)*Dt^9 + 717516800/59049*t^10 + 1/230565129272659484089311535886251438373381186913952655539165544052373858227978240000*(512879320893182541274224105382528199581676818001925795988232687992058293524902707200000*t^18 - 319035772143423487312270365528605807356670417226948070643340151225325532443993545113600*t^17 - 73499966289102874481212875079298164456995552877741273927323564217461210881248586760192*t^16 + 19233737356370542777921427617892754348161918892684404876695754300827178795580737519616*t^15 + 1868645858109340944721435932902495785941616456859809512686265883866516862218087146496*t^14 + 85176770635475080925981928515028277893668725035060294968438555865656124015197210112*t^13 - 2295444602671724506205321722105839300335893384029991465451323308137202254342315520*t^12 - 213952084079935475868313466305669853849815406080203862429075623053434094382645408*t^11 + 223357452810921141979857101485435209070222065822508808300050582435774765927344*t^10 + 98492761285818601668422881048760035243986441874512544032385956390309554571208*t^9 + 659151116548501371346797309691164149173691245037861335596924437947833556838*t^8 - 18735401552785030875121551880700830066080833841526956329411929621455662067*t^7 - 43033063569577283683083637456171296559739998563684927715930682863121745*t^6 + 5260470022876488838493241463374366158336713412299202428136414586610908*t^5 - 25108403139683691150078358901729737656445819732664974431731998028616*t^4 - 213756762755814465148787120391814929335199132926503296375632866176*t^3 + 6544124852773368186151518211038191423422774103322720475108224*t^2 + 117570304621706701917182119335621119142008409664538933299200*t)*Dt^8 - 16586419041363915366995910591260211355444670365608644411320004106035200/1367340480413221718052236359131522998448552758971843481563016879343*t^9 + 1/115282564636329742044655767943125719186690593456976327769582772026186929113989120000*(3893220299507340199672519345403736787733637663923709451365220858848806137211761459200000*t^17 - 2612474169984949752665383532159704889946063535400815094297725781328682827165159653376000*t^16 - 427522103656065844212271336780689919660482116662074637229180193888346348587341861683200*t^15 + 129172790790424324539640283457577442488745030386133500245988131873687327137113356042240*t^14 + 13887561820540542660503741363340200206450734554974503543373005525522591827160160117248*t^13 + 730856176025824284047827463822537690656877355299071618986819600191106784285155217920*t^12 - 5886031890193355637456878838843583744946810056912636455035703411459195498481130496*t^11 - 1260176385405183036641856415967718965983670972794592774468378301587669746079239440*t^10 - 6414911998153312926831190058487879228513600423619059048543881056997058683302344*t^9 + 360884827597702845048906147623393000002770726600210558438018885891114927268560*t^8 + 4407987475787215641944608445319399090987896705837017912872569793062445786382*t^7 - 83827056689950969045572380109562883952496286824415349695161560175665493851*t^6 - 152049044861743494074106143524456249564092495429231052769130670518118947*t^5 + 21379389648252271313908000826237002715922395257794368270669723105635026*t^4 - 43497183517166488791258783164688481097556553222115350427165254189680*t^3 - 627153314204367339267105384427635339453299614164880492395996036960*t^2 - 15173745626778608530426929187357316046175255973207502833281920*t + 97975253851422251597651766113017599285007008053782444416000)*Dt^7 + 379200531652885012655471940765256280018424968214105896441253456793600/455780160137740572684078786377174332816184252990614493854338959781*t^8 + 1/115282564636329742044655767943125719186690593456976327769582772026186929113989120000*(32767289945953329025853206732772634973273796705678592521470421732825946530757672960000000*t^16 - 23583053895174173879473459028547902369106167077816248074734753552320035805055633588224000*t^15 - 2536070391624915886912776186875671873395026982318160703396180227473292496643247570944000*t^14 + 965511996058603084557355619505278684943887148824208856662674613987534644336837941002240*t^13 + 113055862485686483165214197425924224421914718771780052477990787352929264095972391379968*t^12 + 6549156250765303190214691271576016690553785592451123517738018544194201751662812453376*t^11 + 31798645602685526010470923797733603018608493159739557582086725201394283114967870592*t^10 - 7404982221255244534346432809881638712895868418665585674576121758408604917234302528*t^9 - 74303779373640391828847611263800309196352494497499416999058047409184894764257568*t^8 + 1131553671803506872487421452775998980973523335966303205686838226092995278636240*t^7 + 25147578110799618066373513012967230790488585490146777222883662782020738223302*t^6 - 480860370743696918142887940646095679763207440328673404257548331659134319837*t^5 - 869233407416250358601638584532338429410524063046543205209313242118749697*t^4 + 83204568224515559445221286482919875032684521711849367239625365040428072*t^3 + 35293103608400850820416310213251465016458701841488471869342346160480*t^2 - 1552507049940620132161589936616868216997665597757922911168792881920*t - 37827494597662100201214360472589267753737256100250238393247360)*Dt^6 + 375089610177727312482690903792964985059902165088710041411251478691200/1367340480413221718052236359131522998448552758971843481563016879343*t^7 + 1/9606880386360811837054647328593809932224216121414693980798564335515577426165760000*(12921709556947682359325590655054252139460579831326297137814640222466579784085743206400000*t^15 - 9924272598508418218329812853416884678807612554119639688575147319855533278205360996352000*t^14 - 597327269155086606093764860456744640425376483141082761085756337768218240043161636372480*t^13 + 340911578812789215329364759025335442126183024468312339759197004029461159818640625008640*t^12 + 42576743958242369133029239279978712098857480396315121162714784232575925047093926741248*t^11 + 2619628998824232450693744036909111497518882406857206897515812134778929877896908147712*t^10 + 36947630934142143512202250269468681525525601418076557271460160678662414847654223840*t^9 - 1730348248571788010117744284305543441517564877715577638457020789156300110674530816*t^8 - 25770781983345181608531387518754742582029408158577431908595319201249366799388832*t^7 + 101538445706450313527136668413693619330592344693214864029427192860222646618350*t^6 + 6245929013153513649578001182417426286181345805667888526047153133057831403067*t^5 - 138800029082489796518339629091215060890108055309905968845684138660563243318*t^4 - 329583350140293574307223936009082279529320248971410005970582259815952697*t^3 + 12170774783101208186211589017605500121551596853941886168362102035489028*t^2 + 29756749670090845083297910639809979352625089647506677226095236367880*t - 104703299632897123342273843160407921962025131620154026602764224400)*Dt^5 + 136532478554606340500462285596719820567156452103479083281213743349600/4102021441239665154156709077394568995345658276915530444689050638029*t^6 + 1/228735247294305043739396364966519284100576574319397475733299150845608986337280000*(794998124292587478243996212732862641326879018302024896387795524596743308207325184000000*t^14 - 648636757007954999664001621587548909018547032745737039902134752014886865545200140288000*t^13 - 13172498162826916448547550243913906597100770470023342368386141626963123619765003223040*t^12 + 19047902552574016684324555466752790953735226911129742424944980393571087720304670801920*t^11 + 2470807864447615438852074430070554527957806726984472436117142097875624432052878641152*t^10 + 156242722410252489336568909291010219743572438194116899193373521527726397818870469632*t^9 + 3225283824711059620647478694843109967281495339332869171187048203885243374230014272*t^8 - 44398707569741569534261789078644082138515897394266379763915915937871121001222256*t^7 - 1041082652614618322430411303586247961999367845964717927383168096258253769940384*t^6 + 505579673921902998200943801632919803860305704520889696343200787729098518960*t^5 + 272632400133677907376968381656548405836431889735243484716000351493220591188*t^4 - 5838905171179121223966108220822094114796440553780880726723910740428339021*t^3 - 19843536409461729657156736027445062844394592388057084816515504633047468*t^2 + 192438032488603872192916897054516464440412563596900365728252261270376*t + 652232122801059182178345280738771633116598297107902199224013263280)*Dt^4 + 2195769639735187325775960213193191020492572894416325113560061222800/1367340480413221718052236359131522998448552758971843481563016879343*t^5 + 13/114367623647152521869698182483259642050288287159698737866649575422804493168640000*(40008901032910047650614416035131422433789739567373128841844226780007260721315840000000*t^13 - 34534982315020705831426435774577433482133875136351708033180280040140460895889784832000*t^12 + 445179256051661587024136607413425581447555027823591800565096047402924880484110172160*t^11 + 889831757540114449263625584078251089962299406532274169424299087443178120629761474560*t^10 + 116179787287112380213657744374967448708846436894997260345752200845999439983395017728*t^9 + 7275753017059090171106276119200140694498226157122742605001557258971319197089025024*t^8 + 179397638101271764066674285507500770545834976535190503229276725418450499207612896*t^7 + 50122305294178070157435551076635756101140208552772991938967826410024323898248*t^6 - 22735916438409929994590240979723950770788945975689479906361491462163176888940*t^5 + 91014985289661398467318313084758606286920699657539379426081476398382726930*t^4 + 11610355892531394169848117956571755811925970840328863980858054895713624207*t^3 - 158726461230158328602937304192624424170875222926076366304197901237954860*t^2 - 815609218124778502181998341302415365204845064286661835413966557580632*t + 1193985623743060035526300272077565686906232639605749777781361569520)*Dt^3 + 11316971377377312558710387841420937306327753542784255035626474600/455780160137740572684078786377174332816184252990614493854338959781*t^4 + 13/23826588259823442056187121350679092093810059824937237055551994879750936076800*(4763987052675189964480925723989973453155675734851819152665473773858441941319680000*t^12 - 4333695875349177961878522696372407978485638103537782821279243061515538113288601600*t^11 + 171755270222228702294194967550594735459813164834241665578445183648392789720117248*t^10 + 101479777878072689212339360733783708857764841148162952714061964278997061264297984*t^9 + 12924112128452670724458664932188743519724527858300785268408607406893038378583424*t^8 + 764538913117307381373596339695772546973149404708545697543058278708499136505536*t^7 + 19556050434740365073255103844980228868923814189855265364526314272385388064216*t^6 + 151744602285826090216039268773561783728256905073111302960882942857897288844*t^5 + 29703049128346101509957758668418104552640183568179518720536214691985874*t^4 + 30258105725343488289804520255436073286166633956496014734288010383317445*t^3 + 1196336045357716833966618414840533110349817145817451386849137223018940*t^2 - 4908197475957884898270505915836525877832562378410097474365402145288*t - 52706094502925433518772839837116380045863930221998090014234721456)*Dt^2 + 258203118520300071685268965513482670708892625882920089572972500/1367340480413221718052236359131522998448552758971843481563016879343*t^3 + 13/77183635438365539540612638000256210216423906138442620847269176805153664*(2840677765062186185634935973662226427889089081399484307457976323958857728000*t^11 - 2712862872262948291921623092642491248204157622970764514082275004237166936064*t^10 + 160692073754859146699188570279887412923743701876629041630970740920442231808*t^9 + 60472913491445681479858735935300105233123415966215795373494677847651555328*t^8 + 7386937878994624737726240953585038966860325481234488056032996810476303392*t^7 + 391440427237321984611607325659274989966996844306422035790072432636675832*t^6 + 8617032930135686973818971202180376810506525807853783771233533671444908*t^5 + 93603146399947518600541425693750351681760907985583012733658557029518*t^4 + 788907366382298519089660812464435248009485624540066943069821506945*t^3 + 30250565546354912632131364600842121865182718208496060865243302380*t^2 + 539143117665277849992159173269994447079121982896741598800946904*t + 2085609234542433855510714658402999049236910428014719659745808)*Dt + 124441061814875751243891357048761053575678116063677399154000/50642240015304508076008754041908259201798250332290499317148773309*t^2 + 2696346717648073873451523479918066993901612935955126066400/16880746671768169358669584680636086400599416777430166439049591103*t + 4399919239895822813767035165021282540991879828570289600/5626915557256056452889861560212028800199805592476722146349863701
NONSMOOTH_slow_diag[2] = 120*t^11 - 352/7*t^10 + 4065/28*t^9 + 4457/448*t^8 + 17/56*t^7 + 1/172032*(172032*t^16 - 155648*t^15 + 20480*t^14 + 78016*t^13 - 188224*t^12 + 78144*t^11 - 6528*t^10 + 701*t^9 + 2131*t^8 - 1183*t^7 + 97*t^6 - 30*t^5 + 12*t^4)*Dt^5 - 10189/1792*t^6 + 1/172032*(4300800*t^15 - 3473408*t^14 + 1988608*t^13 + 777472*t^12 - 3375296*t^11 + 1221168*t^10 - 155184*t^9 + 62906*t^8 + 8591*t^7 - 5998*t^6 + 113*t^5 - 396*t^4 + 84*t^3)*Dt^4 - 1865/1792*t^5 + 1/43008*(8601600*t^14 - 6111232*t^13 + 6392064*t^12 + 151568*t^11 - 4343472*t^10 + 1231548*t^9 - 214392*t^8 + 119374*t^7 - 12060*t^6 + 3370*t^5 - 759*t^4 - 327*t^3 + 12*t^2)*Dt^3 - 1087/3584*t^4 + 1/14336*(8601600*t^13 - 5275648*t^12 + 8268544*t^11 - 397648*t^10 - 2331040*t^9 + 349396*t^8 - 102532*t^7 + 73356*t^6 - 10556*t^5 + 4388*t^4 - 135*t^3 - 50*t^2 - 8*t)*Dt^2 - 523/7168*t^3 + 1/7168*(4300800*t^12 - 2220032*t^11 + 4803328*t^10 - 90512*t^9 - 419568*t^8 - 92460*t^7 - 18144*t^6 + 11209*t^5 - 1083*t^4 + 729*t^3 + 229*t^2 + 27*t + 4)*Dt - 103/3584*t^2 - 23/7168*t - 1/1792


In [4]:
def certificate_test_runner(diag, run_verify=False, tweak=False):
    """Compare ordinary and certified operators for one diagonal."""
    plain_start = time.perf_counter()
    plain = compute_diagonal_annihilator(diag, certify=False, minimize_denom_deg=True)
    plain_elapsed = time.perf_counter() - plain_start

    certified_start = time.perf_counter()
    operator, cert = compute_diagonal_annihilator(
        diag, certify=True, minimize_denom_deg=True
    )
    certified_elapsed = time.perf_counter() - certified_start

    result = {
        "operators_agree": plain == operator,
        "plain_seconds": plain_elapsed,
        "certified_seconds": certified_elapsed,
    }
    print(f"Operators agree: {result['operators_agree']}")

    if run_verify:
        result["certificate_valid"] = verify_certificate(cert)
        print(f"Certificate verification passed: {result['certificate_valid']}")

    if tweak:
        # Copy the fields we might edit so the original valid certificate stays intact.
        altered = replace(cert, rho0=list(cert.rho0), B=matrix(cert.B))
        entries = [("rho0", i) for i in range(len(altered.rho0))]
        entries += [("B", (i, j)) for i in range(altered.B.nrows())
                    for j in range(altered.B.ncols())]

        if entries:
            field, index = random.choice(entries)
            old = altered.rho0[index] if field == "rho0" else altered.B[index]
            new = 1 if old == 0 else 1 - old
            if new == old:  # 1 - 1/2 = 1/2 would leave this entry unchanged.
                new = old + 1
            if field == "rho0":
                altered.rho0[index] = new
            else:
                altered.B[index] = new
        else:
            # The zero-dimensional reduction has no rho0 or B entries.
            field = "a"
            terms = cert.a.dict()
            index = random.choice(list(terms)) if terms else (0,) * cert.a.parent().ngens()
            old = terms.get(index, 0)
            new = 1 if old == 0 else 1 - old
            if new == old:
                new = old + 1
            altered.a = cert.a + cert.a.parent()({index: new - old})

        result["tampered_certificate_valid"] = verify_certificate(altered)
        print(f"Tweaked {field}[{index}] certificate verification passed: "
              f"{result['tampered_certificate_valid']}")

    print(f"Plain time: {plain_elapsed:.3f} s")
    print(f"Certified time: {certified_elapsed:.3f} s")
    return result

certificate_test_runner(SMOOTH_fast[0], True, False)


Operators agree: True
Certificate verification passed: True
Plain time: 0.279 s
Certified time: 0.105 s


{'operators_agree': True,
 'plain_seconds': 0.2787942528957501,
 'certified_seconds': 0.10502990300301462,
 'certificate_valid': True}

In [5]:
fast_results = []
for group_name, examples in (("SMOOTH_fast", SMOOTH_fast),
                             ("NONSMOOTH_fast", NONSMOOTH_fast)):
    for index, diag in enumerate(examples):
        label = f"{group_name}[{index}]"
        print(f"\n{label}")
        try:
            result = certificate_test_runner(diag, run_verify=True, tweak=True)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
            print(f"FAILED: {result['error']}")
        fast_results.append((label, result))

print("\nSummary (seconds)")
print(f"{'Example':<22} {'Plain':>9} {'Certified':>10} {'Cert/plain':>10} {'Agree':>7} {'Valid':>7} {'Tweaked':>8}")
for label, result in fast_results:
    if "error" in result:
        print(f"{label:<22} ERROR: {result['error']}")
        continue
    plain = result["plain_seconds"]
    certified = result["certified_seconds"]
    ratio = certified / plain if plain else float("inf")
    print(f"{label:<22} {plain:9.3f} {certified:10.3f} {ratio:10.2f} "
          f"{str(result['operators_agree']):>7} "
          f"{str(result['certificate_valid']):>7} "
          f"{str(result['tampered_certificate_valid']):>8}")

passed = sum("error" not in result and result["operators_agree"]
             and result["certificate_valid"]
             and not result["tampered_certificate_valid"]
             for _, result in fast_results)
print(f"\n{passed}/{len(fast_results)} cases: operators agree, certificates verify, tweaked certificates fail.")



SMOOTH_fast[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(0, 0)] certificate verification passed: False
Plain time: 0.110 s
Certified time: 0.116 s

SMOOTH_fast[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(0, 0)] certificate verification passed: False
Plain time: 0.202 s
Certified time: 0.256 s

SMOOTH_fast[2]
Operators agree: True
Certificate verification passed: True
Tweaked B[(0, 1)] certificate verification passed: False
Plain time: 0.133 s
Certified time: 0.153 s

SMOOTH_fast[3]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[1] certificate verification passed: False
Plain time: 0.136 s
Certified time: 0.163 s

SMOOTH_fast[4]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[2] certificate verification passed: False
Plain time: 0.187 s
Certified time: 0.215 s

SMOOTH_fast[5]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[0] certificate verification pa

In [20]:
import traceback

med_results = []
for group_name, examples in (("SMOOTH_med", SMOOTH_med),
                             ("NONSMOOTH_med", NONSMOOTH_med)):
    for index, diag in enumerate(examples):
        label = f"{group_name}[{index}]"
        print(f"\n{label}")
        try:
            result = certificate_test_runner(diag, run_verify=True, tweak=True)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
            print(f"FAILED: {result['error']}")
            traceback.print_exc()
        med_results.append((label, result))

print("\nSummary (seconds)")
print(f"{'Example':<22} {'Plain':>9} {'Certified':>10} {'Cert/plain':>10} {'Agree':>7} {'Valid':>7} {'Tweaked':>8}")
for label, result in med_results:
    if "error" in result:
        print(f"{label:<22} ERROR: {result['error']}")
        continue
    plain = result["plain_seconds"]
    certified = result["certified_seconds"]
    ratio = certified / plain if plain else float("inf")
    print(f"{label:<22} {plain:9.3f} {certified:10.3f} {ratio:10.2f} "
          f"{str(result['operators_agree']):>7} "
          f"{str(result['certificate_valid']):>7} "
          f"{str(result['tampered_certificate_valid']):>8}")

passed = sum("error" not in result and result["operators_agree"]
             and result["certificate_valid"]
             and not result["tampered_certificate_valid"]
             for _, result in med_results)
print(f"\n{passed}/{len(med_results)} cases: operators agree, certificates verify, tweaked certificates fail.")


SMOOTH_med[0]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[0] certificate verification passed: False
Plain time: 0.199 s
Certified time: 0.244 s

SMOOTH_med[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(0, 4)] certificate verification passed: False
Plain time: 0.742 s
Certified time: 1.818 s

NONSMOOTH_med[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(5, 2)] certificate verification passed: False
Plain time: 0.704 s
Certified time: 1.563 s

NONSMOOTH_med[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(12, 2)] certificate verification passed: False
Plain time: 31.704 s
Certified time: 33.982 s

NONSMOOTH_med[2]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[11] certificate verification passed: False
Plain time: 1.145 s
Certified time: 2.741 s

NONSMOOTH_med[3]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 5)] certificate ver

In [6]:
import traceback

slow_results = []
for group_name, examples in (("SMOOTH_slow", SMOOTH_slow),
                             ("NONSMOOTH_slow", NONSMOOTH_slow)):
    for index, diag in enumerate(examples):
        label = f"{group_name}[{index}]"
        print(f"\n{label}")
        try:
            result = certificate_test_runner(diag, run_verify=True, tweak=True)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
            print(f"FAILED: {result['error']}")
            traceback.print_exc()
        slow_results.append((label, result))

print("\nSummary (seconds)")
print(f"{'Example':<22} {'Plain':>9} {'Certified':>10} {'Cert/plain':>10} {'Agree':>7} {'Valid':>7} {'Tweaked':>8}")
for label, result in slow_results:
    if "error" in result:
        print(f"{label:<22} ERROR: {result['error']}")
        continue
    plain = result["plain_seconds"]
    certified = result["certified_seconds"]
    ratio = certified / plain if plain else float("inf")
    print(f"{label:<22} {plain:9.3f} {certified:10.3f} {ratio:10.2f} "
          f"{str(result['operators_agree']):>7} "
          f"{str(result['certificate_valid']):>7} "
          f"{str(result['tampered_certificate_valid']):>8}")

passed = sum("error" not in result and result["operators_agree"]
             and result["certificate_valid"]
             and not result["tampered_certificate_valid"]
             for _, result in slow_results)
print(f"\n{passed}/{len(slow_results)} cases: operators agree, certificates verify, tweaked certificates fail.")


SMOOTH_slow[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(5, 2)] certificate verification passed: False
Plain time: 3.588 s
Certified time: 12.579 s

SMOOTH_slow[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(13, 14)] certificate verification passed: False
Plain time: 29.031 s
Certified time: 44.047 s

SMOOTH_slow[2]
Operators agree: True
Certificate verification passed: True
Tweaked B[(3, 2)] certificate verification passed: False
Plain time: 0.344 s
Certified time: 0.767 s

NONSMOOTH_slow[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 2)] certificate verification passed: False
Plain time: 0.399 s
Certified time: 1.271 s

NONSMOOTH_slow[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(15, 3)] certificate verification passed: False
Plain time: 22.258 s
Certified time: 64.261 s

NONSMOOTH_slow[2]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 5)] cert

In [1]:
######################################################################

In [2]:
# Now, new runs of the above, following further improvements by Claude

In [3]:
######################################################################


In [8]:
fast_results = []
for group_name, examples in (("SMOOTH_fast", SMOOTH_fast),
                             ("NONSMOOTH_fast", NONSMOOTH_fast)):
    for index, diag in enumerate(examples):
        label = f"{group_name}[{index}]"
        print(f"\n{label}")
        try:
            result = certificate_test_runner(diag, run_verify=True, tweak=True)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
            print(f"FAILED: {result['error']}")
        fast_results.append((label, result))

print("\nSummary (seconds)")
print(f"{'Example':<22} {'Plain':>9} {'Certified':>10} {'Cert/plain':>10} {'Agree':>7} {'Valid':>7} {'Tweaked':>8}")
for label, result in fast_results:
    if "error" in result:
        print(f"{label:<22} ERROR: {result['error']}")
        continue
    plain = result["plain_seconds"]
    certified = result["certified_seconds"]
    ratio = certified / plain if plain else float("inf")
    print(f"{label:<22} {plain:9.3f} {certified:10.3f} {ratio:10.2f} "
          f"{str(result['operators_agree']):>7} "
          f"{str(result['certificate_valid']):>7} "
          f"{str(result['tampered_certificate_valid']):>8}")

passed = sum("error" not in result and result["operators_agree"]
             and result["certificate_valid"]
             and not result["tampered_certificate_valid"]
             for _, result in fast_results)
print(f"\n{passed}/{len(fast_results)} cases: operators agree, certificates verify, tweaked certificates fail.")



SMOOTH_fast[0]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[0] certificate verification passed: False
Plain time: 0.117 s
Certified time: 0.103 s

SMOOTH_fast[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 0)] certificate verification passed: False
Plain time: 0.196 s
Certified time: 0.191 s

SMOOTH_fast[2]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[0] certificate verification passed: False
Plain time: 0.145 s
Certified time: 0.128 s

SMOOTH_fast[3]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[1] certificate verification passed: False
Plain time: 0.144 s
Certified time: 0.147 s

SMOOTH_fast[4]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[2] certificate verification passed: False
Plain time: 0.196 s
Certified time: 0.190 s

SMOOTH_fast[5]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[0] certificate verification passed

In [9]:
import traceback

med_results = []
for group_name, examples in (("SMOOTH_med", SMOOTH_med),
                             ("NONSMOOTH_med", NONSMOOTH_med)):
    for index, diag in enumerate(examples):
        label = f"{group_name}[{index}]"
        print(f"\n{label}")
        try:
            result = certificate_test_runner(diag, run_verify=True, tweak=True)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
            print(f"FAILED: {result['error']}")
            traceback.print_exc()
        med_results.append((label, result))

print("\nSummary (seconds)")
print(f"{'Example':<22} {'Plain':>9} {'Certified':>10} {'Cert/plain':>10} {'Agree':>7} {'Valid':>7} {'Tweaked':>8}")
for label, result in med_results:
    if "error" in result:
        print(f"{label:<22} ERROR: {result['error']}")
        continue
    plain = result["plain_seconds"]
    certified = result["certified_seconds"]
    ratio = certified / plain if plain else float("inf")
    print(f"{label:<22} {plain:9.3f} {certified:10.3f} {ratio:10.2f} "
          f"{str(result['operators_agree']):>7} "
          f"{str(result['certificate_valid']):>7} "
          f"{str(result['tampered_certificate_valid']):>8}")

passed = sum("error" not in result and result["operators_agree"]
             and result["certificate_valid"]
             and not result["tampered_certificate_valid"]
             for _, result in med_results)
print(f"\n{passed}/{len(med_results)} cases: operators agree, certificates verify, tweaked certificates fail.")


SMOOTH_med[0]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[0] certificate verification passed: False
Plain time: 0.198 s
Certified time: 0.205 s

SMOOTH_med[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 4)] certificate verification passed: False
Plain time: 0.427 s
Certified time: 0.422 s

NONSMOOTH_med[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(1, 4)] certificate verification passed: False
Plain time: 0.491 s
Certified time: 0.796 s

NONSMOOTH_med[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 7)] certificate verification passed: False
Plain time: 31.465 s
Certified time: 32.521 s

NONSMOOTH_med[2]
Operators agree: True
Certificate verification passed: True
Tweaked B[(7, 8)] certificate verification passed: False
Plain time: 0.873 s
Certified time: 1.153 s

NONSMOOTH_med[3]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[10] certificate veri

In [10]:
import traceback

slow_results = []
for group_name, examples in (("SMOOTH_slow", SMOOTH_slow),
                             ("NONSMOOTH_slow", NONSMOOTH_slow)):
    for index, diag in enumerate(examples):
        label = f"{group_name}[{index}]"
        print(f"\n{label}")
        try:
            result = certificate_test_runner(diag, run_verify=True, tweak=True)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
            print(f"FAILED: {result['error']}")
            traceback.print_exc()
        slow_results.append((label, result))

print("\nSummary (seconds)")
print(f"{'Example':<22} {'Plain':>9} {'Certified':>10} {'Cert/plain':>10} {'Agree':>7} {'Valid':>7} {'Tweaked':>8}")
for label, result in slow_results:
    if "error" in result:
        print(f"{label:<22} ERROR: {result['error']}")
        continue
    plain = result["plain_seconds"]
    certified = result["certified_seconds"]
    ratio = certified / plain if plain else float("inf")
    print(f"{label:<22} {plain:9.3f} {certified:10.3f} {ratio:10.2f} "
          f"{str(result['operators_agree']):>7} "
          f"{str(result['certificate_valid']):>7} "
          f"{str(result['tampered_certificate_valid']):>8}")

passed = sum("error" not in result and result["operators_agree"]
             and result["certificate_valid"]
             and not result["tampered_certificate_valid"]
             for _, result in slow_results)
print(f"\n{passed}/{len(slow_results)} cases: operators agree, certificates verify, tweaked certificates fail.")


SMOOTH_slow[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(4, 1)] certificate verification passed: False
Plain time: 3.230 s
Certified time: 3.549 s

SMOOTH_slow[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(6, 13)] certificate verification passed: False
Plain time: 27.585 s
Certified time: 34.288 s

SMOOTH_slow[2]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 2)] certificate verification passed: False
Plain time: 0.350 s
Certified time: 0.468 s

NONSMOOTH_slow[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(3, 3)] certificate verification passed: False
Plain time: 0.365 s
Certified time: 0.693 s

NONSMOOTH_slow[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(10, 6)] certificate verification passed: False
Plain time: 19.685 s
Certified time: 30.108 s

NONSMOOTH_slow[2]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[0] certific

In [ ]:
### Trying one more time, 11-13, after redoing the work that Claude did: ###

In [ ]:
fast_results = []
for group_name, examples in (("SMOOTH_fast", SMOOTH_fast),
                             ("NONSMOOTH_fast", NONSMOOTH_fast)):
    for index, diag in enumerate(examples):
        label = f"{group_name}[{index}]"
        print(f"\n{label}")
        try:
            result = certificate_test_runner(diag, run_verify=True, tweak=True)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
            print(f"FAILED: {result['error']}")
        fast_results.append((label, result))

print("\nSummary (seconds)")
print(f"{'Example':<22} {'Plain':>9} {'Certified':>10} {'Cert/plain':>10} {'Agree':>7} {'Valid':>7} {'Tweaked':>8}")
for label, result in fast_results:
    if "error" in result:
        print(f"{label:<22} ERROR: {result['error']}")
        continue
    plain = result["plain_seconds"]
    certified = result["certified_seconds"]
    ratio = certified / plain if plain else float("inf")
    print(f"{label:<22} {plain:9.3f} {certified:10.3f} {ratio:10.2f} "
          f"{str(result['operators_agree']):>7} "
          f"{str(result['certificate_valid']):>7} "
          f"{str(result['tampered_certificate_valid']):>8}")

passed = sum("error" not in result and result["operators_agree"]
             and result["certificate_valid"]
             and not result["tampered_certificate_valid"]
             for _, result in fast_results)
print(f"\n{passed}/{len(fast_results)} cases: operators agree, certificates verify, tweaked certificates fail.")


SMOOTH_fast[0]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[0] certificate verification passed: False
Plain time: 0.133 s
Certified time: 0.099 s

SMOOTH_fast[1]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[0] certificate verification passed: False
Plain time: 0.204 s
Certified time: 0.189 s

SMOOTH_fast[2]
Operators agree: True
Certificate verification passed: True
Tweaked B[(1, 1)] certificate verification passed: False
Plain time: 0.141 s
Certified time: 0.127 s

SMOOTH_fast[3]
Operators agree: True
Certificate verification passed: True
Tweaked B[(1, 1)] certificate verification passed: False
Plain time: 0.144 s
Certified time: 0.137 s

SMOOTH_fast[4]
Operators agree: True
Certificate verification passed: True
Tweaked B[(3, 0)] certificate verification passed: False
Plain time: 0.193 s
Certified time: 0.174 s

SMOOTH_fast[5]
Operators agree: True
Certificate verification passed: True
Tweaked rho0[0] certificate verification pa

In [6]:
import traceback

med_results = []
for group_name, examples in (("SMOOTH_med", SMOOTH_med),
                             ("NONSMOOTH_med", NONSMOOTH_med)):
    for index, diag in enumerate(examples):
        label = f"{group_name}[{index}]"
        print(f"\n{label}")
        try:
            result = certificate_test_runner(diag, run_verify=True, tweak=True)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
            print(f"FAILED: {result['error']}")
            traceback.print_exc()
        med_results.append((label, result))

print("\nSummary (seconds)")
print(f"{'Example':<22} {'Plain':>9} {'Certified':>10} {'Cert/plain':>10} {'Agree':>7} {'Valid':>7} {'Tweaked':>8}")
for label, result in med_results:
    if "error" in result:
        print(f"{label:<22} ERROR: {result['error']}")
        continue
    plain = result["plain_seconds"]
    certified = result["certified_seconds"]
    ratio = certified / plain if plain else float("inf")
    print(f"{label:<22} {plain:9.3f} {certified:10.3f} {ratio:10.2f} "
          f"{str(result['operators_agree']):>7} "
          f"{str(result['certificate_valid']):>7} "
          f"{str(result['tampered_certificate_valid']):>8}")

passed = sum("error" not in result and result["operators_agree"]
             and result["certificate_valid"]
             and not result["tampered_certificate_valid"]
             for _, result in med_results)
print(f"\n{passed}/{len(med_results)} cases: operators agree, certificates verify, tweaked certificates fail.")


SMOOTH_med[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 0)] certificate verification passed: False
Plain time: 0.216 s
Certified time: 0.225 s

SMOOTH_med[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 2)] certificate verification passed: False
Plain time: 0.435 s
Certified time: 0.435 s

NONSMOOTH_med[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 0)] certificate verification passed: False
Plain time: 0.511 s
Certified time: 0.883 s

NONSMOOTH_med[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(8, 9)] certificate verification passed: False
Plain time: 31.437 s
Certified time: 32.618 s

NONSMOOTH_med[2]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 11)] certificate verification passed: False
Plain time: 1.153 s
Certified time: 1.839 s

NONSMOOTH_med[3]
Operators agree: True
Certificate verification passed: True
Tweaked B[(8, 3)] certificate 

In [7]:
import traceback

slow_results = []
for group_name, examples in (("SMOOTH_slow", SMOOTH_slow),
                             ("NONSMOOTH_slow", NONSMOOTH_slow)):
    for index, diag in enumerate(examples):
        label = f"{group_name}[{index}]"
        print(f"\n{label}")
        try:
            result = certificate_test_runner(diag, run_verify=True, tweak=True)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
            print(f"FAILED: {result['error']}")
            traceback.print_exc()
        slow_results.append((label, result))

print("\nSummary (seconds)")
print(f"{'Example':<22} {'Plain':>9} {'Certified':>10} {'Cert/plain':>10} {'Agree':>7} {'Valid':>7} {'Tweaked':>8}")
for label, result in slow_results:
    if "error" in result:
        print(f"{label:<22} ERROR: {result['error']}")
        continue
    plain = result["plain_seconds"]
    certified = result["certified_seconds"]
    ratio = certified / plain if plain else float("inf")
    print(f"{label:<22} {plain:9.3f} {certified:10.3f} {ratio:10.2f} "
          f"{str(result['operators_agree']):>7} "
          f"{str(result['certificate_valid']):>7} "
          f"{str(result['tampered_certificate_valid']):>8}")

passed = sum("error" not in result and result["operators_agree"]
             and result["certificate_valid"]
             and not result["tampered_certificate_valid"]
             for _, result in slow_results)
print(f"\n{passed}/{len(slow_results)} cases: operators agree, certificates verify, tweaked certificates fail.")


SMOOTH_slow[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(4, 0)] certificate verification passed: False
Plain time: 3.485 s
Certified time: 3.986 s

SMOOTH_slow[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(8, 9)] certificate verification passed: False
Plain time: 29.614 s
Certified time: 36.947 s

SMOOTH_slow[2]
Operators agree: True
Certificate verification passed: True
Tweaked B[(0, 0)] certificate verification passed: False
Plain time: 0.378 s
Certified time: 0.366 s

NONSMOOTH_slow[0]
Operators agree: True
Certificate verification passed: True
Tweaked B[(2, 3)] certificate verification passed: False
Plain time: 0.385 s
Certified time: 0.696 s

NONSMOOTH_slow[1]
Operators agree: True
Certificate verification passed: True
Tweaked B[(4, 11)] certificate verification passed: False
Plain time: 19.060 s
Certified time: 32.952 s

NONSMOOTH_slow[2]
Operators agree: True
Certificate verification passed: True
Tweaked B[(1, 4)] certifi